# Z-DNA-induced genomic instability in the human pangenome

Analysis pipeline for Z-DNA forming sequences across the human pangenome (HPRC release 2, T2T-CHM13, GRCh38, GRCh37), their genomic distribution, and their association with germline and de novo mutation.

> **Note on paths.** All file paths in this notebook are short placeholders (`data/...`, `out/...`). Replace them with your own locations. Heavy steps (ZSeeker prediction, per-haplotype tables, control generation) were run on an HPC cluster; the cells below show the analysis logic, not a turn-key driver.

## 1. Input data

**HPRC release 2** (minigraph-cactus assemblies): `data.humanpangenome.org`, `42basepairs.com/.../release2/minigraph-cactus`, `github.com/human-pangenomics/hprc_intermediate_assembly`.

**T2T-CHM13v2.0**: NCBI `GCF_009914755.1` (FASTA, GFF, UCSC-named contigs) and marbl/CHM13 annotation tracks (CenSat, RepeatMasker, telomere, Liftoff genes).

**GRCh38.p14**: NCBI `GCF_000001405.40` (primary, MT-removed, UCSC-named) + MANE v1.5.

**GRCh37**: NCBI `GCF_000001405.13` (primary chromosomes only, UCSC-named).

## 2. Z-DNA prediction (ZSeeker)

In [ ]:
%%bash
# Predict Z-DNA across all HPRC haplotype assemblies (run as a 20-way parallel SLURM array).
source ~/.bashrc
conda activate zseeker_env

INPUT_DIR="data/assemblies"
OUTPUT_DIR="data/zdna_csvs"

mkdir -p "$OUTPUT_DIR"

mapfile -t FILES < <(find "$INPUT_DIR" -type f \( -name "*.fa.gz" -o -name "*.fasta.gz" \) | sort)

run_zseeker() {
    fasta_path="$1"
    output_dir="$2"
    base=$(basename "$fasta_path" .gz)
    base=${base%.fasta}
    base=${base%.fa}

    tmpdir=$(mktemp -d tmp/zseeker_${base}_XXXX)
    out_csv="${output_dir}/${base}.zseeker.csv"

    if [[ -f "$out_csv" ]]; then
        echo "⚠️ Skipping existing output: $out_csv"
    else
        gunzip -c "$fasta_path" > "${tmpdir}/${base}.fasta"
        ZSeeker --fasta "${tmpdir}/${base}.fasta" --output_dir "$tmpdir" --n_jobs 1
        csv_file=$(find "$tmpdir" -type f -name '*.csv' | head -n 1)
        if [[ -n "$csv_file" ]]; then
            mv "$csv_file" "$out_csv"
            echo "✅ Output saved: $out_csv"
        else
            echo "⚠️ No CSV produced for $base"
        fi
    fi

    rm -rf "$tmpdir"
    echo "✅ Finished: $base"
}
export -f run_zseeker

printf "%s\n" "${FILES[@]}" | xargs -P20 -I{} bash -c 'run_zseeker "{}" "$1"' _ "$OUTPUT_DIR"

In [ ]:
%%bash
# Predict Z-DNA in each reference assembly (CHM13, GRCh38, GRCh37).
conda activate zseeker_env
for ASM in chm13 grch38 grch37; do
    ZSeeker --fasta data/${ASM}/${ASM}.fa --output_dir data/${ASM} --n_jobs 1
    mv data/${ASM}/*.csv data/${ASM}/${ASM}.zdna.csv
done

## 3. Metadata and Z-DNA metrics

In [ ]:
# Join HPRC sample metadata with per-assembly Z-DNA counts, mean length and mean score.
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

meta_file = Path("data/hprc_metadata.csv")
csv_dir   = Path("data/zdna_csvs")
out_tsv   = Path("data/hprc_zdna_metrics.tsv")

meta = pd.read_csv(meta_file)
if "sample_id" not in meta.columns:
    raise SystemExit("sample_id column missing")

records = []
for f in sorted(csv_dir.glob("*.csv")):
    name = f.name
    low  = name.lower()
    records.append({
        "filename": name,
        "sample_id": name.split("_")[0],
        "pat_or_mat": "pat" if "pat" in low else "mat" if "mat" in low else "0",
        "hap1_or_hap2": "hap1" if "hap1" in low else "hap2" if "hap2" in low else "0",
    })

base = pd.DataFrame(records).merge(meta, on="sample_id", how="left", validate="many_to_one")

stats = []
for f in tqdm(sorted(csv_dir.glob("*.csv")), desc="Processing Z-DNA CSVs"):
    try:
        d = pd.read_csv(f, dtype={"Chromosome": str})
    except Exception:
        continue
    d = d.dropna(subset=["Chromosome", "Start", "End", "Z-DNA Score", "Sequence"])
    d["Start"] = pd.to_numeric(d["Start"], errors="coerce")
    d["End"] = pd.to_numeric(d["End"], errors="coerce")
    d["Z-DNA Score"] = pd.to_numeric(d["Z-DNA Score"], errors="coerce")
    d = d.dropna(subset=["Start", "End", "Z-DNA Score"])
    stats.append({
        "filename": f.name,
        "n_predictions": len(d),
        "average_length": round((d["End"] - d["Start"] + 1).mean(), 3) if len(d) else np.nan,
        "average_score": round(d["Z-DNA Score"].mean(), 3) if len(d) else np.nan,
    })

out = base.merge(pd.DataFrame(stats), on="filename", how="left", validate="one_to_one")
cols = ["filename", "sample_id", "pat_or_mat", "hap1_or_hap2"]
out = out[cols + [c for c in out.columns if c not in cols]]
out.to_csv(out_tsv, sep="\t", index=False)
print(f"Wrote {len(out)} rows to {out_tsv}")

## 4. Effective genome size and Z-DNA density

### 4.1 Reference genomes

In [ ]:
# Effective genome size (ACGT-only segments at 3 length tiers) and per-chromosome Z-DNA density for the reference genomes.
from __future__ import annotations

import gzip
from pathlib import Path
from typing import Iterator

import numpy as np
import pandas as pd
from Bio.SeqIO.FastaIO import SimpleFastaParser

ASSEMBLIES: dict[str, dict[str, str]] = {
    "GRCh38": {
        "fasta": "data/grch38/"
                 "grch38.fa",
        "zseeker_csv": "data/grch38/"
                       "grch38",
        "out_dir": "data/grch38",
    },
    "CHM13": {
        "fasta": "data/chm13/"
                 "chm13.fa",
        "zseeker_csv": "data/chm13/"
                       "chm13.zdna.csv",
        "out_dir": "data/chm13",
    },
}

MIN_SEGMENT_TIERS: tuple[int, ...] = (1, 9, 50)

_ACGT_BYTES = frozenset(b"ACGT")

def _open_fasta(path: Path):
    if path.suffix == ".gz":
        return gzip.open(path, "rt")
    return open(path, "rt", encoding="utf-8")

def _segment_lengths_from_record(seq: str) -> np.ndarray:
    upper_bytes = seq.upper().encode("ascii", errors="replace")
    arr = np.frombuffer(upper_bytes, dtype=np.uint8)

    mask = (arr == ord("A")) | (arr == ord("C")) | (arr == ord("G")) | (arr == ord("T"))

    if not mask.any():
        return np.empty(0, dtype=np.int64)

    padded = np.concatenate(([False], mask, [False]))
    diffs = np.diff(padded.astype(np.int8))
    starts = np.flatnonzero(diffs == 1)
    ends = np.flatnonzero(diffs == -1)
    return (ends - starts).astype(np.int64)

def compute_effective_lengths(
    fasta_path: Path,
    min_segment_tiers: tuple[int, ...],
) -> pd.DataFrame:
    rows: list[dict] = []

    with _open_fasta(fasta_path) as fh:
        for header, seq in SimpleFastaParser(fh):
            chrom = header.split()[0]
            raw_len = len(seq)
            seg_lens = _segment_lengths_from_record(seq)

            row: dict = {"ucsc_chr": chrom, "raw_length_bp": raw_len}
            for tier in min_segment_tiers:
                if seg_lens.size == 0:
                    row[f"eff_len_min{tier}_bp"] = 0
                else:
                    row[f"eff_len_min{tier}_bp"] = int(seg_lens[seg_lens >= tier].sum())
            rows.append(row)

            del seq, seg_lens

    df = pd.DataFrame(rows)
    return df

def chr_sort_key(c: str) -> tuple:
    c = str(c)
    if c.startswith("chr"):
        rest = c[3:]
        if rest.isdigit():
            return (0, int(rest))
        if rest == "X":
            return (1, 23)
        if rest == "Y":
            return (1, 24)
        if rest in ("M", "MT"):
            return (2, 25)
        return (3, rest)
    return (4, c)

def compute_density(
    zseeker_csv: Path,
    eff_df: pd.DataFrame,
    min_segment_tiers: tuple[int, ...],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    z = pd.read_csv(zseeker_csv)

    required = {"Chromosome", "Start", "End"}
    missing = required - set(z.columns)
    if missing:
        raise KeyError(f"Missing columns in {zseeker_csv}: {sorted(missing)}")

    z["ucsc_chr"] = z["Chromosome"].astype(str).str.strip()
    z["Start"] = pd.to_numeric(z["Start"], errors="coerce")
    z["End"] = pd.to_numeric(z["End"], errors="coerce")

    bad = z[z[["ucsc_chr", "Start", "End"]].isna().any(axis=1)]
    if len(bad):
        raise ValueError(f"NA in ucsc_chr/Start/End:\n{bad.head(10)}")

    z["zdna_bp"] = (z["End"] - z["Start"] + 1).astype("int64")
    if (z["zdna_bp"] <= 0).any():
        raise ValueError("Non-positive zdna_bp detected.")

    per_chr_zdna = (
        z.groupby("ucsc_chr", as_index=False)["zdna_bp"]
        .sum()
        .rename(columns={"zdna_bp": "total_zdna_bp"})
    )

    per_chr = eff_df.merge(per_chr_zdna, on="ucsc_chr", how="left")
    per_chr["total_zdna_bp"] = per_chr["total_zdna_bp"].fillna(0).astype("int64")

    for tier in min_segment_tiers:
        eff_col = f"eff_len_min{tier}_bp"
        denom = per_chr[eff_col].astype("float64")
        with np.errstate(divide="ignore", invalid="ignore"):
            per_chr[f"density_fraction_min{tier}"] = np.where(
                denom > 0, per_chr["total_zdna_bp"] / denom, np.nan
            )
            per_chr[f"density_bp_per_Mb_min{tier}"] = np.where(
                denom > 0, per_chr["total_zdna_bp"] / (denom / 1e6), np.nan
            )

    per_chr = (
        per_chr.assign(__k=per_chr["ucsc_chr"].map(chr_sort_key))
        .sort_values("__k")
        .drop(columns="__k")
        .reset_index(drop=True)
    )

    summary_rows = []
    total_zdna = int(per_chr["total_zdna_bp"].sum())
    total_raw = int(per_chr["raw_length_bp"].sum())
    for tier in min_segment_tiers:
        total_eff = int(per_chr[f"eff_len_min{tier}_bp"].sum())
        summary_rows.append({
            "min_segment": tier,
            "total_zdna_bp": total_zdna,
            "total_raw_length_bp": total_raw,
            "total_effective_length_bp": total_eff,
            "fraction_of_raw_excluded": (
                1.0 - total_eff / total_raw if total_raw > 0 else np.nan
            ),
            "density_fraction": total_zdna / total_eff if total_eff > 0 else np.nan,
            "density_bp_per_Mb": (
                total_zdna / (total_eff / 1e6) if total_eff > 0 else np.nan
            ),
        })
    summary = pd.DataFrame(summary_rows)
    return per_chr, summary

def main() -> None:
    pd.set_option("display.max_rows", 400)
    pd.set_option("display.width", 200)

    for assembly_name, paths in ASSEMBLIES.items():
        fasta_path = Path(paths["fasta"])
        zseeker_csv = Path(paths["zseeker_csv"])
        out_dir = Path(paths["out_dir"])
        out_dir.mkdir(parents=True, exist_ok=True)

        assert fasta_path.is_file(), f"FASTA not found: {fasta_path}"
        assert zseeker_csv.is_file(), f"ZSeeker CSV not found: {zseeker_csv}"

        print("\n" + "=" * 100)
        print(assembly_name)
        print("=" * 100)
        print(f"FASTA:       {fasta_path}")
        print(f"ZSeeker CSV: {zseeker_csv}")
        print(f"Tiers:       {MIN_SEGMENT_TIERS}")

        eff_df = compute_effective_lengths(fasta_path, MIN_SEGMENT_TIERS)

        eff_out = out_dir / f"{assembly_name}_effective_chrom_sizes.tsv"
        eff_df.to_csv(eff_out, sep="\t", index=False)
        print(f"\nWrote effective lengths -> {eff_out}")

        per_chr, summary = compute_density(zseeker_csv, eff_df, MIN_SEGMENT_TIERS)

        per_chr_out = out_dir / f"{assembly_name}_zdna_density_per_chr.tsv"
        per_chr.to_csv(per_chr_out, sep="\t", index=False)

        summary_out = out_dir / f"{assembly_name}_zdna_density_genome_summary.tsv"
        summary.to_csv(summary_out, sep="\t", index=False)

        print(f"Wrote per-chr density   -> {per_chr_out}")
        print(f"Wrote genome summary    -> {summary_out}")

        print("\nPer-chromosome (showing key columns):")
        display_cols = ["ucsc_chr", "raw_length_bp", "total_zdna_bp"] + [
            c for tier in MIN_SEGMENT_TIERS
            for c in (f"eff_len_min{tier}_bp", f"density_bp_per_Mb_min{tier}")
        ]
        print(per_chr[display_cols].to_string(index=False))

        print("\nGenome-wide summary across tiers:")
        print(summary.to_string(index=False))

if __name__ == "__main__":
    main()

### 4.2 Per-haplotype and per-contig tables (HPRC)

The builders below run per assembly (PanSN headers; primary / random / unplaced / mitochondrial classes); results are then aggregated across samples.

In [ ]:
# Per-haplotype Z-DNA / effective-size table for one HPRC assembly.
import gzip
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

MIN_SEGMENT = 50  # ACGT segment-length floor (bp) for effective genome size

META_PATH = Path("data/hprc_zdna_metrics.tsv")
ZDNA_DIR  = Path("data/zdna_csvs")
FASTA_DIR = Path("data/assemblies")
ALIAS_DIR = Path("data")
OUTDIR    = Path("data/out")
OUT_CHUNKS = OUTDIR / "haplotype_chunks"

PRIMARY_CHRS = {f"chr{i}" for i in range(1, 23)} | {"chrX", "chrY"}
MITO_PREFIX = "chrM"

CATEGORIES = ("primary_chr", "chr_random", "mitochondrial", "chr_un", "no_alias")

def fasta_from_zseeker(filename: str) -> str:
    return filename.replace(".zseeker.csv", ".fa.gz")

def infer_alias_key(name: str) -> str:
    n = name.lower()
    if "_pat" in n or "#1#" in n or "_hap1" in n:
        return "pat"
    if "_mat" in n or "#2#" in n or "_hap2" in n:
        return "mat"
    return ""

def find_alias_file(stem: str, prefix: str, key: str):
    exact = list(ALIAS_DIR.glob(f"{stem}.chromAlias.txt"))
    if exact:
        return exact[0]
    if key:
        cand = list(ALIAS_DIR.glob(f"{prefix}*{key}*.chromAlias.txt"))
        if cand:
            return cand[0]
    cand = list(ALIAS_DIR.glob(f"{prefix}*.chromAlias.txt"))
    return cand[0] if cand else None

def load_alias_map(alias_file: Path) -> dict:
    lines = [l.strip() for l in open(alias_file) if l.strip() and not l.startswith("#")]
    parts = [l.split() for l in lines]
    df = pd.DataFrame(parts, columns=["assembly", "ucsc", "genbank"]).apply(lambda x: x.str.strip())
    return dict(zip(df["assembly"], df["ucsc"]))

def chr_class(ucsc) -> str:
    if ucsc is None or (isinstance(ucsc, float) and np.isnan(ucsc)):
        return "no_alias"
    if ucsc in PRIMARY_CHRS:
        return "primary_chr"
    if ucsc.startswith(MITO_PREFIX):
        return "mitochondrial"
    if ucsc.startswith("chrUn"):
        return "chr_un"
    if ucsc.startswith("chr") and "_random" in ucsc:
        return "chr_random"
    if ucsc.startswith("chr"):
        return "chr_random"
    return "no_alias"

def effective_length_from_seq(seq: str, min_segment: int = MIN_SEGMENT) -> int:
    if not seq:
        return 0
    upper_bytes = seq.upper().encode("ascii", errors="replace")
    arr = np.frombuffer(upper_bytes, dtype=np.uint8)
    mask = (arr == ord("A")) | (arr == ord("C")) | (arr == ord("G")) | (arr == ord("T"))
    if not mask.any():
        return 0
    padded = np.concatenate(([False], mask, [False]))
    diffs = np.diff(padded.astype(np.int8))
    starts = np.flatnonzero(diffs == 1)
    ends = np.flatnonzero(diffs == -1)
    seg_lens = (ends - starts).astype(np.int64)
    return int(seg_lens[seg_lens >= min_segment].sum())

def genome_sizes_from_fasta(fasta_gz: Path, alias_map: dict):
    bp_eff = {c: 0 for c in CATEGORIES}
    bp_raw = {c: 0 for c in CATEGORIES}

    contig = None
    seq_chunks: list[str] = []

    def flush(cname, chunks):
        if cname is None or not chunks:
            return
        seq = "".join(chunks)
        ucsc = alias_map.get(cname)
        cls = chr_class(ucsc)
        bp_raw[cls] += len(seq)
        bp_eff[cls] += effective_length_from_seq(seq, MIN_SEGMENT)

    with gzip.open(fasta_gz, "rt") as fh:
        for line in fh:
            if line.startswith(">"):
                flush(contig, seq_chunks)
                contig = line[1:].strip().split()[0]
                seq_chunks = []
            else:
                seq_chunks.append(line.strip())
        flush(contig, seq_chunks)

    return bp_eff, bp_raw

def zdna_burden_from_zseeker(csv_path: Path, alias_map: dict):
    bp = {c: 0 for c in CATEGORIES}

    df = pd.read_csv(csv_path, dtype=str)
    if not {"Chromosome", "Start", "End"}.issubset(df.columns):
        return bp, 0, 0, 0

    df = df.dropna(subset=["Chromosome", "Start", "End"])
    df["Start"] = pd.to_numeric(df["Start"], errors="coerce")
    df["End"]   = pd.to_numeric(df["End"], errors="coerce")
    df = df.dropna(subset=["Start", "End"])
    df["len"] = df["End"].astype(int) - df["Start"].astype(int) + 1

    mapped = df["Chromosome"].astype(str).str.strip().map(alias_map)

    for L, ucsc in zip(df["len"].tolist(), mapped.tolist()):
        if pd.isna(ucsc):
            bp["no_alias"] += int(L)
        else:
            bp[chr_class(ucsc)] += int(L)

    n_total = len(df)
    n_unmapped = int(mapped.isna().sum())
    n_mapped = n_total - n_unmapped
    return bp, n_total, n_mapped, n_unmapped

def main(target_csv: str) -> None:
    OUTDIR.mkdir(parents=True, exist_ok=True)
    OUT_CHUNKS.mkdir(parents=True, exist_ok=True)

    meta = pd.read_csv(META_PATH, sep="\t", dtype=str).fillna("")
    row = meta.loc[meta["filename"] == target_csv]
    if row.empty:
        raise SystemExit(f"❌ filename not found in metadata: {target_csv}")

    fname  = target_csv
    zpath  = ZDNA_DIR  / fname
    fasta  = FASTA_DIR / fasta_from_zseeker(fname)

    stem   = fasta.name.replace(".fa.gz", "")
    prefix = fasta.name.split("_")[0]
    key    = infer_alias_key(fasta.name)

    alias_file = find_alias_file(stem, prefix, key)
    if alias_file is None:
        raise SystemExit(f"❌ Alias file not found for {fname}")
    if not zpath.exists():
        raise SystemExit(f"❌ ZSeeker CSV not found: {zpath}")
    if not fasta.exists():
        raise SystemExit(f"❌ FASTA not found: {fasta}")

    alias_map = load_alias_map(alias_file)

    zdna_bp, n_total, n_mapped, n_unmapped = zdna_burden_from_zseeker(zpath, alias_map)
    g_eff, g_raw = genome_sizes_from_fasta(fasta, alias_map)

    out_row = {
        "filename": fname,
        "fasta_file": fasta.name,
        "alias_file": alias_file.name,
        "min_segment_bp": MIN_SEGMENT,
        "n_intervals_total": n_total,
        "n_intervals_mapped": n_mapped,
        "n_intervals_unmapped_to_alias": n_unmapped,

        "zdna_bp_primary_chr":   zdna_bp["primary_chr"],
        "zdna_bp_chr_random":    zdna_bp["chr_random"],
        "zdna_bp_mitochondrial": zdna_bp["mitochondrial"],
        "zdna_bp_chr_un":        zdna_bp["chr_un"],
        "zdna_bp_no_alias":      zdna_bp["no_alias"],

        "genome_bp_primary_chr":   g_eff["primary_chr"],
        "genome_bp_chr_random":    g_eff["chr_random"],
        "genome_bp_mitochondrial": g_eff["mitochondrial"],
        "genome_bp_chr_un":        g_eff["chr_un"],
        "genome_bp_no_alias":      g_eff["no_alias"],

        "genome_bp_raw_primary_chr":   g_raw["primary_chr"],
        "genome_bp_raw_chr_random":    g_raw["chr_random"],
        "genome_bp_raw_mitochondrial": g_raw["mitochondrial"],
        "genome_bp_raw_chr_un":        g_raw["chr_un"],
        "genome_bp_raw_no_alias":      g_raw["no_alias"],
    }

    out_row["zdna_bp_per_mb_primary_chr"] = (
        out_row["zdna_bp_primary_chr"] /
        (out_row["genome_bp_primary_chr"] / 1e6)
        if out_row["genome_bp_primary_chr"] > 0 else float("nan")
    )

    out_file = OUT_CHUNKS / f"{fname}.tsv"
    pd.DataFrame([out_row]).to_csv(out_file, sep="\t", index=False)
    print(f"✅ Wrote {out_file}")

if __name__ == "__main__":
    if len(sys.argv) != 2:
        raise SystemExit("Usage: build_hprc_haplotype_tables.py <zseeker_csv_filename>")
    main(sys.argv[1])

In [ ]:
# Per-contig (chromosome-level) Z-DNA / effective-size table for one HPRC assembly.
import gzip
import sys
from pathlib import Path

import numpy as np
import pandas as pd

MIN_SEGMENT = 50

META_PATH  = Path("data/hprc_zdna_metrics.tsv")
ZDNA_DIR   = Path("data/zdna_csvs")
FASTA_DIR  = Path("data/assemblies")
ALIAS_DIR  = Path("data")
OUTDIR     = Path("data/tmp/haplotype_chromosome_tables")
OUT_CHUNKS = OUTDIR / "chromosome_chunks"

PRIMARY_CHRS = {f"chr{i}" for i in range(1, 23)} | {"chrX", "chrY"}
MITO_PREFIX = "chrM"

def fasta_from_zseeker(filename: str) -> str:
    return filename.replace(".zseeker.csv", ".fa.gz")

def infer_alias_key(name: str) -> str:
    n = name.lower()
    if "_pat" in n or "#1#" in n or "_hap1" in n:
        return "pat"
    if "_mat" in n or "#2#" in n or "_hap2" in n:
        return "mat"
    return ""

def find_alias_file(stem: str, prefix: str, key: str):
    exact = list(ALIAS_DIR.glob(f"{stem}.chromAlias.txt"))
    if exact:
        return exact[0]
    if key:
        cand = list(ALIAS_DIR.glob(f"{prefix}*{key}*.chromAlias.txt"))
        if cand:
            return cand[0]
    cand = list(ALIAS_DIR.glob(f"{prefix}*.chromAlias.txt"))
    return cand[0] if cand else None

def load_alias_map(alias_file: Path) -> dict:
    lines = [l.strip() for l in open(alias_file) if l.strip() and not l.startswith("#")]
    parts = [l.split() for l in lines]
    df = pd.DataFrame(parts, columns=["assembly", "ucsc", "genbank"]).apply(lambda x: x.str.strip())
    return dict(zip(df["assembly"], df["ucsc"]))

def chr_class(ucsc) -> str:
    if ucsc is None or (isinstance(ucsc, float) and np.isnan(ucsc)):
        return "no_alias"
    if ucsc in PRIMARY_CHRS:
        return "primary_chr"
    if ucsc.startswith(MITO_PREFIX):
        return "mitochondrial"
    if ucsc.startswith("chrUn"):
        return "chr_un"
    if ucsc.startswith("chr") and "_random" in ucsc:
        return "chr_random"
    if ucsc.startswith("chr"):
        return "chr_random"
    return "no_alias"

def effective_length_from_seq(seq: str, min_segment: int = MIN_SEGMENT) -> int:
    if not seq:
        return 0
    upper_bytes = seq.upper().encode("ascii", errors="replace")
    arr = np.frombuffer(upper_bytes, dtype=np.uint8)
    mask = (arr == ord("A")) | (arr == ord("C")) | (arr == ord("G")) | (arr == ord("T"))
    if not mask.any():
        return 0
    padded = np.concatenate(([False], mask, [False]))
    diffs = np.diff(padded.astype(np.int8))
    starts = np.flatnonzero(diffs == 1)
    ends = np.flatnonzero(diffs == -1)
    seg_lens = (ends - starts).astype(np.int64)
    return int(seg_lens[seg_lens >= min_segment].sum())

def genome_bp_effective_by_contig(fasta_gz: Path, alias_map: dict) -> pd.DataFrame:
    rows = []
    contig = None
    seq_chunks: list[str] = []

    def flush(cname, chunks):
        if cname is None or not chunks:
            return
        seq = "".join(chunks)
        ucsc = alias_map.get(cname)
        rows.append({
            "assembly_contig": cname,
            "ucsc_name": ucsc if ucsc is not None else "NA",
            "category": chr_class(ucsc),
            "genome_bp_raw": len(seq),
            "genome_bp_effective": effective_length_from_seq(seq, MIN_SEGMENT),
        })

    with gzip.open(fasta_gz, "rt") as fh:
        for line in fh:
            if line.startswith(">"):
                flush(contig, seq_chunks)
                contig = line[1:].strip().split()[0]
                seq_chunks = []
            else:
                seq_chunks.append(line.strip())
        flush(contig, seq_chunks)

    return pd.DataFrame(rows)

def zdna_bp_by_contig(csv_path: Path, alias_map: dict) -> pd.DataFrame:
    df = pd.read_csv(csv_path, usecols=["Chromosome", "Start", "End"], dtype=str)
    df = df.dropna(subset=["Chromosome", "Start", "End"])
    df["Start"] = pd.to_numeric(df["Start"], errors="coerce")
    df["End"]   = pd.to_numeric(df["End"], errors="coerce")
    df = df.dropna(subset=["Start", "End"])
    df["len"] = df["End"].astype(np.int64) - df["Start"].astype(np.int64) + 1

    df["assembly_contig"] = df["Chromosome"].astype(str).str.strip()
    df["ucsc_name"] = df["assembly_contig"].map(alias_map).fillna("NA")
    df["category"] = df["ucsc_name"].apply(
        lambda u: chr_class(u) if u != "NA" else "no_alias"
    )

    agg = df.groupby(
        ["assembly_contig", "ucsc_name", "category"], as_index=False
    ).agg(zdna_bp=("len", "sum"), n_intervals=("len", "size"))
    agg["zdna_bp"] = agg["zdna_bp"].astype(np.int64)
    agg["n_intervals"] = agg["n_intervals"].astype(np.int64)
    return agg

def main(target_csv: str) -> None:
    OUTDIR.mkdir(parents=True, exist_ok=True)
    OUT_CHUNKS.mkdir(parents=True, exist_ok=True)

    meta = pd.read_csv(META_PATH, sep="\t", dtype=str).fillna("")
    if meta.loc[meta["filename"] == target_csv].empty:
        raise SystemExit(f"❌ filename not found in metadata: {target_csv}")

    fname = target_csv
    zpath = ZDNA_DIR / fname
    fasta = FASTA_DIR / fasta_from_zseeker(fname)

    if not zpath.exists():
        raise SystemExit(f"❌ ZSeeker CSV not found: {zpath}")
    if not fasta.exists():
        raise SystemExit(f"❌ FASTA not found: {fasta}")

    stem   = fasta.name.replace(".fa.gz", "")
    prefix = fasta.name.split("_")[0]
    key    = infer_alias_key(fasta.name)
    alias_file = find_alias_file(stem, prefix, key)
    if alias_file is None:
        raise SystemExit(f"❌ Alias file not found for {fname}")

    alias_map = load_alias_map(alias_file)
    genome_df = genome_bp_effective_by_contig(fasta, alias_map)
    zdna_df   = zdna_bp_by_contig(zpath, alias_map)

    df = genome_df.merge(
        zdna_df,
        on=["assembly_contig", "ucsc_name", "category"],
        how="left",
    )
    df["zdna_bp"]     = df["zdna_bp"].fillna(0).astype(np.int64)
    df["n_intervals"] = df["n_intervals"].fillna(0).astype(np.int64)

    df["zdna_bp_per_mb"] = np.where(
        df["genome_bp_effective"] > 0,
        df["zdna_bp"] / (df["genome_bp_effective"] / 1e6),
        np.nan,
    )

    df.insert(0, "filename", fname)
    df["min_segment_bp"] = MIN_SEGMENT
    df["alias_file"]     = alias_file.name
    df["fasta_file"]     = fasta.name

    out_file = OUT_CHUNKS / f"{fname}.tsv"
    df.to_csv(out_file, sep="\t", index=False)
    print(f"✅ Wrote {out_file}")

if __name__ == "__main__":
    if len(sys.argv) != 2:
        raise SystemExit("Usage: build_hprc_chromosome_table.py <zseeker_csv_filename>")
    main(sys.argv[1])

In [ ]:
# Aggregate per-haplotype and per-contig chunk tables across all HPRC samples.
import gzip
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

MIN_SEGMENT = 50  # ACGT segment-length floor (bp) for effective genome size

META_PATH = Path("data/hprc_zdna_metrics.tsv")
ZDNA_DIR  = Path("data/zdna_csvs")
FASTA_DIR = Path("data/assemblies")
ALIAS_DIR = Path("data")
OUTDIR    = Path("data/out")
OUT_CHUNKS = OUTDIR / "haplotype_chunks"

PRIMARY_CHRS = {f"chr{i}" for i in range(1, 23)} | {"chrX", "chrY"}
MITO_PREFIX = "chrM"

CATEGORIES = ("primary_chr", "chr_random", "mitochondrial", "chr_un", "no_alias")

def fasta_from_zseeker(filename: str) -> str:
    return filename.replace(".zseeker.csv", ".fa.gz")

def infer_alias_key(name: str) -> str:
    n = name.lower()
    if "_pat" in n or "#1#" in n or "_hap1" in n:
        return "pat"
    if "_mat" in n or "#2#" in n or "_hap2" in n:
        return "mat"
    return ""

def find_alias_file(stem: str, prefix: str, key: str):
    exact = list(ALIAS_DIR.glob(f"{stem}.chromAlias.txt"))
    if exact:
        return exact[0]
    if key:
        cand = list(ALIAS_DIR.glob(f"{prefix}*{key}*.chromAlias.txt"))
        if cand:
            return cand[0]
    cand = list(ALIAS_DIR.glob(f"{prefix}*.chromAlias.txt"))
    return cand[0] if cand else None

def load_alias_map(alias_file: Path) -> dict:
    lines = [l.strip() for l in open(alias_file) if l.strip() and not l.startswith("#")]
    parts = [l.split() for l in lines]
    df = pd.DataFrame(parts, columns=["assembly", "ucsc", "genbank"]).apply(lambda x: x.str.strip())
    return dict(zip(df["assembly"], df["ucsc"]))

def chr_class(ucsc) -> str:
    if ucsc is None or (isinstance(ucsc, float) and np.isnan(ucsc)):
        return "no_alias"
    if ucsc in PRIMARY_CHRS:
        return "primary_chr"
    if ucsc.startswith(MITO_PREFIX):
        return "mitochondrial"
    if ucsc.startswith("chrUn"):
        return "chr_un"
    if ucsc.startswith("chr") and "_random" in ucsc:
        return "chr_random"
    if ucsc.startswith("chr"):
        return "chr_random"
    return "no_alias"

def effective_length_from_seq(seq: str, min_segment: int = MIN_SEGMENT) -> int:
    if not seq:
        return 0
    upper_bytes = seq.upper().encode("ascii", errors="replace")
    arr = np.frombuffer(upper_bytes, dtype=np.uint8)
    mask = (arr == ord("A")) | (arr == ord("C")) | (arr == ord("G")) | (arr == ord("T"))
    if not mask.any():
        return 0
    padded = np.concatenate(([False], mask, [False]))
    diffs = np.diff(padded.astype(np.int8))
    starts = np.flatnonzero(diffs == 1)
    ends = np.flatnonzero(diffs == -1)
    seg_lens = (ends - starts).astype(np.int64)
    return int(seg_lens[seg_lens >= min_segment].sum())

def genome_sizes_from_fasta(fasta_gz: Path, alias_map: dict):
    bp_eff = {c: 0 for c in CATEGORIES}
    bp_raw = {c: 0 for c in CATEGORIES}

    contig = None
    seq_chunks: list[str] = []

    def flush(cname, chunks):
        if cname is None or not chunks:
            return
        seq = "".join(chunks)
        ucsc = alias_map.get(cname)
        cls = chr_class(ucsc)
        bp_raw[cls] += len(seq)
        bp_eff[cls] += effective_length_from_seq(seq, MIN_SEGMENT)

    with gzip.open(fasta_gz, "rt") as fh:
        for line in fh:
            if line.startswith(">"):
                flush(contig, seq_chunks)
                contig = line[1:].strip().split()[0]
                seq_chunks = []
            else:
                seq_chunks.append(line.strip())
        flush(contig, seq_chunks)

    return bp_eff, bp_raw

def zdna_burden_from_zseeker(csv_path: Path, alias_map: dict):
    bp = {c: 0 for c in CATEGORIES}

    df = pd.read_csv(csv_path, dtype=str)
    if not {"Chromosome", "Start", "End"}.issubset(df.columns):
        return bp, 0, 0, 0

    df = df.dropna(subset=["Chromosome", "Start", "End"])
    df["Start"] = pd.to_numeric(df["Start"], errors="coerce")
    df["End"]   = pd.to_numeric(df["End"], errors="coerce")
    df = df.dropna(subset=["Start", "End"])
    df["len"] = df["End"].astype(int) - df["Start"].astype(int) + 1

    mapped = df["Chromosome"].astype(str).str.strip().map(alias_map)

    for L, ucsc in zip(df["len"].tolist(), mapped.tolist()):
        if pd.isna(ucsc):
            bp["no_alias"] += int(L)
        else:
            bp[chr_class(ucsc)] += int(L)

    n_total = len(df)
    n_unmapped = int(mapped.isna().sum())
    n_mapped = n_total - n_unmapped
    return bp, n_total, n_mapped, n_unmapped

def main(target_csv: str) -> None:
    OUTDIR.mkdir(parents=True, exist_ok=True)
    OUT_CHUNKS.mkdir(parents=True, exist_ok=True)

    meta = pd.read_csv(META_PATH, sep="\t", dtype=str).fillna("")
    row = meta.loc[meta["filename"] == target_csv]
    if row.empty:
        raise SystemExit(f"❌ filename not found in metadata: {target_csv}")

    fname  = target_csv
    zpath  = ZDNA_DIR  / fname
    fasta  = FASTA_DIR / fasta_from_zseeker(fname)

    stem   = fasta.name.replace(".fa.gz", "")
    prefix = fasta.name.split("_")[0]
    key    = infer_alias_key(fasta.name)

    alias_file = find_alias_file(stem, prefix, key)
    if alias_file is None:
        raise SystemExit(f"❌ Alias file not found for {fname}")
    if not zpath.exists():
        raise SystemExit(f"❌ ZSeeker CSV not found: {zpath}")
    if not fasta.exists():
        raise SystemExit(f"❌ FASTA not found: {fasta}")

    alias_map = load_alias_map(alias_file)

    zdna_bp, n_total, n_mapped, n_unmapped = zdna_burden_from_zseeker(zpath, alias_map)
    g_eff, g_raw = genome_sizes_from_fasta(fasta, alias_map)

    out_row = {
        "filename": fname,
        "fasta_file": fasta.name,
        "alias_file": alias_file.name,
        "min_segment_bp": MIN_SEGMENT,
        "n_intervals_total": n_total,
        "n_intervals_mapped": n_mapped,
        "n_intervals_unmapped_to_alias": n_unmapped,

        "zdna_bp_primary_chr":   zdna_bp["primary_chr"],
        "zdna_bp_chr_random":    zdna_bp["chr_random"],
        "zdna_bp_mitochondrial": zdna_bp["mitochondrial"],
        "zdna_bp_chr_un":        zdna_bp["chr_un"],
        "zdna_bp_no_alias":      zdna_bp["no_alias"],

        "genome_bp_primary_chr":   g_eff["primary_chr"],
        "genome_bp_chr_random":    g_eff["chr_random"],
        "genome_bp_mitochondrial": g_eff["mitochondrial"],
        "genome_bp_chr_un":        g_eff["chr_un"],
        "genome_bp_no_alias":      g_eff["no_alias"],

        "genome_bp_raw_primary_chr":   g_raw["primary_chr"],
        "genome_bp_raw_chr_random":    g_raw["chr_random"],
        "genome_bp_raw_mitochondrial": g_raw["mitochondrial"],
        "genome_bp_raw_chr_un":        g_raw["chr_un"],
        "genome_bp_raw_no_alias":      g_raw["no_alias"],
    }

    out_row["zdna_bp_per_mb_primary_chr"] = (
        out_row["zdna_bp_primary_chr"] /
        (out_row["genome_bp_primary_chr"] / 1e6)
        if out_row["genome_bp_primary_chr"] > 0 else float("nan")
    )

    out_file = OUT_CHUNKS / f"{fname}.tsv"
    pd.DataFrame([out_row]).to_csv(out_file, sep="\t", index=False)
    print(f"✅ Wrote {out_file}")

if __name__ == "__main__":
    if len(sys.argv) != 2:
        raise SystemExit("Usage: build_hprc_haplotype_tables.py <zseeker_csv_filename>")
    main(sys.argv[1])

## 5. Human genomes and Z-DNA density

### 5.1 Private vs shared Z-DNA predictions

In [ ]:
# Identify haplotype-private vs shared Z-DNA predictions across the pangenome.
from pathlib import Path
from collections import Counter
import pandas as pd

BASE = Path("data")

HPRC_DIR = BASE / "hprc_zseeker"

CHROM_TABLE = BASE / "haplotype_chromosome_zdna.tsv"

OUTFILE = BASE / "haplotype_private_sequences.tsv"

SPECIAL = {
    "GRCh38_primary": Path(
        "data/grch38/"
        "grch38"
    ),
    "CHM13": Path(
        "data/chm13/"
        "chm13.zdna.csv"
    ),
}

KEEP_CATEGORIES = {
    "primary_chr",
    "chr_random",
    "chr_un",
}

chrom = pd.read_csv(CHROM_TABLE, sep="\t", dtype=str)

chrom = chrom[
    chrom["category"].isin(KEEP_CATEGORIES)
]

allowed_contigs = (
    chrom.groupby("filename")["assembly_contig"]
    .apply(set)
    .to_dict()
)

input_files = {}

EXCLUDE = {
    "CHM13_v2.0_maskedY_rCRS.fa.PanSN.zseeker.csv",
    "GRCh38_GCA_000001405.15_no_alt_analysis_set.PanSN.zseeker.csv",
}

for f in sorted(HPRC_DIR.glob("*.zseeker.csv")):

    if f.name in EXCLUDE:
        continue

    input_files[f.name] = f

for name, path in SPECIAL.items():
    input_files[name] = path

print(f"Input datasets: {len(input_files)}")

dataset_sets = {}

global_counter = Counter()

for dataset_name, csv_path in input_files.items():

    print(f"Loading {dataset_name}")

    df = pd.read_csv(
        csv_path,
        usecols=["Chromosome", "Sequence"],
        dtype=str,
    )

    df = df.dropna(subset=["Chromosome", "Sequence"])

    if dataset_name in allowed_contigs:

        keep = allowed_contigs[dataset_name]

        df = df[
            df["Chromosome"].isin(keep)
        ]

    else:

        chroms = df["Chromosome"].astype(str)

        keep_mask = (
            chroms.str.startswith("chr")
            &
            ~chroms.str.startswith("chrM")
        )

        df = df[keep_mask]

    seq_set = set(df["Sequence"].astype(str))

    dataset_sets[dataset_name] = seq_set

    global_counter.update(seq_set)

    print(
        f"  unique={len(seq_set):,}"
    )

rows = []

for dataset_name, seq_set in dataset_sets.items():

    private_n = sum(
        1
        for s in seq_set
        if global_counter[s] == 1
    )

    total_unique = len(seq_set)

    rows.append({
        "dataset": dataset_name,
        "n_unique_sequences": total_unique,
        "n_private_sequences": private_n,
        "fraction_private": (
            private_n / total_unique
            if total_unique > 0 else 0
        ),
    })

out = pd.DataFrame(rows)

out = out.sort_values(
    "n_private_sequences",
    ascending=False,
)

out.to_csv(
    OUTFILE,
    sep="\t",
    index=False,
)

print(f"\nWrote: {OUTFILE}")
print(out.head())

In [ ]:
# Plot the number of private Z-DNA sequences per haplotype.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path("data")

PRIVATE = BASE / "haplotype_private_sequences.tsv"

META = BASE / "hprc_zdna_metrics.tsv"

OUTPNG = BASE / "private_sequences_by_haplotype.png"

priv = pd.read_csv(PRIVATE, sep="\t")

meta = pd.read_csv(META, sep="\t", dtype=str)

meta = meta[
    ["filename", "superpopulation"]
].copy()

meta["superpopulation"] = (
    meta["superpopulation"]
    .fillna("REFERENCE")
    .replace("", "REFERENCE")
)

df = priv.merge(
    meta,
    left_on="dataset",
    right_on="filename",
    how="left",
)

df.loc[
    df["dataset"] == "GRCh38_primary",
    "superpopulation"
] = "REFERENCE"

df.loc[
    df["dataset"] == "CHM13",
    "superpopulation"
] = "REFERENCE"

df = df.sort_values(
    "n_private_sequences",
    ascending=False,
).reset_index(drop=True)

color_map = {
    "AFR": "#81BA65",
    "AMR": "#B98ED1",
    "EAS": "#CFCA61",
    "EUR": "#97B3F0",
    "SAS": "#ED959A",
    "REFERENCE": "#999999"
}
COL_CHM13  = "#911423"
COL_GRCH38 = "#1C421C"

REFERENCE_BAR_WIDTH = 3.5
colors = [
    color_map.get(x, "#cccccc")
    for x in df["superpopulation"]
]

fig, ax = plt.subplots(figsize=(12, 5))

x = range(len(df))

regular_mask = ~df["dataset"].isin([
    "CHM13",
    "GRCh38_primary"
])

regular_df = df[regular_mask]

regular_x = regular_df.index

regular_colors = [
    color_map.get(x, "#cccccc")
    for x in regular_df["superpopulation"]
]

ax.bar(
    regular_x,
    regular_df["n_private_sequences"],
    color=regular_colors,
    width=1.0,
    linewidth=0,
    zorder=1
)

ref_specs = {
    "CHM13": COL_CHM13,
    "GRCh38_primary": COL_GRCH38,
}

for ref_name, ref_color in ref_specs.items():

    sub = df[df["dataset"] == ref_name]

    if len(sub) == 0:
        continue

    xpos = sub.index[0]
    ypos = sub["n_private_sequences"].iloc[0]

    ax.bar(
        xpos,
        ypos,
        color=ref_color,
        width=REFERENCE_BAR_WIDTH,
        linewidth=0,
        zorder=10
    )

    label = (
        "GRCh38"
        if ref_name == "GRCh38_primary"
        else "CHM13"
    )

    ax.text(
        xpos + 15,
        ypos + 35,
        label,
        ha="center",
        va="bottom",
        fontsize=17,
        fontweight="bold",
        color=ref_color,
        rotation=45,
        zorder=20
    )

ax.set_ylabel("Unique Z-DNA predictions", fontsize=22)

ax.set_xlabel("T2T Haplotypes", fontsize=22)

ax.set_xticks([])
ax.tick_params(axis='y', labelsize=17)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig(OUTPNG, dpi=600)

print(f"Wrote {OUTPNG}")

print("\nTop datasets:")
print(
    df[
        [
            "dataset",
            "superpopulation",
            "n_private_sequences",
            "fraction_private"
        ]
    ].head(20)
)

### 5.2 Score and length distributions

In [ ]:
# Z-DNA length and score distributions by superpopulation (violins + group statistics).
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import scikit_posthocs as sp_post

BASE = Path("data")

META = BASE / "hprc_zdna_metrics.tsv"

CHROM_TABLE = BASE / "haplotype_chromosome_zdna.tsv"

HPRC_DIR = BASE / "hprc_zseeker"

OUTDIR = BASE / "superpopulation_length_score"

OUTDIR.mkdir(exist_ok=True)

SPECIAL = {
    "GRCh38": Path(
        "data/grch38/"
        "grch38"
    ),
    "CHM13": Path(
        "data/chm13/"
        "chm13.zdna.csv"
    ),
}

KEEP_CATEGORIES = {
    "primary_chr",
    "chr_random",
    "chr_un",
}

SP_ORDER = [
    "AFR",
    "AMR",
    "EAS",
    "EUR",
    "SAS",
    "CHM13",
    "GRCh38",
]

SUPERPOP_COLORS = {
    "AFR": "#81BA65",
    "AMR": "#B98ED1",
    "EAS": "#CFCA61",
    "EUR": "#97B3F0",
    "SAS": "#ED959A",
    "CHM13": "#911423",
    "GRCh38": "#1C421C",
}

MAX_PER_GROUP = 200000

RNG = np.random.default_rng(1)

meta = pd.read_csv(
    META,
    sep="\t",
    dtype=str,
)

meta = meta[
    ["filename", "superpopulation"]
].copy()

meta = meta.dropna(subset=["filename"])

chrom = pd.read_csv(
    CHROM_TABLE,
    sep="\t",
    dtype=str,
)

chrom = chrom[
    chrom["category"].isin(KEEP_CATEGORIES)
]

allowed_contigs = (
    chrom.groupby("filename")["assembly_contig"]
    .apply(set)
    .to_dict()
)

input_files = {}

EXCLUDE = {
    "CHM13_v2.0_maskedY_rCRS.fa.PanSN.zseeker.csv",
    "GRCh38_GCA_000001405.15_no_alt_analysis_set.PanSN.zseeker.csv",
}

for f in sorted(HPRC_DIR.glob("*.zseeker.csv")):

    if f.name in EXCLUDE:
        continue

    input_files[f.name] = {
        "path": f,
        "superpopulation": None,
    }

input_files["CHM13"] = {
    "path": SPECIAL["CHM13"],
    "superpopulation": "CHM13",
}

input_files["GRCh38"] = {
    "path": SPECIAL["GRCh38"],
    "superpopulation": "GRCh38",
}

print(f"Input datasets: {len(input_files)}")

dfs = []

for dataset_name, info in input_files.items():

    path = info["path"]

    if info["superpopulation"] is not None:

        sp = info["superpopulation"]

    else:

        hit = meta[
            meta["filename"] == dataset_name
        ]

        if len(hit) == 0:
            continue

        sp = hit["superpopulation"].iloc[0]

        if pd.isna(sp) or sp == "":
            continue

    df = pd.read_csv(
        path,
        usecols=[
            "Chromosome",
            "Start",
            "End",
            "Z-DNA Score",
        ],
    )

    for c in ["Start", "End", "Z-DNA Score"]:

        df[c] = pd.to_numeric(
            df[c],
            errors="coerce",
        )

    df = df.dropna(
        subset=[
            "Chromosome",
            "Start",
            "End",
            "Z-DNA Score",
        ]
    )

    if dataset_name in allowed_contigs:

        keep = allowed_contigs[dataset_name]

        df = df[
            df["Chromosome"].isin(keep)
        ].copy()

    else:

        chroms = df["Chromosome"].astype(str)

        keep_mask = (
            chroms.str.startswith("chr")
            &
            ~chroms.str.startswith("chrM")
        )

        df = df[keep_mask].copy()

    df["length"] = (
        df["End"] - df["Start"] + 1
    ).astype(int)

    df["superpopulation"] = sp

    dfs.append(
        df[
            [
                "superpopulation",
                "length",
                "Z-DNA Score",
            ]
        ]
    )

 

all_df = pd.concat(
    dfs,
    ignore_index=True,
)

print("\nCombined dataframe:")
print(all_df.shape)

subs = []

for sp in SP_ORDER:

    sub = all_df[
        all_df["superpopulation"] == sp
    ].copy()

    print(
        f"{sp}: full_n={len(sub):,}"
    )

    if len(sub) > MAX_PER_GROUP:

        idx = RNG.choice(
            len(sub),
            size=MAX_PER_GROUP,
            replace=False,
        )

        sub = sub.iloc[idx]

    print(
        f"  plot_n={len(sub):,}"
    )

    subs.append(sub)

plot_df = pd.concat(
    subs,
    ignore_index=True,
)

print("\n")
print("=" * 80)
print("LENGTH STATISTICS")
print("=" * 80)

for sp in SP_ORDER:

    vals = all_df.loc[
        all_df["superpopulation"] == sp,
        "length"
    ]

    print(
        f"{sp:8s}",
        f"n={len(vals):,}",
        f"median={vals.median():.2f}",
        f"IQR=[{vals.quantile(0.25):.2f}, {vals.quantile(0.75):.2f}]",
        f"mean={vals.mean():.2f}",
    )

groups = [
    all_df.loc[
        all_df["superpopulation"] == sp,
        "length"
    ].values
    for sp in SP_ORDER
]

kw = stats.kruskal(*groups)

print("\nKruskal-Wallis (length)")
print(kw)

dunn = sp_post.posthoc_dunn(
    all_df,
    val_col="length",
    group_col="superpopulation",
    p_adjust="bonferroni",
)

dunn = dunn.reindex(
    index=SP_ORDER,
    columns=SP_ORDER,
)

print("\nDunn posthoc (length)")
print(dunn.round(4).to_string())

print("\n")
print("=" * 80)
print("SCORE STATISTICS")
print("=" * 80)

for sp in SP_ORDER:

    vals = all_df.loc[
        all_df["superpopulation"] == sp,
        "Z-DNA Score"
    ]

    print(
        f"{sp:8s}",
        f"n={len(vals):,}",
        f"median={vals.median():.2f}",
        f"IQR=[{vals.quantile(0.25):.2f}, {vals.quantile(0.75):.2f}]",
        f"mean={vals.mean():.2f}",
    )

groups = [
    all_df.loc[
        all_df["superpopulation"] == sp,
        "Z-DNA Score"
    ].values
    for sp in SP_ORDER
]

kw = stats.kruskal(*groups)

print("\nKruskal-Wallis (score)")
print(kw)

dunn = sp_post.posthoc_dunn(
    all_df,
    val_col="Z-DNA Score",
    group_col="superpopulation",
    p_adjust="bonferroni",
)

dunn = dunn.reindex(
    index=SP_ORDER,
    columns=SP_ORDER,
)

print("\nDunn posthoc (score)")
print(dunn.round(4).to_string())

BASE_FONT = 21          # previous 13 -> +1
AXIS_LABEL_SIZE = 26    # previous 18 -> +1
XTICK_SIZE = 24         # previous 15 -> +2
YTICK_SIZE = 22         # previous 13 -> +2

plt.rcParams.update({
    "font.size": BASE_FONT,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

LENGTH_Y_TICKS = np.array(
    [9, 50, 100, 500, 5000],
    dtype=float,
)

SCORE_Y_TICKS = np.array(
    [50, 75, 100, 500, 5000],
    dtype=float,
)

def forward_custom_y(y, ticks):

    y = np.asarray(y, dtype=float)

    out = np.zeros_like(y)

    for i in range(len(ticks) - 1):

        lo = ticks[i]
        hi = ticks[i + 1]

        mask = (y >= lo) & (y <= hi)

        if lo == 0:

            out[mask] = (
                i + (y[mask] - lo) / (hi - lo)
            )

        else:

            out[mask] = (
                i
                + np.log10(y[mask] / lo)
                / np.log10(hi / lo)
            )

    out[y > ticks[-1]] = len(ticks) - 1

    return out

def inverse_custom_y(z, ticks):

    z = np.asarray(z, dtype=float)

    out = np.zeros_like(z)

    for i in range(len(ticks) - 1):

        lo = ticks[i]
        hi = ticks[i + 1]

        mask = (z >= i) & (z <= i + 1)

        frac = z[mask] - i

        if lo == 0:

            out[mask] = (
                lo + frac * (hi - lo)
            )

        else:

            out[mask] = (
                lo * (hi / lo) ** frac
            )

    out[z > len(ticks) - 1] = ticks[-1]

    return out

def run_group_statistics(df, value_col, label):

    print("\n")
    print("=" * 80)
    print(f"{label.upper()} — SUPERPOPULATION STATISTICS")
    print("=" * 80)

    for sp in SP_ORDER:

        vals = df.loc[
            df["superpopulation"] == sp,
            value_col
        ]

        print(
            f"{sp:8s}",
            f"n={len(vals):,}",
            f"median={vals.median():.2f}",
            f"IQR=[{vals.quantile(0.25):.2f}, "
            f"{vals.quantile(0.75):.2f}]",
            f"mean={vals.mean():.2f}",
        )

    groups = [
        df.loc[
            df["superpopulation"] == sp,
            value_col
        ].values
        for sp in SP_ORDER
    ]

    kw = stats.kruskal(*groups)

    print(f"\nKruskal-Wallis ({label})")
    print(
        f"H = {kw.statistic:.4f}, "
        f"p = {kw.pvalue:.4g}, "
        f"df = {len(SP_ORDER)-1}"
    )

    dunn = sp_post.posthoc_dunn(
        df,
        val_col=value_col,
        group_col="superpopulation",
        p_adjust="bonferroni",
    )

    dunn = dunn.reindex(
        index=SP_ORDER,
        columns=SP_ORDER,
    )

    print(f"\nDunn posthoc ({label}, Bonferroni-corrected)")
    print(dunn.round(4).to_string())

    def cliffs_delta(x, y):

        x = np.asarray(x)
        y = np.asarray(y)

        u, _ = stats.mannwhitneyu(
            x,
            y,
            alternative="two-sided",
        )

        return (
            (2 * u) / (len(x) * len(y))
        ) - 1

    sig_pairs = []

    for i, a in enumerate(SP_ORDER):

        for b in SP_ORDER[i + 1:]:

            p = dunn.loc[a, b]

            x = df.loc[
                df["superpopulation"] == a,
                value_col
            ].values

            y = df.loc[
                df["superpopulation"] == b,
                value_col
            ].values

            delta = cliffs_delta(x, y)

            if p < 0.05:

                sig_pairs.append(
                    (a, b, p, delta)
                )

    if sig_pairs:

        print(
            "\nSignificant pairs after "
            "Bonferroni correction:"
        )

        for a, b, p, delta in sig_pairs:

            print(
                f"  {a} vs {b}: "
                f"p = {p:.4g}, "
                f"Cliff's delta = {delta:.5f}"
            )

    else:

        print(
            "\nNo pairs significant after "
            "Bonferroni correction."
        )

def distribution_plot(
    value_col,
    ylabel,
    outfile,
    y_ticks,
):

    fig, ax = plt.subplots(
        figsize=(14, 7)
    )

    data = [
        plot_df.loc[
            plot_df["superpopulation"] == sp,
            value_col
        ].values
        for sp in SP_ORDER
    ]

    for i, sp in enumerate(SP_ORDER, start=1):

        vals = plot_df.loc[
            plot_df["superpopulation"] == sp,
            value_col
        ].values

        if len(vals) > 15000:

            idx = RNG.choice(
                len(vals),
                size=15000,
                replace=False,
            )

            vals = vals[idx]

        x = i + RNG.normal(
            0,
            0.05,
            size=len(vals),
        )

        ax.scatter(
            x,
            vals,
            s=3,
            alpha=0.12,
            color=SUPERPOP_COLORS[sp],
            edgecolors="none",
            rasterized=True,
            zorder=1,
        )

    bp = ax.boxplot(
        data,
        patch_artist=True,
        widths=0.55,
        showfliers=False,
        zorder=5,
    )

    for box in bp["boxes"]:

        box.set_facecolor("none")
        box.set_edgecolor("black")
        box.set_linewidth(1.8)
        box.set_zorder(6)

    for item in (
        bp["medians"]
        + bp["whiskers"]
        + bp["caps"]
    ):

        item.set_color("black")
        item.set_linewidth(1.8)
        item.set_zorder(7)

    ax.set_xticks(
        range(1, len(SP_ORDER) + 1)
    )

    ax.set_xticklabels(
        SP_ORDER,
        fontsize=XTICK_SIZE,
    )

    ax.set_ylabel(
        ylabel,
        fontsize=AXIS_LABEL_SIZE,
    )

    ax.tick_params(
        axis="y",
        labelsize=YTICK_SIZE,
    )

    ax.set_yscale(
        "function",
        functions=(
            lambda y: forward_custom_y(y, y_ticks),
            lambda z: inverse_custom_y(z, y_ticks),
        ),
    )

    ax.set_yticks(y_ticks)

    ax.set_yticklabels(
        [str(int(x)) for x in y_ticks]
    )

    plt.tight_layout()

    plt.savefig(
        OUTDIR / outfile,
        dpi=600,
    )

    plt.show()

    print(f"Wrote {(OUTDIR / outfile)}")

distribution_plot(
    value_col="length",
    ylabel="Z-DNA length (bp)",
    outfile="superpopulation_length_violin.png",
    y_ticks=LENGTH_Y_TICKS,
)

distribution_plot(
    value_col="Z-DNA Score",
    ylabel="Z-DNA score",
    outfile="superpopulation_score_violin.png",
    y_ticks=SCORE_Y_TICKS,
)
print("\nDone.")

### 5.3 Density distribution and superpopulation comparison

In [ ]:
# Figure 1A/B: genome-wide Z-DNA density histogram and per-superpopulation density.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scikit_posthocs as sp_post
from matplotlib.patches import Patch
from scipy import stats

INCLUDED_CATS = ["primary_chr", "chr_random", "chr_un"]

WORK_DIR     = Path("data")
HAP_TABLE    = WORK_DIR / "hprc_haplotype_tables.final.tsv"
CONTIG_TABLE = WORK_DIR / "haplotype_chromosome_zdna.tsv"
META_PATH    = WORK_DIR / "hprc_zdna_metrics.tsv"
OUT_DIR      = WORK_DIR

CHM13_REF_DENSITY        = 4896.585494
GRCH38_REF_DENSITY       = 4682.554743

CHM13_AUTO_REF_DENSITY   = 4968.1480
GRCH38_AUTO_REF_DENSITY  = 4694.1008

SEX_CHR_PATTERN = r"(?i)^chr[XY]([._]|$)"

COL_HIST    = "#5CB4E0"
COL_CHM13   = "#911423"
COL_GRCH38  = "#1C421C"

SUPERPOP_COLORS = {
    "AFR": "#81BA65",
    "AMR": "#B98ED1",
    "EAS": "#CFCA61",
    "EUR": "#97B3F0",
    "SAS": "#ED959A",
}

REFERENCE_BAR_WIDTH = 3.5
SP_ORDER = ["AFR", "AMR", "EAS", "EUR", "SAS"]

BASE_FONT = 15  # previous default 12 -> +1 globally

AXIS_LABEL_SIZE = 23   # previous 20 -> +1
TICK_SIZE       = 17   # previous 14 -> +2
TEXT_SIZE       = 16   # previous 14 -> +1

BOX_XTICK_SIZE  = 22   # previous 18 -> +2
N_LABEL_SIZE    = 16  # previous 13 -> +1

LEGEND_SIZE     = 17   # previous 14 -> +1
LEGEND_TITLE    = 19   # previous 16 -> +1

plt.rcParams.update({
    "font.size": BASE_FONT,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

tbl  = pd.read_csv(HAP_TABLE, sep="\t")
meta = pd.read_csv(META_PATH, sep="\t", dtype=str).fillna("")
keep = meta[["filename", "sample_id", "superpopulation"]].copy()

df = tbl.merge(keep, on="filename", how="left", validate="one_to_one")

zdna_cols   = [f"zdna_bp_{c}"   for c in INCLUDED_CATS]
genome_cols = [f"genome_bp_{c}" for c in INCLUDED_CATS]

for c in zdna_cols + genome_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["zdna_bp_chrom"]   = df[zdna_cols].sum(axis=1)
df["genome_bp_chrom"] = df[genome_cols].sum(axis=1)

df = df[df["genome_bp_chrom"] > 0].copy()

df["density"] = (
    df["zdna_bp_chrom"] /
    (df["genome_bp_chrom"] / 1e6)
)

print(f"Loaded {len(df)} haplotypes (all chromosomal contigs, "
      f"scheme: {' + '.join(INCLUDED_CATS)}).")

ct = pd.read_csv(CONTIG_TABLE, sep="\t")

for c in ["genome_bp_effective", "zdna_bp"]:
    ct[c] = pd.to_numeric(ct[c], errors="coerce").fillna(0)

ct_chrom = ct[ct["category"].isin(INCLUDED_CATS)].copy()

sex_mask = ct_chrom["ucsc_name"].astype(str).str.match(SEX_CHR_PATTERN)

print(f"\nSex-chromosome contig filter:")
print(f"  Total chromosomal contigs:         {len(ct_chrom)}")
print(f"  Matched as chrX/chrY (excluded):   {int(sex_mask.sum())}")
print(f"  Distinct sex-contig names dropped: "
      f"{ct_chrom.loc[sex_mask, 'ucsc_name'].nunique()}")

ct_auto = ct_chrom[~sex_mask].copy()

per_hap_auto = (
    ct_auto.groupby("filename", as_index=False)
           .agg(
               zdna_bp_auto=("zdna_bp", "sum"),
               genome_bp_auto=("genome_bp_effective", "sum")
           )
)

per_hap_auto = per_hap_auto[
    per_hap_auto["genome_bp_auto"] > 0
].copy()

per_hap_auto["density_auto"] = (
    per_hap_auto["zdna_bp_auto"] /
    (per_hap_auto["genome_bp_auto"] / 1e6)
)

df_auto = per_hap_auto.merge(keep, on="filename", how="inner")

print(f"  Autosomal-only haplotypes:         {len(df_auto)}")

def print_summary(label: str,
                  d: pd.Series,
                  ref_chm13: float,
                  ref_grch38: float) -> None:

    print(f"\n{'='*70}\n{label}\n{'='*70}")

    print(f"  n           : {len(d)}")
    print(f"  Mean        : {d.mean():.2f} bp/Mb")
    print(f"  Median      : {d.median():.2f} bp/Mb")

    print(f"  Std         : {d.std():.2f} bp/Mb  "
          f"(CV = {100*d.std()/d.mean():.2f}%)")

    print(f"  IQR         : [{d.quantile(0.25):.2f}, "
          f"{d.quantile(0.75):.2f}]")

    print(f"  Range       : {d.max() - d.min():.2f}  "
          f"({d.min():.2f} -- {d.max():.2f})")

    print(f"  CHM13 ref   : {ref_chm13:.2f} bp/Mb  "
          f"(HPRC-CHM13 Δ = {ref_chm13 - d.median():+.2f})")

    print(f"  GRCh38 ref  : {ref_grch38:.2f} bp/Mb  "
          f"(HPRC-GRCh38 Δ = {ref_grch38 - d.median():+.2f})")

    n_above = int((d > ref_chm13).sum())
    n_below = int((d < ref_grch38).sum())

    print(f"  > CHM13     : {n_above} haplotypes "
          f"({100*n_above/len(d):.1f}%)")

    print(f"  < GRCh38    : {n_below} haplotypes "
          f"({100*n_below/len(d):.1f}%)")

print_summary(
    "All chromosomal contigs (Panel A)",
    df["density"],
    CHM13_REF_DENSITY,
    GRCH38_REF_DENSITY
)

print_summary(
    "Autosomes only (Suppl Fig 1A)",
    df_auto["density_auto"],
    CHM13_AUTO_REF_DENSITY,
    GRCH38_AUTO_REF_DENSITY
)

sd_all  = df["density"].std()
sd_auto = df_auto["density_auto"].std()

range_all  = df["density"].max() - df["density"].min()
range_auto = df_auto["density_auto"].max() - df_auto["density_auto"].min()

lev_stat, lev_p = stats.levene(
    df["density"].values,
    df_auto["density_auto"].values
)

print(f"\nAll-chrom vs autosomal-only comparison:")
print(f"  Std         : {sd_all:.2f} -> {sd_auto:.2f} bp/Mb  "
      f"({100*(sd_auto/sd_all - 1):+.1f}%)")

print(f"  Range       : {range_all:.1f} -> {range_auto:.1f} bp/Mb  "
      f"({100*(range_auto/range_all - 1):+.1f}%)")

print(f"  Median shift: {df['density'].median():.2f} -> "
      f"{df_auto['density_auto'].median():.2f}  "
      f"(+{df_auto['density_auto'].median() - df['density'].median():.2f} bp/Mb)")

print(f"  Levene's W  : {lev_stat:.3f}, p = {lev_p:.3g}")

gap_all  = CHM13_REF_DENSITY - GRCH38_REF_DENSITY
gap_auto = CHM13_AUTO_REF_DENSITY - GRCH38_AUTO_REF_DENSITY

print(f"\nGRCh38 -> CHM13 reference gap:")
print(f"  All-chrom   : {gap_all:.2f} bp/Mb")
print(f"  Autosomes   : {gap_auto:.2f} bp/Mb  "
      f"({100*(gap_auto/gap_all - 1):+.1f}%)")

sp_df = df[df["superpopulation"].isin(SUPERPOP_COLORS.keys())].copy()

sp_summary = (
    sp_df.groupby("superpopulation")["density"]
    .agg(
        n="size",
        mean="mean",
        median="median",
        std="std",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        min="min",
        max="max"
    )
    .round(2)
    .reindex(SP_ORDER)
)

print("\n" + "=" * 70)
print("Z-DNA density by superpopulation (all chromosomal contigs)")
print("=" * 70)

print(sp_summary.to_string())

groups = [
    sp_df.loc[sp_df["superpopulation"] == sp, "density"].values
    for sp in SP_ORDER
]

kw_stat, kw_p = stats.kruskal(*groups)

gp_medians = sp_summary["median"]
median_range = gp_medians.max() - gp_medians.min()

print(f"\nKruskal-Wallis H = {kw_stat:.3f}, "
      f"p = {kw_p:.4g}, df = {len(SP_ORDER)-1}")

print(f"  Range of group medians: {median_range:.2f} bp/Mb "
      f"({100*median_range/df['density'].median():.3f}% "
      f"of grand median)")

dunn = sp_post.posthoc_dunn(
    sp_df,
    val_col="density",
    group_col="superpopulation",
    p_adjust="bonferroni"
).reindex(index=SP_ORDER, columns=SP_ORDER)

print("\nDunn's post-hoc (Bonferroni-corrected):")
print(dunn.round(4).to_string())

sig_pairs = [
    (a, b, dunn.loc[a, b])
    for i, a in enumerate(SP_ORDER)
    for b in SP_ORDER[i+1:]
    if dunn.loc[a, b] < 0.05
]

if sig_pairs:
    print("\n  Significant pairs after Bonferroni:")
    for a, b, p in sig_pairs:
        print(f"    {a} vs {b}: p = {p:.4f}")
else:
    print("\n  No pairs significant after Bonferroni correction.")

def plot_histogram(values: pd.Series,
                   fname: str,
                   chm13_ref: float,
                   grch38_ref: float,
                   xlabel: str = "Z-DNA density (bp per Mb)",
                   bins: int = 16) -> None:

    fig, ax = plt.subplots(figsize=(11, 7))

    ax.hist(
        values,
        bins=bins,
        color=COL_HIST,
        edgecolor="black",
        linewidth=0.6
    )

    ax.axvline(
        chm13_ref,
        linestyle="--",
        linewidth=2.5,
        color=COL_CHM13
    )

    ax.axvline(
        grch38_ref,
        linestyle="--",
        linewidth=2.5,
        color=COL_GRCH38
    )

    ymax = ax.get_ylim()[1]

    x_offset = 5

    ax.text(
        chm13_ref + x_offset,
        ymax * 0.975,
        f"CHM13 ({chm13_ref:.0f} bp/Mb)",
        rotation=0,
        va="top",
        ha="left",
        fontsize=TEXT_SIZE,
        color=COL_CHM13,
        fontweight="bold"
    )

    ax.text(
        grch38_ref + x_offset,
        ymax * 0.975,
        f"GRCh38 ({grch38_ref:.0f} bp/Mb)",
        rotation=0,
        va="top",
        ha="left",
        fontsize=TEXT_SIZE,
        color=COL_GRCH38,
        fontweight="bold"
    )

    ax.set_xlabel(xlabel, fontsize=AXIS_LABEL_SIZE)
    ax.set_ylabel("Haplotypes (count)", fontsize=AXIS_LABEL_SIZE)

    ax.tick_params(axis="both", labelsize=TICK_SIZE)

    plt.tight_layout()

    plt.savefig(
        OUT_DIR / fname,
        dpi=600
    )

    plt.show()

plot_histogram(
    df["density"],
    fname="fig1_panelA_density_distribution.png",
    chm13_ref=CHM13_REF_DENSITY,
    grch38_ref=GRCH38_REF_DENSITY,
    xlabel="Z-DNA density (bp per Mb)",
)

plot_histogram(
    df_auto["density_auto"],
    fname="suppfig1A_autosomal_density_distribution.png",
    chm13_ref=CHM13_AUTO_REF_DENSITY,
    grch38_ref=GRCH38_AUTO_REF_DENSITY,
    xlabel="Z-DNA density (bp per Mb), autosomes only",
)

fig, ax = plt.subplots(figsize=(10, 7))

data = [
    sp_df.loc[sp_df["superpopulation"] == sp, "density"].values
    for sp in SP_ORDER
]

bp_artists = ax.boxplot(
    data,
    patch_artist=True,
    widths=0.5,
    showfliers=False
)

for box in bp_artists["boxes"]:
    box.set_facecolor("none")
    box.set_edgecolor("black")
    box.set_linewidth(2)

for el in ("medians", "whiskers", "caps"):
    for art in bp_artists[el]:
        art.set_color("black")
        if el == "medians":
            art.set_linewidth(2)

rng = np.random.default_rng(1)

for i, sp in enumerate(SP_ORDER, start=1):

    y = sp_df.loc[
        sp_df["superpopulation"] == sp,
        "density"
    ].values

    x = i + rng.normal(0, 0.05, size=len(y))

    ax.scatter(
        x,
        y,
        s=20,
        alpha=0.6,
        color=SUPERPOP_COLORS[sp],
        edgecolors="none",
        zorder=2
    )

ymax_panel = ax.get_ylim()[1]

for i, sp in enumerate(SP_ORDER, start=1):

    n = len(
        sp_df.loc[
            sp_df["superpopulation"] == sp
        ]
    )

    ax.text(
        i,
        ymax_panel * 1.005,
        f"n={n}",
        ha="center",
        va="bottom",
        fontsize=N_LABEL_SIZE,
        fontweight="bold"
    )

ax.set_xticks(range(1, len(SP_ORDER) + 1))

ax.set_xticklabels(
    SP_ORDER,
    fontsize=BOX_XTICK_SIZE
)

ax.set_ylabel(
    "Z-DNA density (bp per Mb)",
    fontsize=AXIS_LABEL_SIZE
)

ax.tick_params(axis="y", labelsize=TICK_SIZE)

plt.tight_layout()

plt.savefig(
    OUT_DIR / "fig1_panelB_superpopulation.png",
    dpi=600
)

plt.show()

fig, ax = plt.subplots(figsize=(3, 4))

ax.axis("off")

legend_elements = [
    Patch(
        facecolor=SUPERPOP_COLORS[sp],
        edgecolor="black",
        label=sp
    )
    for sp in SP_ORDER
]

ax.legend(
    handles=legend_elements,
    loc="center",
    frameon=True,
    fontsize=LEGEND_SIZE,
    title="Superpopulation",
    title_fontsize=LEGEND_TITLE
)

plt.tight_layout()

plt.savefig(
    OUT_DIR / "fig1_superpopulation_legend.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("\nAll figures saved to:", OUT_DIR)

### 5.4 Per-chromosome density and parental differences

In [ ]:
# Figure 1C: per-chromosome Z-DNA density (autosomes and sex chromosomes).
import os
os.chdir("data")
import re
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

INCLUDED_CATS = ["primary_chr", "chr_random", "chr_un"]

MIN_SEX_EFF_BP = 10_000_000

CHROM_ORDER = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
AUTOSOMES   = [f"chr{i}" for i in range(1, 23)]
CHROM_LABELS = [str(i) for i in range(1, 23)] + ["X", "Y"]
AUTO_LABELS  = [str(i) for i in range(1, 23)]

WORK_DIR     = Path("data")
CONTIG_TABLE = WORK_DIR / "haplotype_chromosome_zdna.tsv"

GRCH38_REF = {
    "chr1":  4586.05, "chr2":  4503.54, "chr3":  3626.99,
    "chr4":  3917.89, "chr5":  4175.95, "chr6":  4119.90,
    "chr7":  4833.63, "chr8":  4445.11, "chr9":  5011.71,
    "chr10": 4941.70, "chr11": 4545.76, "chr12": 4504.99,
    "chr13": 4218.92, "chr14": 4268.13, "chr15": 4570.52,
    "chr16": 6448.14, "chr17": 6227.94, "chr18": 4355.22,
    "chr19": 8422.65, "chr20": 5469.82, "chr21": 6329.68,
    "chr22": 7945.40, "chrX":  4401.86, "chrY":  5123.67,
}
CHM13_REF = {
    "chr1":  4538.88, "chr2":  4553.19, "chr3":  3652.49,
    "chr4":  3925.39, "chr5":  4216.28, "chr6":  4208.23,
    "chr7":  4825.67, "chr8":  4559.86, "chr9":  4278.55,
    "chr10": 5125.03, "chr11": 4617.81, "chr12": 4544.40,
    "chr13": 6997.97, "chr14": 4974.18, "chr15": 6296.24,
    "chr16": 5636.64, "chr17": 6268.02, "chr18": 4500.75,
    "chr19": 8525.25, "chr20": 5671.37, "chr21": 10918.40,
    "chr22": 8403.66, "chrX":  4553.38, "chrY":  2420.94,
}

COL_VIOLIN = "#5CB4E0"
COL_CHM13  = "#823941"
COL_GRCH38 = "#366942"

df = pd.read_csv(CONTIG_TABLE, sep="\t")
for c in ["genome_bp_effective", "zdna_bp"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df[df["category"].isin(INCLUDED_CATS)].copy()

_RE = re.compile(r"^(chr(?:[0-9]+|X|Y))(?:[._]|$)")
_S  = set(CHROM_ORDER)

def base_chr(name) -> Optional[str]:
    if pd.isna(name):
        return None
    m = _RE.match(str(name))
    if m and m.group(1) in _S:
        return m.group(1)
    return None

df["base_chr"] = df["ucsc_name"].apply(base_chr)
df = df.dropna(subset=["base_chr"]).copy()

per = (
    df.groupby(["filename", "base_chr"], as_index=False)
      .agg(zdna_bp=("zdna_bp", "sum"),
           genome_bp_effective=("genome_bp_effective", "sum"))
)
per = per[per["genome_bp_effective"] > 0].copy()
per["density"] = per["zdna_bp"] / (per["genome_bp_effective"] / 1e6)

print("=" * 70)
print("chrX/chrY haplotype QC diagnostic")
print("=" * 70)

for sex_chr in ["chrX", "chrY"]:
    sub = per[per["base_chr"] == sex_chr].copy()
    print(f"\n{sex_chr}: n = {len(sub)} haplotypes present")
    print(f"  genome_bp_effective: "
          f"min={sub['genome_bp_effective'].min():,.0f}, "
          f"med={sub['genome_bp_effective'].median():,.0f}, "
          f"max={sub['genome_bp_effective'].max():,.0f}")
    print(f"  zdna_bp:             "
          f"min={sub['zdna_bp'].min():,.0f}, "
          f"med={sub['zdna_bp'].median():,.0f}, "
          f"max={sub['zdna_bp'].max():,.0f}")
    print(f"  density:             "
          f"min={sub['density'].min():.1f}, "
          f"med={sub['density'].median():.1f}, "
          f"max={sub['density'].max():.1f}")
    n_below = (sub["genome_bp_effective"] < MIN_SEX_EFF_BP).sum()
    print(f"  Haplotypes with eff < {MIN_SEX_EFF_BP/1e6:.0f} Mb (filtered): "
          f"{n_below}")
    if n_below:
        worst = sub.nsmallest(5, "genome_bp_effective")[
            ["filename", "genome_bp_effective", "zdna_bp", "density"]
        ]
        print(f"  Smallest 5 (likely assembly issues):")
        print(worst.to_string(index=False))

is_sex = per["base_chr"].isin(["chrX", "chrY"])
qc_fail = is_sex & (per["genome_bp_effective"] < MIN_SEX_EFF_BP)
n_filtered = int(qc_fail.sum())
per_qc = per[~qc_fail].copy()
print(f"\nTotal sex-chrom haplotypes filtered: {n_filtered}")

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def plot_violins(per_filtered: pd.DataFrame, chroms: list, labels: list,
                 fname: str, figsize: tuple, show_n: bool) -> None:
    data = []
    ns   = []
    for chrom in chroms:
        vals = per_filtered.loc[per_filtered["base_chr"] == chrom, "density"].dropna().values
        data.append(vals)
        ns.append(len(vals))

    fig, ax = plt.subplots(figsize=figsize)
    positions = list(range(1, len(chroms) + 1))

    vp = ax.violinplot(data, positions=positions,
                       showmedians=True, showextrema=False, widths=0.7)
    for body in vp["bodies"]:
        body.set_facecolor(COL_VIOLIN)
        body.set_edgecolor("black")
        body.set_alpha(0.85)
    if "cmedians" in vp:
        vp["cmedians"].set_color("black")
        vp["cmedians"].set_linewidth(1.5)

    for pos, chrom in zip(positions, chroms):
        ax.scatter(pos, GRCH38_REF[chrom], s=60, marker="D",
                   color=COL_GRCH38, edgecolors="black", linewidth=0.8, zorder=4)
        ax.scatter(pos, CHM13_REF[chrom], s=60, marker="D",
                   color=COL_CHM13, edgecolors="black", linewidth=0.8, zorder=4)

    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=19)
    ax.set_xlabel("Chromosome", fontsize=23)
    ax.set_ylabel("Z-DNA density (bp per Mb)", fontsize=23)
    ax.tick_params(axis="y", labelsize=18)

    if show_n:
        ymax = ax.get_ylim()[1]
        for pos, n in zip(positions, ns):
            ax.text(pos, ymax * 1.005, f"{n}",
                    ha="center", va="bottom", fontsize=15, color="grey")

    legend_handles = [
        Line2D([0], [0], color=COL_VIOLIN, lw=8, alpha=0.85,
               label=f"HPRC haplotypes"),
        Line2D([0], [0], marker="D", color="none",
               markerfacecolor=COL_GRCH38, markeredgecolor="black",
               markersize=15, label="GRCh38"),
        Line2D([0], [0], marker="D", color="none",
               markerfacecolor=COL_CHM13, markeredgecolor="black",
               markersize=15, label="CHM13"),
    ]
    ax.legend(handles=legend_handles, loc="upper left",
              frameon=True, fontsize=15)

    plt.tight_layout()
    out = WORK_DIR / fname
    plt.savefig(out, dpi=600)
    plt.show()
    print(f"Saved {out}")

plot_violins(per_qc, AUTOSOMES, AUTO_LABELS,
             fname="fig1_panelC_autosomes.png",
             figsize=(15, 7), show_n=False)

plot_violins(per_qc, CHROM_ORDER, CHROM_LABELS,
             fname="fig1_panelC_with_sex_chromosomes.png",
             figsize=(16, 7), show_n=True)

print("\n" + "=" * 95)
print("Per-chromosome density summary (after sex-chrom QC filter)")
print("=" * 95)
summary = (
    per_qc.groupby("base_chr")["density"]
    .agg(n="size", median="median", mean="mean", std="std",
         q25=lambda x: x.quantile(0.25), q75=lambda x: x.quantile(0.75),
         min="min", max="max")
    .reindex(CHROM_ORDER)
    .round(2)
)
summary["grch38_ref"] = pd.Series(GRCH38_REF).round(2)
summary["chm13_ref"]  = pd.Series(CHM13_REF).round(2)
print(summary.to_string())

In [ ]:
# Figure 1C: chromosome-level density variance relative to primary chromosomes.
print("\n" + "=" * 80)
print("Primary chromosome representation vs density variability")
print("=" * 80)

primary = df[df["category"] == "primary_chr"].copy()

primary["base_chr"] = primary["ucsc_name"].apply(base_chr)

primary = primary.dropna(subset=["base_chr"])

primary_counts = (
    primary.groupby("base_chr")["filename"]
    .nunique()
    .reindex(CHROM_ORDER)
)

var_df = (
    per_qc.groupby("base_chr")["density"]
    .agg(
        mean="mean",
        std="std",
        variance="var",
        n="size",
    )
    .reindex(CHROM_ORDER)
)

var_df["cv"] = (
    var_df["std"] / var_df["mean"]
)

var_df["primary_n"] = primary_counts

print("\nPer-chromosome variability:")

print(
    var_df[
        ["primary_n", "mean", "std", "cv"]
    ].round(4).to_string()
)

corr_df = var_df.dropna().copy()

r, p = stats.spearmanr(
    corr_df["primary_n"],
    corr_df["cv"],
)

print("\nSpearman correlation:")

print(
    f"rho = {r:.4f}, "
    f"p = {p:.4g}"
)

x = corr_df["primary_n"].values
y = corr_df["cv"].values

slope, intercept = np.polyfit(x, y, 1)

xfit = np.linspace(
    x.min(),
    x.max(),
    200,
)

yfit = slope * xfit + intercept

fig, ax = plt.subplots(
    figsize=(14, 8)
)

ax.scatter(
    x,
    y,
    s=100,
    color="#5CB4E0",
    edgecolors="black",
    linewidth=1,
    zorder=3,
)

ax.plot(
    xfit,
    yfit,
    linestyle="--",
    linewidth=2.2,
    color="black",
    zorder=2,
)

stats_text = (
    f"Spearman ρ = {r:.2f}\n"
    f"p = {p:.2g}\n"
    f"n = {len(corr_df)} chromosomes"
)

ax.text(
    0.98,
    0.98,
    stats_text,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=21,
    bbox=dict(
        boxstyle="round,pad=0.4",
        facecolor="white",
        edgecolor="black",
        alpha=0.95,
    ),
)

ax.set_xlabel(
    "Haplotypes with primary chromosome assembly",
    fontsize=26,
)

ax.set_ylabel(
    "Z-DNA density coeff. of variation",
    fontsize=26,
)

ax.tick_params(
    axis="both",
    labelsize=22,
)

plt.tight_layout()

out = WORK_DIR / "fig1_panelC_variance_vs_primarychr.png"

plt.savefig(
    out,
    dpi=600,
)

plt.show()

print(f"\nSaved {out}")

In [ ]:
# Supplementary: per-chromosome %GC vs Z-DNA density (CHM13).
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import spearmanr

WORK_DIR  = Path("data")

GC_TSV    = WORK_DIR / "reference_chromosome_gc.tsv"

ZDNA_TSV  = (
    WORK_DIR
    / "chm13/CHM13_zdna_density_per_chr.tsv"
)

OUT_PATH  = (
    WORK_DIR
    / "suppfig1_chm13_gc_vs_density.png"
)

ASSEMBLY = "CHM13"

ANNOTATE_EXTRA = ["chr13", "chr21"]

N_GC_EXTREMES = 3

CHROM_ORDER = (
    [f"chr{i}" for i in range(1, 23)]
    + ["chrX", "chrY"]
)

gc = pd.read_csv(
    GC_TSV,
    sep="\t",
)

gc = gc[
    gc["assembly"] == ASSEMBLY
].copy()

zd = pd.read_csv(
    ZDNA_TSV,
    sep="\t",
)

zd = zd[
    [
        "ucsc_chr",
        "density_bp_per_Mb_min50",
    ]
].copy()

zd = zd.rename(
    columns={
        "density_bp_per_Mb_min50": "density_bp_per_mb"
    }
)

df = gc.merge(
    zd,
    on="ucsc_chr",
    how="inner",
)

df = df[
    df["ucsc_chr"].isin(CHROM_ORDER)
].copy()

df["__order"] = df["ucsc_chr"].map(
    {c: i for i, c in enumerate(CHROM_ORDER)}
)

df = (
    df.sort_values("__order")
      .drop(columns="__order")
)

print(
    f"Chromosomes with both GC and density: {len(df)}"
)

print(
    df[
        [
            "ucsc_chr",
            "gc_pct_acgt",
            "density_bp_per_mb",
        ]
    ].to_string(index=False)
)

rho, pval = spearmanr(
    df["gc_pct_acgt"],
    df["density_bp_per_mb"],
)

X = sm.add_constant(
    df["gc_pct_acgt"].values
)

ols = sm.OLS(
    df["density_bp_per_mb"].values,
    X,
).fit()

print(
    f"\nSpearman rho = {rho:.4f}, "
    f"p = {pval:.3g}"
)

print(
    f"OLS: slope = {ols.params[1]:.2f} bp/Mb per %GC, "
    f"intercept = {ols.params[0]:.2f}, "
    f"R^2 = {ols.rsquared:.3f}"
)

df["predicted"] = ols.predict(X)

df["residual"] = (
    df["density_bp_per_mb"]
    - df["predicted"]
)

df_sorted = df.sort_values(
    "residual",
    ascending=False,
)

print(
    "\nTop 5 positive residuals "
    "(more Z-DNA than GC predicts):"
)

print(
    df_sorted.head(5)[
        [
            "ucsc_chr",
            "gc_pct_acgt",
            "density_bp_per_mb",
            "predicted",
            "residual",
        ]
    ].to_string(index=False)
)

print(
    "\nTop 5 negative residuals "
    "(less Z-DNA than GC predicts):"
)

print(
    df_sorted.tail(5)[
        [
            "ucsc_chr",
            "gc_pct_acgt",
            "density_bp_per_mb",
            "predicted",
            "residual",
        ]
    ].to_string(index=False)
)

low_gc = (
    df.nsmallest(
        N_GC_EXTREMES,
        "gc_pct_acgt",
    )["ucsc_chr"].tolist()
)

high_gc = (
    df.nlargest(
        N_GC_EXTREMES,
        "gc_pct_acgt",
    )["ucsc_chr"].tolist()
)

annotate = sorted(
    set(low_gc + high_gc + ANNOTATE_EXTRA),
    key=lambda c: CHROM_ORDER.index(c),
)

print(
    f"\nAnnotated chromosomes: {annotate}"
)

plt.rcParams.update({
    "font.size": 16,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

fig, ax = plt.subplots(
    figsize=(14, 8)
)

ax.scatter(
    df["gc_pct_acgt"],
    df["density_bp_per_mb"],
    s=60,
    color="#4FA3F7",
    edgecolors="black",
    linewidth=0.5,
    zorder=2,
)

x_smooth = np.linspace(
    df["gc_pct_acgt"].min(),
    df["gc_pct_acgt"].max(),
    200,
)

y_smooth = ols.predict(
    sm.add_constant(x_smooth)
)

ax.plot(
    x_smooth,
    y_smooth,
    "--",
    color="black",
    linewidth=2.2,
    zorder=3,
)

y_offset = (
    (
        df["density_bp_per_mb"].max()
        - df["density_bp_per_mb"].min()
    )
    * 0.025
)

for _, row in df.iterrows():

    if row["ucsc_chr"] in annotate:

        ax.text(
            row["gc_pct_acgt"],
            row["density_bp_per_mb"] + y_offset,
            row["ucsc_chr"],
            ha="center",
            va="bottom",
            fontsize=18,
            fontweight="bold",
        )

ax.set_xlabel(
    "GC content (%)",
    fontsize=24,
)

ax.set_ylabel(
    "Z-DNA density (bp per Mb)",
    fontsize=24,
)

ax.tick_params(
    axis="both",
    labelsize=20,
)

p_text = (
    "p < 0.001"
    if pval < 0.001
    else f"p = {pval:.2e}"
)

stats_text = (
    f"Spearman ρ = {rho:.2f}\n"
    f"{p_text}\n"
    f"n = {len(df)} chromosomes"
)

ax.text(
    0.97,
    0.97,
    stats_text,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=19,
    bbox=dict(
        facecolor="white",
        edgecolor="black",
        boxstyle="round,pad=0.4",
        alpha=0.95,
    ),
)

plt.tight_layout()

plt.savefig(
    OUT_PATH,
    dpi=600,
)

plt.show()

print(f"\nSaved {OUT_PATH}")

In [ ]:
# Supplementary: paternal vs maternal autosomal Z-DNA density.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

INCLUDED_CATS = ["primary_chr", "chr_random", "chr_un"]
SEX_CHR_PATTERN = r"(?i)^chr[XY]([._]|$)"

WORK_DIR     = Path("data")
CONTIG_TABLE = WORK_DIR / "haplotype_chromosome_zdna.tsv"
META_PATH    = WORK_DIR / "hprc_zdna_metrics.tsv"
OUT_PATH     = WORK_DIR / "suppfig1_pat_vs_mat_autosomal.png"

ct = pd.read_csv(CONTIG_TABLE, sep="\t")
for c in ["genome_bp_effective", "zdna_bp"]:
    ct[c] = pd.to_numeric(ct[c], errors="coerce").fillna(0)

meta = pd.read_csv(META_PATH, sep="\t", dtype=str).fillna("")
meta = meta[["filename", "sample_id", "pat_or_mat"]].copy()

ct_chrom = ct[ct["category"].isin(INCLUDED_CATS)].copy()
sex_mask = ct_chrom["ucsc_name"].astype(str).str.match(SEX_CHR_PATTERN)
ct_auto  = ct_chrom[~sex_mask].copy()

print(f"Chromosomal contigs: {len(ct_chrom)}")
print(f"  After dropping chrX/chrY-derived contigs: {len(ct_auto)}")

per_hap = (
    ct_auto.groupby("filename", as_index=False)
           .agg(zdna_bp=("zdna_bp", "sum"),
                genome_bp=("genome_bp_effective", "sum"))
)
per_hap = per_hap[per_hap["genome_bp"] > 0].copy()
per_hap["density"] = per_hap["zdna_bp"] / (per_hap["genome_bp"] / 1e6)

per_hap = per_hap.merge(meta, on="filename", how="inner")

per_hap = per_hap[per_hap["pat_or_mat"].isin(["pat", "mat"])].copy()

print(f"\nHaplotypes with pat/mat label: {len(per_hap)}")
print(per_hap["pat_or_mat"].value_counts().to_string())

wide = per_hap.pivot_table(
    index="sample_id", columns="pat_or_mat", values="density", aggfunc="first"
)

n_samples_total       = len(wide)
n_samples_pat_only    = int(wide["mat"].isna().sum() & ~wide["pat"].isna().sum())
n_samples_mat_only    = int(wide["pat"].isna().sum() & ~wide["mat"].isna().sum())
n_samples_both_pat    = int(wide["pat"].notna().sum())
n_samples_both_mat    = int(wide["mat"].notna().sum())

paired = wide.dropna(subset=["pat", "mat"]).copy()
n_pairs = len(paired)
n_excluded_singletons = n_samples_total - n_pairs

print(f"\nSamples appearing in pat/mat data:                {n_samples_total}")
print(f"  Samples with both pat AND mat haplotype (paired): {n_pairs}")
print(f"  Samples excluded as singletons:                   {n_excluded_singletons}")

pat = paired["pat"].values
mat = paired["mat"].values
delta = pat - mat

med_delta  = float(np.median(delta))
mean_delta = float(np.mean(delta))
sd_delta   = float(np.std(delta, ddof=1))
mean_abs   = float(np.mean(np.abs(delta)))

wilc = wilcoxon(pat, mat, alternative="two-sided")

print(f"\n{'='*70}")
print(f"Paired Wilcoxon: paternal vs maternal autosomal Z-DNA density")
print(f"{'='*70}")
print(f"  n pairs               : {n_pairs}")
print(f"  Median Δ (pat - mat)  : {med_delta:+.3f} bp/Mb")
print(f"  Mean Δ (pat - mat)    : {mean_delta:+.3f} ± {sd_delta:.3f} bp/Mb (SD)")
print(f"  Mean |Δ|              : {mean_abs:.3f} bp/Mb")
print(f"  Wilcoxon W            : {wilc.statistic:.1f}")
print(f"  Wilcoxon p            : {wilc.pvalue:.4g}")

grand_median = (np.median(pat) + np.median(mat)) / 2
print(f"  Mean |Δ| as % of grand median ({grand_median:.1f}): "
      f"{100*mean_abs/grand_median:.3f}%")

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
})

fig, ax = plt.subplots(figsize=(20, 7))

ax.scatter(mat, pat, s=35, alpha=0.7, color="#4C72B0",
           edgecolors="black", linewidth=0.4, zorder=2)

lim_lo = min(mat.min(), pat.min())
lim_hi = max(mat.max(), pat.max())
pad    = (lim_hi - lim_lo) * 0.03
ax.plot([lim_lo - pad, lim_hi + pad], [lim_lo - pad, lim_hi + pad],
        color="black", linewidth=1, zorder=1)

ax.set_xlim(lim_lo - pad, lim_hi + pad)
ax.set_ylim(lim_lo - pad, lim_hi + pad)
ax.set_aspect("equal", adjustable="box")

ax.set_xlabel("Maternal autosomal Z-DNA density (bp/Mb)", fontsize=19)
ax.set_ylabel("Paternal autosomal Z-DNA density (bp/Mb)", fontsize=19)
ax.tick_params(axis="both", labelsize=16)

annotation = (
    f"n pairs = {n_pairs}\n"
    f"Median Δ = {med_delta:+.2f} bp/Mb\n"
    f"Mean |Δ| = {mean_abs:.2f} bp/Mb\n"
    f"Wilcoxon p = {wilc.pvalue:.3g}"
)
ax.text(0.03, 0.97, annotation, transform=ax.transAxes,
        ha="left", va="top", fontsize=16,
        bbox=dict(facecolor="white", edgecolor="black",
                  boxstyle="round,pad=0.4"))

plt.savefig(OUT_PATH, dpi=600)
plt.show()
print(f"\nSaved {OUT_PATH}")

## 6. GC-matched controls

Random control intervals matched to Z-DNA on length and GC content, at three GC-tolerance levels (5/10/15%).

In [ ]:
# Generate GC-matched random control intervals for each Z-DNA call.
import os
import gc
import time
import random
import bisect
from pathlib import Path

import numpy as np
import pandas as pd
from pyfaidx import Fasta
from tqdm import tqdm

BASE = "data"
proc = int(os.environ.get("SLURM_PROCID", "0"))

if proc == 0:
    GC_TOL = DINUC_TOL = 0.05
    OUT_DIR = f"{BASE}/controls_5pct_chm13"
elif proc == 1:
    GC_TOL = DINUC_TOL = 0.10
    OUT_DIR = f"{BASE}/controls_10pct_chm13"
elif proc == 2:
    GC_TOL = DINUC_TOL = 0.15
    OUT_DIR = f"{BASE}/controls_15pct_chm13"
else:
    raise SystemExit("Unexpected SLURM_PROCID")

os.makedirs(OUT_DIR, exist_ok=True)

random.seed(42)
np.random.seed(42)

MAX_RANDOM_TRIES = 10_000

FASTA_FILE = f"{BASE}/chm13_refseq/chm13.fa"
ZD_FILE    = f"{BASE}/chm13_zdna_files/chm13.zdna.csv"

CHR_TO_CONTIG = {
    "chr1":  "NC_060925.1", "chr2":  "NC_060926.1", "chr3":  "NC_060927.1",
    "chr4":  "NC_060928.1", "chr5":  "NC_060929.1", "chr6":  "NC_060930.1",
    "chr7":  "NC_060931.1", "chr8":  "NC_060932.1", "chr9":  "NC_060933.1",
    "chr10": "NC_060934.1", "chr11": "NC_060935.1", "chr12": "NC_060936.1",
    "chr13": "NC_060937.1", "chr14": "NC_060938.1", "chr15": "NC_060939.1",
    "chr16": "NC_060940.1", "chr17": "NC_060941.1", "chr18": "NC_060942.1",
    "chr19": "NC_060943.1", "chr20": "NC_060944.1", "chr21": "NC_060945.1",
    "chr22": "NC_060946.1", "chrX":  "NC_060947.1", "chrY":  "NC_060948.1",
}

def within_tol(val, target, tol):
    if target <= 0:
        return val <= tol
    return abs(val - target) <= tol * target

def overlaps(intervals, s, e):
    if not intervals:
        return False
    i = bisect.bisect_right(intervals, (s, e)) - 1
    for j in (i, i + 1):
        if 0 <= j < len(intervals):
            s0, e0 = intervals[j]
            if not (e <= s0 or e0 <= s):
                return True
    return False

def random_candidate(L, Lchr):
    return np.random.randint(0, Lchr - L + 1)

fasta = Fasta(FASTA_FILE)
zd = pd.read_csv(ZD_FILE)

zd["Start0"] = zd["Start"]
zd["End0"]   = zd["End"] + 1
zd["Length"] = zd["End0"] - zd["Start0"]

for chrom in sorted(zd["Chromosome"].unique()):
    if chrom not in CHR_TO_CONTIG:
        continue

    contig = CHR_TO_CONTIG[chrom]
    seq = str(fasta[contig]).upper()
    Lchr = len(seq)

    print(f"\n=== {chrom} ({Lchr:,} bp) ===")

    zdf = zd[zd["Chromosome"] == chrom].copy()

    zdf = zdf.sort_values("Length", ascending=False).reset_index(drop=True)

    b = np.frombuffer(seq.encode("ascii"), dtype=np.uint8)

    is_gc = ((b == ord("G")) | (b == ord("C"))).astype(np.int8)
    gc_cum = np.concatenate([[0], np.cumsum(is_gc)])

    dinuc = np.zeros(Lchr - 1, dtype=np.int8)
    dinuc[(b[:-1] == ord("C")) & (b[1:] == ord("G"))] = 1
    dinuc[(b[:-1] == ord("G")) & (b[1:] == ord("C"))] = 1
    dinuc_cum = np.concatenate([[0], np.cumsum(dinuc)])

    def gc_frac(s, e):
        return (gc_cum[e] - gc_cum[s]) / (e - s)

    def dinuc_frac(s, e):
        if e - s < 2:
            return 0.0
        return (dinuc_cum[e - 1] - dinuc_cum[s]) / (e - s - 1)

    zdf["GC_pct"] = [
        gc_frac(s, e) for s, e in zip(zdf["Start0"], zdf["End0"])
    ]
    zdf["Dinuc_density"] = [
        dinuc_frac(s, e) for s, e in zip(zdf["Start0"], zdf["End0"])
    ]

    Z_intervals = sorted(zip(zdf["Start0"], zdf["End0"]))
    control_intervals = []

    zdf["Ctrl_Start"] = np.nan
    zdf["Ctrl_End"]   = np.nan
    zdf["Ctrl_Seq"]   = ""
    zdf["fallback"]   = False

    for i, r in zdf.iterrows():
        L = int(r.Length)
        t_gc = r.GC_pct
        t_dn = r.Dinuc_density

        placed = False

        for _ in range(MAX_RANDOM_TRIES):
            s = random_candidate(L, Lchr)
            e = s + L
            if overlaps(Z_intervals, s, e): continue
            if overlaps(control_intervals, s, e): continue
            if not within_tol(gc_frac(s, e), t_gc, GC_TOL): continue
            if not within_tol(dinuc_frac(s, e), t_dn, DINUC_TOL): continue

            placed = True
            break

        if not placed:
            forbidden = sorted(Z_intervals + control_intervals)
            free = []
            prev = 0
            for s0, e0 in forbidden:
                if s0 > prev:
                    free.append((prev, s0))
                prev = max(prev, e0)
            if prev < Lchr:
                free.append((prev, Lchr))

            random.shuffle(free)

            for fs, fe in free:
                if fe - fs < L:
                    continue
                start = random.randint(fs, fe - L)
                e = start + L
                if not within_tol(gc_frac(start, e), t_gc, GC_TOL): continue
                if not within_tol(dinuc_frac(start, e), t_dn, DINUC_TOL): continue
                s = start
                placed = True
                zdf.at[i, "fallback"] = True
                break

        if placed:
            zdf.at[i, "Ctrl_Start"] = s
            zdf.at[i, "Ctrl_End"]   = e
            zdf.at[i, "Ctrl_Seq"]   = seq[s:e]
            bisect.insort(control_intervals, (s, e))

    out = f"{OUT_DIR}/{chrom}_controls.tsv"
    zdf.to_csv(out, sep="\t", index=False)
    print(f"Saved {out}")

    del zdf, gc_cum, dinuc_cum
    gc.collect()

print("\nDONE.")

## 7. Ideograms (CHM13)

In [ ]:
# CHM13 ideogram coloured by Z-DNA density.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
import matplotlib.patches as mpatches

os.chdir("data/chm13/chm13_marbl_ann_files")

CYTO_BED   = "chm13v2.0_cytobands_allchrs.bed"
ZDNA_BED   = "chm13_zdna_all.bed"
CENSAT_BED = "chm13_censat.bed"

BIN_SIZE      = 150_000
CHROM_HEIGHT  = 0.30    # half-height of cytoband bar
CENSAT_HEIGHT = 0.3    # height of censat strip below bar
DENSITY_MAX   = 0.75    # max y-extent of Z-DNA ridge above bar
ROW_SPACING   = 1.9     # increased slightly to accommodate censat strip
LABEL_FONTSIZE = 30

STAIN_COLORS = {
    "gneg":    "#FFFFFF",
    "gpos25":  "#C0C0C0",
    "gpos50":  "#808080",
    "gpos75":  "#404040",
    "gpos100": "#000000",
    "acen":    "#B22222",
    "gvar":    "#4682B4",
    "stalk":   "#32CD32",
}

CENSAT_COLORS = {
    "hor":    "#FF7F00",   # orange
    "dhor":   "#FDBF6F",   # light orange
    "hsat1A": "#6A3D9A",   # purple
    "hsat1B": "#CAB2D6",   # lavender
    "hsat2":  "#1F78B4",   # strong blue   (distinct from gvar steel-blue)
    "hsat3":  "#A6CEE3",   # pale blue
    "bsat":   "#33A02C",   # forest green  (distinct from stalk lime)
    "gsat":   "#B2DF8A",   # pale green
    "censat": "#E31A1C",   # crimson (distinct from acen brick-red & zdna red)
    "ct":     "#FB9A99",   # pink
    "mon":    "#FFFF99",   # yellow
    "rDNA":   "#B15928",   # brown
}

CHROM_ORDER = (
    [f"chr{i}" for i in range(1, 8)]
    + ["chrX"]
    + [f"chr{i}" for i in range(8, 23)]
    + ["chrY"]
)

def load_cytobands(path):
    df = pd.read_csv(
        path, sep="\t", header=None,
        names=["chrom", "start", "end", "name", "stain"],
        usecols=[0, 1, 2, 3, 4],
    )
    df = df[df["chrom"].isin(CHROM_ORDER)].copy()
    df["start"] = df["start"].astype(np.int64)
    df["end"]   = df["end"].astype(np.int64)
    return df

def load_zdna(path):
    df = pd.read_csv(
        path, sep="\t", header=None,
        names=["chrom", "start", "end"],
        usecols=[0, 1, 2],
    )
    df = df[df["chrom"].isin(CHROM_ORDER)].copy()
    df["start"] = df["start"].astype(np.int64)
    df["end"]   = df["end"].astype(np.int64)
    return df

def load_censat(path):
    df = pd.read_csv(
        path, sep="\t", header=None,
        names=["chrom", "start", "end", "name"],
        usecols=[0, 1, 2, 3],
    )
    df = df[df["chrom"].isin(CHROM_ORDER)].copy()
    df["start"]    = df["start"].astype(np.int64)
    df["end"]      = df["end"].astype(np.int64)
    df["category"] = df["name"].str.split("_").str[0]
    df = df[df["category"].isin(CENSAT_COLORS)].copy()
    return df

def chrom_sizes_from_cytobands(cyto_df):
    sizes = {}
    for chrom in CHROM_ORDER:
        sub = cyto_df[cyto_df["chrom"] == chrom]
        if len(sub):
            sizes[chrom] = int(sub["end"].max())
    return sizes

def compute_zdna_density(zdna_df, chrom_sizes, bin_size):
    densities = {}

    for chrom, clen in chrom_sizes.items():
        sub = zdna_df[zdna_df["chrom"] == chrom]
        n_bins = int(np.ceil(clen / bin_size))
        covered = np.zeros(n_bins, dtype=np.float64)

        if len(sub):
            s = np.clip(sub["start"].values, 0, clen).astype(np.int64)
            e = np.clip(sub["end"].values,   0, clen).astype(np.int64)
            mask = e > s
            s, e = s[mask], e[mask]

            if len(s):
                b0 = s // bin_size
                b1 = (e - 1) // bin_size

                same = b0 == b1
                if same.any():
                    np.add.at(covered, b0[same], (e[same] - s[same]).astype(float))

                multi = ~same
                if multi.any():
                    sm, em, bsm, bem = s[multi], e[multi], b0[multi], b1[multi]
                    np.add.at(covered, bsm, ((bsm + 1) * bin_size - sm).astype(float))
                    np.add.at(covered, bem, (em - bem * bin_size).astype(float))

                    for i in range(len(sm)):
                        for b in range(bsm[i] + 1, bem[i]):
                            covered[b] += bin_size

        bin_starts = np.arange(n_bins, dtype=np.int64) * bin_size
        bin_ends   = np.minimum(bin_starts + bin_size, clen)
        widths     = (bin_ends - bin_starts).astype(np.float64)
        mids       = bin_starts + widths / 2.0

        chrom_max_bp = covered.max() if len(covered) else 0.0
        dens_norm = covered / chrom_max_bp if chrom_max_bp > 0 else covered

        densities[chrom] = (mids, dens_norm, chrom_max_bp)
        print("\nMaximum Z-DNA coverage per chromosome:")
        print("(bp of Z-DNA contained in the densest 150 kb bin)\n")
        
        for chrom, (_, _, chrom_max_bp) in densities.items():
            print(f"{chrom:>5}: {chrom_max_bp:,.0f} bp / {bin_size:,} bp")

    return densities

def plot_ideogram(cyto, censat, densities, chrom_sizes):
    chroms  = [c for c in CHROM_ORDER if c in chrom_sizes]
    max_bp  = max(chrom_sizes.values())
    n_chr   = len(chroms)

    fig_w = 20
    fig_h = n_chr * ROW_SPACING * 0.52 + 2
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    ax.set_xlim(-max_bp * 0.07, max_bp * 1.02)
    ax.set_ylim(-ROW_SPACING * 0.9, n_chr * ROW_SPACING + 0.5)
    ax.axis("off")

    for ci, chrom in enumerate(chroms):
        clen     = chrom_sizes[chrom]
        y_center = (n_chr - 1 - ci) * ROW_SPACING

        sub_cyto = cyto[cyto["chrom"] == chrom].sort_values("start")
        for _, band in sub_cyto.iterrows():
            color = STAIN_COLORS.get(band["stain"], "#FFFFFF")
            ax.add_patch(Rectangle(
                (band["start"], y_center - CHROM_HEIGHT),
                band["end"] - band["start"],
                2 * CHROM_HEIGHT,
                facecolor=color, edgecolor="none",
            ))

        ax.add_patch(FancyBboxPatch(
            (0, y_center - CHROM_HEIGHT), clen, 2 * CHROM_HEIGHT,
            boxstyle=f"round,pad=0,rounding_size={CHROM_HEIGHT * 0.8}",
            facecolor="none", edgecolor="black", linewidth=0.8,
        ))

        y_censat_top = y_center - CHROM_HEIGHT - 0.04
        y_censat_bot = y_censat_top - CENSAT_HEIGHT

        sub_cen = censat[censat["chrom"] == chrom].sort_values("start")
        for _, seg in sub_cen.iterrows():
            color = CENSAT_COLORS.get(seg["category"], "#AAAAAA")
            ax.add_patch(Rectangle(
                (seg["start"], y_censat_bot),
                seg["end"] - seg["start"],
                CENSAT_HEIGHT,
                facecolor=color, edgecolor="none", alpha=0.85,
            ))

        if len(sub_cen):
            cen_lo = int(sub_cen["start"].min())
            cen_hi = int(sub_cen["end"].max())
            ax.add_patch(Rectangle(
                (cen_lo, y_censat_bot),
                cen_hi - cen_lo,
                CENSAT_HEIGHT,
                facecolor="none", edgecolor="#555555", linewidth=0.5,
            ))
        if chrom in densities:
            mids, dens, chrom_max_bp = densities[chrom]

            y_base = y_center + CHROM_HEIGHT + 0.02
            y_top  = y_base + DENSITY_MAX
            y_vals = y_base + dens * DENSITY_MAX

            poly_x = np.concatenate([[mids[0]], mids, [mids[-1]]])
            poly_y = np.concatenate([[y_base], y_vals, [y_base]])

            ax.fill(poly_x, poly_y, color="#513AB5", alpha=0.55, linewidth=0)
            ax.plot(mids, y_vals, color="#513AB5", linewidth=0.3)

            x_axis = -max_bp * 0.0005 # y axis to the right needs smaller value
            tick_len = max_bp * 0.003

            ax.plot(
                [x_axis, x_axis],
                [y_base, y_top],
                color="black",
                linewidth=0.8
            )

            ax.plot(
                [x_axis, x_axis + tick_len],
                [y_top, y_top],
                color="black",
                linewidth=0.8
            )

            ax.text(
                x_axis - tick_len * 0.5,
                y_top,
                f"{chrom_max_bp/1000:.1f}",
                ha="right",
                va="center",
                fontsize=14
            )

        ax.text(
            -max_bp * 0.01, y_center,
            chrom.replace("chr", ""),
            ha="right", va="center",
            fontsize=LABEL_FONTSIZE, fontweight="bold",
        )

    scale_y   = -ROW_SPACING * 0.65
    scale_len = 50_000_000
    ax.plot([0, scale_len], [scale_y, scale_y], color="black", linewidth=1.5)
    ax.text(scale_len / 2, scale_y - 0.22, "50 Mb",
            ha="center", va="top", fontsize=16)

    plt.tight_layout()
    fig.savefig("chm13_ideogram_zdna.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("[SAVED] chm13_ideogram_zdna.png / .pdf")

def save_legend():
    legend_items = [
        ("#FFFFFF", "gneg — GC-rich euchromatin",           "black"),
        ("#C0C0C0", "gpos25 — moderately AT-rich",          "black"),
        ("#808080", "gpos50 — AT-rich, lower gene density", "black"),
        ("#404040", "gpos75 — strongly AT-rich",            "black"),
        ("#000000", "gpos100 — condensed heterochromatin",  "black"),
        ("#B22222", "acen — centromere (alpha-satellite)",  "black"),
        ("#4682B4", "gvar — variable heterochromatin",      "black"),
        ("#32CD32", "stalk — NOR / rDNA stalk",             "black"),
        ("#513AB5", "Z-DNA density",                        "black"),
    ] + [
        (color, f"CenSat: {cat}", "black")
        for cat, color in CENSAT_COLORS.items()
    ]

    patches = [
        mpatches.Patch(
            facecolor=fc,
            edgecolor=ec,
            label=lbl,
            alpha=0.55 if "Z-DNA" in lbl else 0.85 if "CenSat" in lbl else 1.0,
        )
        for fc, lbl, ec in legend_items
    ]

    n      = len(patches)
    n_cols = 3
    n_rows = int(np.ceil(n / n_cols))
    fig, ax = plt.subplots(figsize=(12, n_rows * 0.55 + 0.8))
    ax.axis("off")
    ax.legend(
        handles=patches,
        loc="center",
        ncol=n_cols,
        frameon=True,
        fontsize=11,
        handlelength=1.5,
        handleheight=1.2,
    )
    plt.tight_layout()
    fig.savefig("chm13_ideogram_legend.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

def main():
    cyto   = load_cytobands(CYTO_BED)
    zdna   = load_zdna(ZDNA_BED)
    censat = load_censat(CENSAT_BED)

    chrom_sizes = chrom_sizes_from_cytobands(cyto)
    densities   = compute_zdna_density(zdna, chrom_sizes, BIN_SIZE)

    print("CenSat categories found:", sorted(censat["category"].unique()))

    plot_ideogram(cyto, censat, densities, chrom_sizes)
    save_legend()
    print("[DONE]")

if __name__ == "__main__":
    main()

In [ ]:
# Z-DNA density per cytoband stain category and per CenSat satellite class.
import os
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

os.chdir("data/chm13/chm13_marbl_ann_files")

CYTO_BED   = "chm13v2.0_cytobands_allchrs.bed"
CENSAT_BED = "chm13_censat.bed"
ZDNA_BED   = "chm13_zdna_all.bed"

CHROM_ORDER = [f"chr{c}" for c in list(range(1, 23)) + ["X", "Y"]]

CYTOBAND_ORDER = [
    "gneg",
    "gpos25",
    "gpos50",
    "gpos75",
    "gpos100",
    "acen",
    "gvar",
    "stalk",
]

CENSAT_ORDER = [
    "hor",
    "dhor",
    "hsat1A",
    "hsat1B",
    "hsat2",
    "hsat3",
    "bsat",
    "gsat",
    "censat",
    "ct",
    "mon",
    "rDNA",
]

STAIN_COLORS = {
    "gneg":    "#FFFFFF",
    "gpos25":  "#C0C0C0",
    "gpos50":  "#808080",
    "gpos75":  "#404040",
    "gpos100": "#000000",
    "acen":    "#B22222",
    "gvar":    "#4682B4",
    "stalk":   "#32CD32",
}

CENSAT_COLORS = {
    "hor":    "#FF7F00",
    "dhor":   "#FDBF6F",
    "hsat1A": "#6A3D9A",
    "hsat1B": "#CAB2D6",
    "hsat2":  "#1F78B4",
    "hsat3":  "#A6CEE3",
    "bsat":   "#33A02C",
    "gsat":   "#B2DF8A",
    "censat": "#E31A1C",
    "ct":     "#FB9A99",
    "mon":    "#FFFF99",
    "rDNA":   "#B15928",
}

LIGHT = {
    "#FFFFFF",
    "#FFFF99",
    "#C0C0C0",
    "#FDBF6F",
    "#B2DF8A",
    "#A6CEE3",
    "#CAB2D6",
    "#FB9A99",
}

FS_LABEL = 22
FS_TICK  = 19
FS_COUNT = 15

RNG_SEED = 42

comparisons_cyto = [

    (
        "euchromatin",
        ["gneg", "gpos25"],

        "heterochromatin",
        ["gpos75", "gpos100"]
    ),

    (
        "stalk",
        ["stalk"],

        "other",
        ["gneg", "gpos25", "gpos50",
         "gpos75", "gpos100", "acen", "gvar"]
    ),

    (
        "acen",
        ["acen"],

        "non_acen",
        ["gneg", "gpos25", "gpos50",
         "gpos75", "gpos100", "gvar"]
    ),
]

comparisons_censat = [

    (
        "rDNA",
        ["rDNA"],

        "non_rDNA",
        ["hor", "dhor", "hsat1A", "hsat1B",
         "hsat2", "hsat3", "bsat", "gsat",
         "censat", "ct", "mon"]
    ),

    (
        "censat",
        ["censat"],

        "other_satellites",
        ["hor", "dhor", "hsat1A", "hsat1B",
         "hsat2", "hsat3", "bsat", "gsat",
         "ct", "mon"]
    ),

    (
        "hsat1B",
        ["hsat1B"],

        "other_hsat",
        ["hsat1A", "hsat2", "hsat3"]
    ),
]

def load_zdna(path):

    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["chrom", "start", "end"],
        usecols=[0, 1, 2]
    )

    df = df[df["chrom"].isin(CHROM_ORDER)].copy()

    df["start"] = df["start"].astype(np.int64)
    df["end"]   = df["end"].astype(np.int64)

    return df

def load_cytobands(path):

    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["chrom", "start", "end", "name", "stain"],
        usecols=[0, 1, 2, 3, 4]
    )

    df = df[df["chrom"].isin(CHROM_ORDER)].copy()

    df["start"] = df["start"].astype(np.int64)
    df["end"]   = df["end"].astype(np.int64)

    return df

def load_censat(path):

    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["chrom", "start", "end", "name"],
        usecols=[0, 1, 2, 3]
    )

    df = df[df["chrom"].isin(CHROM_ORDER)].copy()

    df["start"] = df["start"].astype(np.int64)
    df["end"]   = df["end"].astype(np.int64)

    df["category"] = df["name"].str.split("_").str[0]

    df = df[df["category"].isin(CENSAT_COLORS)].copy()

    return df

def build_zdna_coverage(zdna_df, chrom, chrom_len):

    sub = zdna_df[zdna_df["chrom"] == chrom]

    if len(sub) == 0:
        return np.zeros(chrom_len, dtype=bool)

    s = np.clip(sub["start"].values, 0, chrom_len).astype(np.int64)
    e = np.clip(sub["end"].values,   0, chrom_len).astype(np.int64)

    mask = e > s

    s = s[mask]
    e = e[mask]

    diff = np.zeros(chrom_len + 1, dtype=np.int32)

    np.add.at(diff, s,  1)
    np.add.at(diff, e, -1)

    return np.cumsum(diff[:-1]) > 0

def zdna_density_for_intervals(intervals_df, zdna_df, category_col):

    rows = []

    for chrom in intervals_df["chrom"].unique():

        sub_int = intervals_df[intervals_df["chrom"] == chrom]

        if len(sub_int) == 0:
            continue

        chrom_len = int(sub_int["end"].max() + 1)

        cov = build_zdna_coverage(zdna_df, chrom, chrom_len)

        for _, row in sub_int.iterrows():

            s = max(0, int(row["start"]))
            e = min(chrom_len, int(row["end"]))

            if e <= s:
                continue

            length = e - s

            zdna_bp = int(cov[s:e].sum())

            density = zdna_bp / length

            rows.append({
                "category": row[category_col],
                "chrom": chrom,
                "start": s,
                "end": e,
                "length": length,
                "zdna_bp": zdna_bp,
                "density": density,
            })

    return pd.DataFrame(rows)

def get_vals(df, cats):

    return (
        df[df["category"].isin(cats)]["density"].values * 1000
    )

def cliffs_delta(x, y):

    x = np.asarray(x)
    y = np.asarray(y)

    gt = 0
    lt = 0

    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)

    return (gt - lt) / (len(x) * len(y))

def run_pairwise_tests(data_df, comparisons, out_tsv):

    results = []

    for a_name, a_cats, b_name, b_cats in comparisons:

        a = get_vals(data_df, a_cats)
        b = get_vals(data_df, b_cats)

        stat, p = mannwhitneyu(
            a,
            b,
            alternative="two-sided"
        )

        delta = cliffs_delta(a, b)

        results.append({

            "group1": a_name,
            "group2": b_name,

            "n1": len(a),
            "n2": len(b),

            "median1_bp_per_kb": np.median(a),
            "median2_bp_per_kb": np.median(b),

            "mean1_bp_per_kb": np.mean(a),
            "mean2_bp_per_kb": np.mean(b),

            "cliffs_delta": delta,

            "u_statistic": stat,
            "p_value": p,
        })

    res = pd.DataFrame(results)

    res["p_adj_bh"] = multipletests(
        res["p_value"],
        method="fdr_bh"
    )[1]

    res.to_csv(out_tsv, sep="\t", index=False)

    print(f"  [SAVED] {out_tsv}")

    return res

def _draw_boxes_and_jitter(
    ax,
    cats,
    positions,
    color_map,
    y_vals_for_plot
):

    rng = np.random.default_rng(RNG_SEED)

    bp = ax.boxplot(
        y_vals_for_plot,
        positions=positions,
        widths=0.38,
        patch_artist=False,
        showfliers=False,
    )

    for element in ["boxes", "whiskers", "caps", "medians"]:

        for artist in bp[element]:

            artist.set_color("#333333")
            artist.set_linewidth(1.4)
            artist.set_zorder(4)

    for pos, cat, vals in zip(
        positions,
        cats,
        y_vals_for_plot
    ):

        if len(vals) == 0:
            continue

        jitter = rng.uniform(-0.15, 0.15, size=len(vals))

        color = color_map.get(cat, "#999999")

        ec = "#555555" if color in LIGHT else color

        ax.scatter(
            pos + jitter,
            vals,

            s=18,

            color=color,
            edgecolors=ec,

            linewidths=0.5,
            alpha=0.65,

            zorder=2,
        )

def _set_xaxis(ax, cats, positions, cat_counts):

    ax.set_xticks(positions)

    texts = ax.set_xticklabels(
        cats,
        rotation=35,
        ha="right",
        fontsize=FS_TICK
    )

    ax.figure.canvas.draw()

    for t, n in zip(texts, cat_counts):

        x, y = t.get_position()
        N_OFFSET = 1
        ax.text(
            x + 0.85,
            y + N_OFFSET,
        
            f"n={n}",
        
            transform=t.get_transform(),
        
            rotation=t.get_rotation(),
        
            ha=t.get_ha(),
            va="bottom",
        
            fontsize=FS_COUNT,
            color="black"
        )

def _despine(ax):

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(0.9)
    ax.spines["bottom"].set_linewidth(0.9)

def make_boxplot_log2(
    data_df,
    category_order,
    color_map,
    title_label,
    out_png,
    out_tsv
):

    cats = [
        c for c in category_order
        if c in data_df["category"].values
    ]

    out = data_df[data_df["category"].isin(cats)].copy()

    out["density_bp_per_kb"] = out["density"] * 1000

    out.to_csv(out_tsv, sep="\t", index=False)

    print(f"  [SAVED] {out_tsv}")

    cat_data_bpkb = [

        data_df.loc[
            data_df["category"] == c,
            "density"
        ].values * 1000

        for c in cats
    ]

    cat_counts = [len(v) for v in cat_data_bpkb]

    cat_data_log = [
        np.log2(v + 1)
        for v in cat_data_bpkb
    ]

    real_ticks = [0, 2, 4, 8, 16, 32, 64, 128]

    log_ticks = [
        np.log2(v + 1)
        for v in real_ticks
    ]

    tick_labels = [str(v) for v in real_ticks]

    y_max_log = max(
        v.max()
        for v in cat_data_log
        if len(v) > 0
    )

    y_ceil = max(
        log_ticks[-1],
        np.ceil(y_max_log * 1.05)
    )

    fig_w = max(6, len(cats) * 0.55)

    fig, ax = plt.subplots(figsize=(fig_w, 6))

    positions = np.arange(1, len(cats) + 1)

    _draw_boxes_and_jitter(
        ax,
        cats,
        positions,
        color_map,
        cat_data_log
    )

    _set_xaxis(ax, cats, positions, cat_counts)

    ax.set_xlim(0.3, len(cats) + 0.7)

    ax.set_ylim(0, y_ceil)

    ax.set_yticks(log_ticks)
    ax.set_yticklabels(tick_labels, fontsize=FS_TICK)

    ax.set_ylabel(
        "Z-DNA density (bp per kb)",
        fontsize=FS_LABEL
    )

    ax.set_xlabel(
        title_label,
        fontsize=FS_LABEL
    )

    ax.tick_params(axis="both", labelsize=FS_TICK)

    _despine(ax)

    plt.tight_layout()

    fig.savefig(
        out_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    print(f"  [SAVED] {out_png}")

def make_boxplot_log10(
    data_df,
    category_order,
    color_map,
    title_label,
    out_png,
    out_tsv
):

    cats = [
        c for c in category_order
        if c in data_df["category"].values
    ]

    out = data_df[data_df["category"].isin(cats)].copy()

    out["density_bp_per_kb"] = out["density"] * 1000

    out.to_csv(out_tsv, sep="\t", index=False)

    print(f"  [SAVED] {out_tsv}")

    cat_data_bpkb = [

        data_df.loc[
            data_df["category"] == c,
            "density"
        ].values * 1000

        for c in cats
    ]

    cat_counts = [len(v) for v in cat_data_bpkb]

    cat_data_log = [
        np.log10(v + 1)
        for v in cat_data_bpkb
    ]

    real_ticks = [0, 1, 10, 100, 1000]

    log_ticks = [
        np.log10(v + 1)
        for v in real_ticks
    ]

    tick_labels = [str(v) for v in real_ticks]

    y_max_log = max(
        v.max()
        for v in cat_data_log
        if len(v) > 0
    )

    y_ceil = max(
        log_ticks[-1],
        np.ceil(y_max_log * 1.05)
    )

    fig_w = max(6, len(cats) * 0.65)

    fig, ax = plt.subplots(figsize=(fig_w, 6))

    positions = np.arange(1, len(cats) + 1)

    _draw_boxes_and_jitter(
        ax,
        cats,
        positions,
        color_map,
        cat_data_log
    )

    _set_xaxis(ax, cats, positions, cat_counts)

    ax.set_xlim(0.3, len(cats) + 0.7)

    ax.set_ylim(0, y_ceil)

    ax.set_yticks(log_ticks)
    ax.set_yticklabels(tick_labels, fontsize=FS_TICK)

    ax.set_ylabel(
        "Z-DNA density (bp per kb)",
        fontsize=FS_LABEL
    )

    ax.set_xlabel(
        title_label,
        fontsize=FS_LABEL
    )

    ax.tick_params(axis="both", labelsize=FS_TICK)

    _despine(ax)

    plt.tight_layout()

    fig.savefig(
        out_png,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

    print(f"  [SAVED] {out_png}")

def main():

    print("[INFO] Loading data...")

    zdna   = load_zdna(ZDNA_BED)
    cyto   = load_cytobands(CYTO_BED)
    censat = load_censat(CENSAT_BED)

    print("[INFO] Computing cytoband densities...")

    cyto_data = zdna_density_for_intervals(
        cyto.rename(columns={"stain": "category"}),
        zdna,
        category_col="category",
    )
    print(
        cyto_data.groupby("category")["density"]
        .agg(["count", "mean", "median"])
        .to_string()
    )
    make_boxplot_log2(
        cyto_data,
        category_order=CYTOBAND_ORDER,
        color_map=STAIN_COLORS,
        title_label="Cytoband stain category",
        out_png="zdna_density_by_cytoband.png",
        out_tsv="zdna_density_by_cytoband.tsv",
    )
    run_pairwise_tests(
        cyto_data,
        comparisons_cyto,
        "cytoband_selected_comparisons.tsv"
    )
    print("[INFO] Computing CenSat densities...")
    censat_data = zdna_density_for_intervals(
        censat,
        zdna,
        category_col="category",
    )
    print(
        censat_data.groupby("category")["density"]
        .agg(["count", "mean", "median"])
        .to_string()
    )
    make_boxplot_log10(
        censat_data,
        category_order=CENSAT_ORDER,
        color_map=CENSAT_COLORS,
        title_label="CenSat category",
        out_png="zdna_density_by_censat.png",
        out_tsv="zdna_density_by_censat.tsv",
    )
    run_pairwise_tests(
        censat_data,
        comparisons_censat,
        "censat_selected_comparisons.tsv"
    )
    print("[DONE]")
if __name__ == "__main__":
    main()

In [ ]:
# Z-DNA karyogram for the sex chromosomes (X / Y).
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
import matplotlib.patches as mpatches

os.chdir("data/chm13/chm13_marbl_ann_files")

CYTO_BED = "chm13v2.0_cytobands_allchrs.bed"
ZDNA_BED = "chm13_zdna_all.bed"
SEQ_BED  = "chm13v2.0_chrXY_sequence_class_v1.bed"

BIN_SIZE = 150_000
CHROMS = ["chrX", "chrY"]

CHROM_HEIGHT = 0.22      # smaller cytoband than full ideogram
TRACK_HEIGHT = 0.35      # taller XY annotation track
DENSITY_MAX  = 0.75
ROW_SPACING  = 2.0
ZDNA_COLOR = "#513AB5"

STAIN_COLORS = {
    "gneg": "#FFFFFF",
    "gpos25": "#C0C0C0",
    "gpos50": "#808080",
    "gpos75": "#404040",
    "gpos100": "#000000",
    "acen": "#B22222",
    "gvar": "#4682B4",
    "stalk": "#32CD32",
}

SEQ_COLLAPSE = {
    "PAR1": "PAR",
    "PAR2": "PAR",
    "X-DEG": "X-degenerate",
    "XTR": "X-transposed",
    "XTR1": "X-transposed",
    "XTR2": "X-transposed",
    "AMPL": "Ampliconic",
    "SAT": "Satellite",
    "DYZ17": "Satellite",
    "DYZ19": "Satellite",
    "CEN": "Centromere",
    "HET": "Heterochromatin",
    "OTHER": "Other"
}

SEQ_COLORS = {
    "PAR": "#E6B95E",
    "X-degenerate": "#ABA737",
    "X-transposed": "#56B4E9",
    "Ampliconic": "#009E73",
    "Satellite": "#CC79A7",
    "Centromere": "#B22222",
    "Heterochromatin": "#999999",
    "Other": "#D1C7C7",
}

def load_cytobands():
    df = pd.read_csv(
        CYTO_BED, sep="\t", header=None,
        names=["chrom", "start", "end", "name", "stain"]
    )
    return df[df.chrom.isin(CHROMS)].copy()

def load_zdna():
    df = pd.read_csv(
        ZDNA_BED, sep="\t", header=None,
        names=["chrom", "start", "end"],
        usecols=[0, 1, 2]
    )
    return df[df.chrom.isin(CHROMS)].copy()

def load_seq():
    df = pd.read_csv(
        SEQ_BED, sep="\t", header=None,
        usecols=[0, 1, 2, 3],
        names=["chrom", "start", "end", "category"]
    )
    df = df[df.chrom.isin(CHROMS)].copy()
    df["major"] = df["category"].map(SEQ_COLLAPSE)

    missing = df[df["major"].isna()]["category"].unique()
    if len(missing):
        raise ValueError(f"Unmapped sequence classes: {missing}")

    return df

def compute_density(df, chrom_len):
    n_bins = int(np.ceil(chrom_len / BIN_SIZE))
    covered = np.zeros(n_bins, dtype=np.float64)

    if len(df):
        s = np.clip(df["start"].values, 0, chrom_len).astype(np.int64)
        e = np.clip(df["end"].values,   0, chrom_len).astype(np.int64)

        mask = e > s
        s, e = s[mask], e[mask]

        if len(s):
            b0 = s // BIN_SIZE
            b1 = (e - 1) // BIN_SIZE

            same = b0 == b1
            if same.any():
                np.add.at(covered, b0[same], (e[same] - s[same]).astype(float))

            multi = ~same
            if multi.any():
                sm, em, bsm, bem = s[multi], e[multi], b0[multi], b1[multi]

                np.add.at(
                    covered,
                    bsm,
                    ((bsm + 1) * BIN_SIZE - sm).astype(float)
                )

                np.add.at(
                    covered,
                    bem,
                    (em - bem * BIN_SIZE).astype(float)
                )

                for i in range(len(sm)):
                    for b in range(bsm[i] + 1, bem[i]):
                        covered[b] += BIN_SIZE

    bin_starts = np.arange(n_bins, dtype=np.int64) * BIN_SIZE
    bin_ends = np.minimum(bin_starts + BIN_SIZE, chrom_len)
    widths = (bin_ends - bin_starts).astype(np.float64)
    mids = bin_starts + widths / 2.0

    chrom_max_bp = covered.max() if len(covered) else 0.0
    dens_norm = covered / chrom_max_bp if chrom_max_bp > 0 else covered

    return mids, dens_norm, chrom_max_bp

cyto = load_cytobands()
zdna = load_zdna()
seq  = load_seq()

chrom_sizes = cyto.groupby("chrom")["end"].max().to_dict()
max_bp = max(chrom_sizes.values())

fig, ax = plt.subplots(figsize=(16, 5))

print("\nMaximum Z-DNA coverage per chromosome:")
print("(kb of Z-DNA contained in the densest 150 kb bin)\n")

for i, chrom in enumerate(CHROMS):

    clen = chrom_sizes[chrom]
    y = (len(CHROMS) - 1 - i) * ROW_SPACING

    sub_cyto = cyto[cyto.chrom == chrom].sort_values("start")

    for _, r in sub_cyto.iterrows():
        ax.add_patch(Rectangle(
            (r.start, y - CHROM_HEIGHT),
            r.end - r.start,
            2 * CHROM_HEIGHT,
            facecolor=STAIN_COLORS.get(r.stain, "white"),
            edgecolor="none"
        ))

    ax.add_patch(FancyBboxPatch(
        (0, y - CHROM_HEIGHT),
        clen,
        2 * CHROM_HEIGHT,
        boxstyle=f"round,pad=0,rounding_size={CHROM_HEIGHT * 0.8}",
        facecolor="none",
        edgecolor="black",
        linewidth=0.8
    ))

    sub_zdna = zdna[zdna.chrom == chrom]
    mids, dens_norm, chrom_max_bp = compute_density(sub_zdna, clen)

    print(f"{chrom:>5}: {chrom_max_bp / 1000:,.1f} kb / {BIN_SIZE / 1000:,.0f} kb")

    y_base = y + CHROM_HEIGHT + 0.05
    y_top  = y_base + DENSITY_MAX
    y_vals = y_base + dens_norm * DENSITY_MAX

    ax.fill_between(
        mids,
        y_base,
        y_vals,
        color=ZDNA_COLOR,
        alpha=0.6,
        linewidth=0
    )

    ax.plot(
        mids,
        y_vals,
        color=ZDNA_COLOR,
        linewidth=0.3
    )

    x_axis = -max_bp * 0.0005
    tick_len = max_bp * 0.003

    ax.plot(
        [x_axis, x_axis],
        [y_base, y_top],
        color="black",
        linewidth=0.8
    )

    ax.plot(
        [x_axis, x_axis + tick_len],
        [y_top, y_top],
        color="black",
        linewidth=0.8
    )

    ax.text(
        x_axis - tick_len * 0.5,
        y_top,
        f"{chrom_max_bp / 1000:,.1f}",
        ha="right",
        va="center",
        fontsize=12
    )

    track_top = y - CHROM_HEIGHT - 0.05
    track_bot = track_top - TRACK_HEIGHT

    sub_seq = seq[seq.chrom == chrom].sort_values("start")

    for _, r in sub_seq.iterrows():
        ax.add_patch(Rectangle(
            (r.start, track_bot),
            r.end - r.start,
            TRACK_HEIGHT,
            facecolor=SEQ_COLORS[r.major],
            edgecolor="none",
            alpha=0.90
        ))

    ax.add_patch(Rectangle(
        (0, track_bot),
        clen,
        TRACK_HEIGHT,
        facecolor="none",
        edgecolor="#555555",
        linewidth=0.5
    ))

    ax.text(
        -max_bp * 0.012,
        y,
        chrom.replace("chr", ""),
        ha="right",
        va="center",
        fontsize=30,
        fontweight="bold"
    )

ax.set_xlim(-max_bp * 0.035, max_bp * 1.02)
ax.set_ylim(-1.5, ROW_SPACING * len(CHROMS))
ax.axis("off")

plt.tight_layout()
plt.savefig(
    "data/xy_zdna_karyogram.png",
    dpi=600,
    bbox_inches="tight"
)
plt.close(fig)

legend_items = [
    mpatches.Patch(color=ZDNA_COLOR, label="Z-DNA density")
]

for k, c in SEQ_COLORS.items():
    legend_items.append(mpatches.Patch(color=c, label=k))

fig, ax = plt.subplots(figsize=(7, 4))
ax.axis("off")

ax.legend(
    handles=legend_items,
    loc="center",
    frameon=True,
    fontsize=12
)

plt.tight_layout()
plt.savefig(
    "data/xy_karyogram_legend.png",
    dpi=600,
    bbox_inches="tight"
)
plt.close(fig)

print("\n[SAVED] xy_zdna_karyogram.png")
print("[SAVED] xy_karyogram_legend.png")

In [ ]:
# Z-DNA density across CHM13 sequence-class bins.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.chdir("data/chm13/chm13_marbl_ann_files")

SEQ_BED  = "chm13v2.0_chrXY_sequence_class_v1.bed"
ZDNA_BED = "chm13_zdna_all.bed"

SEQ_COLLAPSE = {
    "PAR1": "PAR",
    "PAR2": "PAR",
    "X-DEG": "X-degenerate",
    "XTR": "X-transposed",
    "XTR1": "X-transposed",
    "XTR2": "X-transposed",
    "AMPL": "Ampliconic",
    "SAT": "Satellite",
    "DYZ17": "Satellite",
    "DYZ19": "Satellite",
    "CEN": "Centromere",
    "HET": "Heterochromatin",
    "OTHER": "Other"
}

SEQ_COLORS = {
    "PAR": "#E6B95E",
    "X-degenerate": "#ABA737",
    "X-transposed": "#56B4E9",
    "Ampliconic": "#009E73",
    "Satellite": "#CC79A7",
    "Centromere": "#B22222",
    "Heterochromatin": "#999999",
    "Other": "#D1C7C7",
}

seq = pd.read_csv(
    SEQ_BED,
    sep="\t",
    header=None,
    usecols=[0,1,2,3],
    names=["chrom","start","end","cat"]
)

zdna = pd.read_csv(
    ZDNA_BED,
    sep="\t",
    header=None,
    names=["chrom","start","end"]
)

seq = seq[seq.chrom.isin(["chrX","chrY"])].copy()
zdna = zdna[zdna.chrom.isin(["chrX","chrY"])].copy()

seq["feature"] = seq["cat"].map(SEQ_COLLAPSE)

records = []

for _, row in seq.iterrows():

    chrom = row.chrom
    ftype = row.feature

    s = int(row.start)
    e = int(row.end)

    if e <= s:
        continue

    length = e - s

    zdna_chr = zdna[zdna.chrom == chrom]

    sub = zdna_chr[
        (zdna_chr.start < e) &
        (zdna_chr.end   > s)
    ]

    overlap = 0

    for _, z in sub.iterrows():
        overlap += max(
            0,
            min(e, z.end) - max(s, z.start)
        )

    density = overlap / length * 1_000_000

    records.append({
        "chrom": chrom,
        "feature": ftype,
        "density": density
    })

df = pd.DataFrame(records)

df["group"] = df["feature"]

mask = df["feature"].isin(["PAR","X-transposed"])
df.loc[mask, "group"] = df.loc[mask,"feature"] + " " + df.loc[mask,"chrom"]

df["density_log"] = np.log10(df["density"] + 1)

groups = [
    "PAR chrX","PAR chrY",
    "X-transposed chrX","X-transposed chrY",
    "X-degenerate",
    "Ampliconic",
    "Satellite",
    "Centromere",
    "Heterochromatin",
    "Other"
]

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

cats = groups
positions = np.arange(len(cats))

data = [df[df.group==c].density_log.values for c in cats]
counts = [len(v) for v in data]

fig, ax = plt.subplots(figsize=(10,6))

for pos,cat,vals in zip(positions,cats,data):

    if len(vals)==0:
        continue

    jitter = rng.uniform(-0.15,0.15,len(vals))
    color = SEQ_COLORS.get(cat.split()[0],"#888888")
    ax.scatter(
        pos + jitter,
        vals,
        s=100,                     # larger points
        color=color,
        edgecolors="black",       # consistent outline
        linewidths=0.9,
        alpha=0.9,                # less transparency
        zorder=3
    )

labels = cats # labels = [f"{c}\nn={n}" for c,n in zip(cats,counts)] for counts

ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=17)

ax.set_xlabel("Sequence class", fontsize=20)
ax.set_ylabel("Z-DNA density (bp per Mb)", fontsize=20)

ticks = [0,1,2,3,4,5]
tick_labels = ["0","1","10","100","1000","10000"]

ax.set_yticks(ticks)
ax.set_yticklabels(tick_labels, fontsize=17)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(0.9)
ax.spines["bottom"].set_linewidth(0.9)

plt.tight_layout()
plt.show
plt.savefig(
"data/zdna_seqclass_bins.png",
dpi=600
)
print(df.groupby("feature")["density"].agg(["count","median","mean"]))

## 8. Genomic-feature enrichment (CHM13)

In [ ]:
%%bash
# Build feature BED tracks (genes, introns, repeats, CenSat, telomere) and compute Z-DNA overlap/enrichment.
set -euo pipefail

ANNDIR="data/chm13/annotation"

GFF="${ANNDIR}/chm13_genes.gff3"
ZDNA="${ANNDIR}/chm13_zdna_all.bed"
CENSAT="${ANNDIR}/chm13_censat.bed"
TELO="${ANNDIR}/chm13_telomere.bed"
RMSK="${ANNDIR}/chm13_rmsk.bed"
INTRONS="${ANNDIR}/pc_transcript_introns.bed"
CHROMSIZES="${ANNDIR}/chm13_chrom_sizes.txt"

OUTDIR="enrichment_investigation"
FEATDIR="${OUTDIR}/features"
mkdir -p "${FEATDIR}"

for t in bedtools awk sort zgrep cut; do
  command -v "$t" >/dev/null || { echo "Missing tool: $t" >&2; exit 1; }
done

sort_bed3()  { LC_ALL=C sort -k1,1 -k2,2n; }
merge_bed3() { cut -f1-3 | sort_bed3 | bedtools merge -i -; }
bed_len()    { awk '{s+=($3-$2)} END{print s+0}'; }

merge_bed3 < "${ZDNA}" > "${OUTDIR}/Z.merged.bed"

zgrep -v '^#' "${GFF}" \
| awk -F'\t' '$3=="gene"' \
| awk -F'[;\t]' '
    {
      for(i=1;i<=NF;i++) if($i~/^gene_biotype=/){
        sub(/^gene_biotype=/,"",$i)
        s=$4-1; e=$5
        if(e>s) print $1,s,e,$i
      }
    }' OFS='\t' \
> "${OUTDIR}/_genes_all.bed4"

for bt in protein_coding lncRNA miRNA snoRNA; do
  awk -v B="${bt}" '$4==B{print $1,$2,$3}' OFS='\t' "${OUTDIR}/_genes_all.bed4" \
    | merge_bed3 > "${FEATDIR}/gene_${bt}.merged.bed"
done

awk '$4=="pseudogene" || $4=="transcribed_pseudogene" {print $1,$2,$3}' OFS='\t' \
  "${OUTDIR}/_genes_all.bed4" \
  | merge_bed3 > "${FEATDIR}/gene_pseudogene.merged.bed"

awk '
  $4!="protein_coding" && $4!="lncRNA" &&
  $4!="pseudogene"     && $4!="transcribed_pseudogene" &&
  $4!="miRNA"          && $4!="snoRNA" &&
  $4!="rRNA"           && $4!="tRNA" {
    print $1,$2,$3
  }' OFS='\t' "${OUTDIR}/_genes_all.bed4" \
  | merge_bed3 > "${FEATDIR}/gene_other.merged.bed"

awk '$4!="rRNA" && $4!="tRNA"{print $1,$2,$3}' OFS='\t' "${OUTDIR}/_genes_all.bed4" \
  | merge_bed3 > "${FEATDIR}/gene_body.merged.bed"

awk '$4=="rRNA"{print $1,$2,$3}' OFS='\t' "${OUTDIR}/_genes_all.bed4" \
  | merge_bed3 > "${OUTDIR}/_gene_rRNA.tmp.bed"
awk '$4=="tRNA"{print $1,$2,$3}' OFS='\t' "${OUTDIR}/_genes_all.bed4" \
  | merge_bed3 > "${OUTDIR}/_gene_tRNA.tmp.bed"

zgrep -v '^#' "${GFF}" \
  | awk -F'\t' '$3=="exon"{print $1,$4-1,$5}' OFS='\t' \
  | merge_bed3 > "${FEATDIR}/pc_exon.merged.bed"

zgrep -v '^#' "${GFF}" \
  | awk -F'\t' '$3=="CDS"{print $1,$4-1,$5}' OFS='\t' \
  | merge_bed3 > "${FEATDIR}/pc_CDS.merged.bed"

merge_bed3 < "${INTRONS}" > "${FEATDIR}/pc_intron.merged.bed"

awk -F'\t' '{
  name=$4
  sub(/\(.*/,"",name)
  split(name, a, "_")
  print $1,$2,$3,a[1]
}' OFS='\t' "${CENSAT}" > "${OUTDIR}/_censat.bed4"

for p in ct censat hor dhor mon hsat1A hsat1B hsat2 hsat3 gsat bsat rDNA; do
  awk -v P="${p}" '$4==P{print $1,$2,$3}' OFS='\t' "${OUTDIR}/_censat.bed4" \
    | merge_bed3 > "${FEATDIR}/censat_${p}.merged.bed"
done

cp "${FEATDIR}/censat_rDNA.merged.bed" "${OUTDIR}/_censat_rDNA.tmp.bed" 2>/dev/null \
  || : > "${OUTDIR}/_censat_rDNA.tmp.bed"

awk '$7=="rRNA"{print $1,$2,$3}' OFS='\t' "${RMSK}" | merge_bed3 > "${OUTDIR}/_rmsk_rRNA.tmp.bed"
awk '$7=="tRNA"{print $1,$2,$3}' OFS='\t' "${RMSK}" | merge_bed3 > "${OUTDIR}/_rmsk_tRNA.tmp.bed"

awk '$7=="DNA"{print $1,$2,$3}' OFS='\t' "${RMSK}" \
  | merge_bed3 > "${FEATDIR}/rmsk_DNA_transposon.merged.bed"
awk '$7!="DNA" && $7 ~ /^DNA/{print $1,$2,$3}' OFS='\t' "${RMSK}" \
  | merge_bed3 > "${FEATDIR}/rmsk_DNA_other.merged.bed"

for cls in LINE SINE LTR Simple_repeat Satellite Low_complexity \
           Retroposon RC Beta snRNA srpRNA scRNA Unknown; do
  awk -v C="${cls}" '$7==C{print $1,$2,$3}' OFS='\t' "${RMSK}" \
    | merge_bed3 > "${FEATDIR}/rmsk_${cls}.merged.bed"
done

cut -f1-3 "${RMSK}" | merge_bed3 > "${FEATDIR}/rmsk_all.merged.bed"

merge_bed3 < "${TELO}" > "${FEATDIR}/telomere.merged.bed"

cat \
  "${OUTDIR}/_gene_rRNA.tmp.bed" \
  "${OUTDIR}/_rmsk_rRNA.tmp.bed" \
  "${OUTDIR}/_censat_rDNA.tmp.bed" \
  | merge_bed3 > "${FEATDIR}/rDNA_arrays.merged.bed"

cat \
  "${OUTDIR}/_gene_tRNA.tmp.bed" \
  "${OUTDIR}/_rmsk_tRNA.tmp.bed" \
  | merge_bed3 > "${FEATDIR}/tRNA_arrays.merged.bed"

cat "${FEATDIR}"/*.merged.bed | merge_bed3 > "${OUTDIR}/_UNION_annot.merged.bed"

awk 'BEGIN{OFS="\t"} {print $1, 0, $2}' "${CHROMSIZES}" | sort_bed3 \
  > "${OUTDIR}/_genome.bed"

bedtools subtract -a "${OUTDIR}/_genome.bed" -b "${OUTDIR}/_UNION_annot.merged.bed" \
  | sort_bed3 > "${FEATDIR}/unannotated_intergenic.merged.bed"

GENOME_BP=$(awk '{s+=$2} END{print s}' "${CHROMSIZES}")
Z_TOTAL_BP=$(bed_len < "${OUTDIR}/Z.merged.bed")

OUTTSV="${OUTDIR}/zdna_genome_enrichment.tsv"
{
  echo -e "feature\tfeature_bp\tzdna_overlap_bp\tgenome_bp\tzdna_total_bp"
  for f in "${FEATDIR}"/*.merged.bed; do
    name=$(basename "$f" .merged.bed)
    feat_bp=$(bed_len < "$f")
    z_overlap=$(bedtools intersect -a "${OUTDIR}/Z.merged.bed" -b "$f" -wo \
                | awk '{s+=$NF} END{print s+0}')
    printf "%s\t%d\t%d\t%d\t%d\n" \
      "${name}" "${feat_bp}" "${z_overlap}" "${GENOME_BP}" "${Z_TOTAL_BP}"
  done
} > "${OUTTSV}"

rm -f "${OUTDIR}"/_*.bed* "${OUTDIR}"/_*.tmp.bed 2>/dev/null || true

echo "[DONE] Per-feature merged BEDs: ${FEATDIR}/"
echo "[DONE] Enrichment input TSV:    ${OUTTSV}"
echo
echo "Next: run 02_plot_genome_enrichment.py with --in_tsv ${OUTTSV}"

In [ ]:
# Plot genome-wide Z-DNA feature enrichment and check strand mirror-symmetry of the metaprofile.
import os
os.chdir("data/chm13/annotation")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_TSV  = Path("enrichment_investigation/zdna_genome_enrichment.tsv")
OUT_PNG = Path("enrichment_investigation/zdna_genome_enrichment.png")
OUT_PDF = Path("enrichment_investigation/zdna_genome_enrichment.pdf")

ENRICHMENT_FLOOR = 0.06

LOG2_YMIN = -4.2
LOG2_YMAX =  4.2

BAR_WIDTH = 0.80
PANEL_GAP = 1.0

FIG_HEIGHT        = 8
FIG_WIDTH_PER_BAR = 0.55   # bar slot width
FIG_WIDTH_MIN     = 25.0

FS_TICK  = 19
FS_LABEL = 22
FS_FOLD  = 14   # fold-change labels above bars
FS_BASE  = 20

LABEL_PAD_POS   = 0.28  # gap between positive-bar tip and label
LABEL_PAD_NEG   = 1  # gap between negative-bar tip and label (more air)
LABEL_X_SHIFT   = -0.5 # shift label start slightly left of bar center

Y_BOTTOM_PAD       = 1.5   # extra y-axis range below the bar floor
LABEL_OFFSET_FLOOR = 1  # vertical drop below LOG2_YMIN for floor labels

PANEL_COLORS = {
    "tandem_repeats":       "#7BCFB7",  # mint
    "interspersed_repeats": "#5FBFB5",  # teal
    "gene_biotypes":        "#7B7BC9",  # slate purple
    "gene_parts":           "#B9A4D8",  # lavender
}

PANEL_ORDER = [
    "tandem_repeats",
    "interspersed_repeats",
    "gene_biotypes",
    "gene_parts",
]

FEATURE_MAP: dict[str, tuple[str, str]] = {
    "censat_hsat1A": ("HSAT1A",         "tandem_repeats"),
    "censat_hsat1B": ("HSAT1B",         "tandem_repeats"),
    "censat_hsat2":  ("HSAT2",          "tandem_repeats"),
    "censat_hsat3":  ("HSAT3",          "tandem_repeats"),
    "censat_hor":    ("HOR",            "tandem_repeats"),
    "censat_dhor":   ("dHOR",           "tandem_repeats"),
    "censat_mon":    ("Monomers",       "tandem_repeats"),
    "censat_gsat":   ("GSAT",           "tandem_repeats"),
    "censat_bsat":   ("BSAT",           "tandem_repeats"),
    "censat_ct":     ("CT",  "tandem_repeats"),
    "censat_censat": ("CenSat",         "tandem_repeats"),
    "telomere":      ("Telomeres",      "tandem_repeats"),
    "tRNA_arrays":   ("tRNA",    "tandem_repeats"),
    "rDNA_arrays":   ("rDNA",    "tandem_repeats"),

    "rmsk_LINE":          ("LINEs",                 "interspersed_repeats"),
    "rmsk_SINE":          ("SINEs",                 "interspersed_repeats"),
    "rmsk_LTR":           ("LTR retr.",  "interspersed_repeats"),
    "rmsk_DNA_transposon":("DNA transp.",       "interspersed_repeats"),
    "rmsk_Retroposon":    ("Retroposons",           "interspersed_repeats"),
    "rmsk_RC":            ("Rolling-circle",        "interspersed_repeats"),
    "rmsk_Simple_repeat": ("Simple repeats",        "interspersed_repeats"),
    "rmsk_Beta":          ("Beta repeats",          "interspersed_repeats"),
    "rmsk_snRNA":         ("snRNA repeats",         "interspersed_repeats"),
    "rmsk_srpRNA":        ("srpRNA repeats",        "interspersed_repeats"),
    "rmsk_scRNA":         ("scRNA repeats",         "interspersed_repeats"),

    "gene_protein_coding":("Protein-coding",  "gene_biotypes"),
    "gene_lncRNA":        ("lncRNA",          "gene_biotypes"),
    "gene_pseudogene":    ("Pseudogenes",           "gene_biotypes"),
    "gene_miRNA":         ("miRNA",           "gene_biotypes"),
    "gene_snoRNA":        ("snoRNA",          "gene_biotypes"),

    "gene_body":              ("Gene body",          "gene_parts"),
    "pc_intron":              ("Introns",            "gene_parts"),
    "pc_exon":                ("Exons",              "gene_parts"),
    "pc_CDS":                 ("CDS",                "gene_parts"),
}

PANEL_TITLES = {
    "tandem_repeats":       "Satellites & tandem repeats",
    "interspersed_repeats": "Interspersed & other repeats",
    "gene_biotypes":        "Gene biotypes",
    "gene_parts":           "Gene parts",
}

df = pd.read_csv(IN_TSV, sep="\t")

for c in ("feature_bp", "zdna_overlap_bp", "genome_bp", "zdna_total_bp"):
    df[c] = pd.to_numeric(df[c], errors="coerce")

z_density_feat = np.where(
    df["feature_bp"] > 0,
    df["zdna_overlap_bp"] / df["feature_bp"],
    np.nan,
)
z_density_genome = np.where(
    df["genome_bp"] > 0,
    df["zdna_total_bp"] / df["genome_bp"],
    np.nan,
)
with np.errstate(divide="ignore", invalid="ignore"):
    df["enrichment"] = z_density_feat / z_density_genome
    df["log2_enrichment"] = np.where(
        df["enrichment"] > 0, np.log2(df["enrichment"]), np.nan
    )

enriched_tsv = IN_TSV.with_name(IN_TSV.stem + "_with_log2.tsv")
df.sort_values("log2_enrichment").to_csv(enriched_tsv, sep="\t", index=False)
print(f"[SAVED] {enriched_tsv}")

print("\nTop 10 enriched (genome background):")
print(df.sort_values("log2_enrichment", ascending=False)
        [["feature", "enrichment", "log2_enrichment"]]
        .head(10).to_string(index=False))

print("\nBottom 10 depleted (genome background):")
print(df.sort_values("log2_enrichment", ascending=True)
        [["feature", "enrichment", "log2_enrichment"]]
        .head(10).to_string(index=False))

mapping = df["feature"].map(FEATURE_MAP)
unmapped = df.loc[mapping.isna(), "feature"].tolist()
if unmapped:
    print(f"\n[WARN] {len(unmapped)} feature(s) in TSV not mapped to a "
          f"panel; dropped from plot: {unmapped}")
df_plot = df.loc[mapping.notna()].copy()
df_plot["label"] = mapping.dropna().map(lambda t: t[0])
df_plot["panel"] = mapping.dropna().map(lambda t: t[1])

pieces = []
for p in PANEL_ORDER:
    sub = df_plot[df_plot["panel"] == p].copy()
    sub["_sort_key"] = sub["log2_enrichment"].fillna(-np.inf)
    sub = sub.sort_values("_sort_key").reset_index(drop=True)
    pieces.append(sub)
df_plot = pd.concat(pieces, ignore_index=True)

panel_sizes = [len(p) for p in pieces]
positions = []
cursor = 0.0
panel_boundaries = []
for i, n in enumerate(panel_sizes):
    positions.extend([cursor + j for j in range(n)])
    cursor += n
    if i < len(panel_sizes) - 1:
        panel_boundaries.append(cursor + PANEL_GAP / 2 - 0.5)
        cursor += PANEL_GAP
df_plot["x"] = positions

def fold_label(enr: float) -> str:
    if enr is None or np.isnan(enr) or enr == 0:
        return "None"
    if enr < ENRICHMENT_FLOOR:
        return f"<{ENRICHMENT_FLOOR:.2f}x"
    return f"{enr:.2f}x"

plt.rcParams.update({
    "font.size":       FS_BASE,
    "axes.labelsize":  FS_LABEL,
    "xtick.labelsize": FS_TICK,
    "ytick.labelsize": FS_BASE,
    "pdf.fonttype":    42,
    "ps.fonttype":     42,
})

fig_width = max(FIG_WIDTH_MIN, FIG_WIDTH_PER_BAR * len(df_plot))
fig, ax = plt.subplots(figsize=(fig_width, FIG_HEIGHT))

for _, row in df_plot.iterrows():
    col = PANEL_COLORS[row["panel"]]
    l2  = row["log2_enrichment"]
    enr = row["enrichment"]

    if np.isnan(l2):
        h = LOG2_YMIN
    else:
        h = max(LOG2_YMIN, min(LOG2_YMAX, l2))

    ax.bar(row["x"], h, width=BAR_WIDTH,
           color=col, edgecolor="none", linewidth=0, zorder=2)

    lab = fold_label(enr)

    is_floor = np.isnan(l2) or enr == 0 or h <= LOG2_YMIN + 1e-9

    if is_floor:
        y_text = LOG2_YMIN - LABEL_OFFSET_FLOOR
    elif l2 >= 0:
        y_text = h + LABEL_PAD_POS
    else:
        y_text = h - LABEL_PAD_NEG

    ax.text(row["x"] + LABEL_X_SHIFT, y_text, lab,
            ha="left", va="bottom",
            rotation=45, fontsize=FS_FOLD)

for xb in panel_boundaries:
    ax.axvline(xb, color="black", lw=0.8, zorder=1)

ax.axhline(0, color="black", lw=0.8, linestyle="--", zorder=1)

ax.set_xticks(df_plot["x"].to_list())
ax.set_xticklabels(df_plot["label"].to_list(),
                   rotation=45, ha="right", rotation_mode="anchor")
ax.set_ylabel(r"$\log_{2}$ enrichment (Z-DNA / genome)")
ax.set_ylim(LOG2_YMIN - Y_BOTTOM_PAD, LOG2_YMAX + 0.6)
ax.set_yticks(np.arange(int(LOG2_YMIN), int(LOG2_YMAX) + 1, 2))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(axis="y", width=1.2, length=5)
ax.tick_params(axis="x", width=1.2, length=5)

ymax_for_titles = LOG2_YMAX + 0.35
cursor = 0.0
for i, p in enumerate(PANEL_ORDER):
    n = panel_sizes[i]
    if n == 0:
        cursor += PANEL_GAP
        continue
    xmid = cursor + (n - 1) / 2.0
    ax.text(xmid, ymax_for_titles, PANEL_TITLES[p],
            ha="center", va="bottom",
            fontsize=FS_BASE, fontweight="bold")
    cursor += n + PANEL_GAP

plt.tight_layout()
fig.savefig(OUT_PNG, dpi=600, bbox_inches="tight")
fig.savefig(OUT_PDF, bbox_inches="tight")
print(f"\n[SAVED] {OUT_PNG}")
print(f"[SAVED] {OUT_PDF}")
plt.show()

flipped_neg = prof_neg_mean[::-1]
peak_pos = x_bp[np.argmax(prof_pos_mean)]
peak_neg = x_bp[np.argmax(prof_neg_mean)]
peak_neg_flipped = x_bp[np.argmax(flipped_neg)]
print(f"Plus-strand peak at:        {peak_pos:+.0f} bp")
print(f"Minus-strand peak at:       {peak_neg:+.0f} bp")
print(f"Minus-strand peak (flipped) {peak_neg_flipped:+.0f} bp")
print(f"Asymmetry offset:           {peak_pos + peak_neg:+.0f} bp "
      f"(should be ~0 if perfectly centered)")

## 9. Mutation-anchored Z-DNA metaprofiles (HPRC v2 MC, CHM13)

### 9.1 VCF parsing

In [ ]:
# Parse the HPRC v2 minigraph-cactus wave VCF into per-ALT mutation BED + metadata (class, size bin, AF).
from pathlib import Path
import gzip
import csv
from collections import defaultdict

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

VCF_PATH = Path(
    "data/mutations/"
    "hprc_mutations"
)
OUT_DIR = Path(
    "data/mutations"
)
META_TSV = Path(
    "data/hprc_zdna_metrics.tsv"
)
PREFIX = "hprc_mutations"

CARRIER_BATCH_SIZE = 500_000

def get_size_bin(x: int) -> str:
    if x == 1:       return "1bp"
    elif x <= 5:     return "2_5bp"
    elif x <= 20:    return "6_20bp"
    elif x <= 50:    return "21_50bp"
    elif x <= 200:   return "51_200bp"
    else:            return "gt200bp"

SUPERPOPS = ("AFR", "EUR", "EAS", "SAS", "AMR", "OTH")

def load_sample_metadata(meta_tsv: Path) -> dict:
    df = pd.read_csv(meta_tsv, sep="\t", dtype=str, keep_default_na=False)
    sample_info = {}

    for sid, group in df.groupby("sample_id"):
        if sid in ("CHM13", "GRCh38"):
            continue
        superpop = ""
        for v in group["superpopulation"]:
            if v:
                superpop = v
                break
        if not superpop:
            superpop = "OTH"
        elif superpop not in SUPERPOPS:
            superpop = "OTH"
        trio = any(v == "True" for v in group["trio_available"])
        sample_info[sid] = {"superpop": superpop, "trio": trio}

    return sample_info

def stream_vcf(vcf_path: Path):
    opener = gzip.open if str(vcf_path).endswith(".gz") else open
    with opener(vcf_path, "rt") as f:
        for line in f:
            if line.startswith("##"):
                continue
            if line.startswith("#CHROM"):
                header = line.rstrip("\n").split("\t")
                yield {"sample_names": header[9:]}
                continue
            fields = line.rstrip("\n").split("\t")
            yield {
                "CHROM":  fields[0],
                "POS":    int(fields[1]),
                "ID":     fields[2],
                "REF":    fields[3],
                "ALT":    fields[4],
                "QUAL":   fields[5],
                "FILTER": fields[6],
                "INFO":   fields[7],
                "FORMAT": fields[8],
                "GTS":    fields[9:],
            }

def parse_attributes(attrs: str) -> defaultdict:
    attributes: defaultdict = defaultdict(dict)
    valid = {"AC", "AF", "AN", "LEN", "TYPE", "NS"}
    for aa in attrs.split(";"):
        if "=" not in aa:
            continue
        key, val = aa.split("=", 1)
        if key not in valid:
            continue
        if key in {"NS", "AN"}:
            try:    attributes[key][0] = int(val)
            except ValueError: attributes[key][0] = val
            continue
        for i, v in enumerate(val.split(",")):
            if key in {"AC", "LEN"}:
                try:    attributes[key][i] = int(v)
                except ValueError: attributes[key][i] = v
            elif key == "AF":
                try:    attributes[key][i] = float(v)
                except ValueError: attributes[key][i] = v
            else:
                attributes[key][i] = v
    return attributes

def mutation_metadata(ref: str, alt: str) -> dict:
    ref_len = len(ref)
    alt_len = len(alt)

    if ref_len == alt_len == 1:
        struct = "snp"
    elif ref_len == alt_len:
        struct = "mnp" if ref_len <= 20 else "substitution"
    elif ref_len < alt_len:
        struct = "ins" if ref_len == 1 else "complex"
    elif ref_len > alt_len:
        struct = "del" if alt_len == 1 else "complex"
    else:
        raise RuntimeError("unreachable")

    if struct in ("complex", "substitution"):
        if ref_len > alt_len:   dominant = "del_like"
        elif alt_len > ref_len: dominant = "ins_like"
        else:                   dominant = "balanced"
    else:
        dominant = struct

    if struct in ("snp", "mnp", "substitution"):
        ref_span = ref_len
        alt_span = alt_len
    else:
        ref_span = ref_len - 1
        alt_span = alt_len - 1

    if struct in ("snp", "mnp", "substitution", "del") or dominant == "del_like":
        event_size = ref_span
    elif struct == "ins" or dominant == "ins_like":
        event_size = alt_span
    elif dominant == "balanced":
        event_size = ref_span
    else:
        event_size = max(ref_span, alt_span)

    size_bin = get_size_bin(event_size)

    if struct == "snp":
        anchor = "single_base"
    elif struct in ("mnp", "substitution"):
        anchor = "midpoint"
    elif dominant == "del_like":
        anchor = "left_breakpoint"
    else:
        anchor = "insertion_site"

    return {
        "struct_class": struct, "dominant_op": dominant,
        "ref_span": ref_span, "alt_span": alt_span,
        "size_bin": size_bin, "anchor_type": anchor,
    }

def scan_record(
    gts: list,
    sample_names: list,
    n_alts: int,
    sample_info: dict,
):
    per_alt = [None]  # index 0 unused
    for _ in range(n_alts):
        d = {
            "AC_global": 0, "AN_global": 0,
            "n_het": 0, "n_hom_alt": 0, "n_hom_ref": 0,
            "n_missing_full": 0, "n_missing_partial": 0,
            "AC_pat": 0, "AN_pat": 0, "AC_mat": 0, "AN_mat": 0,
            "n_trio_pat_carriers": 0, "n_trio_mat_carriers": 0,
        }
        for sp in SUPERPOPS:
            d[f"AC_{sp}"] = 0
            d[f"AN_{sp}"] = 0
        per_alt.append(d)

    carriers = []  # (sample, slot, alt_idx, zyg, superpop, parent, is_trio)

    for sample, gt in zip(sample_names, gts):
        if sample == "GRCh38":
            continue
        info = sample_info.get(sample)
        if info is None:
            superpop = "OTH"
            is_trio = False
        else:
            superpop = info["superpop"]
            is_trio = info["trio"]

        if "|" in gt:
            slots = gt.split("|")
        elif "/" in gt:
            slots = gt.split("/")
        else:
            slots = [gt]

        n_slots = len(slots)

        if n_slots == 1:
            a = slots[0]
            if a == ".":
                for i in range(1, n_alts + 1):
                    per_alt[i]["n_missing_full"] += 1
                continue
            try:
                a_i = int(a)
            except ValueError:
                continue
            for i in range(1, n_alts + 1):
                pa_d = per_alt[i]
                pa_d["AN_global"] += 1
                pa_d[f"AN_{superpop}"] += 1
                if a_i == i:
                    pa_d["AC_global"] += 1
                    pa_d[f"AC_{superpop}"] += 1
                    carriers.append(
                        (sample, 0, i, "haploid", superpop,
                         "unknown_phase", is_trio)
                    )
            continue

        if n_slots != 2:
            for i in range(1, n_alts + 1):
                per_alt[i]["n_missing_full"] += 1
            continue

        a, b = slots
        a_missing = (a == ".")
        b_missing = (b == ".")

        if a_missing and b_missing:
            for i in range(1, n_alts + 1):
                per_alt[i]["n_missing_full"] += 1
            continue

        a_i = None if a_missing else int(a)
        b_i = None if b_missing else int(b)

        an_contrib = (0 if a_missing else 1) + (0 if b_missing else 1)
        partial = (a_missing != b_missing)

        an_pat = (1 if (is_trio and not a_missing) else 0)
        an_mat = (1 if (is_trio and not b_missing) else 0)

        for i in range(1, n_alts + 1):
            pa_d = per_alt[i]
            pa_d["AN_global"] += an_contrib
            pa_d[f"AN_{superpop}"] += an_contrib
            pa_d["AN_pat"] += an_pat
            pa_d["AN_mat"] += an_mat

            if partial:
                pa_d["n_missing_partial"] += 1

            a_is_alt = (a_i == i)
            b_is_alt = (b_i == i)

            if not a_is_alt and not b_is_alt:
                if a_i == 0 and b_i == 0:
                    pa_d["n_hom_ref"] += 1
                continue

            if a_is_alt and b_is_alt:
                pa_d["AC_global"] += 2
                pa_d[f"AC_{superpop}"] += 2
                pa_d["n_hom_alt"] += 1
                if is_trio:
                    pa_d["AC_pat"] += 1
                    pa_d["AC_mat"] += 1
                    pa_d["n_trio_pat_carriers"] += 1
                    pa_d["n_trio_mat_carriers"] += 1
                pat_str = "paternal" if is_trio else "unknown_phase"
                mat_str = "maternal" if is_trio else "unknown_phase"
                carriers.append(
                    (sample, 0, i, "hom_alt", superpop, pat_str, is_trio)
                )
                carriers.append(
                    (sample, 1, i, "hom_alt", superpop, mat_str, is_trio)
                )
            elif a_is_alt:
                pa_d["AC_global"] += 1
                pa_d[f"AC_{superpop}"] += 1
                zyg = "het_partial" if partial else "het"
                pa_d["n_het"] += 1
                if is_trio:
                    pa_d["AC_pat"] += 1
                    pa_d["n_trio_pat_carriers"] += 1
                pat_str = "paternal" if is_trio else "unknown_phase"
                carriers.append(
                    (sample, 0, i, zyg, superpop, pat_str, is_trio)
                )
            elif b_is_alt:
                pa_d["AC_global"] += 1
                pa_d[f"AC_{superpop}"] += 1
                zyg = "het_partial" if partial else "het"
                pa_d["n_het"] += 1
                if is_trio:
                    pa_d["AC_mat"] += 1
                    pa_d["n_trio_mat_carriers"] += 1
                mat_str = "maternal" if is_trio else "unknown_phase"
                carriers.append(
                    (sample, 1, i, zyg, superpop, mat_str, is_trio)
                )

    return per_alt, carriers

EXPLODED_COLS = [
    "seqID", "start", "end", "mut_id",
    "struct_class", "dominant_op",
    "ref_span", "alt_span", "size_bin", "anchor_type",
    "mutation", "general_mut",
    "reference", "variant",
    "subvariants", "is_multiallelic",
    "AC", "AF", "AN", "NS", "LEN", "TYPE",
    "AC_computed", "AN_computed",
    "n_hom_alt", "n_het", "n_hom_ref",
    "n_missing_full", "n_missing_partial",
    "AC_AFR", "AN_AFR", "AC_EUR", "AN_EUR",
    "AC_EAS", "AN_EAS", "AC_SAS", "AN_SAS",
    "AC_AMR", "AN_AMR", "AC_OTH", "AN_OTH",
    "AC_pat", "AN_pat", "AC_mat", "AN_mat",
    "n_trio_pat_carriers", "n_trio_mat_carriers",
]

META_COLS = [
    "mut_id", "seqID", "start", "end",
    "struct_class", "dominant_op", "ref_span", "alt_span",
    "anchor_type", "size_bin",
    "mutation", "general_mut",
    "reference", "variant",
    "AC", "AF", "AN", "NS", "LEN", "TYPE",
    "AC_computed", "AN_computed",
    "n_hom_alt", "n_het", "n_hom_ref",
    "n_missing_full", "n_missing_partial",
    "AC_AFR", "AN_AFR", "AC_EUR", "AN_EUR",
    "AC_EAS", "AN_EAS", "AC_SAS", "AN_SAS",
    "AC_AMR", "AN_AMR", "AC_OTH", "AN_OTH",
    "AC_pat", "AN_pat", "AC_mat", "AN_mat",
    "n_trio_pat_carriers", "n_trio_mat_carriers",
    "subvariants", "is_multiallelic",
]

CARRIER_SCHEMA = pa.schema([
    ("mut_id",            pa.string()),
    ("seqID",             pa.string()),
    ("anchor_pos",        pa.int64()),
    ("sample_id",         pa.string()),
    ("slot_idx",          pa.int8()),
    ("alt_idx",           pa.int16()),
    ("zygosity",          pa.string()),
    ("superpop",          pa.string()),
    ("parent_of_origin",  pa.string()),
    ("is_trio",           pa.bool_()),
    ("struct_class",      pa.string()),
    ("size_bin",          pa.string()),
])

def run(vcf_path: Path, out_dir: Path, prefix: str, sample_info: dict):
    exploded_path = out_dir / f"{prefix}.exploded.tsv.gz"
    bed_path      = out_dir / f"{prefix}.mutations.bed.gz"
    meta_path     = out_dir / f"{prefix}.mutations.meta.tsv.gz"
    threecol_path = out_dir / f"{prefix}.mutations.3col.tsv.gz"
    carrier_path  = out_dir / f"{prefix}.carriers.parquet"

    f_exploded = gzip.open(exploded_path, "wt", compresslevel=4, newline="")
    f_bed      = gzip.open(bed_path,      "wt", compresslevel=4, newline="")
    f_meta     = gzip.open(meta_path,     "wt", compresslevel=4, newline="")
    f_threecol = gzip.open(threecol_path, "wt", compresslevel=4, newline="")

    w_exploded = csv.writer(f_exploded, delimiter="\t",
                            lineterminator="\n",
                            quoting=csv.QUOTE_MINIMAL)
    w_bed      = csv.writer(f_bed,      delimiter="\t",
                            lineterminator="\n",
                            quoting=csv.QUOTE_MINIMAL)
    w_meta     = csv.writer(f_meta,     delimiter="\t",
                            lineterminator="\n",
                            quoting=csv.QUOTE_MINIMAL)
    w_threecol = csv.writer(f_threecol, delimiter="\t",
                            lineterminator="\n",
                            quoting=csv.QUOTE_MINIMAL)

    w_exploded.writerow(EXPLODED_COLS)
    w_meta.writerow(META_COLS)
    w_threecol.writerow(["chrom", "pos", "mutation"])

    pq_writer = pq.ParquetWriter(
        carrier_path, CARRIER_SCHEMA, compression="zstd",
        compression_level=3,
    )
    carrier_buffer = []  # list of tuples matching CARRIER_SCHEMA order

    def flush_carriers():
        if not carrier_buffer:
            return
        cols = list(zip(*carrier_buffer))
        arrays = [
            pa.array(cols[0], type=pa.string()),
            pa.array(cols[1], type=pa.string()),
            pa.array(cols[2], type=pa.int64()),
            pa.array(cols[3], type=pa.string()),
            pa.array(cols[4], type=pa.int8()),
            pa.array(cols[5], type=pa.int16()),
            pa.array(cols[6], type=pa.string()),
            pa.array(cols[7], type=pa.string()),
            pa.array(cols[8], type=pa.string()),
            pa.array(cols[9], type=pa.bool_()),
            pa.array(cols[10], type=pa.string()),
            pa.array(cols[11], type=pa.string()),
        ]
        batch = pa.RecordBatch.from_arrays(arrays, schema=CARRIER_SCHEMA)
        pq_writer.write_batch(batch)
        carrier_buffer.clear()

    sample_names = None
    n_variants_written = 0
    n_carriers_total = 0

    try:
        for record in tqdm(stream_vcf(vcf_path), unit=" records"):
            if "sample_names" in record:
                sample_names = record["sample_names"]
                n_in_meta = sum(
                    1 for s in sample_names
                    if s != "GRCh38" and s in sample_info
                )
                n_trio = sum(
                    1 for s in sample_names
                    if s in sample_info and sample_info[s]["trio"]
                )
                print(f"VCF samples: {len(sample_names)} "
                      f"(reference 'GRCh38' present: "
                      f"{'GRCh38' in sample_names})")
                print(f"  matched in metadata: {n_in_meta}")
                print(f"  trio-phased:         {n_trio}")
                continue

            seqID     = record["CHROM"]
            reference = record["REF"]
            alts      = record["ALT"]
            info      = record["INFO"]
            gts       = record["GTS"]
            pos0      = record["POS"] - 1

            if not alts or alts == ".":
                continue

            alt_list = alts.split(",")
            n_alts = len(alt_list)
            attributes = parse_attributes(info)
            ref_len = len(reference)

            per_alt_stats, carriers = scan_record(
                gts, sample_names, n_alts, sample_info
            )

            for i, alt in enumerate(alt_list):
                alt_idx = i + 1
                meta = mutation_metadata(reference, alt)
                start0 = pos0

                if meta["struct_class"] == "snp":
                    center = start0
                elif meta["anchor_type"] == "midpoint":
                    center = start0 + 1 + (ref_len - 1) // 2
                elif meta["anchor_type"] == "left_breakpoint":
                    center = start0 + 1
                elif meta["anchor_type"] == "right_breakpoint":
                    center = start0 + ref_len - 1
                elif meta["anchor_type"] == "insertion_site":
                    center = start0 + 1
                else:
                    raise RuntimeError("Unknown anchor type")

                start = center
                end = center + 1
                mut_id = f"{seqID}:{pos0+1}:{reference}:{alt}"

                stats = per_alt_stats[alt_idx]

                AC   = attributes["AC"].get(i)
                AF   = attributes["AF"].get(i)
                LEN  = attributes["LEN"].get(i)
                TYPE = attributes["TYPE"].get(i)
                AN   = attributes["AN"].get(0)
                NS   = attributes["NS"].get(0)

                mutation_str = f"{meta['struct_class']}_{meta['size_bin']}"
                general_mut  = meta["struct_class"]

                exploded_row = [
                    seqID, start, end, mut_id,
                    meta["struct_class"], meta["dominant_op"],
                    meta["ref_span"], meta["alt_span"],
                    meta["size_bin"], meta["anchor_type"],
                    mutation_str, general_mut,
                    reference, alt,
                    n_alts, n_alts > 1,
                    AC, AF, AN, NS, LEN, TYPE,
                    stats["AC_global"], stats["AN_global"],
                    stats["n_hom_alt"], stats["n_het"], stats["n_hom_ref"],
                    stats["n_missing_full"], stats["n_missing_partial"],
                    stats["AC_AFR"], stats["AN_AFR"],
                    stats["AC_EUR"], stats["AN_EUR"],
                    stats["AC_EAS"], stats["AN_EAS"],
                    stats["AC_SAS"], stats["AN_SAS"],
                    stats["AC_AMR"], stats["AN_AMR"],
                    stats["AC_OTH"], stats["AN_OTH"],
                    stats["AC_pat"], stats["AN_pat"],
                    stats["AC_mat"], stats["AN_mat"],
                    stats["n_trio_pat_carriers"],
                    stats["n_trio_mat_carriers"],
                ]
                w_exploded.writerow(exploded_row)

                w_bed.writerow([seqID, start, end, mut_id, 0, "."])

                w_meta.writerow([
                    mut_id, seqID, start, end,
                    meta["struct_class"], meta["dominant_op"],
                    meta["ref_span"], meta["alt_span"],
                    meta["anchor_type"], meta["size_bin"],
                    mutation_str, general_mut,
                    reference, alt,
                    AC, AF, AN, NS, LEN, TYPE,
                    stats["AC_global"], stats["AN_global"],
                    stats["n_hom_alt"], stats["n_het"], stats["n_hom_ref"],
                    stats["n_missing_full"], stats["n_missing_partial"],
                    stats["AC_AFR"], stats["AN_AFR"],
                    stats["AC_EUR"], stats["AN_EUR"],
                    stats["AC_EAS"], stats["AN_EAS"],
                    stats["AC_SAS"], stats["AN_SAS"],
                    stats["AC_AMR"], stats["AN_AMR"],
                    stats["AC_OTH"], stats["AN_OTH"],
                    stats["AC_pat"], stats["AN_pat"],
                    stats["AC_mat"], stats["AN_mat"],
                    stats["n_trio_pat_carriers"],
                    stats["n_trio_mat_carriers"],
                    n_alts, n_alts > 1,
                ])

                w_threecol.writerow([seqID, start, mutation_str])

                for (sample, slot, c_alt_idx, zyg,
                     superpop, parent, is_trio) in carriers:
                    if c_alt_idx != alt_idx:
                        continue
                    carrier_buffer.append((
                        mut_id, seqID, int(center),
                        sample, int(slot), int(alt_idx),
                        zyg, superpop, parent, bool(is_trio),
                        meta["struct_class"], meta["size_bin"],
                    ))

                n_variants_written += 1

            if len(carrier_buffer) >= CARRIER_BATCH_SIZE:
                n_carriers_total += len(carrier_buffer)
                flush_carriers()

        n_carriers_total += len(carrier_buffer)
        flush_carriers()

    finally:
        f_exploded.close()
        f_bed.close()
        f_meta.close()
        f_threecol.close()
        pq_writer.close()

    print(f"\nWrote: {exploded_path}")
    print(f"Wrote: {bed_path}")
    print(f"Wrote: {meta_path}")
    print(f"Wrote: {threecol_path}")
    print(f"Wrote: {carrier_path}")
    print(f"Variant rows: {n_variants_written:,}")
    print(f"Carrier rows: {n_carriers_total:,}")

if __name__ == "__main__":
    sample_info = load_sample_metadata(META_TSV)
    print(f"Loaded metadata for {len(sample_info)} samples")
    n_trio = sum(1 for v in sample_info.values() if v["trio"])
    print(f"  of which trio-phased: {n_trio}")
    pop_counts = defaultdict(int)
    for v in sample_info.values():
        pop_counts[v["superpop"]] += 1
    print(f"  superpop breakdown: {dict(pop_counts)}")
    run(VCF_PATH, OUT_DIR, PREFIX, sample_info)

### 9.2 Mutation composition

In [ ]:
# Count and aggregate mutation composition (class x size bin) from the meta TSV.
from pathlib import Path
import numpy as np
import polars as pl

BASE = "data"
MUT_META = f"{BASE}/mutations/hprc_mutations.meta.tsv.gz"
OUTDIR = Path(f"{BASE}/mutation_summary")
OUTDIR.mkdir(exist_ok=True)

print("Scanning meta TSV...")
lf = pl.scan_csv(
    MUT_META, separator="\t",
    schema_overrides={
        "struct_class": pl.Categorical,
        "dominant_op":  pl.Categorical,
        "reference":    pl.Utf8,
        "variant":      pl.Utf8,
        "ref_span":     pl.Int64,
        "alt_span":     pl.Int64,
    },
).select([
    "struct_class", "dominant_op", "ref_span", "alt_span",
    "reference", "variant",
])

print("Computing event_size and stratum...")
lf = lf.with_columns([
    pl.when(
        (pl.col("struct_class") == "ins")
        | ((pl.col("struct_class") == "complex") & (pl.col("dominant_op") == "ins_like"))
    )
    .then(pl.col("alt_span"))
    .otherwise(pl.col("ref_span"))
    .alias("event_size")
])

L = pl.col("event_size")
sc = pl.col("struct_class").cast(pl.Utf8)
dom = pl.col("dominant_op").cast(pl.Utf8)

size_bin = (
    pl.when(L == 1).then(pl.lit("1bp"))
      .when(L <= 5).then(pl.lit("2_5bp"))
      .when(L <= 50).then(pl.lit("6_50bp"))
      .when(L <= 200).then(pl.lit("51_200bp"))
      .otherwise(pl.lit("gt200bp"))
)
sub_bin = (
    pl.when(L <= 50).then(pl.lit("20_50"))
      .when(L <= 200).then(pl.lit("50_200"))
      .otherwise(pl.lit("gt200"))
)
class_key = (
    pl.when(sc == "snp").then(pl.lit("SNP"))
      .when(sc == "mnp").then(pl.lit("MNP"))
      .when(sc == "substitution").then(pl.lit("SUB"))
      .when(sc == "ins").then(pl.lit("INS"))
      .when(sc == "del").then(pl.lit("DEL"))
      .when((sc == "complex") & (dom == "ins_like")).then(pl.lit("CINS"))
      .when((sc == "complex") & (dom == "del_like")).then(pl.lit("CDEL"))
      .otherwise(pl.lit("OTHER"))
)

lf = lf.with_columns([class_key.alias("class_key"),
                       size_bin.alias("size_bin"),
                       sub_bin.alias("sub_bin")])

print("Collecting SNP subtypes...")
COMP = {"A": "T", "T": "A", "C": "G", "G": "C"}
VALID = ["A", "C", "G", "T"]

snp_lf = (
    lf.filter(pl.col("class_key") == "SNP")
      .with_columns([
          pl.col("reference").str.to_uppercase().alias("ref_u"),
          pl.col("variant").str.to_uppercase().alias("alt_u"),
      ])
      .filter(pl.col("ref_u").is_in(VALID) & pl.col("alt_u").is_in(VALID))
)
ref_strand_flip = pl.col("ref_u").is_in(["A", "G"])
norm_ref = (
    pl.when(ref_strand_flip).then(
        pl.col("ref_u").replace_strict(COMP, default=pl.col("ref_u"))
    ).otherwise(pl.col("ref_u"))
)
norm_alt = (
    pl.when(ref_strand_flip).then(
        pl.col("alt_u").replace_strict(COMP, default=pl.col("alt_u"))
    ).otherwise(pl.col("alt_u"))
)
snp_counts = (
    snp_lf.with_columns((norm_ref + pl.lit(">") + norm_alt).alias("subtype"))
          .group_by("subtype").len().rename({"len": "count"})
          .sort("count", descending=True)
          .collect(engine="streaming")
)
snp_counts.write_csv(OUTDIR / "counts_snp_subtypes.tsv", separator="\t")
print(snp_counts)

print("\nCollecting MNP / SUB / indel counts...")
class_size_counts = (
    lf.filter(pl.col("class_key").is_in(["INS", "DEL", "CINS", "CDEL"]))
      .group_by(["class_key", "size_bin"]).len().rename({"len": "count"})
      .sort(["class_key", "size_bin"])
      .collect(engine="streaming")
)
class_size_counts.write_csv(OUTDIR / "counts_class_size.tsv", separator="\t")
print(class_size_counts)

sub_counts = (
    lf.filter(pl.col("class_key") == "SUB")
      .group_by("sub_bin").len().rename({"len": "count"})
      .sort("sub_bin")
      .collect(engine="streaming")
)
sub_counts.write_csv(OUTDIR / "counts_substitution.tsv", separator="\t")
print(sub_counts)

mnp_count = lf.filter(pl.col("class_key") == "MNP").select(pl.len()).collect(engine="streaming").item()
pl.DataFrame({"class_key": ["MNP"], "count": [mnp_count]}).write_csv(
    OUTDIR / "counts_mnp.tsv", separator="\t"
)
print(f"MNP total: {mnp_count:,}")

totals_per_class = (
    lf.group_by("class_key").len().rename({"len": "count"})
      .sort("class_key")
      .collect(engine="streaming")
)
totals_per_class.write_csv(OUTDIR / "counts_class_totals.tsv", separator="\t")
print("\nPer-class totals:")
print(totals_per_class)

print(f"\n[DONE] aggregates written to {OUTDIR}")

In [ ]:
# Plot mutation composition from the aggregated TSVs.
from pathlib import Path
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

BASE = "data"
OUTDIR = Path(f"{BASE}/mutation_summary")

LABEL_FONTSIZE = 28
TICK_FONTSIZE = 20
COUNT_FONTSIZE = 18
LEGEND_FONTSIZE = 18

FIGSIZE = (15, 7)

SNP_COLORS = {
    "C>A": "#4E79A7", "C>G": "#F28E2B", "C>T": "#E15759",
    "T>A": "#76B7B2", "T>C": "#59A14F", "T>G": "#EDC948",
}

SIZE_COLORS = {
    "1bp":      "#e0e0e0",
    "2_5bp":    "#bdbdbd",
    "6_50bp":   "#888888",
    "51_200bp": "#525252",
    "gt200bp":  "#1a1a1a",
}

SUB_COLORS = {
    "20_50":  "#9ecae1",
    "50_200": "#6baed6",
    "gt200":  "#2171b5",
}

MNP_COLOR = "#F4A261"

LABEL_MAP = {
    "SNP": "SNV",
    "MNP": "MNV\n2–20 bp",
    "SUB": "Substitution",
    "DEL": "Deletion",
    "INS": "Insertion",
    "CINS": "Complex\ninsertion",
    "CDEL": "Complex\ndeletion",
}

DISPLAY_LABELS = {
    "C>A": "C>A", "C>G": "C>G", "C>T": "C>T",
    "T>A": "T>A", "T>C": "T>C", "T>G": "T>G",
    "MNP": "MNV 2–20 bp",
    "20_50": "20–50 bp",
    "50_200": "50–200 bp",
    "gt200": ">200 bp",
    "1bp": "1 bp",
    "2_5bp": "2–5 bp",
    "6_50bp": "6–50 bp",
    "51_200bp": "51–200 bp",
    "gt200bp": ">200 bp",
}

snp = pl.read_csv(OUTDIR / "counts_snp_subtypes.tsv", separator="\t").to_pandas()
class_size = pl.read_csv(OUTDIR / "counts_class_size.tsv", separator="\t").to_pandas()
sub = pl.read_csv(OUTDIR / "counts_substitution.tsv", separator="\t").to_pandas()
mnp = pl.read_csv(OUTDIR / "counts_mnp.tsv", separator="\t").to_pandas()

snp_counts = dict(zip(snp["subtype"], snp["count"]))
mnp_count = int(mnp["count"].iloc[0])
sub_counts = dict(zip(sub["sub_bin"], sub["count"]))

indel_pivot = (
    class_size
    .pivot(index="class_key", columns="size_bin", values="count")
    .fillna(0)
)

def norm(v):
    v = np.asarray(v, dtype=float)
    total = v.sum()
    return v / total if total else v

def get_vals(cls, keys):
    if cls not in indel_pivot.index:
        return [0] * len(keys)
    return [int(indel_pivot.loc[cls].get(k, 0)) for k in keys]

def print_category_table(class_name, keys, vals):
    total = int(np.sum(vals))
    print(f"\n[{LABEL_MAP.get(class_name, class_name).replace(chr(10), ' ')}]")
    print(f"total = {total:,}")
    print(f"{'category':<18} {'count':>15} {'within_class_%':>16}")

    for k, v in zip(keys, vals):
        pct = (100 * v / total) if total else 0
        label = DISPLAY_LABELS.get(k, k)
        print(f"{label:<18} {int(v):>15,} {pct:>15.2f}%")

classes = ["SNP", "MNP", "SUB", "DEL", "INS", "CINS", "CDEL"]
bars = []

snp_order = sorted(snp_counts, key=snp_counts.get, reverse=True)
snp_vals = [int(snp_counts[k]) for k in snp_order]
bars.append(("SNP", snp_order, snp_vals, norm(snp_vals), SNP_COLORS, sum(snp_vals)))

bars.append(("MNP", ["MNP"], [mnp_count], [1.0], {"MNP": MNP_COLOR}, mnp_count))

sub_keys = ["20_50", "50_200", "gt200"]
sub_vals = [int(sub_counts.get(k, 0)) for k in sub_keys]
bars.append(("SUB", sub_keys, sub_vals, norm(sub_vals), SUB_COLORS, sum(sub_vals)))

del_keys = ["1bp", "2_5bp", "6_50bp", "51_200bp", "gt200bp"]
del_vals = get_vals("DEL", del_keys)
bars.append(("DEL", del_keys, del_vals, norm(del_vals), SIZE_COLORS, sum(del_vals)))

ins_keys = ["1bp", "2_5bp", "6_50bp", "51_200bp", "gt200bp"]
ins_vals = get_vals("INS", ins_keys)
bars.append(("INS", ins_keys, ins_vals, norm(ins_vals), SIZE_COLORS, sum(ins_vals)))

cins_keys = ["2_5bp", "6_50bp", "51_200bp", "gt200bp"]
cins_vals = get_vals("CINS", cins_keys)
bars.append(("CINS", cins_keys, cins_vals, norm(cins_vals), SIZE_COLORS, sum(cins_vals)))

cdel_keys = ["2_5bp", "6_50bp", "51_200bp", "gt200bp"]
cdel_vals = get_vals("CDEL", cdel_keys)
bars.append(("CDEL", cdel_keys, cdel_vals, norm(cdel_vals), SIZE_COLORS, sum(cdel_vals)))

print("\n==============================")
print("MUTATION COMPOSITION SUMMARY")
print("==============================")

grand_total = sum(total for _, _, _, _, _, total in bars)
print(f"\nGrand total across displayed classes = {grand_total:,}")

print(f"\n{'class':<22} {'count':>15} {'of_displayed_total_%':>22}")
for cls, _, _, _, _, total in bars:
    pct = 100 * total / grand_total if grand_total else 0
    print(f"{LABEL_MAP.get(cls, cls).replace(chr(10), ' '):<22} {total:>15,} {pct:>21.2f}%")

for cls, keys, vals, _, _, _ in bars:
    print_category_table(cls, keys, vals)

fig, ax = plt.subplots(figsize=FIGSIZE)

x = np.arange(len(classes))

for i, (cls, subs, raw_vals, fracs, cmap, total) in enumerate(bars):
    bottom = 0.0

    for j, sub_name in enumerate(subs):
        color = MNP_COLOR if cls == "MNP" else cmap.get(sub_name, "#cccccc")
        ax.bar(
            i,
            fracs[j],
            bottom=bottom,
            color=color,
            edgecolor="white",
            linewidth=0.9,
            width=0.72,
        )
        bottom += fracs[j]

    ax.text(
        i,
        1.045,
        f"n={total:,}",
        ha="center",
        va="bottom",
        fontsize=COUNT_FONTSIZE,
    )

ax.set_xticks(x)
ax.set_xticklabels(
    [LABEL_MAP[c] for c in classes],
    fontsize=TICK_FONTSIZE,
    rotation=45,
    ha="right",
    rotation_mode="anchor",
)

ax.tick_params(axis="x", pad=18)
ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)

ax.set_ylim(0, 1.13)
ax.set_ylabel("Fraction within class", fontsize=LABEL_FONTSIZE, labelpad=12)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

fig.savefig(OUTDIR / "mutation_composition_updated.png", dpi=400, bbox_inches="tight")
fig.savefig(OUTDIR / "mutation_composition_updated.pdf", bbox_inches="tight")

plt.show()

snp_order_legend = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]

fig_leg1 = plt.figure(figsize=(6, 3))
ax_leg1 = fig_leg1.add_subplot(111)
ax_leg1.axis("off")
ax_leg1.legend(
    handles=[Patch(color=SNP_COLORS[k], label=k) for k in snp_order_legend],
    loc="center",
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
    ncol=3,
)
fig_leg1.savefig(OUTDIR / "legend_snp.png", dpi=600, bbox_inches="tight")
fig_leg1.savefig(OUTDIR / "legend_snp.pdf", bbox_inches="tight")
plt.show()

size_order = ["1bp", "2_5bp", "6_50bp", "51_200bp", "gt200bp"]
size_labels = {
    "1bp": "1 bp",
    "2_5bp": "2–5 bp",
    "6_50bp": "6–50 bp",
    "51_200bp": "51–200 bp",
    "gt200bp": ">200 bp",
}

fig_leg2 = plt.figure(figsize=(6, 3))
ax_leg2 = fig_leg2.add_subplot(111)
ax_leg2.axis("off")
ax_leg2.legend(
    handles=[Patch(color=SIZE_COLORS[k], label=size_labels[k]) for k in size_order],
    loc="center",
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
    ncol=2,
)
fig_leg2.savefig(OUTDIR / "legend_size.png", dpi=600, bbox_inches="tight")
fig_leg2.savefig(OUTDIR / "legend_size.pdf", bbox_inches="tight")
plt.show()

sub_order_legend = ["20_50", "50_200", "gt200"]
sub_labels = {
    "20_50": "20–50 bp",
    "50_200": "50–200 bp",
    "gt200": ">200 bp",
}

fig_leg3 = plt.figure(figsize=(5, 3))
ax_leg3 = fig_leg3.add_subplot(111)
ax_leg3.axis("off")
ax_leg3.legend(
    handles=[Patch(color=SUB_COLORS[k], label=sub_labels[k]) for k in sub_order_legend],
    loc="center",
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
    ncol=1,
)
fig_leg3.savefig(OUTDIR / "legend_substitution.png", dpi=600, bbox_inches="tight")
fig_leg3.savefig(OUTDIR / "legend_substitution.pdf", bbox_inches="tight")
plt.show()

print("\n[DONE]")
print(f"Saved: {OUTDIR / 'mutation_composition_updated.png'}")
print(f"Saved: {OUTDIR / 'mutation_composition_updated.pdf'}")

### 9.3 Window bar charts

In [ ]:
# Compute windowed Z-DNA vs control mutation density around mutations.
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

BASE = "data"
INDIR_LEFT  = Path(f"{BASE}/mutation_profiles_refined_tol10_left")
INDIR_RIGHT = Path(f"{BASE}/mutation_profiles_refined_tol10_right")
OUTDIR = Path(f"{BASE}/mutation_window_bars")
OUTDIR.mkdir(parents=True, exist_ok=True)

WINDOWS = [(10, "pm10"), (25, "pm25"), (50, "pm50")]
N_BOOTS = 50_000
SEED = 42
MIN_CHROMS = 6  # minimum chromosomes for a meaningful test

DUAL_ANCHOR_STRATA = {"DEL_gt50bp", "CDEL_gt50bp", "SUB_50_200", "SUB_gt200"}

STRATUM_ORDER = [
    "SNP", "MNP_2_20",
    "DEL_1bp", "DEL_2_5bp", "DEL_6_50bp", "DEL_gt50bp",
    "INS_1bp", "INS_2_5bp", "INS_6_50bp", "INS_51_200bp", "INS_gt200bp",
    "CDEL_1_5bp", "CDEL_6_50bp", "CDEL_gt50bp",
    "CINS_1_5bp", "CINS_6_50bp", "CINS_51_200bp", "CINS_gt200bp",
    "SUB_20_50", "SUB_50_200", "SUB_gt200",
]

def read_profile(indir: Path, stratum: str):
    path = indir / f"profile_{stratum}.tsv"
    if not path.exists():
        return None
    return pd.read_csv(path, sep="\t")

def read_bychrom(indir: Path, stratum: str):
    path = indir / f"profile_{stratum}_bychrom.tsv"
    if not path.exists():
        return None
    return pd.read_csv(path, sep="\t")

def gather_data() -> dict:
    out = {}
    for stratum in STRATUM_ORDER:
        if stratum in DUAL_ANCHOR_STRATA:
            for indir, lab in [(INDIR_LEFT, "left"), (INDIR_RIGHT, "right")]:
                prof = read_profile(indir, stratum)
                bychr = read_bychrom(indir, stratum)
                if prof is not None and bychr is not None:
                    out[(stratum, lab)] = {"profile": prof, "bychrom": bychr}
        else:
            prof = read_profile(INDIR_LEFT, stratum)
            bychr = read_bychrom(INDIR_LEFT, stratum)
            if prof is not None and bychr is not None:
                out[(stratum, "left")] = {"profile": prof, "bychrom": bychr}
    return out

def per_chrom_window_stats(bychrom_df, radius):
    chroms = bychrom_df["chrom"].unique()
    rows = []
    for chrom in chroms:
        cd = bychrom_df[bychrom_df["chrom"] == chrom]
        z_all = cd["Z_raw"].to_numpy(dtype=np.float64)
        c_all = cd["C_raw"].to_numpy(dtype=np.float64)

        z_full_mean = z_all.mean()
        c_full_mean = c_all.mean()

        if z_full_mean == 0 or c_full_mean == 0:
            continue

        win = cd[(cd["offset"] >= -radius) & (cd["offset"] <= radius)]
        z_win_mean = win["Z_raw"].to_numpy(dtype=np.float64).mean()
        c_win_mean = win["C_raw"].to_numpy(dtype=np.float64).mean()

        rows.append({
            "chrom": chrom,
            "Z_win_mean": z_win_mean,
            "C_win_mean": c_win_mean,
            "Z_full_mean": z_full_mean,
            "C_full_mean": c_full_mean,
            "Z_norm": z_win_mean / z_full_mean,
            "C_norm": c_win_mean / c_full_mean,
        })
    return pd.DataFrame(rows)

def chromosome_wilcoxon_onesided(bychrom_df, radius):
    cs = per_chrom_window_stats(bychrom_df, radius)

    if len(cs) < MIN_CHROMS:
        return np.nan, np.nan, len(cs)

    z_norm = cs["Z_norm"].to_numpy()
    c_norm = cs["C_norm"].to_numpy()
    diff = z_norm - c_norm

    if np.allclose(diff, 0):
        return 0.0, 1.0, len(cs)

    try:
        stat, p = wilcoxon(z_norm, c_norm,
                           alternative="greater", zero_method="wilcox")
        return float(stat), float(p), len(cs)
    except ValueError:
        return np.nan, np.nan, len(cs)

def chromosome_block_bootstrap(bychrom_df, radius, n_boots, seed):
    rng = np.random.default_rng(seed)
    chroms = bychrom_df["chrom"].unique()
    n = len(chroms)

    z_win_arr = np.empty(n)
    c_win_arr = np.empty(n)
    z_tot_arr = np.empty(n)
    c_tot_arr = np.empty(n)

    for j, chrom in enumerate(chroms):
        cd = bychrom_df[bychrom_df["chrom"] == chrom]
        z_all = cd["Z_raw"].to_numpy(dtype=np.float64)
        c_all = cd["C_raw"].to_numpy(dtype=np.float64)

        win = cd[(cd["offset"] >= -radius) & (cd["offset"] <= radius)]
        z_win_arr[j] = win["Z_raw"].sum()
        c_win_arr[j] = win["C_raw"].sum()
        z_tot_arr[j] = z_all.sum()
        c_tot_arr[j] = c_all.sum()

    Z_win_total = z_win_arr.sum()
    C_win_total = c_win_arr.sum()
    Z_tot_total = z_tot_arr.sum()
    C_tot_total = c_tot_arr.sum()

    if Z_tot_total == 0 or C_tot_total == 0 or C_win_total == 0 or Z_win_total == 0:
        return np.nan, np.nan, np.nan, np.nan

    observed = np.log((Z_win_total / Z_tot_total) / (C_win_total / C_tot_total))

    idx = rng.integers(0, n, size=(n_boots, n))

    zw_b = z_win_arr[idx].sum(axis=1)
    cw_b = c_win_arr[idx].sum(axis=1)
    zt_b = z_tot_arr[idx].sum(axis=1)
    ct_b = c_tot_arr[idx].sum(axis=1)

    valid = (zt_b > 0) & (ct_b > 0) & (cw_b > 0) & (zw_b > 0)
    if not valid.any():
        return observed, np.nan, np.nan, np.nan

    boots = np.log((zw_b[valid] / zt_b[valid]) / (cw_b[valid] / ct_b[valid]))

    ci_lo = np.percentile(boots, 2.5)
    ci_hi = np.percentile(boots, 97.5)

    centered = boots - boots.mean()
    p_one = (np.sum(centered >= observed) + 1) / (len(centered) + 1)

    return observed, ci_lo, ci_hi, p_one

def build_table(radius):
    data = gather_data()
    rows = []

    for (stratum, anchor_lab), d in data.items():
        prof = d["profile"]
        bychr = d["bychrom"]

        w = prof[(prof["offset"] >= -radius) & (prof["offset"] <= radius)]
        z_mean = w["Z_mean"].mean()
        c_mean = w["C_mean"].mean()
        z_lo_mean = w["Z_lo"].mean()
        z_hi_mean = w["Z_hi"].mean()
        c_lo_mean = w["C_lo"].mean()
        c_hi_mean = w["C_hi"].mean()
        fold = (z_mean / c_mean) if (c_mean and c_mean > 0) else np.nan

        n_events_used = (
            int(prof["n_events_used"].iloc[0])
            if "n_events_used" in prof.columns
            else np.nan
        )

        wilc_stat, p_wilc, n_chroms = chromosome_wilcoxon_onesided(
            bychr, radius
        )
        log_ratio, ci_lo, ci_hi, p_boot = chromosome_block_bootstrap(
            bychr, radius, N_BOOTS, SEED
        )

        rows.append({
            "stratum": stratum,
            "anchor": anchor_lab,
            "stratum_anchor": (
                f"{stratum}_{anchor_lab}"
                if stratum in DUAL_ANCHOR_STRATA
                else stratum
            ),
            "fold_enrichment": fold,
            "Z_mean": z_mean,
            "Z_lo_mean": z_lo_mean,
            "Z_hi_mean": z_hi_mean,
            "C_mean": c_mean,
            "C_lo_mean": c_lo_mean,
            "C_hi_mean": c_hi_mean,
            "log_ratio_ZC": log_ratio,
            "log_ratio_lo": ci_lo,
            "log_ratio_hi": ci_hi,
            "p_bootstrap_onesided": p_boot,
            "wilcoxon_stat": wilc_stat,
            "p_wilcoxon_onesided": p_wilc,
            "n_chromosomes_tested": n_chroms,
            "n_events_used": n_events_used,
        })

    tbl = pd.DataFrame(rows)

    for col in ["p_bootstrap_onesided", "p_wilcoxon_onesided"]:
        q_col = "q_bh_" + col[len("p_"):]
        tbl[q_col] = np.nan
        mask = tbl[col].notna()
        if mask.any():
            tbl.loc[mask, q_col] = multipletests(
                tbl.loc[mask, col].values,
                method="fdr_bh"
            )[1]

    def sig_marker(p, direction_pos):
        if pd.isna(p):
            return ""
        if not direction_pos:
            return "ns_dir"
        if p < 1e-3:
            return "***"
        if p < 1e-2:
            return "**"
        if p < 5e-2:
            return "*"
        return "ns"

    tbl["direction_positive"] = tbl["log_ratio_ZC"] > 0
    tbl["sig_marker_primary"] = tbl.apply(
        lambda r: sig_marker(
            r["q_bh_wilcoxon_onesided"], r["direction_positive"]
        ),
        axis=1,
    )
    tbl["sig_marker_bootstrap"] = tbl.apply(
        lambda r: sig_marker(
            r["q_bh_bootstrap_onesided"], r["direction_positive"]
        ),
        axis=1,
    )
    return tbl

for radius, tag in WINDOWS:
    print(f"\n=== Window ±{radius} bp ({tag}) ===")
    tbl = build_table(radius)
    out_path = OUTDIR / f"window_stats_{tag}.tsv"
    tbl.to_csv(out_path, sep="\t", index=False)
    print(f"  wrote: {out_path}")
    print(
        tbl[
            [
                "stratum_anchor",
                "Z_mean",
                "C_mean",
                "log_ratio_ZC",
                "fold_enrichment",
                "p_wilcoxon_onesided",
                "q_bh_wilcoxon_onesided",
                "sig_marker_primary",
                "p_bootstrap_onesided",
                "q_bh_bootstrap_onesided",
                "sig_marker_bootstrap",
                "n_chromosomes_tested",
                "n_events_used",
            ]
        ].to_string(index=False)
    )
print("\n[DONE]")

In [ ]:
# Plot mutation-window bar charts.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from pathlib import Path

BASE = "data"
INDIR = Path(f"{BASE}/mutation_window_bars")
OUTDIR = INDIR

WINDOWS = [(10, "pm10"), (25, "pm25"), (50, "pm50")]
DUAL_ANCHOR_STRATA = {"DEL_gt50bp", "CDEL_gt50bp", "SUB_50_200", "SUB_gt200"}
STRATUM_ORDER = [
    "SNP", "MNP_2_20",
    "DEL_1bp", "DEL_2_5bp", "DEL_6_50bp", "DEL_gt50bp",
    "INS_1bp", "INS_2_5bp", "INS_6_50bp", "INS_51_200bp", "INS_gt200bp",
    "CDEL_1_5bp", "CDEL_6_50bp", "CDEL_gt50bp",
    "CINS_1_5bp", "CINS_6_50bp", "CINS_51_200bp", "CINS_gt200bp",
    "SUB_20_50", "SUB_50_200", "SUB_gt200",
]

X_LABEL_MAP = {
    "SNP": "SNV", "MNP_2_20": "MNV 2-20",
    "SUB_20_50": "Sub 20-50",
    "SUB_50_200_left": "Sub 50-200 (L)", "SUB_50_200_right": "Sub 50-200 (R)",
    "SUB_gt200_left": "Sub >200 (L)", "SUB_gt200_right": "Sub >200 (R)",
    "DEL_1bp": "Del 1", "DEL_2_5bp": "Del 2-5", "DEL_6_50bp": "Del 6-50",
    "DEL_gt50bp_left": "Del >50 (L)", "DEL_gt50bp_right": "Del >50 (R)",
    "INS_1bp": "Ins 1", "INS_2_5bp": "Ins 2-5", "INS_6_50bp": "Ins 6-50",
    "INS_51_200bp": "Ins 51-200", "INS_gt200bp": "Ins >200",
    "CDEL_1_5bp": "CDel 1-5", "CDEL_6_50bp": "CDel 6-50",
    "CDEL_gt50bp_left": "CDel >50 (L)", "CDEL_gt50bp_right": "CDel >50 (R)",
    "CINS_1_5bp": "CIns 1-5", "CINS_6_50bp": "CIns 6-50",
    "CINS_51_200bp": "CIns 51-200", "CINS_gt200bp": "CIns >200",
}

Z_COLOR = "tab:blue"
C_COLOR = "tab:gray"
LABEL_FONTSIZE = 22
TICK_FONTSIZE  = 18
SIG_FONTSIZE   = 18

def expand_order_with_dual(tbl):
    present = set(tbl["stratum_anchor"].tolist())
    out = []
    for s in STRATUM_ORDER:
        if s in DUAL_ANCHOR_STRATA:
            for tag in ("_left", "_right"):
                lab = s + tag
                if lab in present:
                    out.append(lab)
        else:
            if s in present:
                out.append(s)
    return out

def plot_one_window(tag, radius):
    tbl_path = INDIR / f"window_stats_{tag}.tsv"
    tbl = pd.read_csv(tbl_path, sep="\t")
    order = expand_order_with_dual(tbl)
    tbl = tbl.set_index("stratum_anchor").loc[order].reset_index()
    n = len(tbl)
    fig_w = max(14, 0.65 * n + 4)
    fig, ax = plt.subplots(figsize=(fig_w, 7.5))
    x = np.arange(n)
    width = 0.38

    z_vals = tbl["Z_mean"].to_numpy()
    c_vals = tbl["C_mean"].to_numpy()
    z_err_lo = np.clip(z_vals - tbl["Z_lo_mean"].to_numpy(), 0, None)
    z_err_hi = np.clip(tbl["Z_hi_mean"].to_numpy() - z_vals, 0, None)
    c_err_lo = np.clip(c_vals - tbl["C_lo_mean"].to_numpy(), 0, None)
    c_err_hi = np.clip(tbl["C_hi_mean"].to_numpy() - c_vals, 0, None)

    ax.bar(x - width/2, z_vals, width,
           yerr=[z_err_lo, z_err_hi], capsize=2,
           color=Z_COLOR, edgecolor="white", linewidth=0.5,
           error_kw={"linewidth": 1.0, "ecolor": "black"})
    ax.bar(x + width/2, c_vals, width,
           yerr=[c_err_lo, c_err_hi], capsize=2,
           color=C_COLOR, edgecolor="white", linewidth=0.5,
           error_kw={"linewidth": 1.0, "ecolor": "black"})

    ax.axhline(1.0, ls=":", color="black", lw=1.5, alpha=0.6)

    
    bar_top = np.maximum(z_vals + z_err_hi, c_vals + c_err_hi)
    bracket_y_offset = 0.06 * (bar_top.max() if bar_top.max() > 0 else 1.0)
    
    for i, row in enumerate(tbl.itertuples()):
        marker = row.sig_marker_primary if isinstance(row.sig_marker_primary, str) else ""
        z_val = z_vals[i]
        c_val = c_vals[i]
        fold = z_val / c_val if c_val > 0 else np.nan
    
        y_bracket = bar_top[i] + bracket_y_offset
        x_left = i - width/2
        x_right = i + width/2
    
        if marker in ("", "ns", "ns_dir"):
            ax.text(i, y_bracket, "n.s.",
                    ha="center", va="bottom", fontsize=SIG_FONTSIZE - 4,
                    color="#666666", fontstyle="italic")
        else:
            ax.plot([x_left, x_left, x_right, x_right],
                    [y_bracket, y_bracket + bracket_y_offset*0.35,
                     y_bracket + bracket_y_offset*0.35, y_bracket],
                    lw=1.2, color="black")
            ax.text(i, y_bracket + bracket_y_offset*0.4, marker,
                    ha="center", va="bottom", fontsize=SIG_FONTSIZE)
            if np.isfinite(fold):
                ax.text(i, y_bracket + bracket_y_offset*1.5,
                        f"{fold:.1f}×",
                        ha="center", va="bottom",
                        fontsize=SIG_FONTSIZE - 4, color="#333333")
    
    ymax = (bar_top + bracket_y_offset * 4.0).max()
    ax.set_ylim(0, ymax * 1.05)

    ax.set_xticks(x)
    ax.set_xticklabels(
        [X_LABEL_MAP.get(s, s) for s in tbl["stratum_anchor"]],
        rotation=45, ha="right", fontsize=TICK_FONTSIZE,
    )
    ax.set_ylabel(f"Relative density (\u00b1{radius} bp window)",
                  fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.margins(x=0.005)

    plt.tight_layout()
    fig.savefig(OUTDIR / f"mutation_window_bars_{tag}.png",
                dpi=400, bbox_inches="tight")
    plt.show(fig)
    print(f"[SAVED] mutation_window_bars_{tag}.png/pdf")

def save_combined_legend():
    fig_leg = plt.figure(figsize=(5.5, 2.6))
    ax_leg = fig_leg.add_subplot(111)
    ax_leg.axis("off")
    color_handles = [
        mpatches.Patch(color=Z_COLOR, label="Z-DNA"),
        mpatches.Patch(color=C_COLOR, label="Matched control"),
    ]
    sig_handles = [
        Line2D([0], [0], color="none", marker="$\\ast\\ast\\ast$",
               markersize=14, label=r"$P_{\mathrm{bonf}} < 10^{-3}$"),
        Line2D([0], [0], color="none", marker="$\\ast\\ast$",
               markersize=14, label=r"$P_{\mathrm{bonf}} < 10^{-2}$"),
        Line2D([0], [0], color="none", marker="$\\ast$",
               markersize=14, label=r"$P_{\mathrm{bonf}} < 0.05$"),
        Line2D([0], [0], color="none", marker="$\\mathrm{n.s.}$",
               markersize=14, label="not significant (one-sided Z > C)"),
        Line2D([0], [0], color="none", marker="$2.5\\times$",
               markersize=14, label="fold enrichment (Z / C)"),
    ]
    ax_leg.legend(handles=color_handles + sig_handles,
                  loc="center", frameon=False, fontsize=12, ncol=1,
                  handletextpad=0.6, borderpad=0.4)
    fig_leg.savefig(OUTDIR / "mutation_window_bars_legend.png",
                    dpi=600, bbox_inches="tight")
    plt.show(fig_leg)
    print("[SAVED] mutation_window_bars_legend.png/pdf")

for radius, tag in WINDOWS:
    plot_one_window(tag, radius)
save_combined_legend()
print("[DONE]")

### 9.4 Metaprofiles

In [ ]:
# Compute Z-DNA vs control mutation metaprofiles around mutation sites.
import numpy as np
import pandas as pd
from pathlib import Path

BASE = "data"

MASTER         = f"{BASE}/controls/CHM13_controls_MASTER.tsv"
MUT_BED        = f"{BASE}/mutations/hprc_mutations.bed.gz"
MUT_META       = f"{BASE}/mutations/hprc_mutations.meta.tsv.gz"
CHROM_LEN_TSV  = f"{BASE}/chm13/chm13_seqreport_zdna.tsv"

TOL = "10%"
WINDOW_RADIUS = 500
N_BOOTS = 500
CI_LOW, CI_HIGH = 2.5, 97.5
MIN_MUTS = 500

ANCHOR_MODE = "right"   # "left" or "right"

OUTDIR = Path(f"{BASE}/mutation_profiles_refined_tol10_{ANCHOR_MODE}")
OUTDIR.mkdir(parents=True, exist_ok=True)

def chrom_block_bootstrap_windowmean_norm(counts_by_chr, n_boot):
    chroms = list(counts_by_chr)
    mat = np.stack([counts_by_chr[c].astype(float) for c in chroms], axis=0)
    rng = np.random.default_rng(42)

    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(chroms), len(chroms))
        v = mat[idx].sum(axis=0)
        m = v.mean()
        boots.append(v / m if m else np.full_like(v, np.nan, dtype=float))

    boots = np.stack(boots, axis=0)
    return (
        np.nanpercentile(boots, CI_LOW, axis=0),
        np.nanpercentile(boots, CI_HIGH, axis=0),
    )

def build_bool_coverage(starts, ends, chrom_len):
    starts = np.asarray(starts, dtype=np.int64)
    ends   = np.asarray(ends,   dtype=np.int64)

    starts = np.clip(starts, 0, chrom_len)
    ends   = np.clip(ends,   0, chrom_len)

    mask = ends > starts
    if not np.any(mask):
        return np.zeros(chrom_len, dtype=bool)

    starts = starts[mask]
    ends   = ends[mask]

    diff = np.zeros(chrom_len + 1, dtype=np.int32)
    np.add.at(diff, starts,  1)
    np.add.at(diff, ends,   -1)

    return np.cumsum(diff[:-1]) > 0

def midpoint_from_left_start_and_span(start_arr, ref_span_arr):
    return start_arr + (ref_span_arr - 1) // 2

def right_breakpoint_from_left_start_and_span(start_arr, ref_span_arr):
    return start_arr + ref_span_arr - 1

chrom_len = dict(
    pd.read_csv(
        CHROM_LEN_TSV,
        sep="\t",
        usecols=["ucsc_pansn_name", "gcf_seq_length"]
    ).values
)

m = pd.read_csv(MASTER, sep="\t")
m = m[(m["tol_level"] == TOL) & (m["has_control"])].drop_duplicates(
    ["Chromosome", "Start0", "End0"]
)

z_intervals = {
    chrom: d[["Start0", "End0"]].to_numpy(dtype=np.int64)
    for chrom, d in m.groupby("Chromosome")
}
c_intervals = {
    chrom: d[["Ctrl_Start", "Ctrl_End"]].to_numpy(dtype=np.int64)
    for chrom, d in m.groupby("Chromosome")
}

mut = pd.read_csv(
    MUT_BED,
    sep="\t",
    header=None,
    names=["Chromosome", "Start", "End", "mut_id", "score", "strand"]
)

meta = pd.read_csv(MUT_META, sep="\t", compression="gzip")

need = [
    "mut_id",
    "struct_class",
    "dominant_op",
    "ref_span",
    "alt_span",
]
mut = mut.merge(meta[need], on="mut_id", how="left").dropna()

mut["Start"]    = mut["Start"].astype(np.int64)
mut["ref_span"] = mut["ref_span"].astype(np.int64)
mut["alt_span"] = mut["alt_span"].astype(np.int64)

mut["event_size"] = np.where(
    (mut["struct_class"] == "ins") |
    ((mut["struct_class"] == "complex") & (mut["dominant_op"] == "ins_like")),
    mut["alt_span"],
    mut["ref_span"]
).astype(np.int64)

def assign_stratum(row):
    sc  = row["struct_class"]
    dom = row["dominant_op"]
    L   = int(row["event_size"])
    if sc == "snp":
        return "SNP"
    if sc == "mnp":
        return "MNP_2_20"
    
    if sc == "substitution":
        if L <= 50:
            return "SUB_20_50"
        elif L <= 200:
            return "SUB_50_200"
        else:
            return "SUB_gt200"

    if sc == "del":
        if L == 1:
            return "DEL_1bp"
        elif 2 <= L <= 5:
            return "DEL_2_5bp"
        elif 6 <= L <= 50:
            return "DEL_6_50bp"
        else:
            return "DEL_gt50bp"

    if sc == "ins":
        if L == 1:
            return "INS_1bp"
        elif 2 <= L <= 5:
            return "INS_2_5bp"
        elif 6 <= L <= 50:
            return "INS_6_50bp"
        elif 51 <= L <= 200:
            return "INS_51_200bp"
        else:
            return "INS_gt200bp"

    if sc == "complex" and dom == "del_like":
        if 1 <= L <= 5:
            return "CDEL_1_5bp"
        elif 6 <= L <= 50:
            return "CDEL_6_50bp"
        else:
            return "CDEL_gt50bp"

    if sc == "complex" and dom == "ins_like":
        if 1 <= L <= 5:
            return "CINS_1_5bp"
        elif 6 <= L <= 50:
            return "CINS_6_50bp"
        elif 51 <= L <= 200:
            return "CINS_51_200bp"
        else:
            return "CINS_gt200bp"

    return None

mut["stratum"] = mut.apply(assign_stratum, axis=1)
mut = mut.dropna(subset=["stratum"]).copy()

DUAL_ANCHOR_STRATA = {
    "DEL_gt50bp",
    "CDEL_gt50bp",
    "SUB_50_200",
    "SUB_gt200",
}

MIDPOINT_ANCHOR_STRATA = {
    "DEL_1bp",
    "DEL_2_5bp",
    "DEL_6_50bp",
    "CDEL_1_5bp",
    "CDEL_6_50bp",
    "SUB_20_50",   # ← behaves like compact substitution
}

SINGLE_ANCHOR_STRATA = {
    "INS_1bp",
    "INS_2_5bp",
    "INS_6_50bp",
    "INS_51_200bp",
    "INS_gt200bp",
    "CINS_1_5bp",
    "CINS_6_50bp",
    "CINS_51_200bp",
    "CINS_gt200bp",
    "SNP",
    "MNP_2_20",    # updated name
}

offsets = np.arange(-WINDOW_RADIUS, WINDOW_RADIUS + 1, dtype=np.int64)

summary_rows = []

for stratum, sub in mut.groupby("stratum", sort=True):

    if ANCHOR_MODE == "right" and stratum not in DUAL_ANCHOR_STRATA:
        continue

    if len(sub) < MIN_MUTS:
        continue

    Z_by_chr = {}
    C_by_chr = {}

    Z_total = np.zeros(len(offsets), dtype=np.int64)
    C_total = np.zeros(len(offsets), dtype=np.int64)

    n_events_total = 0
    n_events_used  = 0

    for chrom, d in sub.groupby("Chromosome", sort=False):
        if chrom not in z_intervals or chrom not in c_intervals or chrom not in chrom_len:
            continue

        chrom_L = int(chrom_len[chrom])

        starts = d["Start"].to_numpy(dtype=np.int64)
        refs   = d["ref_span"].to_numpy(dtype=np.int64)

        if stratum in DUAL_ANCHOR_STRATA:
            left_anchor  = starts
            right_anchor = right_breakpoint_from_left_start_and_span(starts, refs)

            if ANCHOR_MODE == "left":
                anchors = left_anchor
            elif ANCHOR_MODE == "right":
                anchors = right_anchor
            else:
                raise ValueError("ANCHOR_MODE must be 'left' or 'right'")

        elif stratum in MIDPOINT_ANCHOR_STRATA:
            anchors = midpoint_from_left_start_and_span(starts, refs)

        elif stratum in SINGLE_ANCHOR_STRATA:
            anchors = starts

        else:
            continue

        z_cov = build_bool_coverage(
            z_intervals[chrom][:, 0],
            z_intervals[chrom][:, 1],
            chrom_L
        )
        c_cov = build_bool_coverage(
            c_intervals[chrom][:, 0],
            c_intervals[chrom][:, 1],
            chrom_L
        )

        Zc = np.zeros(len(offsets), dtype=np.int64)
        Cc = np.zeros(len(offsets), dtype=np.int64)

        for i, k in enumerate(offsets):
            pos = anchors + k
            mask = (pos >= 0) & (pos < chrom_L)
            if np.any(mask):
                pos_valid = pos[mask]
                Zc[i] = z_cov[pos_valid].sum()
                Cc[i] = c_cov[pos_valid].sum()

        Z_total += Zc
        C_total += Cc
        Z_by_chr[chrom] = Zc
        C_by_chr[chrom] = Cc

        n_events_total += len(d)
        n_events_used  += len(anchors)

    if len(Z_by_chr) == 0:
        continue

    Z_mean = Z_total / Z_total.mean() if Z_total.mean() else np.full_like(Z_total, np.nan, dtype=float)
    C_mean = C_total / C_total.mean() if C_total.mean() else np.full_like(C_total, np.nan, dtype=float)

    Z_lo, Z_hi = chrom_block_bootstrap_windowmean_norm(Z_by_chr, N_BOOTS)
    C_lo, C_hi = chrom_block_bootstrap_windowmean_norm(C_by_chr, N_BOOTS)

    out = pd.DataFrame({
        "offset": offsets,
        "Z_mean": Z_mean,
        "Z_lo": Z_lo,
        "Z_hi": Z_hi,
        "C_mean": C_mean,
        "C_lo": C_lo,
        "C_hi": C_hi,
        "stratum": stratum,
        "anchor_mode": ANCHOR_MODE,
        "n_events_total": n_events_total,
        "n_events_used": n_events_used,
    })
    chrom_rows = []
    for chrom in Z_by_chr:
        for i, off in enumerate(offsets):
            chrom_rows.append((chrom, int(off), int(Z_by_chr[chrom][i]), int(C_by_chr[chrom][i])))
    chrom_df = pd.DataFrame(chrom_rows, columns=["chrom", "offset", "Z_raw", "C_raw"])
    chrom_df.to_csv(OUTDIR / f"profile_{stratum}_bychrom.tsv", sep="\t", index=False)
    out.to_csv(OUTDIR / f"profile_{stratum}.tsv", sep="\t", index=False)

    summary_rows.append({
        "stratum": stratum,
        "anchor_mode": ANCHOR_MODE,
        "n_events_total": n_events_total,
        "n_events_used": n_events_used,
        "n_chromosomes": len(Z_by_chr),
        "mean_Z_raw": float(Z_total.mean()) if len(Z_total) else np.nan,
        "mean_C_raw": float(C_total.mean()) if len(C_total) else np.nan,
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["stratum", "anchor_mode"])
summary_df.to_csv(OUTDIR / "profile_summary.tsv", sep="\t", index=False)

print("[DONE] Refined metaprofiles computed and cached.")
print(summary_df)

In [ ]:
# Plot the mutation metaprofiles.
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

BASE = "data"

ANCHOR_MODES = ["right", "left"]

INDIRS = {
    a: Path(f"{BASE}/mutation_profiles_refined_tol10_{a}")
    for a in ANCHOR_MODES
}

OUTDIR = Path(f"{BASE}/mutation_profiles_plots_tol10")
OUTDIR.mkdir(exist_ok=True)

LABEL_FONTSIZE = 22
TICK_FONTSIZE  = 20
TITLE_FONTSIZE = 24
Z_COLOR = "tab:blue"
C_COLOR = "tab:gray"

DUAL_ANCHOR_STRATA = {
    "DEL_gt50bp",
    "CDEL_gt50bp",
    "SUB_50_200",
    "SUB_gt200",
}
SINGLE_ANCHOR_STRATA = {
    "SNP",
    "MNP_2_20",

    "DEL_1bp", "DEL_2_5bp", "DEL_6_50bp",
    "INS_1bp", "INS_2_5bp", "INS_6_50bp", "INS_51_200bp", "INS_gt200bp",
    "CDEL_1_5bp", "CDEL_6_50bp",
    "CINS_1_5bp", "CINS_6_50bp", "CINS_51_200bp", "CINS_gt200bp",

    "SUB_20_50",   # ← NEW (midpoint class but treated as single plot)
}
TITLE_MAP = {
    "SNP": "Single-nucleotide variants",
    "MNP_2_20": "Multi-nucleotide variants (2–20 bp)",

    "SUB_20_50": "Substitutions (20–50 bp)",
    "SUB_50_200": "Substitutions (50–200 bp)",
    "SUB_gt200": "Substitutions (>200 bp)",

    "DEL_1bp": "1 bp deletions",
    "DEL_2_5bp": "2–5 bp deletions",
    "DEL_6_50bp": "6–50 bp deletions",
    "DEL_gt50bp": ">50 bp deletions",

    "INS_1bp": "1 bp insertions",
    "INS_2_5bp": "2–5 bp insertions",
    "INS_6_50bp": "6–50 bp insertions",
    "INS_51_200bp": "51–200 bp insertions",
    "INS_gt200bp": ">200 bp insertions",

    "CDEL_1_5bp": "Complex deletion-like (1–5 bp)",
    "CDEL_6_50bp": "Complex deletion-like (6–50 bp)",
    "CDEL_gt50bp": "Complex deletion-like (>50 bp)",

    "CINS_1_5bp": "Complex insertion-like (1–5 bp)",
    "CINS_6_50bp": "Complex insertion-like (6–50 bp)",
    "CINS_51_200bp": "Complex insertion-like (51–200 bp)",
    "CINS_gt200bp": "Complex insertion-like (>200 bp)",
}

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def plot_profile(df, title, outpath):
    plt.figure(figsize=(8.6, 6.2))

    plt.plot(df["offset"], df["Z_mean"], lw=4.0, color=Z_COLOR)
    plt.fill_between(df["offset"], df["Z_lo"], df["Z_hi"], alpha=0.35, color=Z_COLOR)
    
    plt.plot(df["offset"], df["C_mean"], lw=4.0, color=C_COLOR)
    plt.fill_between(df["offset"], df["C_lo"], df["C_hi"], alpha=0.35, color=C_COLOR)
    
    plt.axvline(0, ls="--", color="black", lw=2)
    plt.axhline(1, ls=":", color="black", lw=2)

    plt.xlabel("Position relative to mutation anchor (bp)", fontsize=LABEL_FONTSIZE)
    plt.ylabel("Relative density",
               fontsize=LABEL_FONTSIZE)

    plt.title(title, fontsize=TITLE_FONTSIZE, pad=18)

    plt.tick_params(labelsize=TICK_FONTSIZE)
    ax = plt.gca()
    ax.set_xticks([-400, 0, 400])
    yticks = ax.get_yticks()
    ax.set_yticks(yticks[::2])
    despine(plt.gca())

    plt.tight_layout()
    plt.savefig(outpath, dpi=600)
    plt.close()

for anchor in ANCHOR_MODES:

    indir = INDIRS[anchor]
    if not indir.exists():
        continue

    for path in sorted(indir.glob("profile_*.tsv")):
        df = pd.read_csv(path, sep="\t")
        stratum = path.stem.replace("profile_", "")

        if stratum not in TITLE_MAP:
            continue
        if stratum in DUAL_ANCHOR_STRATA:
            anchor_label = (
                "right breakpoint" if anchor == "right"
                else "left breakpoint"
            )
            title = f"{TITLE_MAP[stratum]} ({anchor_label})"
        
        else:
            if anchor == "right":
                continue
        
            title = TITLE_MAP[stratum]

        outname = f"{stratum}_{anchor}.png"
        outpath = OUTDIR / outname
        plot_profile(df, title, outpath)

        window = df[(df["offset"] >= -5) & (df["offset"] <= 5)]
        if window.empty:
            print(f"[WARN] No bins in ±5 bp window for {stratum}")
        else:
            z_win = window["Z_mean"].mean()
            c_win = window["C_mean"].mean()
            ratio = z_win / c_win if c_win > 0 else np.nan

            print(
                f"[±5bp WINDOW] {stratum:15s} | "            
                f"Z/C = {ratio:.3f}"
            )

legend_fig = plt.figure(figsize=(3.0, 1.8))
ax_leg = legend_fig.add_subplot(111)
ax_leg.axis("off")

legend_handles = [
    Line2D([0], [0], lw=4.0, label="Z-DNA", color=Z_COLOR),
    Line2D([0], [0], lw=4.0, label="Control", color=C_COLOR),
]

ax_leg.legend(
    handles=legend_handles,
    loc="center",
    frameon=False,
    fontsize=13
)

legend_out = OUTDIR / "mutation_profiles_legend.png"
legend_fig.savefig(legend_out, dpi=600, bbox_inches="tight")

print("[DONE] All plots generated.")

### 9.5 SNP-type profiles

In [ ]:
# Z-DNA vs control density profiles by SNP type (windowed counts + block-bootstrap CIs).
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

BASE = "data"

MASTER   = f"{BASE}/controls/CHM13_controls_MASTER.tsv"
MUT_BED  = f"{BASE}/mutations/hprc_mutations.bed.gz"
MUT_META = f"{BASE}/mutations/hprc_mutations.meta.tsv.gz"
CHROM_LEN_TSV = f"{BASE}/chm13/chm13_seqreport_zdna.tsv"

WINDOW_RADIUS = 500
N_BOOTS = 300
CI_LOW, CI_HIGH = 2.5, 97.5

GATHER_CHUNK = 50_000

OUTDIR = Path(f"{BASE}/mutation_profiles_snp6")
OUTDIR.mkdir(exist_ok=True)

def build_bool_coverage(starts, ends, L):
    starts = np.asarray(starts, dtype=np.int64)
    ends   = np.asarray(ends,   dtype=np.int64)

    starts = np.clip(starts, 0, L)
    ends   = np.clip(ends,   0, L)

    mask = ends > starts
    if not np.any(mask):
        return np.zeros(L, dtype=bool)

    starts = starts[mask]
    ends   = ends[mask]

    diff = np.zeros(L + 1, dtype=np.int32)
    np.add.at(diff, starts, 1)
    np.add.at(diff, ends, -1)

    return np.cumsum(diff[:-1]) > 0

def windowed_counts(anchors, cov, offsets, L, chunk=GATHER_CHUNK):
    W = offsets.shape[0]
    out = np.zeros(W, dtype=np.int64)
    if anchors.size == 0:
        return out

    off = offsets[None, :]
    for s in range(0, anchors.size, chunk):
        idx = anchors[s:s + chunk][:, None] + off      # (b, W)
        valid = (idx >= 0) & (idx < L)
        np.clip(idx, 0, L - 1, out=idx)                # avoid OOB indexing
        vals = cov[idx]                                # (b, W) bool
        vals &= valid                                  # zero out-of-range
        out += vals.sum(axis=0)
    return out

def bootstrap_ci(mat, n_boot):
    rng = np.random.default_rng(42)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, mat.shape[0], mat.shape[0])
        v = mat[idx].sum(axis=0)
        m = v.mean()
        boots.append(v / m if m else np.full_like(v, np.nan))
    boots = np.stack(boots)
    return np.nanpercentile(boots, CI_LOW, axis=0), np.nanpercentile(boots, CI_HIGH, axis=0)

chrom_len = dict(pd.read_csv(
    CHROM_LEN_TSV, sep="\t",
    usecols=["ucsc_pansn_name", "gcf_seq_length"]
).values)

m = pd.read_csv(MASTER, sep="\t")
m = m[m["has_control"]].copy()
m = m.dropna(subset=["Chromosome", "Start0", "End0", "Ctrl_Start", "Ctrl_End"])
m = m.drop_duplicates(["Chromosome", "Start0", "End0"])

z_intervals = {
    c: d[["Start0", "End0"]].to_numpy(dtype=np.int64)
    for c, d in m.groupby("Chromosome")
}
c_intervals = {
    c: d[["Ctrl_Start", "Ctrl_End"]].dropna().to_numpy(dtype=np.int64)
    for c, d in m.groupby("Chromosome")
}

mut = pd.read_csv(MUT_BED, sep="\t", header=None,
    names=["Chromosome", "Start", "End", "mut_id", "score", "strand"])
meta = pd.read_csv(MUT_META, sep="\t", compression="gzip")
mut = mut.merge(meta[["mut_id", "struct_class", "reference", "variant"]],
                on="mut_id").dropna()

snp = mut[mut["struct_class"] == "snp"].copy()
snp["reference"] = snp["reference"].str.upper()
snp["variant"]   = snp["variant"].str.upper()

valid = {"A", "C", "G", "T"}
snp = snp[snp["reference"].isin(valid) & snp["variant"].isin(valid)].copy()

comp = {"A": "T", "T": "A", "C": "G", "G": "C"}

def normalize(ref, alt):
    if ref in ("A", "G"):
        ref = comp[ref]
        alt = comp[alt]
    return f"{ref}>{alt}"

snp["type"] = [normalize(r, a) for r, a in zip(snp["reference"], snp["variant"])]

types = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
offsets = np.arange(-WINDOW_RADIUS, WINDOW_RADIUS + 1, dtype=np.int64)
W = offsets.shape[0]

type_counts = snp["type"].value_counts()
active_types = [t for t in types if type_counts.get(t, 0) >= 1000]

anchors_by = {
    (chrom, t): d["Start"].to_numpy(np.int64)
    for (chrom, t), d in snp.groupby(["Chromosome", "type"])
}

Z_total  = {t: np.zeros(W) for t in active_types}
C_total  = {t: np.zeros(W) for t in active_types}
Z_by_chr = {t: {} for t in active_types}
C_by_chr = {t: {} for t in active_types}

chroms = [c for c in z_intervals if c in chrom_len]
for chrom in chroms:
    L = int(chrom_len[chrom])

    z_cov = build_bool_coverage(z_intervals[chrom][:, 0], z_intervals[chrom][:, 1], L)
    c_cov = build_bool_coverage(c_intervals[chrom][:, 0], c_intervals[chrom][:, 1], L)

    for t in active_types:
        anchors = anchors_by.get((chrom, t))
        if anchors is None or anchors.size == 0:
            continue

        Zc = windowed_counts(anchors, z_cov, offsets, L)
        Cc = windowed_counts(anchors, c_cov, offsets, L)

        Z_total[t]  += Zc
        C_total[t]  += Cc
        Z_by_chr[t][chrom] = Zc
        C_by_chr[t][chrom] = Cc

for t in active_types:
    if not Z_by_chr[t]:
        print(f"[WARN] no chromosomes contributed for {t}")
        continue

    Z_mean = Z_total[t] / Z_total[t].mean()
    C_mean = C_total[t] / C_total[t].mean()

    Z_lo, Z_hi = bootstrap_ci(np.stack(list(Z_by_chr[t].values())), N_BOOTS)
    C_lo, C_hi = bootstrap_ci(np.stack(list(C_by_chr[t].values())), N_BOOTS)

    df = pd.DataFrame({
        "offset": offsets,
        "Z_mean": Z_mean, "Z_lo": Z_lo, "Z_hi": Z_hi,
        "C_mean": C_mean, "C_lo": C_lo, "C_hi": C_hi,
    })
    df.to_csv(OUTDIR / f"profile_{t}.tsv", sep="\t", index=False)

    comp_label = f"{comp[t[0]]}>{comp[t[-1]]}"
    plt.figure(figsize=(7, 5))
    plt.plot(offsets, Z_mean, lw=3)
    plt.fill_between(offsets, Z_lo, Z_hi, alpha=0.3)
    plt.plot(offsets, C_mean, lw=3, color="gray")
    plt.fill_between(offsets, C_lo, C_hi, alpha=0.3, color="gray")
    plt.axvline(0, ls="--", color="black")
    plt.axhline(1, ls=":", color="black")
    plt.title(f"{t} / {comp_label}")
    plt.xlabel("Position relative to mutation (bp)")
    plt.ylabel("Relative density")
    plt.tight_layout()
    plt.savefig(OUTDIR / f"{t}.png", dpi=400)
    plt.close()

print("[DONE] SNP-specific profiles computed.")

In [ ]:
# SNP-subtype Z-DNA enrichment: profiles to per-chromosome window bars.
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

BASE = "data"

MASTER        = f"{BASE}/controls/CHM13_controls_MASTER.tsv"
MUT_BED       = f"{BASE}/mutations/hprc_mutations.bed.gz"
MUT_META      = f"{BASE}/mutations/hprc_mutations.meta.tsv.gz"
CHROM_LEN_TSV = f"{BASE}/chm13/chm13_seqreport_zdna.tsv"

TOL = "10%"
WINDOW_RADIUS = 500
N_BOOTS_CI    = 500       # bootstrap iterations for profile CI bands
N_BOOTS_STAT  = 50_000    # bootstrap iterations for the statistical test
SEED          = 42
CI_LOW, CI_HIGH = 2.5, 97.5
MIN_CHROMS    = 6
GATHER_CHUNK  = 50_000

WINDOWS = [(10, "pm10"), (25, "pm25"), (50, "pm50")]

PROFILE_DIR = Path(f"{BASE}/mutation_profiles_snp6")
STATS_DIR   = Path(f"{BASE}/mutation_window_bars_snp6")
PROFILE_DIR.mkdir(exist_ok=True)
STATS_DIR.mkdir(exist_ok=True)

SNP_TYPES = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]

Z_COLOR = "tab:blue"
C_COLOR = "tab:gray"
LABEL_FONTSIZE = 22
TICK_FONTSIZE  = 18
SIG_FONTSIZE   = 18

def build_bool_coverage(starts, ends, L):
    starts = np.asarray(starts, dtype=np.int64)
    ends   = np.asarray(ends,   dtype=np.int64)
    starts = np.clip(starts, 0, L)
    ends   = np.clip(ends,   0, L)
    mask = ends > starts
    if not np.any(mask):
        return np.zeros(L, dtype=bool)
    starts, ends = starts[mask], ends[mask]
    diff = np.zeros(L + 1, dtype=np.int32)
    np.add.at(diff, starts, 1)
    np.add.at(diff, ends, -1)
    return np.cumsum(diff[:-1]) > 0

def windowed_counts(anchors, cov, offsets, L, chunk=GATHER_CHUNK):
    W = offsets.shape[0]
    out = np.zeros(W, dtype=np.int64)
    if anchors.size == 0:
        return out
    off = offsets[None, :]
    for s in range(0, anchors.size, chunk):
        idx = anchors[s:s + chunk][:, None] + off
        valid = (idx >= 0) & (idx < L)
        np.clip(idx, 0, L - 1, out=idx)
        vals = cov[idx]
        vals &= valid
        out += vals.sum(axis=0)
    return out

def bootstrap_ci(mat, n_boot):
    rng = np.random.default_rng(42)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, mat.shape[0], mat.shape[0])
        v = mat[idx].sum(axis=0)
        m = v.mean()
        boots.append(v / m if m else np.full_like(v, np.nan, dtype=float))
    boots = np.stack(boots)
    return (np.nanpercentile(boots, CI_LOW,  axis=0),
            np.nanpercentile(boots, CI_HIGH, axis=0))

def per_chrom_window_stats(bychrom_df, radius):
    chroms = bychrom_df["chrom"].unique()
    rows = []
    for chrom in chroms:
        cd = bychrom_df[bychrom_df["chrom"] == chrom]
        z_all = cd["Z_raw"].to_numpy(dtype=np.float64)
        c_all = cd["C_raw"].to_numpy(dtype=np.float64)
        z_fm = z_all.mean()
        c_fm = c_all.mean()
        if z_fm == 0 or c_fm == 0:
            continue
        win = cd[(cd["offset"] >= -radius) & (cd["offset"] <= radius)]
        z_wm = win["Z_raw"].to_numpy(dtype=np.float64).mean()
        c_wm = win["C_raw"].to_numpy(dtype=np.float64).mean()
        rows.append({"chrom": chrom,
                     "Z_norm": z_wm / z_fm, "C_norm": c_wm / c_fm})
    return pd.DataFrame(rows)

def chromosome_wilcoxon_onesided(bychrom_df, radius):
    cs = per_chrom_window_stats(bychrom_df, radius)
    if len(cs) < MIN_CHROMS:
        return np.nan, np.nan, len(cs)
    z_n = cs["Z_norm"].to_numpy()
    c_n = cs["C_norm"].to_numpy()
    if np.allclose(z_n - c_n, 0):
        return 0.0, 1.0, len(cs)
    try:
        stat, p = wilcoxon(z_n, c_n, alternative="greater", zero_method="wilcox")
        return float(stat), float(p), len(cs)
    except ValueError:
        return np.nan, np.nan, len(cs)

def chromosome_block_bootstrap(bychrom_df, radius, n_boots, seed):
    rng = np.random.default_rng(seed)
    chroms = bychrom_df["chrom"].unique()
    n = len(chroms)
    z_win_arr = np.empty(n)
    c_win_arr = np.empty(n)
    z_tot_arr = np.empty(n)
    c_tot_arr = np.empty(n)
    for j, chrom in enumerate(chroms):
        cd = bychrom_df[bychrom_df["chrom"] == chrom]
        z_all = cd["Z_raw"].to_numpy(dtype=np.float64)
        c_all = cd["C_raw"].to_numpy(dtype=np.float64)
        win = cd[(cd["offset"] >= -radius) & (cd["offset"] <= radius)]
        z_win_arr[j] = win["Z_raw"].sum()
        c_win_arr[j] = win["C_raw"].sum()
        z_tot_arr[j] = z_all.sum()
        c_tot_arr[j] = c_all.sum()

    Zw, Cw = z_win_arr.sum(), c_win_arr.sum()
    Zt, Ct = z_tot_arr.sum(), c_tot_arr.sum()
    if Zt == 0 or Ct == 0 or Cw == 0 or Zw == 0:
        return np.nan, np.nan, np.nan, np.nan
    observed = np.log((Zw / Zt) / (Cw / Ct))

    idx = rng.integers(0, n, size=(n_boots, n))
    zw_b = z_win_arr[idx].sum(axis=1)
    cw_b = c_win_arr[idx].sum(axis=1)
    zt_b = z_tot_arr[idx].sum(axis=1)
    ct_b = c_tot_arr[idx].sum(axis=1)
    valid = (zt_b > 0) & (ct_b > 0) & (cw_b > 0) & (zw_b > 0)
    if not valid.any():
        return observed, np.nan, np.nan, np.nan
    boots = np.log((zw_b[valid] / zt_b[valid]) / (cw_b[valid] / ct_b[valid]))
    ci_lo = np.percentile(boots, 2.5)
    ci_hi = np.percentile(boots, 97.5)
    centered = boots - boots.mean()
    p_one = (np.sum(centered >= observed) + 1) / (len(centered) + 1)
    return observed, ci_lo, ci_hi, p_one

print("Loading data...")
chrom_len = dict(pd.read_csv(
    CHROM_LEN_TSV, sep="\t",
    usecols=["ucsc_pansn_name", "gcf_seq_length"]
).values)

m = pd.read_csv(MASTER, sep="\t")
m = m[(m["tol_level"] == TOL) & (m["has_control"])].copy()
m = m.dropna(subset=["Chromosome", "Start0", "End0", "Ctrl_Start", "Ctrl_End"])
m = m.drop_duplicates(["Chromosome", "Start0", "End0"])

z_intervals = {c: d[["Start0","End0"]].to_numpy(np.int64) for c, d in m.groupby("Chromosome")}
c_intervals = {c: d[["Ctrl_Start","Ctrl_End"]].to_numpy(np.int64) for c, d in m.groupby("Chromosome")}

mut = pd.read_csv(MUT_BED, sep="\t", header=None,
                  names=["Chromosome","Start","End","mut_id","score","strand"])
meta = pd.read_csv(MUT_META, sep="\t", compression="gzip")
mut = mut.merge(meta[["mut_id","struct_class","reference","variant"]],
                on="mut_id").dropna()

snp = mut[mut["struct_class"] == "snp"].copy()
snp["reference"] = snp["reference"].str.upper()
snp["variant"]   = snp["variant"].str.upper()

valid_bases = {"A", "C", "G", "T"}
snp = snp[snp["reference"].isin(valid_bases) & snp["variant"].isin(valid_bases)].copy()

comp = {"A": "T", "T": "A", "C": "G", "G": "C"}

def normalize_snp(ref, alt):
    if ref in ("A", "G"):
        ref, alt = comp[ref], comp[alt]
    return f"{ref}>{alt}"

snp["type"] = [normalize_snp(r, a) for r, a in zip(snp["reference"], snp["variant"])]

type_counts = snp["type"].value_counts()
active_types = [t for t in SNP_TYPES if type_counts.get(t, 0) >= 1000]
print(f"Active SNP types: {active_types}")
for t in active_types:
    print(f"  {t}: {type_counts[t]:,}")

anchors_by = {
    (chrom, t): d["Start"].to_numpy(np.int64)
    for (chrom, t), d in snp.groupby(["Chromosome", "type"])
}

print("\nComputing profiles...")
offsets = np.arange(-WINDOW_RADIUS, WINDOW_RADIUS + 1, dtype=np.int64)
W = offsets.shape[0]

Z_total  = {t: np.zeros(W, dtype=np.int64) for t in active_types}
C_total  = {t: np.zeros(W, dtype=np.int64) for t in active_types}
Z_by_chr = {t: {} for t in active_types}
C_by_chr = {t: {} for t in active_types}
n_events = {t: 0 for t in active_types}

chroms = [c for c in z_intervals if c in chrom_len]
for ci, chrom in enumerate(chroms):
    L = int(chrom_len[chrom])
    print(f"  [{ci+1}/{len(chroms)}] {chrom} (L={L:,})")

    z_cov = build_bool_coverage(z_intervals[chrom][:, 0], z_intervals[chrom][:, 1], L)
    c_cov = build_bool_coverage(c_intervals[chrom][:, 0], c_intervals[chrom][:, 1], L)

    for t in active_types:
        anch = anchors_by.get((chrom, t))
        if anch is None or anch.size == 0:
            continue
        Zc = windowed_counts(anch, z_cov, offsets, L)
        Cc = windowed_counts(anch, c_cov, offsets, L)
        Z_total[t]  += Zc
        C_total[t]  += Cc
        Z_by_chr[t][chrom] = Zc
        C_by_chr[t][chrom] = Cc
        n_events[t] += anch.size

for t in active_types:
    if not Z_by_chr[t]:
        print(f"[WARN] no data for {t}")
        continue

    Z_mean = Z_total[t] / Z_total[t].mean()
    C_mean = C_total[t] / C_total[t].mean()
    Z_lo, Z_hi = bootstrap_ci(np.stack(list(Z_by_chr[t].values())), N_BOOTS_CI)
    C_lo, C_hi = bootstrap_ci(np.stack(list(C_by_chr[t].values())), N_BOOTS_CI)

    safe_name = t.replace(">", "_")

    prof_df = pd.DataFrame({
        "offset": offsets,
        "Z_mean": Z_mean, "Z_lo": Z_lo, "Z_hi": Z_hi,
        "C_mean": C_mean, "C_lo": C_lo, "C_hi": C_hi,
        "n_events_used": n_events[t],
    })
    prof_df.to_csv(PROFILE_DIR / f"profile_{safe_name}.tsv", sep="\t", index=False)

    chrom_rows = []
    for chrom in Z_by_chr[t]:
        for i, off in enumerate(offsets):
            chrom_rows.append((chrom, int(off),
                               int(Z_by_chr[t][chrom][i]),
                               int(C_by_chr[t][chrom][i])))
    chrom_df = pd.DataFrame(chrom_rows, columns=["chrom", "offset", "Z_raw", "C_raw"])
    chrom_df.to_csv(PROFILE_DIR / f"profile_{safe_name}_bychrom.tsv", sep="\t", index=False)

    print(f"  {t}: {n_events[t]:,} events, {len(Z_by_chr[t])} chroms → saved")

print("[PROFILES DONE]")

print("\nComputing chromosome-level statistics...")

def build_snp6_table(radius):
    rows = []
    for t in active_types:
        safe_name = t.replace(">", "_")
        prof_path = PROFILE_DIR / f"profile_{safe_name}.tsv"
        bychr_path = PROFILE_DIR / f"profile_{safe_name}_bychrom.tsv"
        if not prof_path.exists() or not bychr_path.exists():
            continue

        prof  = pd.read_csv(prof_path, sep="\t")
        bychr = pd.read_csv(bychr_path, sep="\t")

        w = prof[(prof["offset"] >= -radius) & (prof["offset"] <= radius)]
        z_mean    = w["Z_mean"].mean()
        c_mean    = w["C_mean"].mean()
        z_lo_mean = w["Z_lo"].mean()
        z_hi_mean = w["Z_hi"].mean()
        c_lo_mean = w["C_lo"].mean()
        c_hi_mean = w["C_hi"].mean()
        fold = (z_mean / c_mean) if (c_mean and c_mean > 0) else np.nan

        n_ev = int(prof["n_events_used"].iloc[0]) if "n_events_used" in prof.columns else np.nan

        wilc_stat, p_wilc, n_chr = chromosome_wilcoxon_onesided(bychr, radius)
        log_r, ci_lo, ci_hi, p_boot = chromosome_block_bootstrap(
            bychr, radius, N_BOOTS_STAT, SEED)

        rows.append({
            "snp_type": t,
            "fold_enrichment": fold,
            "Z_mean": z_mean, "Z_lo_mean": z_lo_mean, "Z_hi_mean": z_hi_mean,
            "C_mean": c_mean, "C_lo_mean": c_lo_mean, "C_hi_mean": c_hi_mean,
            "log_ratio_ZC": log_r, "log_ratio_lo": ci_lo, "log_ratio_hi": ci_hi,
            "p_bootstrap_onesided": p_boot,
            "wilcoxon_stat": wilc_stat,
            "p_wilcoxon_onesided": p_wilc,
            "n_chromosomes_tested": n_chr,
            "n_events_used": n_ev,
        })

    tbl = pd.DataFrame(rows)
    if tbl.empty:
        print("  WARNING: no SNP types loaded")
        return tbl

    for col in ["p_bootstrap_onesided", "p_wilcoxon_onesided"]:
        bonf_col = "p_bonf_" + col[len("p_"):]
        tbl[bonf_col] = np.nan
        mask = tbl[col].notna()
        if mask.any():
            tbl.loc[mask, bonf_col] = multipletests(
                tbl.loc[mask, col].values, method="bonferroni")[1]

    def sig_marker(p, direction_pos):
        if pd.isna(p): return ""
        if not direction_pos: return "ns_dir"
        if p < 1e-3: return "***"
        if p < 1e-2: return "**"
        if p < 5e-2: return "*"
        return "ns"

    tbl["direction_positive"] = tbl["log_ratio_ZC"] > 0
    tbl["sig_marker_primary"] = tbl.apply(
        lambda r: sig_marker(r["p_bonf_wilcoxon_onesided"], r["direction_positive"]), axis=1)
    tbl["sig_marker_bootstrap"] = tbl.apply(
        lambda r: sig_marker(r["p_bonf_bootstrap_onesided"], r["direction_positive"]), axis=1)
    return tbl

for radius, tag in WINDOWS:
    print(f"\n=== Window ±{radius} bp ({tag}) ===")
    tbl = build_snp6_table(radius)
    if tbl.empty:
        continue
    out_path = STATS_DIR / f"snp6_window_stats_{tag}.tsv"
    tbl.to_csv(out_path, sep="\t", index=False)
    print(f"  wrote: {out_path}")
    print(tbl[["snp_type", "Z_mean", "C_mean", "fold_enrichment",
               "p_wilcoxon_onesided", "p_bonf_wilcoxon_onesided",
               "sig_marker_primary",
               "p_bootstrap_onesided", "p_bonf_bootstrap_onesided",
               "sig_marker_bootstrap",
               "n_chromosomes_tested", "n_events_used"]].to_string(index=False))

print("\n[STATS DONE]")

print("\nPlotting...")

def plot_snp6_bars(tag, radius):
    tbl_path = STATS_DIR / f"snp6_window_stats_{tag}.tsv"
    if not tbl_path.exists():
        print(f"  [SKIP] {tbl_path} not found")
        return
    tbl = pd.read_csv(tbl_path, sep="\t")
    type_order = [t for t in SNP_TYPES if t in tbl["snp_type"].values]
    tbl = tbl.set_index("snp_type").loc[type_order].reset_index()

    n = len(tbl)
    fig, ax = plt.subplots(figsize=(max(8, 0.9 * n + 3), 7))
    x = np.arange(n)
    width = 0.38

    z_vals = tbl["Z_mean"].to_numpy()
    c_vals = tbl["C_mean"].to_numpy()
    z_err_lo = np.clip(z_vals - tbl["Z_lo_mean"].to_numpy(), 0, None)
    z_err_hi = np.clip(tbl["Z_hi_mean"].to_numpy() - z_vals, 0, None)
    c_err_lo = np.clip(c_vals - tbl["C_lo_mean"].to_numpy(), 0, None)
    c_err_hi = np.clip(tbl["C_hi_mean"].to_numpy() - c_vals, 0, None)

    ax.bar(x - width/2, z_vals, width,
           yerr=[z_err_lo, z_err_hi], capsize=3,
           color=Z_COLOR, edgecolor="white", linewidth=0.5,
           error_kw={"linewidth": 1.0, "ecolor": "black"})
    ax.bar(x + width/2, c_vals, width,
           yerr=[c_err_lo, c_err_hi], capsize=3,
           color=C_COLOR, edgecolor="white", linewidth=0.5,
           error_kw={"linewidth": 1.0, "ecolor": "black"})

    ax.axhline(1.0, ls=":", color="black", lw=1.5, alpha=0.6)

    bar_top = np.maximum(z_vals + z_err_hi, c_vals + c_err_hi)
    bracket_y_offset = 0.06 * (bar_top.max() if bar_top.max() > 0 else 1.0)

    for i, row in enumerate(tbl.itertuples()):
        marker = row.sig_marker_primary if isinstance(row.sig_marker_primary, str) else ""
        fold = z_vals[i] / c_vals[i] if c_vals[i] > 0 else np.nan
        y_bracket = bar_top[i] + bracket_y_offset
        x_left  = i - width/2
        x_right = i + width/2

        if marker in ("", "ns", "ns_dir"):
            ax.text(i, y_bracket, "n.s.",
                    ha="center", va="bottom", fontsize=SIG_FONTSIZE - 4,
                    color="#666666", fontstyle="italic")
        else:
            ax.plot([x_left, x_left, x_right, x_right],
                    [y_bracket, y_bracket + bracket_y_offset*0.35,
                     y_bracket + bracket_y_offset*0.35, y_bracket],
                    lw=1.2, color="black")
            ax.text(i, y_bracket + bracket_y_offset*0.4, marker,
                    ha="center", va="bottom", fontsize=SIG_FONTSIZE)
            if np.isfinite(fold):
                ax.text(i, y_bracket + bracket_y_offset*1.5,
                        f"{fold:.2f}×",
                        ha="center", va="bottom",
                        fontsize=SIG_FONTSIZE - 4, color="#333333")

    ymax = (bar_top + bracket_y_offset * 4.0).max()
    ax.set_ylim(0, ymax * 1.05)

    ax.set_xticks(x)
    ax.set_xticklabels(type_order, fontsize=TICK_FONTSIZE)
    ax.set_ylabel(f"Relative density (±{radius} bp window)", fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    fig.savefig(STATS_DIR / f"snp6_window_bars_{tag}.png", dpi=400, bbox_inches="tight")
    fig.savefig(STATS_DIR / f"snp6_window_bars_{tag}.pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"  [SAVED] snp6_window_bars_{tag}.png/pdf")

def save_snp6_legend():
    fig_leg = plt.figure(figsize=(5.5, 2.2))
    ax_leg = fig_leg.add_subplot(111)
    ax_leg.axis("off")
    color_handles = [
        mpatches.Patch(color=Z_COLOR, label="Z-DNA"),
        mpatches.Patch(color=C_COLOR, label="Matched control"),
    ]
    sig_handles = [
        Line2D([0], [0], color="none", marker="$\\ast\\ast\\ast$",
               markersize=14, label=r"$P_{\mathrm{bonf}} < 10^{-3}$"),
        Line2D([0], [0], color="none", marker="$\\ast\\ast$",
               markersize=14, label=r"$P_{\mathrm{bonf}} < 10^{-2}$"),
        Line2D([0], [0], color="none", marker="$\\ast$",
               markersize=14, label=r"$P_{\mathrm{bonf}} < 0.05$"),
        Line2D([0], [0], color="none", marker="$\\mathrm{n.s.}$",
               markersize=14, label="not significant"),
        Line2D([0], [0], color="none", marker="$1.25\\times$",
               markersize=14, label="fold enrichment (Z / C)"),
    ]
    ax_leg.legend(handles=color_handles + sig_handles,
                  loc="center", frameon=False, fontsize=12, ncol=1,
                  handletextpad=0.6, borderpad=0.4)
    fig_leg.savefig(STATS_DIR / "snp6_window_bars_legend.png", dpi=600, bbox_inches="tight")
    fig_leg.savefig(STATS_DIR / "snp6_window_bars_legend.pdf", bbox_inches="tight")
    plt.close(fig_leg)
    print("  [SAVED] snp6_window_bars_legend.png/pdf")

for radius, tag in WINDOWS:
    plot_snp6_bars(tag, radius)
save_snp6_legend()

print("\n[ALL DONE]")

## 10. Z-DNA-anchored mutation metaprofiles

In [ ]:
# Z-DNA-centred mutation metaprofile (left/right flanks, stranded).
import numpy as np
import pandas as pd
from pathlib import Path

BASE = "data"

MASTER        = f"{BASE}/controls/CHM13_controls_MASTER.tsv"
MUT_BED       = f"{BASE}/mutations/hprc_mutations.bed.gz"
MUT_META      = f"{BASE}/mutations/hprc_mutations.meta.tsv.gz"
CHROM_LEN_TSV = f"{BASE}/chm13/chm13_seqreport_zdna.tsv"

TOL    = "10%"
FLANK  = 100          # flanking bp on each side

N_BOOTS        = 10000
CI_LO, CI_HI   = 2.5, 97.5
RNG_SEED        = 42
MIN_MUTS        = 500

OUTDIR = Path(f"{BASE}/zdna_centred_refined_tol10_flank{FLANK}")
OUTDIR.mkdir(parents=True, exist_ok=True)

DUAL_ANCHOR_STRATA = {
    "DEL_gt50bp",
    "CDEL_gt50bp",
    "SUB_50_200",
    "SUB_gt200",
}

def assign_stratum(row):
    sc  = row["struct_class"]
    dom = row["dominant_op"]
    L   = int(row["event_size"])

    if sc == "snp":
        return "SNP"

    if sc == "mnp":
        return "MNP_2_20"

    if sc == "substitution":
        if L <= 50:   return "SUB_20_50"
        elif L <= 200: return "SUB_50_200"
        else:          return "SUB_gt200"

    if sc == "del":
        if L == 1:           return "DEL_1bp"
        elif L <= 5:         return "DEL_2_5bp"
        elif L <= 50:        return "DEL_6_50bp"
        else:                return "DEL_gt50bp"

    if sc == "ins":
        if L == 1:           return "INS_1bp"
        elif L <= 5:         return "INS_2_5bp"
        elif L <= 50:        return "INS_6_50bp"
        elif L <= 200:       return "INS_51_200bp"
        else:                return "INS_gt200bp"

    if sc == "complex" and dom == "del_like":
        if L <= 5:   return "CDEL_1_5bp"
        elif L <= 50: return "CDEL_6_50bp"
        else:         return "CDEL_gt50bp"

    if sc == "complex" and dom == "ins_like":
        if L <= 5:   return "CINS_1_5bp"
        elif L <= 50: return "CINS_6_50bp"
        elif L <= 200: return "CINS_51_200bp"
        else:          return "CINS_gt200bp"

    return None

SNP_TYPES = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]
COMP = {"A": "T", "T": "A", "C": "G", "G": "C"}

def normalize_snp(ref, alt):
    ref, alt = ref.upper(), alt.upper()
    if ref in ("A", "G"):
        ref = COMP[ref]
        alt = COMP[alt]
    return f"{ref}>{alt}"

def count_flanks_and_body(intervals, mut_positions, chrom_len, flank, n_body_bins):
    left_counts  = np.zeros(flank,        dtype=np.int64)
    body_counts  = np.zeros(n_body_bins,  dtype=np.int64)
    right_counts = np.zeros(flank,        dtype=np.int64)

    if len(mut_positions) == 0 or len(intervals) == 0:
        return left_counts, body_counts, right_counts

    for start, end in intervals:
        start, end = int(start), int(end)
        iv_len = end - start
        if iv_len <= 0:
            continue

        lf_s = max(0, start - flank)
        lf_e = start
        if lf_s < lf_e:
            lo = np.searchsorted(mut_positions, lf_s, side="left")
            hi = np.searchsorted(mut_positions, lf_e, side="left")
            for m in range(lo, hi):
                b = int(mut_positions[m]) - (start - flank)
                if 0 <= b < flank:
                    left_counts[b] += 1

        lo = np.searchsorted(mut_positions, start, side="left")
        hi = np.searchsorted(mut_positions, end,   side="left")
        for m in range(lo, hi):
            frac = (int(mut_positions[m]) - start) / iv_len
            b = min(int(frac * n_body_bins), n_body_bins - 1)
            body_counts[b] += 1

        rf_s = end
        rf_e = min(chrom_len, end + flank)
        if rf_s < rf_e:
            lo = np.searchsorted(mut_positions, rf_s, side="left")
            hi = np.searchsorted(mut_positions, rf_e, side="left")
            for m in range(lo, hi):
                b = int(mut_positions[m]) - end
                if 0 <= b < flank:
                    right_counts[b] += 1

    return left_counts, body_counts, right_counts

def chrom_block_bootstrap_frequency(counts_by_chr, n_intervals_by_chr, n_boot):
    chroms = list(counts_by_chr)
    if len(chroms) < 2:
        return None, None

    count_mat = np.stack([counts_by_chr[c].astype(np.float64) for c in chroms])
    n_vec     = np.array([n_intervals_by_chr[c] for c in chroms], dtype=np.float64)

    rng   = np.random.default_rng(RNG_SEED)
    boots = np.empty((n_boot, count_mat.shape[1]), dtype=np.float64)

    for b in range(n_boot):
        idx           = rng.integers(0, len(chroms), len(chroms))
        sum_counts    = count_mat[idx].sum(axis=0)
        sum_n         = n_vec[idx].sum()
        boots[b]      = sum_counts / sum_n if sum_n > 0 else np.full(count_mat.shape[1], np.nan)

    return (
        np.nanpercentile(boots, CI_LO, axis=0),
        np.nanpercentile(boots, CI_HI, axis=0),
    )

def run_stratum(label, anchors_by_chr, z_intervals, c_intervals,
                chrom_len_map, flank, n_body_bins, n_boot):
    window_len = flank + n_body_bins + flank
    accum = {
        track: {
            "total":    np.zeros(window_len, dtype=np.int64),
            "by_chr":   {},
            "n_by_chr": {},
            "n_total":  0,
        }
        for track in ("Z", "C")
    }

    for chrom in sorted(set(list(z_intervals) + list(c_intervals))):
        cl = chrom_len_map.get(chrom)
        if cl is None:
            continue
        mut_pos = anchors_by_chr.get(chrom, np.array([], dtype=np.int64))

        for track, iv_dict in [("Z", z_intervals), ("C", c_intervals)]:
            ivs = iv_dict.get(chrom)
            if ivs is None or len(ivs) == 0:
                continue
            lc, bc, rc = count_flanks_and_body(ivs, mut_pos, int(cl), flank, n_body_bins)
            combined = np.concatenate([lc, bc, rc])
            accum[track]["total"]           += combined
            accum[track]["by_chr"][chrom]    = combined
            accum[track]["n_by_chr"][chrom]  = len(ivs)
            accum[track]["n_total"]         += len(ivs)

    rows = []
    for track in ("Z", "C"):
        total = accum[track]["total"].astype(np.float64)
        n_tot = accum[track]["n_total"]
        if n_tot == 0:
            print(f"  [{label}] [{track}] No intervals — skipping")
            return False

        freq   = total / n_tot
        wmean  = total.mean()
        rel    = total / wmean if wmean > 0 else np.full(window_len, np.nan)

        ci_lo, ci_hi = chrom_block_bootstrap_frequency(
            accum[track]["by_chr"], accum[track]["n_by_chr"], n_boot
        )
        if ci_lo is None:
            print(f"  [{label}] [{track}] Bootstrap failed — skipping")
            return False

        for i in range(window_len):
            if i < flank:
                region    = "left_flank"
                pos_label = i - flank
            elif i < flank + n_body_bins:
                region    = "body"
                pos_label = i - flank
            else:
                region    = "right_flank"
                pos_label = i - flank - n_body_bins

            rows.append({
                "stratum":     label,
                "track":       track,
                "region":      region,
                "position":    pos_label,
                "window_idx":  i,
                "raw_count":   int(total[i]),
                "frequency":   freq[i],
                "rel_density": rel[i],
                "ci_lo":       ci_lo[i],
                "ci_hi":       ci_hi[i],
                "n_intervals": n_tot,
            })
    
    chrom_rows = []
    
    for track in ("Z", "C"):
        for chrom, arr in accum[track]["by_chr"].items():
    
            n_intervals_chr = accum[track]["n_by_chr"][chrom]
    
            for i, v in enumerate(arr):
                chrom_rows.append({
                    "track": track,
                    "chrom": chrom,
                    "window_idx": i,
                    "raw_count": int(v),
                    "n_intervals": int(n_intervals_chr),
                })
    
    chrom_df = pd.DataFrame(chrom_rows)
    
    chrom_df.to_csv(
        OUTDIR / f"zdna_centred_{label}_bychrom.tsv",
        sep="\t",
        index=False,
    )

    out_df   = pd.DataFrame(rows)
    tsv_path = OUTDIR / f"zdna_centred_{label}.tsv"
    out_df.to_csv(tsv_path, sep="\t", index=False)
    print(f"  [SAVED] {tsv_path.name}")
    return True

print("[INFO] Loading chromosome lengths...")
chrom_len_map = dict(pd.read_csv(
    CHROM_LEN_TSV, sep="\t",
    usecols=["ucsc_pansn_name", "gcf_seq_length"],
).values)

print("[INFO] Loading Z-DNA / control intervals...")
m = pd.read_csv(MASTER, sep="\t")
m = m[(m["tol_level"] == TOL) & (m["has_control"])].drop_duplicates(
    ["Chromosome", "Start0", "End0"]
)

z_intervals = {c: d[["Start0",    "End0"      ]].values.astype(np.int64) for c, d in m.groupby("Chromosome")}
c_intervals = {c: d[["Ctrl_Start","Ctrl_End"  ]].values.astype(np.int64) for c, d in m.groupby("Chromosome")}

all_z_lens  = np.concatenate([iv[:, 1] - iv[:, 0] for iv in z_intervals.values()])
all_c_lens  = np.concatenate([iv[:, 1] - iv[:, 0] for iv in c_intervals.values()])
N_BODY_BINS = int(min(all_z_lens[all_z_lens > 0].min(),
                      all_c_lens[all_c_lens > 0].min()))
WINDOW_LEN  = FLANK + N_BODY_BINS + FLANK

print(f"  N_BODY_BINS = {N_BODY_BINS}  |  WINDOW_LEN = {WINDOW_LEN}")
print(f"  Z-DNA intervals: {sum(len(v) for v in z_intervals.values()):,}")
print(f"  Control intervals: {sum(len(v) for v in c_intervals.values()):,}")

print("\n[INFO] Loading mutations...")
mut = pd.read_csv(
    MUT_BED, sep="\t", header=None,
    names=["Chromosome", "Start", "End", "mut_id", "score", "strand"],
)
meta = pd.read_csv(MUT_META, sep="\t", compression="gzip")
need = ["mut_id", "struct_class", "dominant_op", "ref_span", "alt_span"]
mut  = mut.merge(meta[need], on="mut_id", how="left").dropna()

mut["Start"]    = mut["Start"].astype(np.int64)
mut["ref_span"] = mut["ref_span"].astype(np.int64)
mut["alt_span"] = mut["alt_span"].astype(np.int64)

mut["event_size"] = np.where(
    (mut["struct_class"] == "ins") |
    ((mut["struct_class"] == "complex") & (mut["dominant_op"] == "ins_like")),
    mut["alt_span"],
    mut["ref_span"],
).astype(np.int64)

mut["stratum"] = mut.apply(assign_stratum, axis=1)
mut = mut.dropna(subset=["stratum"]).copy()
print(f"  Total mutations after stratification: {len(mut):,}")

print(f"\n{'='*70}")
print(f"  FLANK={FLANK}  N_BODY_BINS={N_BODY_BINS}  N_BOOTS={N_BOOTS}")
print(f"{'='*70}\n")

for stratum, sub in mut.groupby("stratum", sort=True):

    if len(sub) < MIN_MUTS:
        print(f"[SKIP] {stratum}: only {len(sub)} mutations (< {MIN_MUTS})")
        continue

    print(f"\n[STRATUM] {stratum}  ({len(sub):,} mutations)")

    anchors_by_chr = {}
    for chrom, g in sub.groupby("Chromosome"):
        anchors = g["Start"].values.astype(np.int64)  # left anchor always
        anchors_by_chr[chrom] = np.sort(anchors)

    run_stratum(
        label           = stratum,
        anchors_by_chr  = anchors_by_chr,
        z_intervals     = z_intervals,
        c_intervals     = c_intervals,
        chrom_len_map   = chrom_len_map,
        flank           = FLANK,
        n_body_bins     = N_BODY_BINS,
        n_boot          = N_BOOTS,
    )

print(f"\n{'='*70}")
print("  SNP sub-stratification by substitution type")
print(f"{'='*70}\n")

snp_meta = meta[meta["mut_id"].isin(mut[mut["stratum"] == "SNP"]["mut_id"])][
    ["mut_id", "reference", "variant"]
]
snp = mut[mut["stratum"] == "SNP"].copy().merge(snp_meta, on="mut_id", how="left")
snp["reference"] = snp["reference"].str.upper()
snp["variant"]   = snp["variant"].str.upper()
valid = {"A", "C", "G", "T"}
snp   = snp[snp["reference"].isin(valid) & snp["variant"].isin(valid)].copy()
snp["snp_type"] = [normalize_snp(r, a) for r, a in zip(snp["reference"], snp["variant"])]

for snp_type in SNP_TYPES:
    sub = snp[snp["snp_type"] == snp_type]

    if len(sub) < MIN_MUTS:
        print(f"[SKIP] SNP {snp_type}: only {len(sub)} mutations")
        continue

    print(f"\n[SNP TYPE] {snp_type}  ({len(sub):,} mutations)")

    anchors_by_chr = {}
    for chrom, g in sub.groupby("Chromosome"):
        anchors_by_chr[chrom] = np.sort(g["Start"].values.astype(np.int64))

    label = "SNP_" + snp_type.replace(">", "to")

    run_stratum(
        label           = label,
        anchors_by_chr  = anchors_by_chr,
        z_intervals     = z_intervals,
        c_intervals     = c_intervals,
        chrom_len_map   = chrom_len_map,
        flank           = FLANK,
        n_body_bins     = N_BODY_BINS,
        n_boot          = N_BOOTS,
    )

print(f"\n[DONE] All strata computed.  Output directory: {OUTDIR}")

In [ ]:
# Plot the Z-DNA-centred mutation metaprofiles.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.ticker import MaxNLocator
from matplotlib.lines import Line2D

BASE = "data"
FLANK_COMPUTED = 100     # must match FLANK used in the compute script

INDIR  = Path(f"{BASE}/zdna_centred_refined_tol10_flank{FLANK_COMPUTED}")
OUTDIR = INDIR           # write PNGs alongside TSVs (or change to a separate path)

DISPLAY_FLANK = 50       # how many bp to show on each flank in the plot

METRIC   = "frequency"   # "frequency" or "rel_density"
SHOW_CI  = True          # only meaningful for "frequency"

FIGSIZE     = (14, 6)
Z_COLOR     = "tab:blue"
C_COLOR     = "tab:gray"
CI_ALPHA    = 0.15
LINE_WIDTH  = 3.2
BODY_MARKER = "."
BODY_MSIZE  = 4
DPI         = 600
BODY_BG     = "#f0f0f0"

LABEL_FONT  = 22
TITLE_FONT  = 26
TICK_FONT   = 20

TITLE_MAP = {
    "SNP":          "Single-nucleotide variants (all)",
    "MNP_2_20":     "Multi-nucleotide variants (2–20 bp)",

    "SUB_20_50":    "Substitutions (20–50 bp)",
    "SUB_50_200":   "Substitutions (50–200 bp)",
    "SUB_gt200":    "Substitutions (> 200 bp)",

    "DEL_1bp":      "1 bp deletions",
    "DEL_2_5bp":    "2–5 bp deletions",
    "DEL_6_50bp":   "6–50 bp deletions",
    "DEL_gt50bp":   "> 50 bp deletions",

    "INS_1bp":      "1 bp insertions",
    "INS_2_5bp":    "2–5 bp insertions",
    "INS_6_50bp":   "6–50 bp insertions",
    "INS_51_200bp": "51–200 bp insertions",
    "INS_gt200bp":  "> 200 bp insertions",

    "CDEL_1_5bp":   "Complex deletion-like (1–5 bp)",
    "CDEL_6_50bp":  "Complex deletion-like (6–50 bp)",
    "CDEL_gt50bp":  "Complex deletion-like (> 50 bp)",

    "CINS_1_5bp":   "Complex insertion-like (1–5 bp)",
    "CINS_6_50bp":  "Complex insertion-like (6–50 bp)",
    "CINS_51_200bp":"Complex insertion-like (51–200 bp)",
    "CINS_gt200bp": "Complex insertion-like (> 200 bp)",

    "SNP_CtoA":     "C>A  /  G>T  substitutions",
    "SNP_CtoG":     "C>G  /  G>C  substitutions",
    "SNP_CtoT":     "C>T  /  G>A  substitutions",
    "SNP_TtoA":     "T>A  /  A>T  substitutions",
    "SNP_TtoC":     "T>C  /  A>G  substitutions",
    "SNP_TtoG":     "T>G  /  A>C  substitutions",
}

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def plot_tsv(tsv_path: Path, outdir: Path):
    df = pd.read_csv(tsv_path, sep="\t")
    stratum = df["stratum"].iloc[0]

    title = TITLE_MAP.get(stratum, stratum)

    z_track     = df[df["track"] == "Z"].sort_values("window_idx")
    n_body_bins = len(z_track[z_track["region"] == "body"])
    total_bins  = len(z_track)
    orig_flank  = (total_bins - n_body_bins) // 2

    if DISPLAY_FLANK > orig_flank:
        raise ValueError(
            f"DISPLAY_FLANK={DISPLAY_FLANK} > orig_flank={orig_flank} "
            f"in {tsv_path.name}"
        )

    fig, axes = plt.subplots(
        1, 3,
        figsize=FIGSIZE,
        sharey=True,
        gridspec_kw={"width_ratios": [DISPLAY_FLANK, n_body_bins, DISPLAY_FLANK]},
    )

    n_z = df[(df["track"] == "Z") & (df["region"] == "body")]["n_intervals"].iloc[0]
    n_c = df[(df["track"] == "C") & (df["region"] == "body")]["n_intervals"].iloc[0]
    n_mut = int(df[(df["track"] == "Z")]["raw_count"].sum())

    for track, color in [("Z", Z_COLOR), ("C", C_COLOR)]:
        td    = df[df["track"] == track].sort_values("window_idx")
        y     = td[METRIC].values
        ci_lo = td["ci_lo"].values
        ci_hi = td["ci_hi"].values

        sl = slice(orig_flank - DISPLAY_FLANK, orig_flank)
        x_left = np.arange(-DISPLAY_FLANK, 0)
        axes[0].plot(x_left, y[sl], color=color, lw=LINE_WIDTH)
        if SHOW_CI:
            axes[0].fill_between(x_left, ci_lo[sl], ci_hi[sl],
                                 color=color, alpha=CI_ALPHA)

        sb     = slice(orig_flank, orig_flank + n_body_bins)
        x_body = np.arange(1, n_body_bins + 1)
        axes[1].plot(x_body, y[sb], color=color, lw=LINE_WIDTH,
                     marker=BODY_MARKER, markersize=BODY_MSIZE)
        if SHOW_CI:
            axes[1].fill_between(x_body, ci_lo[sb], ci_hi[sb],
                                 color=color, alpha=CI_ALPHA)

        sr      = slice(orig_flank + n_body_bins,
                        orig_flank + n_body_bins + DISPLAY_FLANK)
        x_right = np.arange(1, DISPLAY_FLANK + 1)
        axes[2].plot(x_right, y[sr], color=color, lw=LINE_WIDTH)
        if SHOW_CI:
            axes[2].fill_between(x_right, ci_lo[sr], ci_hi[sr],
                                 color=color, alpha=CI_ALPHA)

    if METRIC == "frequency":
        ylabel = "Mutation frequency\n(per interval per bp-bin)"
    else:
        ylabel = "Relative density"

    axes[0].set_xlabel("Upstream (bp)",  fontsize=LABEL_FONT)
    axes[0].set_ylabel(ylabel,           fontsize=LABEL_FONT)
    axes[0].set_xlim(-DISPLAY_FLANK, 0)
    axes[0].set_xticks([-DISPLAY_FLANK, 0])
    axes[0].tick_params(labelsize=TICK_FONT)
    axes[0].yaxis.set_major_locator(MaxNLocator(nbins=4))
    despine(axes[0])

    axes[1].set_xlabel("Z-DNA / Control body (binned)",   fontsize=LABEL_FONT, labelpad=28)
    axes[1].set_facecolor(BODY_BG)
    axes[1].tick_params(labelsize=TICK_FONT)
    axes[1].set_xticks([1, n_body_bins])
    despine(axes[1])

    axes[2].set_xlabel("Downstream (bp)", fontsize=LABEL_FONT)
    axes[2].set_xlim(0, DISPLAY_FLANK)
    axes[2].set_xticks([0, DISPLAY_FLANK])
    axes[2].tick_params(labelsize=TICK_FONT)

    fig.suptitle(
        title,
        fontsize=TITLE_FONT, fontweight="bold", y=1.03,
    )

    plt.tight_layout()

    png_path = outdir / f"zdna_centred_{stratum}.png"
    fig.savefig(png_path, dpi=DPI, bbox_inches="tight")
    plt.show(fig)

    for track, label in [("Z", "Z"), ("C", "C")]:
        td = df[df["track"] == track].sort_values("window_idx")
        y  = td[METRIC].values

        left_slice  = slice(orig_flank - 5, orig_flank)
        body_slice  = slice(orig_flank, orig_flank + n_body_bins)
        right_slice = slice(orig_flank + n_body_bins,
                            orig_flank + n_body_bins + 5)

        vals = np.concatenate([y[left_slice], y[body_slice], y[right_slice]])
        if label == "Z":
            z_win = vals.mean()
        else:
            c_win = vals.mean()

    ratio = z_win / c_win if c_win > 0 else np.nan
    print(
        f"[±5bp WINDOW] {stratum:15s} | "
        f"Z = {z_win:.6f} | C = {c_win:.6f} | "
        f"Z/C = {ratio:.3f}"
    )

if not INDIR.exists():
    raise FileNotFoundError(f"Input directory not found: {INDIR}")

tsv_files = sorted(INDIR.glob("zdna_centred_*.tsv"))
if not tsv_files:
    raise FileNotFoundError(f"No TSV files found in {INDIR}")

OUTDIR.mkdir(parents=True, exist_ok=True)

for tsv_path in tsv_files:
    stratum = tsv_path.stem.replace("zdna_centred_", "")
    if stratum not in TITLE_MAP:
        print(f"[SKIP] {tsv_path.name}  (not in TITLE_MAP)")
        continue
    plot_tsv(tsv_path, OUTDIR)

leg_fig = plt.figure(figsize=(3.0, 1.8))
ax_leg  = leg_fig.add_subplot(111)
ax_leg.axis("off")
ax_leg.legend(
    handles=[
        Line2D([0], [0], lw=LINE_WIDTH, color=Z_COLOR, label="Z-DNA"),
        Line2D([0], [0], lw=LINE_WIDTH, color=C_COLOR, label="Control"),
    ],
    loc="center", frameon=False, fontsize=13,
)
leg_path = OUTDIR / "zdna_centred_legend.png"
leg_fig.savefig(leg_path, dpi=DPI, bbox_inches="tight")
plt.show(leg_fig)
print(f"[OK] {leg_path.name}")

print(f"\n[DONE] All plots written to {OUTDIR}")

In [ ]:
# Summary panel: smoothed log2(Z/control) mutation enrichment across the Z-DNA body.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
from pathlib import Path
from scipy.ndimage import gaussian_filter1d

BASE = "data"

FLANK_COMPUTED = 100
INDIR  = Path(f"{BASE}/zdna_centred_refined_tol10_flank{FLANK_COMPUTED}")
OUTDIR = INDIR

DISPLAY_FLANK = 50

STRATA_TO_PLOT = [
    ("SNP",         "SNV",                       "#0072B2"),
    ("MNP_2_20",   "MNV (2–20 bp)",    "#6A3D9A"),
    ("DEL_6_50bp",  "Deletion 6-50 bp",         "#324F33"),
    ("INS_6_50bp",  "Insertion 6-50 bp",        "#009E73"),
    ("CDEL_6_50bp", "Complex deletion 6-50 bp", "#A89472"),
    ("CINS_6_50bp", "Complex insertion 6-50 bp","#D55E00"),
    
]

LABEL_FONT = 20
TICK_FONT  = 16
LEG_FONT   = 16

LINE_WIDTH = 3.0

BODY_BG = "#e9e9e9"

GAUSS_SIGMA = 1.2

def load_stratum(stratum: str):

    path = INDIR / f"zdna_centred_{stratum}.tsv"

    if not path.exists():
        return None

    df = pd.read_csv(path, sep="\t")

    z = (
        df[df["track"] == "Z"]
        .sort_values("window_idx")
        .reset_index(drop=True)
    )

    c = (
        df[df["track"] == "C"]
        .sort_values("window_idx")
        .reset_index(drop=True)
    )

    z_freq = z["frequency"].to_numpy()
    c_freq = c["frequency"].to_numpy()

    with np.errstate(divide="ignore", invalid="ignore"):

        log2_ratio = np.where(
            (z_freq > 0) & (c_freq > 0),
            np.log2(z_freq / c_freq),
            np.nan,
        )

    n_body = (z["region"] == "body").sum()
    orig_flank = (len(z) - n_body) // 2

    return log2_ratio, orig_flank, n_body

def smooth(arr, sigma):

    arr = np.asarray(arr, dtype=np.float64)

    mask = np.isnan(arr)

    filled = arr.copy()

    if mask.any():
        filled[mask] = np.nanmean(arr)

    smoothed = gaussian_filter1d(
        filled,
        sigma=sigma,
        mode="nearest",
    )

    smoothed[mask] = np.nan

    return smoothed

loaded = []

for stratum, label, color in STRATA_TO_PLOT:

    res = load_stratum(stratum)

    if res is None:
        print(f"[WARN] missing TSV for {stratum}")
        continue

    log2_ratio, orig_flank, n_body = res

    loaded.append(
        (
            stratum,
            label,
            color,
            log2_ratio,
            orig_flank,
            n_body,
        )
    )

if not loaded:
    raise RuntimeError("No strata loaded.")

ref_n_body = loaded[0][5]
ref_flank  = loaded[0][4]

fig = plt.figure(figsize=(18, 5))

gs = GridSpec(
    1,
    3,
    width_ratios=[5, 1.6, 5],
    wspace=0.0,
)

axL = fig.add_subplot(gs[0, 0])
axM = fig.add_subplot(gs[0, 1], sharey=axL)
axR = fig.add_subplot(gs[0, 2], sharey=axL)

axes = [axL, axM, axR]

x_left = np.linspace(-DISPLAY_FLANK, 0, DISPLAY_FLANK + 1)

x_body = np.linspace(0, ref_n_body - 1, ref_n_body)

x_right = np.linspace(ref_n_body - 1, ref_n_body - 1 + DISPLAY_FLANK,
                      DISPLAY_FLANK + 1)

sl_left = slice(
    ref_flank - DISPLAY_FLANK,
    ref_flank + 1,
)

sl_body = slice(
    ref_flank,
    ref_flank + ref_n_body,
)

sl_right = slice(
    ref_flank + ref_n_body - 1,
    ref_flank + ref_n_body + DISPLAY_FLANK,
)

for stratum, label, color, log2_ratio, _, _ in loaded:

    y = smooth(log2_ratio, GAUSS_SIGMA)

    axL.plot(
        x_left,
        y[sl_left],
        color=color,
        lw=LINE_WIDTH,
        alpha=0.98,
        zorder=5,
        solid_capstyle="round",
        clip_on=False,
    )

    axM.plot(
        x_body,
        y[sl_body],
        color=color,
        lw=LINE_WIDTH,
        alpha=0.98,
        zorder=10,
        solid_capstyle="round",
        clip_on=False,
    )

    axR.plot(
        x_right,
        y[sl_right],
        color=color,
        lw=LINE_WIDTH,
        alpha=0.98,
        zorder=5,
        solid_capstyle="round",
        clip_on=False,
    )

for ax in axes:

    ax.axhline(
        0,
        ls=":",
        color="black",
        lw=1.2,
        alpha=0.7,
    )

    ax.tick_params(
        labelsize=TICK_FONT,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axM.spines["left"].set_visible(False)
axR.spines["left"].set_visible(False)

axM.tick_params(axis="y", left=False, labelleft=False)
axR.tick_params(axis="y", left=False, labelleft=False)

axM.set_facecolor(BODY_BG)
axM.set_zorder(0)

axL.set_xlim(-DISPLAY_FLANK, 0)
axR.set_xlim(ref_n_body - 1, ref_n_body - 1 + DISPLAY_FLANK)

axL.set_xticks([-50])
axR.set_xticks([ref_n_body - 1 + DISPLAY_FLANK])

axM.set_xticks([0, ref_n_body - 1])
axM.set_xticklabels(["1", str(ref_n_body)])

axL.set_ylabel(
    "log\u2082(Z / C)\nmutation frequency",
    fontsize=LABEL_FONT,
)

axL.set_xlabel(
    "Upstream (bp)",
    fontsize=LABEL_FONT,
)

axM.set_xlabel(
    "Z-DNA / Control body (binned)",
    fontsize=LABEL_FONT,
    labelpad=18,
)

axR.set_xlabel(
    "Downstream (bp)",
    fontsize=LABEL_FONT,
)

axL.set_xlim(-DISPLAY_FLANK, 0)
axR.set_xlim(ref_n_body - 1, ref_n_body - 1 + DISPLAY_FLANK)

axL.set_xticks([-50])
axR.set_xticks([ref_n_body - 1 + DISPLAY_FLANK])

axM.set_xticks([0, ref_n_body - 1])
axM.set_xticklabels(["1", str(ref_n_body)])

handles = [
    Line2D(
        [0],
        [0],
        color=c,
        lw=LINE_WIDTH,
        label=lbl,
    )
    for _, lbl, c, *_ in loaded
]

axR.legend(
    handles=handles,
    loc="upper right",
    frameon=False,
    fontsize=LEG_FONT,
)

plt.tight_layout()
out_png = OUTDIR / "zdna_centred_summary_log2_kde.png"
fig.savefig(
    out_png,
    dpi=600,
    bbox_inches="tight",
)
plt.show()
print(f"[SAVED] {out_png}")

In [ ]:
# Mutation enrichment summary statistics around Z-DNA.
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon
def chromosome_body_test(
    bychrom_path,
    n_flank,
    n_body,
    n_boots=50000,
    seed=42,
):

    df = pd.read_csv(bychrom_path, sep="\t")

    z_freqs = []
    c_freqs = []

    chroms = sorted(set(df["chrom"]))

    for chrom in chroms:

        z = df[
            (df["track"] == "Z")
            & (df["chrom"] == chrom)
        ].sort_values("window_idx")

        c = df[
            (df["track"] == "C")
            & (df["chrom"] == chrom)
        ].sort_values("window_idx")

        if len(z) == 0 or len(c) == 0:
            continue

        z_body = z["raw_count"].to_numpy()[n_flank:n_flank+n_body]
        c_body = c["raw_count"].to_numpy()[n_flank:n_flank+n_body]

        z_n = z["n_intervals"].iloc[0]
        c_n = c["n_intervals"].iloc[0]

        if z_n == 0 or c_n == 0:
            continue

        z_freqs.append(z_body.mean() / z_n)
        c_freqs.append(c_body.mean() / c_n)

    z_freqs = np.array(z_freqs)
    c_freqs = np.array(c_freqs)

    n = len(z_freqs)

    if n < 6:
        return np.nan, np.nan, np.nan, np.nan

    try:
        _, p_wilc = wilcoxon(
            z_freqs,
            c_freqs,
            alternative="greater",
            zero_method="wilcox",
        )
    except ValueError:
        p_wilc = np.nan

    rng = np.random.default_rng(seed)

    idx = rng.integers(
        0,
        n,
        size=(n_boots, n)
    )

    zb = z_freqs[idx].mean(axis=1)
    cb = c_freqs[idx].mean(axis=1)

    valid = (zb > 0) & (cb > 0)

    log2_boot = np.log2(zb[valid] / cb[valid])

    ci_lo = 2 ** np.percentile(log2_boot, 2.5)
    ci_hi = 2 ** np.percentile(log2_boot, 97.5)

    fold = z_freqs.mean() / c_freqs.mean()

    return fold, ci_lo, ci_hi, p_wilc

BASE = "data"
INDIR = Path(f"{BASE}/zdna_centred_refined_tol10_flank100")

STRATA = ["SNP", "MNP_2_20", "DEL_6_50bp", "INS_6_50bp",
          "CDEL_6_50bp", "CINS_6_50bp"]

rows = []
for stratum in STRATA:
    df = pd.read_csv(INDIR / f"zdna_centred_{stratum}.tsv", sep="\t")
    z = df[df["track"] == "Z"].sort_values("window_idx").reset_index(drop=True)
    c = df[df["track"] == "C"].sort_values("window_idx").reset_index(drop=True)
    z_freq = z["frequency"].to_numpy()
    c_freq = c["frequency"].to_numpy()
    n_body = (z["region"] == "body").sum()
    n_flank = (z["region"] == "left_flank").sum()
    
    body_z = z_freq[n_flank:n_flank + n_body]
    body_c = c_freq[n_flank:n_flank + n_body]
    
    body_mean_z = body_z.mean()
    body_mean_c = body_c.mean()
    
    bychrom_path = (
        INDIR /
        f"zdna_centred_{stratum}_bychrom.tsv"
    )
    
    fold_body, fold_lo, fold_hi, w_p = chromosome_body_test(
        bychrom_path,
        n_flank,
        n_body,
    )
    third = max(1, n_body // 3)
    boundary_bins = np.r_[np.arange(third), np.arange(n_body - third, n_body)]
    interior_bins = np.arange(third, n_body - third)
    boundary_z = body_z[boundary_bins].mean()
    interior_z = body_z[interior_bins].mean()
    boundary_c = body_c[boundary_bins].mean()
    interior_c = body_c[interior_bins].mean()
    log2_boundary_vs_interior_Z = (np.log2(boundary_z / interior_z)
                                    if (boundary_z > 0 and interior_z > 0) else np.nan)
    log2_boundary_vs_interior_C = (np.log2(boundary_c / interior_c)
                                    if (boundary_c > 0 and interior_c > 0) else np.nan)

    left_flank_freq = z_freq[:n_flank]
    right_flank_freq = z_freq[n_flank + n_body:]
    left_flank_c = c_freq[:n_flank]
    right_flank_c = c_freq[n_flank + n_body:]

    left_log2 = np.log2(left_flank_freq / left_flank_c,
                         where=(left_flank_freq > 0) & (left_flank_c > 0),
                         out=np.full_like(left_flank_freq, np.nan))
    right_log2 = np.log2(right_flank_freq / right_flank_c,
                          where=(right_flank_freq > 0) & (right_flank_c > 0),
                          out=np.full_like(right_flank_freq, np.nan))

    near_left  = np.nanmean(left_log2[-5:])
    distal_left = np.nanmean(left_log2[:5])
    near_right  = np.nanmean(right_log2[:5])
    distal_right = np.nanmean(right_log2[-5:])

    def decay_distance(log2_arr, from_idx_zero_at_body):
        idx = np.where(log2_arr < 0.5)[0]
        if len(idx) == 0:
            return np.nan
        if from_idx_zero_at_body == "right":
            return idx[0] + 1
        else:
            return len(log2_arr) - idx[-1]

    decay_upstream   = decay_distance(left_log2[::-1], "right")
    decay_downstream = decay_distance(right_log2,       "right")

    rows.append({
        "stratum": stratum,
        "n_intervals_Z": int(z["n_intervals"].iloc[0]),
        "n_mutations_in_body": int(
            z.iloc[n_flank:n_flank+n_body]["raw_count"].sum()
        ),

        "fold_enrichment_body": fold_body,        
        "fold_enrichment_body_CI_lo": fold_lo,        
        "fold_enrichment_body_CI_hi": fold_hi,        
        "wilcoxon_body_p_onesided": w_p,
        "log2_boundary_vs_interior_Z":   log2_boundary_vs_interior_Z,
        "log2_boundary_vs_interior_C":   log2_boundary_vs_interior_C,
        "log2_boundary_vs_interior_diff": (log2_boundary_vs_interior_Z
                                            - log2_boundary_vs_interior_C),
        "log2_near_upstream":   near_left,
        "log2_distal_upstream": distal_left,
        "log2_near_downstream":   near_right,
        "log2_distal_downstream": distal_right,
        "decay_distance_upstream_bp":   decay_upstream,
        "decay_distance_downstream_bp": decay_downstream,
    })

tbl = pd.DataFrame(rows)
from statsmodels.stats.multitest import multipletests

mask = tbl["wilcoxon_body_p_onesided"].notna()

tbl.loc[mask, "wilcoxon_body_p_bh"] = multipletests(
    tbl.loc[mask, "wilcoxon_body_p_onesided"],
    method="fdr_bh",
)[1]
tbl["significant_bh"] = (
    tbl["wilcoxon_body_p_bh"] < 0.05
)
tbl.to_csv(INDIR / "zdna_centred_summary_metrics.tsv", sep="\t", index=False)
print(tbl.to_string(index=False))

## 11. De novo mutations (denovo-db, GRCh37)

Source: `denovo-db.gs.washington.edu`.

### 11.1 Parsing and metadata

In [ ]:
# Parse denovo-db VCF, extract sequence types and per-variant metadata.
from pathlib import Path
import gzip
import json
from collections import defaultdict
import pandas as pd
from tqdm import tqdm

VCF_PATH = Path(
    "data/denovo_db/"
    "denovo-db.non-ssc-samples.variants.vcf.gz"
)

OUT_DIR = Path(
    "data/denovo_db"
)

def get_size_bin(x: int) -> str:
    if x == 1:       return "1bp"
    elif x <= 5:     return "2_5bp"
    elif x <= 20:    return "6_20bp"
    elif x <= 50:    return "21_50bp"
    elif x <= 200:   return "51_200bp"
    else:            return "gt200bp"

def stream_vcf_first9(vcf_path: Path):
    opener = gzip.open if str(vcf_path).endswith(".gz") else open
    with opener(vcf_path, "rt") as f:
        for line in f:
            if line.startswith("#"):
                continue
            fields = line.split("\t", 9)
            yield {
                "#CHROM":  fields[0],
                "POS":     int(fields[1]),
                "ID":      fields[2],
                "REF":     fields[3],
                "ALT":     fields[4],
                "QUAL":    fields[5],
                "FILTER":  fields[6],
                "INFO":    fields[7],
            }

def extract_sequence_types(info: str):
    for field in info.split(";"):
        if field.startswith("STUDIES="):
            raw = field[len("STUDIES="):]

            try:
                studies = json.loads(raw)
            except json.JSONDecodeError:
                return []

            return [s.get("SequenceType") for s in studies if "SequenceType" in s]

    return []

def mutation_metadata(ref: str, alt: str) -> dict:
    ref_len = len(ref)
    alt_len = len(alt)

    if ref_len == alt_len == 1:
        struct = "snp"
    elif ref_len == alt_len:
        struct = "mnp" if ref_len <= 20 else "substitution"
    elif ref_len < alt_len:
        struct = "ins" if ref_len == 1 else "complex"
    elif ref_len > alt_len:
        struct = "del" if alt_len == 1 else "complex"
    else:
        raise RuntimeError("unreachable")

    if struct in ("complex", "substitution"):
        if ref_len > alt_len:   dominant = "del_like"
        elif alt_len > ref_len: dominant = "ins_like"
        else:                   dominant = "balanced"
    else:
        dominant = struct

    if struct in ("snp", "mnp", "substitution"):
        ref_span = ref_len
        alt_span = alt_len
    else:
        ref_span = ref_len - 1
        alt_span = alt_len - 1

    if struct in ("snp", "mnp", "substitution", "del") or dominant == "del_like":
        event_size = ref_span
    elif struct == "ins" or dominant == "ins_like":
        event_size = alt_span
    elif dominant == "balanced":
        event_size = ref_span
    else:
        event_size = max(ref_span, alt_span)

    size_bin = get_size_bin(event_size)

    if struct == "snp":
        anchor = "single_base"
    elif struct in ("mnp", "substitution"):
        anchor = "midpoint"
    elif dominant == "del_like":
        anchor = "left_breakpoint"
    else:
        anchor = "insertion_site"

    return {
        "struct_class": struct,
        "dominant_op":  dominant,
        "ref_span":     ref_span,
        "alt_span":     alt_span,
        "size_bin":     size_bin,
        "anchor_type":  anchor,
    }

COLUMNS = [
    "seqID", "start", "end", "mut_id",
    "struct_class", "dominant_op",
    "ref_span", "alt_span", "size_bin", "anchor_type",
    "mutation", "general_mut",
    "reference", "variant",
    "subvariants", "is_multiallelic",
    "AC", "AF", "AN", "NS", "LEN", "TYPE",
]

def explode_vcf(vcf_iter) -> pd.DataFrame:
    rows = []

    for row in tqdm(vcf_iter):
        info = row["INFO"]

        if "SequenceType\":\"genome\"" not in info:
            continue
        
        seq_types = extract_sequence_types(info)
        
        if not seq_types:
            continue  # safe given QC
        
        if "genome" not in seq_types:
            continue

        seqID     = row["#CHROM"]
        reference = row["REF"]
        alts      = row["ALT"]
        pos0      = row["POS"] - 1

        if not alts or alts == ".":
            continue

        alt_list   = alts.split(",")
        subvariants = len(alt_list)

        ref_len = len(reference)

        for alt in alt_list:
            meta   = mutation_metadata(reference, alt)
            start0 = pos0

            if meta["struct_class"] == "snp":
                center = start0
            elif meta["anchor_type"] == "midpoint":
                center = start0 + 1 + (ref_len - 1) // 2
            elif meta["anchor_type"] == "left_breakpoint":
                center = start0 + 1
            elif meta["anchor_type"] == "right_breakpoint":
                center = start0 + ref_len - 1
            elif meta["anchor_type"] == "insertion_site":
                center = start0 + 1
            else:
                raise RuntimeError("Unknown anchor type")

            start  = center
            end    = center + 1
            mut_id = f"{seqID}:{pos0+1}:{reference}:{alt}"

            AC = AF = AN = NS = LEN = TYPE = None

            rows.append((
                seqID, start, end, mut_id,
                meta["struct_class"], meta["dominant_op"],
                meta["ref_span"],     meta["alt_span"],
                meta["size_bin"],     meta["anchor_type"],
                f"{meta['struct_class']}_{meta['size_bin']}",
                meta["struct_class"],
                reference, alt,
                subvariants, subvariants > 1,
                AC, AF, AN, NS, LEN, TYPE,
            ))
    assert len(rows) > 0, "No variants passed filtering"
    return pd.DataFrame(rows, columns=COLUMNS)

if __name__ == "__main__":

    vcf_enriched = explode_vcf(stream_vcf_first9(VCF_PATH))

    output_exploded_path = OUT_DIR / "denovo-db.hg19.exploded.tsv.gz"
    vcf_enriched.to_csv(output_exploded_path, sep="\t", index=False, compression="gzip")
    print(f"Wrote: {output_exploded_path}")

    bed_df = pd.DataFrame({
        "chrom":  vcf_enriched["seqID"],
        "start":  vcf_enriched["start"].astype(int),
        "end":    vcf_enriched["end"].astype(int),
        "name":   vcf_enriched["mut_id"],
        "score":  0,
        "strand": ".",
    })
    bed_output_path = OUT_DIR / "denovodb.bed.gz"
    bed_df.to_csv(bed_output_path, sep="\t", header=False, index=False, compression="gzip")

    META_COLS = [
        "mut_id", "seqID", "start", "end",
        "struct_class", "dominant_op", "ref_span", "alt_span", "anchor_type", "size_bin",
        "mutation", "general_mut",
        "reference", "variant",
        "AC", "AF", "AN", "NS", "LEN", "TYPE",
        "subvariants", "is_multiallelic",
    ]
    meta_output_path = OUT_DIR / "denovodb.meta.tsv.gz"
    vcf_enriched[META_COLS].to_csv(meta_output_path, sep="\t", index=False, compression="gzip")

    threecol_df = pd.DataFrame({
        "chrom":    vcf_enriched["seqID"],
        "pos":      vcf_enriched["start"],
        "mutation": vcf_enriched["mutation"],
    })
    threecol_path = OUT_DIR / "denovo-db.hg19.mutations.3col.tsv.gz"
    threecol_df.to_csv(threecol_path, sep="\t", index=False, compression="gzip")

In [ ]:
# Assign Z-DNA score strata and normalise de novo variant profiles.
import pandas as pd
import numpy as np

META = "data/denovo_db/denovodb.meta.tsv.gz"

df = pd.read_csv(
    META,
    sep="\t",
    usecols=["mut_id","struct_class","dominant_op","ref_span","alt_span","reference","variant"]
)

df["event_size"] = np.where(
    (df["struct_class"]=="ins") |
    ((df["struct_class"]=="complex") & (df["dominant_op"]=="ins_like")),
    df["alt_span"],
    df["ref_span"]
)

def assign_stratum(r):
    sc, dom, L = r["struct_class"], r["dominant_op"], int(r["event_size"])

    if sc == "snp":
        return "SNP"

    if sc == "mnp":
        return "MNP_2_20"

    if sc == "del":
        if L==1: return "DEL_1bp"
        if L<=5: return "DEL_2_5bp"
        if L<=50: return "DEL_6_50bp"
        return "DEL_gt50bp"

    if sc == "ins":
        if L==1: return "INS_1bp"
        if L<=5: return "INS_2_5bp"
        if L<=50: return "INS_6_50bp"
        if L<=200: return "INS_51_200bp"
        return "INS_gt200bp"

    if sc=="complex" and dom=="del_like":
        if L<=5: return "CDEL_1_5bp"
        if L<=50: return "CDEL_6_50bp"
        return "CDEL_gt50bp"

    if sc=="complex" and dom=="ins_like":
        if L<=5: return "CINS_1_5bp"
        if L<=50: return "CINS_6_50bp"
        if L<=200: return "CINS_51_200bp"
        return "CINS_gt200bp"

    return None

df["stratum"] = df.apply(assign_stratum, axis=1)

print("\n===== STRATUM COUNTS =====")
print(df["stratum"].value_counts().sort_index())

comp = str.maketrans("ACGT","TGCA")

def norm(ref, alt):
    if ref in ("C","T"):
        return f"{ref}>{alt}"
    return f"{ref.translate(comp)}>{alt.translate(comp)}"

snp = df[df["stratum"]=="SNP"].copy()
snp["mut_class"] = [norm(r,a) for r,a in zip(snp["reference"], snp["variant"])]

print("\n===== SNP SUBCLASSES =====")
counts = snp["mut_class"].value_counts().reindex(
    ["C>A","C>G","C>T","T>A","T>C","T>G"]
)
print(counts)

print("\nFractions:")
print((counts / counts.sum()).round(4))

In [ ]:
# Block-bootstrap windowed de novo mutation density around Z-DNA vs controls.
import numpy as np
import pandas as pd
from pathlib import Path

BASE = "data/denovo_db"

MUT_BED  = f"{BASE}/denovodb.bed.gz"
MUT_META = f"{BASE}/denovodb.meta.tsv.gz"

Z_BED = f"{BASE}/grch37_zdna_tol10_has_control.bed"
C_BED = f"{BASE}/grch37_zdna_tol10_control.bed"

WINDOW_RADIUS = 500
N_BOOTS = 10000
CI_LOW, CI_HIGH = 2.5, 97.5
MIN_MUTS = 500

ANCHOR_MODE = "left"  # run twice manually (left/right)

def chrom_block_bootstrap_windowmean_norm(counts_by_chr, n_boot):
    chroms = list(counts_by_chr)
    mat = np.stack([counts_by_chr[c].astype(float) for c in chroms], axis=0)
    rng = np.random.default_rng(42)

    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(chroms), len(chroms))
        v = mat[idx].sum(axis=0)
        m = v.mean()
        boots.append(v / m if m else np.full_like(v, np.nan))

    boots = np.stack(boots, axis=0)
    return (
        np.nanpercentile(boots, CI_LOW, axis=0),
        np.nanpercentile(boots, CI_HIGH, axis=0),
    )

def build_bool_coverage(starts, ends, chrom_len):
    diff = np.zeros(chrom_len + 1, dtype=np.int32)
    np.add.at(diff, starts, 1)
    np.add.at(diff, ends, -1)
    return np.cumsum(diff[:-1]) > 0

def midpoint(start, span):
    return start + (span - 1) // 2

def right_bp(start, span):
    return start + span - 1

def load_bed(path):
    df = pd.read_csv(path, sep="\t", header=None, names=["chr","start","end"])
    return {
        c: d[["start","end"]].to_numpy(dtype=np.int64)
        for c, d in df.groupby("chr")
    }

z_intervals = load_bed(Z_BED)
c_intervals = load_bed(C_BED)

chrom_len = {}
for chrom in set(z_intervals) | set(c_intervals):
    max_end = 0
    if chrom in z_intervals:
        max_end = max(max_end, z_intervals[chrom][:,1].max())
    if chrom in c_intervals:
        max_end = max(max_end, c_intervals[chrom][:,1].max())
    chrom_len[chrom] = int(max_end + 1000)  # padding

mut = pd.read_csv(
    MUT_BED,
    sep="\t",
    header=None,
    names=["Chromosome","Start","End","mut_id","score","strand"],
    dtype={
        "Chromosome": "string",
        "Start": "int64",
        "End": "int64",
        "mut_id": "string",
        "score": "float64",
        "strand": "string",
    }
)
mut["Chromosome"] = "chr" + mut["Chromosome"].astype(str)
meta = pd.read_csv(
    MUT_META,
    sep="\t",
    usecols=["mut_id","struct_class","dominant_op","ref_span","alt_span"],
    dtype={
        "mut_id": "string",
        "struct_class": "string",
        "dominant_op": "string",
        "ref_span": "int64",
        "alt_span": "int64",
    }
)

mut = mut.merge(
    meta[["mut_id","struct_class","dominant_op","ref_span","alt_span"]],
    on="mut_id"
)

mut["event_size"] = np.where(
    (mut["struct_class"]=="ins") |
    ((mut["struct_class"]=="complex") & (mut["dominant_op"]=="ins_like")),
    mut["alt_span"],
    mut["ref_span"]
)

def assign_stratum(r):
    sc, dom, L = r["struct_class"], r["dominant_op"], int(r["event_size"])
    if sc == "snp":
        return "SNP"

    if sc == "mnp":
        return "MNP_2_20"

    if sc == "del":
        if L==1: return "DEL_1bp"
        if L<=5: return "DEL_2_5bp"
        if L<=50: return "DEL_6_50bp"
        return "DEL_gt50bp"

    if sc == "ins":
        if L==1: return "INS_1bp"
        if L<=5: return "INS_2_5bp"
        if L<=50: return "INS_6_50bp"
        if L<=200: return "INS_51_200bp"
        return "INS_gt200bp"

    if sc=="complex" and dom=="del_like":
        if L<=5: return "CDEL_1_5bp"
        if L<=50: return "CDEL_6_50bp"
        return "CDEL_gt50bp"

    if sc=="complex" and dom=="ins_like":
        if L<=5: return "CINS_1_5bp"
        if L<=50: return "CINS_6_50bp"
        if L<=200: return "CINS_51_200bp"
        return "CINS_gt200bp"

    return None

mut["stratum"] = mut.apply(assign_stratum, axis=1)
mut = mut.dropna(subset=["stratum"])

offsets = np.arange(-WINDOW_RADIUS, WINDOW_RADIUS+1)

summary = []

for stratum, sub in mut.groupby("stratum"):

    if len(sub) < MIN_MUTS:
        continue

    Z_by_chr, C_by_chr = {}, {}
    Z_total = np.zeros(len(offsets))
    C_total = np.zeros(len(offsets))

    for chrom, d in sub.groupby("Chromosome"):
        if chrom not in z_intervals or chrom not in c_intervals:
            continue

        L = chrom_len[chrom]

        starts = d["Start"].values.astype(int)
        spans  = d["ref_span"].values.astype(int)

        anchors = starts  # simplified (no dual-anchor split here)

        z_cov = build_bool_coverage(
            z_intervals[chrom][:,0],
            z_intervals[chrom][:,1],
            L
        )
        c_cov = build_bool_coverage(
            c_intervals[chrom][:,0],
            c_intervals[chrom][:,1],
            L
        )

        Zc = np.zeros(len(offsets))
        Cc = np.zeros(len(offsets))

        for i,k in enumerate(offsets):
            pos = anchors + k
            mask = (pos>=0) & (pos<L)
            pos = pos[mask]
            Zc[i] = z_cov[pos].sum()
            Cc[i] = c_cov[pos].sum()

        Z_total += Zc
        C_total += Cc

        Z_by_chr[chrom] = Zc
        C_by_chr[chrom] = Cc

    if not Z_by_chr:
        continue

    Z_mean = Z_total / Z_total.mean()
    C_mean = C_total / C_total.mean()

    Z_lo, Z_hi = chrom_block_bootstrap_windowmean_norm(Z_by_chr, N_BOOTS)
    C_lo, C_hi = chrom_block_bootstrap_windowmean_norm(C_by_chr, N_BOOTS)

    out = pd.DataFrame({
        "offset": offsets,
        "Z_mean": Z_mean,
        "Z_lo": Z_lo,
        "Z_hi": Z_hi,
        "C_mean": C_mean,
        "C_lo": C_lo,
        "C_hi": C_hi,
        "stratum": stratum,
    })

    out.to_csv(f"{BASE}/metaprofiles/profile_{stratum}_{ANCHOR_MODE}.tsv", sep="\t", index=False)

    summary.append({
        "stratum": stratum,
        "n_events": len(sub)
    })

pd.DataFrame(summary).to_csv(f"{BASE}/metaprofiles/profile_summary_{ANCHOR_MODE}.tsv", sep="\t", index=False)

print("[DONE]")

In [ ]:
# Plot de novo mutation metaprofiles.
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

BASE = Path("data/denovo_db/metaprofiles")

LABEL_FONTSIZE = 22
TICK_FONTSIZE  = 20
TITLE_FONTSIZE = 24

Z_COLOR = "tab:green"
C_COLOR = "tab:gray"

TITLE_MAP = {
    "MNP_2_20": "Multi-nucleotide variants (2–20 bp)",
    "SNP": "Single nucl. pol.",

    "DEL_1bp": "1 bp deletions",
    "DEL_2_5bp": "2–5 bp deletions",
    "DEL_6_50bp": "6–50 bp deletions",
    "DEL_gt50bp": ">50 bp deletions",

    "INS_1bp": "1 bp insertions",
    "INS_2_5bp": "2–5 bp insertions",
    "INS_6_50bp": "6–50 bp insertions",
    "INS_51_200bp": "51–200 bp insertions",
    "INS_gt200bp": ">200 bp insertions",

    "CDEL_1_5bp": "Complex deletion-like (1–5 bp)",
    "CDEL_6_50bp": "Complex deletion-like (6–50 bp)",
    "CDEL_gt50bp": "Complex deletion-like (>50 bp)",

    "CINS_1_5bp": "Complex insertion-like (1–5 bp)",
    "CINS_6_50bp": "Complex insertion-like (6–50 bp)",
    "CINS_51_200bp": "Complex insertion-like (51–200 bp)",
    "CINS_gt200bp": "Complex insertion-like (>200 bp)",
}

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def plot_profile(df, title, outpath):
    plt.figure(figsize=(8.6, 6.2))

    plt.plot(df["offset"], df["Z_mean"], lw=4.0, color=Z_COLOR)
    plt.fill_between(df["offset"], df["Z_lo"], df["Z_hi"], alpha=0.35, color=Z_COLOR)

    plt.plot(df["offset"], df["C_mean"], lw=4.0, color=C_COLOR)
    plt.fill_between(df["offset"], df["C_lo"], df["C_hi"], alpha=0.35, color=C_COLOR)

    plt.axvline(0, ls="--", color="black", lw=2)
    plt.axhline(1, ls=":", color="black", lw=2)

    plt.xlabel("Position relative to mutation anchor (bp)", fontsize=LABEL_FONTSIZE)
    plt.ylabel("Relative density", fontsize=LABEL_FONTSIZE)
    plt.title(title, fontsize=TITLE_FONTSIZE, pad=18)

    plt.tick_params(labelsize=TICK_FONTSIZE)

    ax = plt.gca()
    ax.set_xticks([-400, 0, 400])
    despine(ax)

    plt.savefig(outpath, dpi=600)
    plt.show()

profile_files = sorted(BASE.glob("profile_*.tsv"))

for path in profile_files:
    df = pd.read_csv(path, sep="\t")

    name = path.stem.replace("profile_", "")
    parts = name.split("_")
    
    anchor = parts[-1]
    stratum = "_".join(parts[:-1])

    if stratum not in TITLE_MAP:
        continue

    title = TITLE_MAP[stratum]

    outpath = BASE / f"{stratum}_{anchor}.png"

    plot_profile(df, title, outpath)

    print(f"[OK] {outpath.name}")
    window = df[(df["offset"] >= -5) & (df["offset"] <= 5)]
    if window.empty:
        print(f"[WARN] No bins in ±5 bp window for {stratum}")
    else:
        z_win = window["Z_mean"].mean()
        c_win = window["C_mean"].mean()
        ratio = z_win / c_win if c_win > 0 else float('nan')
        print(
            f"[±5bp WINDOW] {stratum:20s} | "
            f"Z = {z_win:.4f} | C = {c_win:.4f} | "
            f"Z/C = {ratio:.3f}"
        )

legend_fig = plt.figure(figsize=(3.0, 1.8))
ax = legend_fig.add_subplot(111)
ax.axis("off")

legend_handles = [
    Line2D([0], [0], lw=4.0, label="Z-DNA", color=Z_COLOR),
    Line2D([0], [0], lw=4.0, label="Control", color=C_COLOR),
]

ax.legend(handles=legend_handles, loc="center", frameon=False, fontsize=13)

legend_path = BASE / "mutation_profiles_legend.png"
legend_fig.savefig(legend_path, dpi=600, bbox_inches="tight")

plt.show()

print("[DONE] plots generated")

In [ ]:
# Per-stratum significance testing (paired bootstrap / Wilcoxon) for de novo profiles.
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

BASE = "data/denovo_db"
INDIR = Path(f"{BASE}/metaprofiles")
OUTDIR = Path(f"{BASE}/window_bars")
OUTDIR.mkdir(parents=True, exist_ok=True)

WINDOWS = [(10, "pm10"), (25, "pm25"), (50, "pm50")]
N_BOOTS = 50000
SEED = 42
ANCHOR_MODE = "left"

def discover_strata():
    paths = sorted(INDIR.glob(f"profile_*_{ANCHOR_MODE}.tsv"))
    print(f"[discover] found {len(paths)} profile files:")
    for p in paths:
        print(f"  - {p.name}")
    strata = []
    suffix = f"_{ANCHOR_MODE}"
    for p in paths:
        name = p.stem.replace("profile_", "")
        if name.endswith(suffix):
            stratum = name[: -len(suffix)]
            strata.append((stratum, p))
    return strata

def load_profile_safe(path):
    try:
        df = pd.read_csv(path, sep="\t")
    except Exception as e:
        print(f"[ERROR] failed to read {path.name}: {e}")
        return None
    required = {"offset", "Z_mean", "C_mean", "Z_lo", "Z_hi", "C_lo", "C_hi"}
    missing = required - set(df.columns)
    if missing:
        print(f"[ERROR] {path.name} missing columns: {missing}")
        print(f"        present columns: {df.columns.tolist()}")
        return None
    return df

def paired_bootstrap_pvalue_onesided(df, radius, n_boots, seed):
    rng = np.random.default_rng(seed)
    w = df[(df["offset"] >= -radius) & (df["offset"] <= radius)]
    if len(w) == 0:
        return np.nan, np.nan, np.nan, np.nan
    z = w["Z_mean"].to_numpy(dtype=np.float64)
    c = w["C_mean"].to_numpy(dtype=np.float64)
    n = len(z)
    z_mean, c_mean = z.mean(), c.mean()
    if not (z_mean > 0 and c_mean > 0):
        return np.nan, np.nan, np.nan, np.nan
    observed = np.log(z_mean / c_mean)
    idx = rng.integers(0, n, size=(n_boots, n))
    zb = z[idx].mean(axis=1)
    cb = c[idx].mean(axis=1)
    valid = (zb > 0) & (cb > 0)
    if not valid.any():
        return observed, np.nan, np.nan, np.nan
    boots = np.log(zb[valid] / cb[valid])
    centered = boots - boots.mean()
    p_one = (np.sum(centered >= observed) + 1) / (len(centered) + 1)
    ci_lo = np.percentile(boots, 2.5)
    ci_hi = np.percentile(boots, 97.5)
    return observed, ci_lo, ci_hi, p_one

def wilcoxon_onesided(df, radius):
    w = df[(df["offset"] >= -radius) & (df["offset"] <= radius)]
    if len(w) == 0:
        return np.nan, np.nan
    z = w["Z_mean"].to_numpy(dtype=np.float64)
    c = w["C_mean"].to_numpy(dtype=np.float64)
    if np.allclose(z - c, 0):
        return 0.0, 1.0
    try:
        stat, p = wilcoxon(z, c, alternative="greater", zero_method="wilcox")
        return float(stat), float(p)
    except ValueError:
        return np.nan, np.nan

def build_table(radius):
    strata = discover_strata()
    rows = []
    for stratum, path in strata:
        df = load_profile_safe(path)
        if df is None:
            continue
        w = df[(df["offset"] >= -radius) & (df["offset"] <= radius)]
        if len(w) == 0:
            print(f"[WARN] {stratum}: no offsets in \u00b1{radius} window")
            continue
        z_mean = w["Z_mean"].mean()
        c_mean = w["C_mean"].mean()
        z_lo_mean = w["Z_lo"].mean()
        z_hi_mean = w["Z_hi"].mean()
        c_lo_mean = w["C_lo"].mean()
        c_hi_mean = w["C_hi"].mean()
        log_ratio, ci_lo, ci_hi, p_boot = paired_bootstrap_pvalue_onesided(
            df, radius, N_BOOTS, SEED
        )
        wilc_stat, p_wilc = wilcoxon_onesided(df, radius)
        fold = (z_mean / c_mean) if (c_mean and c_mean > 0) else np.nan
        rows.append({
            "stratum": stratum,
            "Z_mean": z_mean, "Z_lo_mean": z_lo_mean, "Z_hi_mean": z_hi_mean,
            "C_mean": c_mean, "C_lo_mean": c_lo_mean, "C_hi_mean": c_hi_mean,
            "fold_enrichment": fold,
            "log_ratio_ZC": log_ratio,
            "log_ratio_lo": ci_lo, "log_ratio_hi": ci_hi,
            "p_bootstrap_onesided": p_boot,
            "p_wilcoxon_onesided": p_wilc,
            "wilcoxon_stat": wilc_stat,
        })

    tbl = pd.DataFrame(rows)
    for col in ["p_bootstrap_onesided", "p_wilcoxon_onesided"]:
        new_col = "p_bonf_" + col[len("p_"):]
        tbl[new_col] = np.nan
        if col not in tbl.columns or len(tbl) == 0:
            continue
        mask = tbl[col].notna()
        if mask.any():
            tbl.loc[mask, new_col] = multipletests(
                tbl.loc[mask, col].values, method="bonferroni"
            )[1]

    def sig_marker(p, direction_pos):
        if pd.isna(p): return ""
        if not direction_pos: return "ns_dir"
        if p < 1e-3:   return "***"
        if p < 1e-2:   return "**"
        if p < 5e-2:   return "*"
        return "ns"

    if len(tbl):
        tbl["direction_positive"] = tbl["log_ratio_ZC"] > 0
        tbl["sig_marker_primary"] = tbl.apply(
            lambda r: sig_marker(r["p_bonf_wilcoxon_onesided"], r["direction_positive"]),
            axis=1,
        )
        tbl["sig_marker_bootstrap"] = tbl.apply(
            lambda r: sig_marker(r["p_bonf_bootstrap_onesided"], r["direction_positive"]),
            axis=1,
        )
    return tbl

for radius, tag in WINDOWS:
    print(f"\n=== Window \u00b1{radius} bp ({tag}) ===")
    tbl = build_table(radius)
    out_path = OUTDIR / f"window_stats_{tag}.tsv"
    tbl.to_csv(out_path, sep="\t", index=False)
    print(f"  wrote: {out_path}")
    if len(tbl):
        print(tbl[["stratum", "Z_mean", "C_mean", "fold_enrichment",
                   "log_ratio_ZC",
                   "p_wilcoxon_onesided", "p_bonf_wilcoxon_onesided",
                   "sig_marker_primary",
                   "p_bootstrap_onesided", "p_bonf_bootstrap_onesided",
                   "sig_marker_bootstrap"]].to_string(index=False))
    else:
        print("  (no strata)")

print("\n[DONE]")

### 11.2 Size-collapsed metaprofiles

In [ ]:
# De novo mutation x Z-DNA metaprofiles collapsed across size bins (relative density).
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.stats.multitest import multipletests

BASE = Path("data/denovo_db")

MUT_BED  = BASE / "denovodb.bed.gz"
MUT_META = BASE / "denovodb.meta.tsv.gz"
Z_BED    = BASE / "grch37_zdna_tol10_has_control.bed"
C_BED    = BASE / "grch37_zdna_tol10_control.bed"

OUTDIR = BASE / "metaprofiles_bytype"
OUTDIR.mkdir(parents=True, exist_ok=True)

WINDOW_RADIUS = 500
OFFSETS = np.arange(-WINDOW_RADIUS, WINDOW_RADIUS + 1)
WINDOWS = [(10, "pm10"), (25, "pm25"), (50, "pm50")]

N_BOOT = 10_000
SEED = 42
MIN_MUTS = 400          # lowered so MNP (n=481) is retained
CI_LOW, CI_HIGH = 2.5, 97.5

CLASS_ORDER = ["SNV", "MNP", "DEL", "INS", "CDEL", "CINS"]

def _ensure_chr(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    return s.where(s.str.startswith("chr"), "chr" + s)

def load_bed(path):
    df = pd.read_csv(path, sep="\t", header=None, usecols=[0, 1, 2],
                     names=["chr", "start", "end"])
    df["chr"] = _ensure_chr(df["chr"])
    return {c: d[["start", "end"]].to_numpy(np.int64)
            for c, d in df.groupby("chr")}

def assign_type(struct_class: str, dominant_op: str):
    sc, dom = struct_class, dominant_op
    if sc == "snp":
        return "SNV"
    if sc == "mnp":
        return "MNP"
    if sc == "del":
        return "DEL"
    if sc == "ins":
        return "INS"
    if sc == "complex" and dom == "del_like":
        return "CDEL"
    if sc == "complex" and dom == "ins_like":
        return "CINS"
    return None  # 'substitution' / balanced complex -> excluded

def bool_coverage(intervals: np.ndarray, L: int) -> np.ndarray:
    diff = np.zeros(L + 1, dtype=np.int32)
    np.add.at(diff, intervals[:, 0], 1)
    np.add.at(diff, intervals[:, 1], -1)
    return np.cumsum(diff[:-1]) > 0

def per_chrom_counts(sub, z_iv, c_iv):
    chroms = sorted(set(sub["Chromosome"]) & set(z_iv) & set(c_iv))
    Zc, Cc = {}, {}
    n_off = len(OFFSETS)

    for chrom in chroms:
        anchors = sub.loc[sub["Chromosome"] == chrom, "Start"].to_numpy(np.int64)
        if anchors.size == 0:
            continue
        region_max = max(int(z_iv[chrom][:, 1].max()), int(c_iv[chrom][:, 1].max()))
        L = int(max(region_max, anchors.max() + WINDOW_RADIUS) + 2)

        zcov = bool_coverage(z_iv[chrom], L)
        ccov = bool_coverage(c_iv[chrom], L)

        zk = np.zeros(n_off)
        ck = np.zeros(n_off)
        for i, k in enumerate(OFFSETS):
            pos = anchors + k
            p = pos[pos >= 0]              # rare: anchors within WINDOW_RADIUS of coord 0
            zk[i] = zcov[p].sum()
            ck[i] = ccov[p].sum()

        Zc[chrom] = zk
        Cc[chrom] = ck

    return Zc, Cc, [c for c in chroms if c in Zc]

def _self_norm(v):
    m = v.mean()
    return v / m if m else np.full_like(v, np.nan, dtype=float)

def relative_density(Zc, Cc, chroms):
    Zt = np.sum([Zc[c] for c in chroms], axis=0)
    Ct = np.sum([Cc[c] for c in chroms], axis=0)
    return _self_norm(Zt), _self_norm(Ct)

def block_bootstrap_rel(Zc, Cc, chroms, n_boot, rng):
    Zmat = np.stack([Zc[c] for c in chroms])   # (n_chr, n_off)
    Cmat = np.stack([Cc[c] for c in chroms])
    n_chr = len(chroms)
    n_off = len(OFFSETS)

    Zb = np.empty((n_boot, n_off))
    Cb = np.empty((n_boot, n_off))
    for b in range(n_boot):
        idx = rng.integers(0, n_chr, n_chr)
        Zb[b] = _self_norm(Zmat[idx].sum(0))
        Cb[b] = _self_norm(Cmat[idx].sum(0))
    return Zb, Cb

def main():
    z_iv = load_bed(Z_BED)
    c_iv = load_bed(C_BED)

    mut = pd.read_csv(
        MUT_BED, sep="\t", header=None,
        names=["Chromosome", "Start", "End", "mut_id", "score", "strand"],
        dtype={"Chromosome": "string", "Start": "int64", "End": "int64",
               "mut_id": "string"},
    )
    mut["Chromosome"] = _ensure_chr(mut["Chromosome"])

    n_before = len(mut)
    mut = mut.drop_duplicates(subset="mut_id")
    if len(mut) != n_before:
        print(f"[dedup] dropped {n_before - len(mut)} duplicate mut_id rows")

    meta = pd.read_csv(
        MUT_META, sep="\t",
        usecols=["mut_id", "struct_class", "dominant_op"],
        dtype={"mut_id": "string", "struct_class": "string",
               "dominant_op": "string"},
    )
    mut = mut.merge(meta, on="mut_id", how="inner")

    mut["mut_type"] = [assign_type(sc, dom)
                       for sc, dom in zip(mut["struct_class"], mut["dominant_op"])]

    n_excluded = mut["mut_type"].isna().sum()
    print(f"[classify] excluded (substitution/balanced-complex/unclassified): {n_excluded}")
    mut = mut.dropna(subset=["mut_type"])

    print("\n[counts by collapsed type]")
    print(mut["mut_type"].value_counts().reindex(CLASS_ORDER))

    rng = np.random.default_rng(SEED)
    stats_rows = []

    for mtype in CLASS_ORDER:
        sub = mut[mut["mut_type"] == mtype]
        n = len(sub)
        if n < MIN_MUTS:
            print(f"[skip] {mtype}: n={n} < MIN_MUTS={MIN_MUTS}")
            continue

        Zc, Cc, chroms = per_chrom_counts(sub, z_iv, c_iv)
        if len(chroms) < 2:
            print(f"[skip] {mtype}: <2 usable chromosomes")
            continue

        Z_rel, C_rel = relative_density(Zc, Cc, chroms)
        Zb, Cb = block_bootstrap_rel(Zc, Cc, chroms, N_BOOT, rng)

        Z_lo = np.nanpercentile(Zb, CI_LOW, axis=0)
        Z_hi = np.nanpercentile(Zb, CI_HIGH, axis=0)
        C_lo = np.nanpercentile(Cb, CI_LOW, axis=0)
        C_hi = np.nanpercentile(Cb, CI_HIGH, axis=0)

        prof = pd.DataFrame({
            "offset": OFFSETS,
            "Z_rel": Z_rel, "Z_lo": Z_lo, "Z_hi": Z_hi,
            "C_rel": C_rel, "C_lo": C_lo, "C_hi": C_hi,
            "mut_type": mtype, "n_events": n,
        })
        prof.to_csv(OUTDIR / f"profile_{mtype}.tsv", sep="\t", index=False)
        print(f"[ok] profile_{mtype}.tsv  (n={n}, chroms={len(chroms)})")

        for radius, tag in WINDOWS:
            m = (OFFSETS >= -radius) & (OFFSETS <= radius)
            z_win = np.nanmean(Z_rel[m])
            c_win = np.nanmean(C_rel[m])

            zb_win = np.nanmean(Zb[:, m], axis=1)
            cb_win = np.nanmean(Cb[:, m], axis=1)
            ok = np.isfinite(zb_win) & np.isfinite(cb_win) & (zb_win > 0) & (cb_win > 0)

            if not (np.isfinite(z_win) and np.isfinite(c_win) and c_win > 0 and ok.any()):
                log_ratio = ci_lo = ci_hi = p_one = np.nan
            else:
                log_ratio = np.log(z_win / c_win)
                boots = np.log(zb_win[ok] / cb_win[ok])
                ci_lo = np.percentile(boots, 2.5)
                ci_hi = np.percentile(boots, 97.5)
                p_one = (np.sum(boots <= 0) + 1) / (boots.size + 1)  # one-sided: Z > C

            stats_rows.append({
                "mut_type": mtype, "window": tag, "radius": radius, "n_events": n,
                "Z_rel_win": z_win, "C_rel_win": c_win,
                "fold_relative": (z_win / c_win) if c_win > 0 else np.nan,
                "log_ratio_ZC": log_ratio,
                "log_ratio_ci_lo": ci_lo, "log_ratio_ci_hi": ci_hi,
                "p_boot_onesided": p_one,
            })

    stats = pd.DataFrame(stats_rows)
    stats["p_bh_within_window"] = np.nan
    for tag in stats["window"].unique():
        m = (stats["window"] == tag) & stats["p_boot_onesided"].notna()
        if m.any():
            stats.loc[m, "p_bh_within_window"] = multipletests(
                stats.loc[m, "p_boot_onesided"].to_numpy(), method="fdr_bh"
            )[1]

    def marker(p, lr):
        if pd.isna(p) or pd.isna(lr):
            return ""
        if lr <= 0:
            return "ns_dir"
        if p < 1e-3: return "***"
        if p < 1e-2: return "**"
        if p < 5e-2: return "*"
        return "ns"

    stats["sig"] = [marker(p, lr)
                    for p, lr in zip(stats["p_bh_within_window"], stats["log_ratio_ZC"])]

    stats.to_csv(OUTDIR / "window_enrichment_stats.tsv", sep="\t", index=False)
    print("\n[window relative-density enrichment stats]")
    cols = ["mut_type", "window", "n_events", "Z_rel_win", "C_rel_win",
            "fold_relative", "log_ratio_ZC", "log_ratio_ci_lo", "log_ratio_ci_hi",
            "p_boot_onesided", "p_bh_within_window", "sig"]
    print(stats[cols].to_string(index=False))
    print(f"\n[wrote] {OUTDIR/'window_enrichment_stats.tsv'}")
    print("[DONE]")
if __name__ == "__main__":
    main()

In [ ]:
# Plot by-type relative-density metaprofiles.
import matplotlib
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

BASE = Path("data/denovo_db/metaprofiles_bytype")
LABEL_FS, TICK_FS, TITLE_FS = 22, 20, 24
Z_COLOR, C_COLOR = "tab:green", "tab:gray"

TITLE_MAP = {
    "SNV":  "Single-nucleotide variants",
    "MNP":  "Multi-nucleotide variants",
    "DEL":  "Deletions",
    "INS":  "Insertions",
    "CDEL": "Complex deletion-like",
    "CINS": "Complex insertion-like",
}

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def plot_profile(df, title, outpath):
    plt.figure(figsize=(8.6, 6.2))
    plt.plot(df["offset"], df["Z_rel"], lw=4.0, color=Z_COLOR)
    plt.fill_between(df["offset"], df["Z_lo"], df["Z_hi"], alpha=0.35, color=Z_COLOR)
    plt.plot(df["offset"], df["C_rel"], lw=4.0, color=C_COLOR)
    plt.fill_between(df["offset"], df["C_lo"], df["C_hi"], alpha=0.35, color=C_COLOR)

    plt.axvline(0, ls="--", color="black", lw=2)
    plt.axhline(1, ls=":", color="black", lw=2)
    plt.xlabel("Position relative to mutation anchor (bp)", fontsize=LABEL_FS)
    plt.ylabel("Relative density", fontsize=LABEL_FS)
    plt.title(title, fontsize=TITLE_FS, pad=18)
    plt.tick_params(labelsize=TICK_FS)
    ax = plt.gca()
    ax.set_xticks([-400, 0, 400])
    despine(ax)
    plt.tight_layout()
    plt.savefig(outpath, dpi=600)
    plt.show()

def main():
    for path in sorted(BASE.glob("profile_*.tsv")):
        mtype = path.stem.replace("profile_", "")
        if mtype not in TITLE_MAP:
            continue
        df = pd.read_csv(path, sep="\t")
        out = BASE / f"{mtype}.png"
        plot_profile(df, TITLE_MAP[mtype], out)
        print(f"[ok] {out.name}")

    fig = plt.figure(figsize=(3.0, 1.6))
    ax = fig.add_subplot(111); ax.axis("off")
    handles = [Line2D([0], [0], lw=4.0, color=Z_COLOR, label="Z-DNA"),
               Line2D([0], [0], lw=4.0, color=C_COLOR, label="Control")]
    ax.legend(handles=handles, loc="center", frameon=False, fontsize=13)
    fig.savefig(BASE / "legend.png", dpi=600, bbox_inches="tight")
    plt.show()
    print("[DONE]")

if __name__ == "__main__":
    main()

In [ ]:
# Summary bar chart for de novo mutation x Z-DNA relative density.
import matplotlib
matplotlib.use("Agg")                # headless-safe; no plt.show()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from pathlib import Path

BASE   = Path("data/denovo_db/metaprofiles_bytype")
STATS  = BASE / "window_enrichment_stats.tsv"
OUTDIR = BASE

WINDOWS = [(10, "pm10"), (25, "pm25"), (50, "pm50")]
CLASS_ORDER = ["SNV", "DEL", "INS", "CDEL", "CINS"]
X_LABEL_MAP = {
    "SNV": "SNV",
    "DEL": "Del", "INS": "Ins", "CDEL": "CDel", "CINS": "CIns",
}

Z_COLOR, C_COLOR = "tab:green", "tab:gray"
LABEL_FS, TICK_FS, SIG_FS = 22, 18, 18
STAR_SET = {"*", "**", "***"}

def window_band(mtype, radius):
    df = pd.read_csv(BASE / f"profile_{mtype}.tsv", sep="\t")
    w = df[(df["offset"] >= -radius) & (df["offset"] <= radius)]
    return (w["Z_lo"].mean(), w["Z_hi"].mean(),
            w["C_lo"].mean(), w["C_hi"].mean())

def plot_window(radius, tag, stats):
    sub = stats[stats["window"] == tag].copy()
    present = [c for c in CLASS_ORDER if c in set(sub["mut_type"])]
    sub = sub.set_index("mut_type").loc[present].reset_index()
    n = len(sub)
    if n == 0:
        print(f"[skip] {tag}: no rows")
        return

    z_vals = sub["Z_rel_win"].to_numpy(float)
    c_vals = sub["C_rel_win"].to_numpy(float)
    folds  = sub["fold_relative"].to_numpy(float)
    sigs   = sub["sig"].fillna("").to_numpy()

    bands = np.array([window_band(m, radius) for m in sub["mut_type"]])
    z_lo, z_hi, c_lo, c_hi = bands[:, 0], bands[:, 1], bands[:, 2], bands[:, 3]
    z_err = [np.clip(z_vals - z_lo, 0, None), np.clip(z_hi - z_vals, 0, None)]
    c_err = [np.clip(c_vals - c_lo, 0, None), np.clip(c_hi - c_vals, 0, None)]

    fig_w = max(7.0, 1.3 * n + 2.5)
    fig, ax = plt.subplots(figsize=(fig_w, 6.5))
    x = np.arange(n)
    width = 0.38

    ax.bar(x - width/2, z_vals, width, yerr=z_err, capsize=2,
           color=Z_COLOR, edgecolor="white", linewidth=0.5,
           error_kw={"linewidth": 1.0, "ecolor": "black"})
    ax.bar(x + width/2, c_vals, width, yerr=c_err, capsize=2,
           color=C_COLOR, edgecolor="white", linewidth=0.5,
           error_kw={"linewidth": 1.0, "ecolor": "black"})
    ax.axhline(1.0, ls=":", color="black", lw=1.5, alpha=0.6)

    bar_top = np.maximum(z_vals + z_err[1], c_vals + c_err[1])
    off = 0.06 * (bar_top.max() if bar_top.max() > 0 else 1.0)
    for i in range(n):
        yb = bar_top[i] + off
        if sigs[i] in STAR_SET:
            ax.plot([x[i]-width/2, x[i]-width/2, x[i]+width/2, x[i]+width/2],
                    [yb, yb+off*0.35, yb+off*0.35, yb], lw=1.2, color="black")
            ax.text(x[i], yb+off*0.4, sigs[i], ha="center", va="bottom", fontsize=SIG_FS)
            if np.isfinite(folds[i]):
                ax.text(x[i], yb+off*1.5, f"{folds[i]:.1f}\u00d7", ha="center",
                        va="bottom", fontsize=SIG_FS-4, color="#333333")
        else:
            ax.text(x[i], yb, "n.s.", ha="center", va="bottom",
                    fontsize=SIG_FS-4, color="#666666", fontstyle="italic")

    ax.set_ylim(0, (bar_top + off*4.0).max() * 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{X_LABEL_MAP.get(m, m)}\n(n={int(nn):,})"
         for m, nn in zip(sub["mut_type"], sub["n_events"])],
        fontsize=TICK_FS,
    )
    ax.set_ylabel(f"Relative density (\u00b1{radius} bp window)", fontsize=LABEL_FS)
    ax.tick_params(axis="y", labelsize=TICK_FS)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.margins(x=0.02)
    plt.tight_layout()
    fig.savefig(OUTDIR / f"denovo_window_bars_{tag}.png", dpi=400, bbox_inches="tight")
    plt.close(fig)
    print(f"[ok] denovo_window_bars_{tag}.png")

def save_legend():
    fig = plt.figure(figsize=(5.5, 2.4))
    ax = fig.add_subplot(111); ax.axis("off")
    handles = [
        mpatches.Patch(color=Z_COLOR, label="Z-DNA"),
        mpatches.Patch(color=C_COLOR, label="Matched control"),
        Line2D([0],[0], color="none", marker="$\\ast\\ast\\ast$", markersize=14,
               label=r"$P_{\mathrm{BH}} < 10^{-3}$"),
        Line2D([0],[0], color="none", marker="$\\ast\\ast$", markersize=14,
               label=r"$P_{\mathrm{BH}} < 10^{-2}$"),
        Line2D([0],[0], color="none", marker="$\\ast$", markersize=14,
               label=r"$P_{\mathrm{BH}} < 0.05$"),
        Line2D([0],[0], color="none", marker="$\\mathrm{n.s.}$", markersize=14,
               label="not significant (one-sided Z > C)"),
        Line2D([0],[0], color="none", marker="$1.2\\times$", markersize=14,
               label="relative fold (Z / C)"),
    ]
    ax.legend(handles=handles, loc="center", frameon=False, fontsize=12,
              handletextpad=0.6, borderpad=0.4)
    fig.savefig(OUTDIR / "denovo_window_bars_legend.png", dpi=600, bbox_inches="tight")
    plt.close(fig)
    print("[ok] denovo_window_bars_legend.png")

def main():
    stats = pd.read_csv(STATS, sep="\t")
    for radius, tag in WINDOWS:
        plot_window(radius, tag, stats)
    save_legend()
    print("[DONE]")

if __name__ == "__main__":
    main()

## 12. Enrichment vs Z-DNA score

In [ ]:
# Prepare score-stratified inputs for mutation enrichment.
import numpy as np
import pandas as pd
from pathlib import Path

BASE = "data"

MASTER = f"{BASE}/controls/CHM13_controls_MASTER.tsv"

TOL = "10%"
N_SCORE_GROUPS = 3

OUTDIR = Path(f"{BASE}/mutation_profiles_scorestrat_tol10_left")
OUTDIR.mkdir(parents=True, exist_ok=True)

OUT_TSV = OUTDIR / "score_bins_length_summary.tsv"

m = pd.read_csv(MASTER, sep="\t")

m = (
    m[m["tol_level"] == TOL]
    .drop_duplicates(["Chromosome", "Start0", "End0"])
    .copy()
)

m["zdna_score"] = m["Z-DNA Score"].astype(float)
m["length_bp"] = m["End0"].astype(int) - m["Start0"].astype(int)

scores_sorted = np.sort(m["zdna_score"].values)

n_total = len(scores_sorted)

boundaries = [
    int(np.ceil(i * n_total / N_SCORE_GROUPS))
    for i in range(1, N_SCORE_GROUPS + 1)
]

boundaries[-1] = n_total

SCORE_BIN_ORDER = []

prev = 0

for i, end in enumerate(boundaries, 1):

    lo = scores_sorted[prev]
    hi = scores_sorted[end - 1]

    name = f"G{i} ({lo:.1f}–{hi:.1f})"

    SCORE_BIN_ORDER.append(name)

    prev = end

m["score_bin"] = pd.qcut(
    m["zdna_score"],
    q=N_SCORE_GROUPS,
    labels=SCORE_BIN_ORDER,
    duplicates="drop",
)

m = m.dropna(subset=["score_bin"]).copy()

summary = (
    m.groupby("score_bin", observed=True)
    .agg(
        n=("length_bp", "size"),
        score_min=("zdna_score", "min"),
        score_max=("zdna_score", "max"),
        length_min=("length_bp", "min"),
        length_q05=("length_bp", lambda x: x.quantile(0.05)),
        length_q25=("length_bp", lambda x: x.quantile(0.25)),
        length_median=("length_bp", "median"),
        length_mean=("length_bp", "mean"),
        length_q75=("length_bp", lambda x: x.quantile(0.75)),
        length_q95=("length_bp", lambda x: x.quantile(0.95)),
        length_max=("length_bp", "max"),
    )
    .reset_index()
)

for col in summary.columns:
    if col.startswith("length_") or col.startswith("score_"):
        summary[col] = summary[col].round(3)

summary.to_csv(OUT_TSV, sep="\t", index=False)

print("\nScore-bin length summary:")
print(summary.to_string(index=False))

print(f"\n[OK] Wrote: {OUT_TSV}")
print("\nScore-length correlation:")
print(
    m[["zdna_score", "length_bp"]]
    .corr(method="spearman")
    .iloc[0, 1]
)

In [ ]:
# Score-stratified Z-DNA mutation metaprofiles (block-bootstrap).
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.signal import fftconvolve

BASE = "data"

MASTER         = f"{BASE}/controls/CHM13_controls_MASTER.tsv"

MUT_BED        = (
    f"{BASE}/mutations/"
    "hprc_mutations.bed.gz"
)

MUT_META       = (
    f"{BASE}/mutations/"
    "hprc_mutations.meta.tsv.gz"
)

CHROM_LEN_TSV  = (
    f"{BASE}/chm13/"
    "chm13_seqreport_zdna.tsv"
)

TOL = "10%"

WINDOW_RADIUS = 500

N_BOOTS = 500

CI_LOW  = 2.5
CI_HIGH = 97.5

MIN_MUTS = 500

ANCHOR_MODE = "left"

OUTDIR = Path(
    f"{BASE}/mutation_profiles_scorestrat_tol10_{ANCHOR_MODE}"
)

OUTDIR.mkdir(parents=True, exist_ok=True)

N_SCORE_GROUPS = 3

def chrom_block_bootstrap_windowmean_norm(
    counts_by_chr,
    n_boot,
):

    chroms = list(counts_by_chr)

    if len(chroms) < 3:

        n = len(next(iter(counts_by_chr.values())))

        return (
            np.full(n, np.nan),
            np.full(n, np.nan),
        )

    mat = np.stack(
        [
            counts_by_chr[c].astype(float)
            for c in chroms
        ],
        axis=0,
    )

    rng = np.random.default_rng(42)

    boots = []

    for _ in range(n_boot):

        idx = rng.integers(
            0,
            len(chroms),
            len(chroms),
        )

        v = mat[idx].sum(axis=0)

        m = v.mean()

        boots.append(
            v / m
            if m
            else np.full_like(v, np.nan, dtype=float)
        )

    boots = np.stack(boots, axis=0)

    return (
        np.nanpercentile(boots, CI_LOW, axis=0),
        np.nanpercentile(boots, CI_HIGH, axis=0),
    )

def build_bool_coverage(
    starts,
    ends,
    chrom_len,
):

    starts = np.clip(
        np.asarray(starts, dtype=np.int64),
        0,
        chrom_len,
    )

    ends = np.clip(
        np.asarray(ends, dtype=np.int64),
        0,
        chrom_len,
    )

    mask = ends > starts

    if not np.any(mask):

        return np.zeros(chrom_len, dtype=bool)

    starts = starts[mask]
    ends   = ends[mask]

    diff = np.zeros(chrom_len + 1, dtype=np.int32)

    np.add.at(diff, starts,  1)
    np.add.at(diff, ends,   -1)

    return np.cumsum(diff[:-1]) > 0

def offset_profile_fast(
    anchors,
    z_cov,
    chrom_L,
    window_radius,
):

    if anchors.size == 0:

        return np.zeros(
            2 * window_radius + 1,
            dtype=np.int64,
        )

    pad = window_radius

    zpad = np.pad(
        z_cov.astype(np.uint8),
        pad_width=pad,
        mode="constant",
        constant_values=0,
    )

    a = anchors + pad

    offsets = np.arange(
        -window_radius,
        window_radius + 1,
        dtype=np.int64,
    )

    counts = np.empty(
        len(offsets),
        dtype=np.int64,
    )

    for i, k in enumerate(offsets):

        counts[i] = zpad[a + k].sum()

    return counts

def midpoint_from_left_start_and_span(
    start_arr,
    ref_span_arr,
):

    return start_arr + (ref_span_arr - 1) // 2

def right_breakpoint_from_left_start_and_span(
    start_arr,
    ref_span_arr,
):

    return start_arr + ref_span_arr - 1

DUAL_ANCHOR_STRATA = {
    "DEL_gt50bp",
    "CDEL_gt50bp",
    "SUB_50_200",
    "SUB_gt200",
}

MIDPOINT_ANCHOR_STRATA = {
    "DEL_1bp",
    "DEL_2_5bp",
    "DEL_6_50bp",
    "CDEL_1_5bp",
    "CDEL_6_50bp",
    "SUB_20_50",
}

SINGLE_ANCHOR_STRATA = {
    "INS_1bp",
    "INS_2_5bp",
    "INS_6_50bp",
    "INS_51_200bp",
    "INS_gt200bp",
    "CINS_1_5bp",
    "CINS_6_50bp",
    "CINS_51_200bp",
    "CINS_gt200bp",
    "SNP",
    "MNP_2_20",
}

def assign_stratum(row):

    sc  = row["struct_class"]
    dom = row["dominant_op"]

    L = int(row["event_size"])

    if sc == "snp":
        return "SNP"

    if sc == "mnp":
        return "MNP_2_20"

    if sc == "substitution":

        if L <= 50:
            return "SUB_20_50"

        elif L <= 200:
            return "SUB_50_200"

        else:
            return "SUB_gt200"

    if sc == "del":

        if L == 1:
            return "DEL_1bp"

        elif L <= 5:
            return "DEL_2_5bp"

        elif L <= 50:
            return "DEL_6_50bp"

        else:
            return "DEL_gt50bp"

    if sc == "ins":

        if L == 1:
            return "INS_1bp"

        elif L <= 5:
            return "INS_2_5bp"

        elif L <= 50:
            return "INS_6_50bp"

        elif L <= 200:
            return "INS_51_200bp"

        else:
            return "INS_gt200bp"

    if sc == "complex" and dom == "del_like":

        if L <= 5:
            return "CDEL_1_5bp"

        elif L <= 50:
            return "CDEL_6_50bp"

        else:
            return "CDEL_gt50bp"

    if sc == "complex" and dom == "ins_like":

        if L <= 5:
            return "CINS_1_5bp"

        elif L <= 50:
            return "CINS_6_50bp"

        elif L <= 200:
            return "CINS_51_200bp"

        else:
            return "CINS_gt200bp"

    return None

print("[INFO] Loading chromosome lengths...")

chrom_len_map = dict(
    pd.read_csv(
        CHROM_LEN_TSV,
        sep="\t",
        usecols=[
            "ucsc_pansn_name",
            "gcf_seq_length",
        ],
    ).values
)

print("[INFO] Loading Z-DNA regions...")

m = pd.read_csv(MASTER, sep="\t")

m = (
    m[m["tol_level"] == TOL]
    .drop_duplicates(
        ["Chromosome", "Start0", "End0"]
    )
    .copy()
)

m["zdna_score"] = m["Z-DNA Score"].astype(float)

print(f"  {len(m):,} unique Z-DNA regions")

scores_sorted = np.sort(m["zdna_score"].values)

n_total = len(scores_sorted)

boundaries = [
    int(np.ceil(i * n_total / N_SCORE_GROUPS))
    for i in range(1, N_SCORE_GROUPS + 1)
]

boundaries[-1] = n_total

SCORE_BINS = {}
SCORE_BIN_ORDER = []

prev = 0

for i, end in enumerate(boundaries, 1):

    lo = scores_sorted[prev]
    hi = scores_sorted[end - 1]

    name = f"G{i} ({lo:.1f}–{hi:.1f})"

    SCORE_BINS[name] = (lo, hi)

    SCORE_BIN_ORDER.append(name)

    prev = end

m["score_bin"] = pd.qcut(
    m["zdna_score"],
    q=N_SCORE_GROUPS,
    labels=SCORE_BIN_ORDER,
    duplicates="drop",
)

m = m.dropna(subset=["score_bin"]).copy()

for lb in SCORE_BIN_ORDER:

    sub = m[m["score_bin"] == lb]

    vals = sub["zdna_score"]

    print(
        f"  {lb}: "
        f"n={len(sub):,}  "
        f"range={vals.min():.2f}–{vals.max():.2f}"
    )

print("[INFO] Building coverage masks...")

z_cov_cache = {}

for lb in SCORE_BIN_ORDER:

    z_sub = m[m["score_bin"] == lb]

    for chrom, d in z_sub.groupby("Chromosome"):

        chrom_L = int(chrom_len_map.get(chrom, 0))

        if chrom_L <= 0:
            continue

        ivs = d[
            ["Start0", "End0"]
        ].to_numpy(dtype=np.int64)

        z_cov_cache[(lb, chrom)] = build_bool_coverage(
            ivs[:, 0],
            ivs[:, 1],
            chrom_L,
        )

print("[INFO] Loading mutations...")

mut = pd.read_csv(
    MUT_BED,
    sep="\t",
    header=None,
    names=[
        "Chromosome",
        "Start",
        "End",
        "mut_id",
        "score",
        "strand",
    ],
)

meta = pd.read_csv(
    MUT_META,
    sep="\t",
    compression="gzip",
)

need = [
    "mut_id",
    "struct_class",
    "dominant_op",
    "ref_span",
    "alt_span",
]

mut = (
    mut.merge(
        meta[need],
        on="mut_id",
        how="left",
    )
    .dropna()
)

mut["Start"] = mut["Start"].astype(np.int64)

mut["ref_span"] = mut["ref_span"].astype(np.int64)
mut["alt_span"] = mut["alt_span"].astype(np.int64)

mut["event_size"] = np.where(
    (
        (mut["struct_class"] == "ins")
        |
        (
            (mut["struct_class"] == "complex")
            &
            (mut["dominant_op"] == "ins_like")
        )
    ),
    mut["alt_span"],
    mut["ref_span"],
).astype(np.int64)

mut["stratum"] = mut.apply(
    assign_stratum,
    axis=1,
)

mut = mut.dropna(subset=["stratum"]).copy()

print(f"  {len(mut):,} mutations after stratification")

offsets = np.arange(
    -WINDOW_RADIUS,
    WINDOW_RADIUS + 1,
    dtype=np.int64,
)

print(
    f"\n[INFO] Computing profiles "
    f"(window=±{WINDOW_RADIUS} bp)...\n"
)

for stratum, sub in mut.groupby(
    "stratum",
    sort=True,
):

    if (
        ANCHOR_MODE == "right"
        and
        stratum not in DUAL_ANCHOR_STRATA
    ):
        continue

    if len(sub) < MIN_MUTS:
        continue

    totals = {
        lb: np.zeros(
            len(offsets),
            dtype=np.int64,
        )
        for lb in SCORE_BIN_ORDER
    }

    by_chr = {
        lb: {}
        for lb in SCORE_BIN_ORDER
    }

    n_used = 0

    for chrom, d in sub.groupby(
        "Chromosome",
        sort=False,
    ):

        chrom_L = int(
            chrom_len_map.get(chrom, 0)
        )

        if chrom_L == 0:
            continue

        starts = d["Start"].to_numpy(dtype=np.int64)

        refs = d["ref_span"].to_numpy(dtype=np.int64)

        if stratum in DUAL_ANCHOR_STRATA:

            if ANCHOR_MODE == "left":

                anchors = starts

            else:

                anchors = (
                    right_breakpoint_from_left_start_and_span(
                        starts,
                        refs,
                    )
                )

        elif stratum in MIDPOINT_ANCHOR_STRATA:

            anchors = midpoint_from_left_start_and_span(
                starts,
                refs,
            )

        else:

            anchors = starts

        if anchors.size == 0:
            continue

        n_used += len(anchors)

        for lb in SCORE_BIN_ORDER:

            z_cov = z_cov_cache.get((lb, chrom))

            if z_cov is None:
                continue

            counts = offset_profile_fast(
                anchors,
                z_cov,
                chrom_L,
                WINDOW_RADIUS,
            )

            totals[lb] += counts

            by_chr[lb][chrom] = counts

    if n_used < MIN_MUTS:
        continue

    frames = []

    for lb in SCORE_BIN_ORDER:

        t = totals[lb]

        tm = t.mean()

        Z_mean = (
            t / tm
            if tm > 0
            else np.full(
                len(offsets),
                np.nan,
            )
        )

        Z_lo, Z_hi = (
            chrom_block_bootstrap_windowmean_norm(
                by_chr[lb],
                N_BOOTS,
            )
        )

        df_out = pd.DataFrame({
            "offset": offsets,
            "score_bin": lb,
            "Z_mean": Z_mean,
            "Z_lo": Z_lo,
            "Z_hi": Z_hi,
            "stratum": stratum,
            "n_mut": n_used,
        })

        frames.append(df_out)

    pd.concat(
        frames,
        ignore_index=True,
    ).to_csv(
        OUTDIR / f"profile_{stratum}.tsv",
        sep="\t",
        index=False,
    )
    chrom_rows = []
    for lb in SCORE_BIN_ORDER:
        for chrom in by_chr[lb]:
            for i, off in enumerate(offsets):
                chrom_rows.append((lb, chrom, int(off), int(by_chr[lb][chrom][i])))
    chrom_df = pd.DataFrame(chrom_rows, columns=["score_bin", "chrom", "offset", "Z_raw"])
    chrom_df.to_csv(OUTDIR / f"profile_{stratum}_bychrom.tsv", sep="\t", index=False)

    print(f"  [OK] {stratum}")

print(f"\n[DONE] Results saved to {OUTDIR}")

In [ ]:
# Plot score-stratified profiles and forest plot of normalized enrichment.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D

BASE = "data"

ANCHOR_MODES = ["left", "right"]

INDIRS = {
    a: Path(f"{BASE}/mutation_profiles_scorestrat_tol10_{a}")
    for a in ANCHOR_MODES
}

OUTDIR = Path(f"{BASE}/mutation_profiles_scorestrat_plots_tol10")
OUTDIR.mkdir(exist_ok=True)

LABEL_FONTSIZE = 26
TICK_FONTSIZE  = 24
TITLE_FONTSIZE = 28

PALETTE = ["#A4C3DE", "#4D82B0", "#18344D"]

DUAL_ANCHOR_STRATA = {
    "DEL_gt50bp",
    "CDEL_gt50bp",
    "SUB_50_200",
    "SUB_gt200",
}

TITLE_MAP = {
    "SNP": "SNV",
    "MNP_2_20": "MNV 2–20 bp",
    "SUB_20_50": "Sub. 20–50 bp",
    "SUB_50_200": "Substitutions 50–200 bp",
    "SUB_gt200": "Substitutions >200 bp",
    "DEL_1bp": "1 bp deletions",
    "DEL_2_5bp": "2–5 bp del.",
    "DEL_6_50bp": "6–50 bp deletions",
    "DEL_gt50bp": ">50 bp deletions",
    "INS_1bp": "1 bp insertions",
    "INS_2_5bp": "2–5 bp insertions",
    "INS_6_50bp": "6–50 bp ins.",
    "INS_51_200bp": "51–200 bp insertions",
    "INS_gt200bp": ">200 bp insertions",
    "CDEL_1_5bp": "Complex deletion 1–5 bp",
    "CDEL_6_50bp": "Comp. del. 6–50 bp",
    "CDEL_gt50bp": "Comp. del. >50 bp",
    "CINS_1_5bp": "Comp. ins. 1–5 bp",
    "CINS_6_50bp": "Comp. ins. 6–50 bp",
    "CINS_51_200bp": "Comp. ins. 51–200 bp",
    "CINS_gt200bp": "Comp. ins. >200 bp",
}

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def plot_scorestrat_profile(df_all, title, outpath):

    bins_present = [b for b in df_all["score_bin"].unique()]

    bins_present.sort(
        key=lambda x: float(x.split("(")[1].split("–")[0])
    )

    plt.figure(figsize=(8.6, 6.2))

    for i, lb in enumerate(bins_present):

        sub = (
            df_all[df_all["score_bin"] == lb]
            .sort_values("offset")
        )

        if sub.empty:
            continue

        c = PALETTE[i % len(PALETTE)]

        plt.plot(
            sub["offset"],
            sub["Z_mean"],
            lw=4.0,
            color=c,
            label=lb,
        )

        plt.fill_between(
            sub["offset"],
            sub["Z_lo"],
            sub["Z_hi"],
            alpha=0.25,
            color=c,
        )

    plt.axvline(0, ls="--", color="black", lw=2)
    plt.axhline(1, ls=":", color="black", lw=2)

    plt.xlabel(
        "Position relative to mutation anchor (bp)",
        fontsize=LABEL_FONTSIZE,
    )

    plt.ylabel(
        "Relative density",
        fontsize=LABEL_FONTSIZE,
    )

    plt.title(
        title,
        fontsize=TITLE_FONTSIZE,
        pad=18,
    )

    plt.tick_params(labelsize=TICK_FONTSIZE)

    ax = plt.gca()

    ax.set_xticks([-400, 0, 400])

    yticks = ax.get_yticks()
    ax.set_yticks(yticks[::2])

    despine(ax)

    plt.tight_layout()

    plt.savefig(outpath, dpi=600)

    plt.show()

for anchor in ANCHOR_MODES:

    indir = INDIRS.get(anchor)

    if indir is None or not indir.exists():
        continue

    for path in sorted(indir.glob("profile_*.tsv")):

        df = pd.read_csv(path, sep="\t")

        stratum = path.stem.replace("profile_", "")

        if stratum not in TITLE_MAP:
            continue

        if stratum in DUAL_ANCHOR_STRATA:

            anchor_label = (
                "right breakpoint"
                if anchor == "right"
                else "left breakpoint"
            )

            title = f"{TITLE_MAP[stratum]} ({anchor_label})"

        else:

            if anchor == "right":
                continue

            title = TITLE_MAP[stratum]

        outname = f"{stratum}_{anchor}.png"

        outpath = OUTDIR / outname

        plot_scorestrat_profile(df, title, outpath)

        bins_in_file = sorted(
            df["score_bin"].unique(),
            key=lambda x: float(x.split("(")[1].split("–")[0]),
        )

        for lb in bins_in_file:

            sub = df[df["score_bin"] == lb]

            window = sub[
                (sub["offset"] >= -5) &
                (sub["offset"] <= 5)
            ]

            if window.empty:
                continue

            z_win = window["Z_mean"].mean()

            print(
                f"[±5bp] "
                f"{stratum:15s} | "
                f"{lb:22s} | "
                f"Z_mean = {z_win:.3f}"
            )

        print()

sample_tsv = next(
    iter(sorted(
        Path(INDIRS["left"]).glob("profile_*.tsv")
    )),
    None,
)

if sample_tsv:

    sample_df = pd.read_csv(sample_tsv, sep="\t")

    bins_all = sorted(
        sample_df["score_bin"].unique(),
        key=lambda x: float(x.split("(")[1].split("–")[0]),
    )

    legend_fig = plt.figure(figsize=(5.0, 1.8))

    ax_leg = legend_fig.add_subplot(111)

    ax_leg.axis("off")

    legend_handles = [
        Line2D(
            [0],
            [0],
            lw=4.0,
            label=lb,
            color=PALETTE[i % len(PALETTE)],
        )
        for i, lb in enumerate(bins_all)
    ]

    ax_leg.legend(
        handles=legend_handles,
        loc="center",
        frameon=False,
        fontsize=13,
    )

    legend_fig.savefig(
        OUTDIR / "scorestrat_legend.png",
        dpi=600,
        bbox_inches="tight",
    )

    plt.show()

print("[DONE] All score-stratified plots generated.")

from matplotlib.ticker import MaxNLocator

forest_data = []

for anchor in ["left"]:

    indir = INDIRS.get(anchor)

    if indir is None or not indir.exists():
        continue

    for path in sorted(indir.glob("profile_*.tsv")):

        df = pd.read_csv(path, sep="\t")

        stratum = path.stem.replace("profile_", "")

        if stratum not in TITLE_MAP:
            continue

        if stratum in DUAL_ANCHOR_STRATA and anchor == "right":
            continue

        bins_in_file = sorted(
            df["score_bin"].unique(),
            key=lambda x: float(x.split("(")[1].split("–")[0]),
        )

        for lb in bins_in_file:

            sub = df[df["score_bin"] == lb]

            window = sub[
                (sub["offset"] >= -5) &
                (sub["offset"] <= 5)
            ]

            if window.empty:
                continue

            forest_data.append({
                "stratum": stratum,
                "score_bin": lb,
                "z_mean":  window["Z_mean"].mean(),
                "z_lo":    window["Z_lo"].mean(),
                "z_hi":    window["Z_hi"].mean(),
            })

fdf = pd.DataFrame(forest_data)

if len(fdf) == 0:

    print("[WARN] No data for forest plot")

else:

    FOREST_STRATA = [
        "CDEL_6_50bp",
        "DEL_2_5bp",
        "INS_6_50bp",
        "CINS_1_5bp",
        "MNP_2_20",
        "SNP",
        "SUB_20_50",
    ]

    strata_order = [
        s for s in FOREST_STRATA
        if s in fdf["stratum"].unique()
    ]

    fdf = fdf[
        fdf["stratum"].isin(strata_order)
    ].copy()

    all_bins = sorted(
        fdf["score_bin"].unique(),
        key=lambda x: float(x.split("(")[1].split("–")[0]),
    )

    n_bins = len(all_bins)

    fdf["excess"]    = fdf["z_mean"] - 1.0
    fdf["excess_lo"] = fdf["z_lo"]   - 1.0
    fdf["excess_hi"] = fdf["z_hi"]   - 1.0

    normalized = []

    for stratum, grp in fdf.groupby("stratum"):

        total_abs = grp["excess"].abs().sum()

        if total_abs == 0:
            continue

        scale = 100.0 / total_abs

        for _, row in grp.iterrows():

            normalized.append({
                "stratum":  row["stratum"],
                "score_bin": row["score_bin"],
                "pct_mean": row["excess"]    * scale,
                "pct_lo":   row["excess_lo"] * scale,
                "pct_hi":   row["excess_hi"] * scale,
            })

    ndf = pd.DataFrame(normalized)

    group_spacing = 1.4
    spread        = 0.35

    x_positions = (
        np.arange(len(strata_order))
        * group_spacing
    )

    offsets_within = np.linspace(
        -spread / 2,
        spread / 2,
        n_bins,
    )

    fig, ax = plt.subplots(figsize=(14, 8))

    for bin_idx, lb in enumerate(all_bins):

        color = PALETTE[bin_idx % len(PALETTE)]

        sub = ndf[
            ndf["score_bin"] == lb
        ]

        for s_idx, stratum in enumerate(strata_order):

            row = sub[
                sub["stratum"] == stratum
            ]

            if row.empty:
                continue

            r = row.iloc[0]

            x = (
                x_positions[s_idx]
                + offsets_within[bin_idx]
            )

            ax.errorbar(
                x,
                r["pct_mean"],

                yerr=[
                    [r["pct_mean"] - r["pct_lo"]],
                    [r["pct_hi"]   - r["pct_mean"]],
                ],

                fmt="o",

                color=color,

                markersize=8,

                capsize=4,
                capthick=1.8,
                linewidth=1.8,

                zorder=3,
            )

    ax.axhline(
        0,
        ls="--",
        color="black",
        lw=1.5,
        alpha=0.6,
        zorder=1,
    )

    ax.set_xticks(x_positions)

    ax.set_xticklabels(
        [TITLE_MAP.get(s, s) for s in strata_order],
        rotation=45,
        ha="right",
        fontsize=TICK_FONTSIZE - 4,
    )

    ax.set_ylabel(
        "Normalized enrichment (%)",
        fontsize=LABEL_FONTSIZE,
    )

    ax.tick_params(
        axis="y",
        labelsize=TICK_FONTSIZE,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()

    forest_out = OUTDIR / "forest_scorestrat_normalized.png"

    fig.savefig(
        forest_out,
        dpi=600,
        bbox_inches="tight",
    )

    plt.show()

    print(f"[OK] Forest plot: {forest_out}")

In [ ]:
# Bar chart of score-stratified normalized enrichment.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from matplotlib.lines import Line2D

BASE = "data"

ANCHOR_MODES = ["left", "right"]

INDIRS = {
    a: Path(f"{BASE}/mutation_profiles_scorestrat_tol10_{a}")
    for a in ANCHOR_MODES
}

OUTDIR = Path(f"{BASE}/mutation_profiles_scorestrat_plots_tol10")
OUTDIR.mkdir(exist_ok=True)

LABEL_FONTSIZE = 26
TICK_FONTSIZE  = 24
SIG_FONTSIZE   = 16

PALETTE = ["#A4C3DE", "#4D82B0", "#18344D"]

DUAL_ANCHOR_STRATA = {
    "DEL_gt50bp", "CDEL_gt50bp", "SUB_50_200", "SUB_gt200",
}

TITLE_MAP = {
    "SNP": "SNV", "MNP_2_20": "MNV 2-20",
    "SUB_20_50": "Sub 20-50",
    "SUB_50_200_left": "Sub 50-200 (L)", "SUB_50_200_right": "Sub 50-200 (R)",
    "SUB_gt200_left": "Sub >200 (L)", "SUB_gt200_right": "Sub >200 (R)",
    "DEL_1bp": "Del 1", "DEL_2_5bp": "Del 2-5", "DEL_6_50bp": "Del 6-50",
    "DEL_gt50bp_left": "Del >50 (L)", "DEL_gt50bp_right": "Del >50 (R)",
    "INS_1bp": "Ins 1", "INS_2_5bp": "Ins 2-5", "INS_6_50bp": "Ins 6-50",
    "INS_51_200bp": "Ins 51-200", "INS_gt200bp": "Ins >200",
    "CDEL_1_5bp": "CDel 1-5", "CDEL_6_50bp": "CDel 6-50",
    "CDEL_gt50bp_left": "CDel >50 (L)", "CDEL_gt50bp_right": "CDel >50 (R)",
    "CINS_1_5bp": "CIns 1-5", "CINS_6_50bp": "CIns 6-50",
    "CINS_51_200bp": "CIns 51-200", "CINS_gt200bp": "CIns >200",
}

FOREST_STRATA = [
    "SNP",
    "MNP_2_20",
    "SUB_20_50",
    "DEL_2_5bp",
    "DEL_6_50bp",    
    "CDEL_6_50bp",
    "INS_6_50bp",    
    "CINS_1_5bp",
    "CINS_6_50bp",    
]

WINDOW_RADIUS_TEST = 5
MIN_CHROMS = 6
N_BOOTS = 50_000
SEED = 42

def load_bychrom_scorestrat(indir, stratum):
    path = indir / f"profile_{stratum}_bychrom.tsv"
    if not path.exists():
        return None
    return pd.read_csv(path, sep="\t")

def per_chrom_normalized_window(bychrom_df, radius):
    result = {}
    for lb in bychrom_df["score_bin"].unique():
        sub = bychrom_df[bychrom_df["score_bin"] == lb]
        result[lb] = {}
        for chrom, cd in sub.groupby("chrom"):
            z_all = cd["Z_raw"].to_numpy(dtype=np.float64)
            z_full_mean = z_all.mean()
            if z_full_mean == 0:
                continue
            win = cd[(cd["offset"] >= -radius) & (cd["offset"] <= radius)]
            z_win_mean = win["Z_raw"].to_numpy(dtype=np.float64).mean()
            result[lb][chrom] = z_win_mean / z_full_mean
    return result

def pairwise_wilcoxon(chrom_vals, bins):
    results = []
    for b1, b2 in combinations(bins, 2):
        chroms = sorted(set(chrom_vals.get(b1, {})) & set(chrom_vals.get(b2, {})))
        n = len(chroms)
        if n < MIN_CHROMS:
            results.append({
                "bin1": b1, "bin2": b2,
                "stat": np.nan, "p": np.nan,
                "n_chroms": n,
                "mean1": np.nan, "mean2": np.nan,
                "direction": "",
            })
            continue

        v1 = np.array([chrom_vals[b1][c] for c in chroms])
        v2 = np.array([chrom_vals[b2][c] for c in chroms])
        diff = v1 - v2

        if np.allclose(diff, 0):
            results.append({
                "bin1": b1, "bin2": b2,
                "stat": 0.0, "p": 1.0,
                "n_chroms": n,
                "mean1": v1.mean(), "mean2": v2.mean(),
                "direction": "=",
            })
            continue

        try:
            stat, p = wilcoxon(v1, v2, alternative="two-sided", zero_method="wilcox")
        except ValueError:
            stat, p = np.nan, np.nan

        direction = ">" if v1.mean() > v2.mean() else "<"

        results.append({
            "bin1": b1, "bin2": b2,
            "stat": float(stat) if not np.isnan(stat) else np.nan,
            "p": float(p) if not np.isnan(p) else np.nan,
            "n_chroms": n,
            "mean1": v1.mean(), "mean2": v2.mean(),
            "direction": direction,
        })

    return results

indir = INDIRS["left"]
forest_data = []
all_test_results = []

all_strata = sorted([
    p.stem.replace("profile_", "").replace("_bychrom", "")
    for p in indir.glob("profile_*_bychrom.tsv")
])
print(f"Found {len(all_strata)} strata with bychrom data: {all_strata}")

for stratum in all_strata:
    prof_path = indir / f"profile_{stratum}.tsv"
    if not prof_path.exists():
        continue

    df = pd.read_csv(prof_path, sep="\t")
    bins_in_file = sorted(
        df["score_bin"].unique(),
        key=lambda x: float(x.split("(")[1].split("–")[0]),
    )

    for lb in bins_in_file:
        sub = df[df["score_bin"] == lb]
        window = sub[(sub["offset"] >= -WINDOW_RADIUS_TEST) &
                     (sub["offset"] <= WINDOW_RADIUS_TEST)]
        if window.empty:
            continue
        forest_data.append({
            "stratum": stratum,
            "score_bin": lb,
            "z_mean": window["Z_mean"].mean(),
            "z_lo":   window["Z_lo"].mean(),
            "z_hi":   window["Z_hi"].mean(),
        })

    bychrom_df = load_bychrom_scorestrat(indir, stratum)
    if bychrom_df is None:
        print(f"  [WARN] No _bychrom.tsv for {stratum}")
        continue

    chrom_vals = per_chrom_normalized_window(bychrom_df, WINDOW_RADIUS_TEST)
    pw_results = pairwise_wilcoxon(chrom_vals, bins_in_file)
    for r in pw_results:
        r["stratum"] = stratum
        all_test_results.append(r)

fdf = pd.DataFrame(forest_data)
tdf = pd.DataFrame(all_test_results)

if tdf.empty:
    print("[WARN] No test results")
else:
    mask = tdf["p"].notna()
    tdf["q_bh"] = np.nan
    
    if mask.any():
        tdf.loc[mask, "q_bh"] = multipletests(
            tdf.loc[mask, "p"].values,
            method="fdr_bh"
        )[1]

    def sig_marker(p):
        if pd.isna(p): return ""
        if p < 1e-3: return "***"
        if p < 1e-2: return "**"
        if p < 5e-2: return "*"
        return "ns"

    tdf["sig"] = tdf["q_bh"].apply(sig_marker)

    print("\n" + "=" * 80)
    print("PAIRWISE CHROMOSOME-LEVEL WILCOXON (two-sided, BH-FDR corrected)")
    print("=" * 80)
    print(tdf[["stratum", "bin1", "bin2", "direction", "mean1", "mean2",
               "p", "q_bh", "sig", "n_chroms"]].to_string(index=False))

    tdf.to_csv(OUTDIR / "scorestrat_pairwise_tests_ALL.tsv", sep="\t", index=False)
    print(f"\n  Wrote: {OUTDIR / 'scorestrat_pairwise_tests_ALL.tsv'}")

    print("\n" + "=" * 80)
    print("PATTERN SUMMARY: which group dominates per stratum?")
    print("=" * 80)
    print(f"{'stratum':<20} {'G1_mean':>8} {'G2_mean':>8} {'G3_mean':>8}  {'pattern':<20} {'G1vG2':>6} {'G2vG3':>6} {'G1vG3':>6}")

    for stratum in all_strata:
        st = tdf[tdf["stratum"] == stratum]
        if st.empty:
            continue

        sf = pd.DataFrame([r for r in forest_data if r["stratum"] == stratum])
        if sf.empty:
            continue
        sf = sf.sort_values("score_bin",
                            key=lambda s: s.map(lambda x: float(x.split("(")[1].split("–")[0])))
        bin_means = sf.set_index("score_bin")["z_mean"].to_dict()
        bins_sorted = list(sf["score_bin"])

        if len(bins_sorted) < 3:
            continue

        g1m = bin_means[bins_sorted[0]]
        g2m = bin_means[bins_sorted[1]]
        g3m = bin_means[bins_sorted[2]]

        ranked = sorted([(g1m, "G1"), (g2m, "G2"), (g3m, "G3")], reverse=True)
        pattern = ">".join([r[1] for r in ranked])

        def get_sig(b1, b2):
            row = st[(st["bin1"] == b1) & (st["bin2"] == b2)]
            if row.empty:
                row = st[(st["bin1"] == b2) & (st["bin2"] == b1)]
            if row.empty:
                return ""
            return row.iloc[0]["sig"]

        s12 = get_sig(bins_sorted[0], bins_sorted[1])
        s23 = get_sig(bins_sorted[1], bins_sorted[2])
        s13 = get_sig(bins_sorted[0], bins_sorted[2])

        print(f"{stratum:<20} {g1m:>8.3f} {g2m:>8.3f} {g3m:>8.3f}  {pattern:<20} {s12:>6} {s23:>6} {s13:>6}")

if fdf.empty:
    print("[WARN] No forest data")
else:
    strata_order = [s for s in FOREST_STRATA if s in fdf["stratum"].unique()]
    fdf = fdf[fdf["stratum"].isin(strata_order)].copy()

    all_bins = sorted(
        fdf["score_bin"].unique(),
        key=lambda x: float(x.split("(")[1].split("–")[0]),
    )
    n_bins = len(all_bins)

    fdf["excess"]    = fdf["z_mean"] - 1.0
    fdf["excess_lo"] = fdf["z_lo"]   - 1.0
    fdf["excess_hi"] = fdf["z_hi"]   - 1.0

    normalized = []
    for stratum, grp in fdf.groupby("stratum"):
        total_abs = grp["excess"].abs().sum()
        if total_abs == 0:
            continue
        scale = 100.0 / total_abs
        for _, row in grp.iterrows():
            normalized.append({
                "stratum":   row["stratum"],
                "score_bin": row["score_bin"],
                "pct_mean":  row["excess"]    * scale,
                "pct_lo":    row["excess_lo"] * scale,
                "pct_hi":    row["excess_hi"] * scale,
            })
    ndf = pd.DataFrame(normalized)

    group_spacing = 1.4
    spread = 0.50
    x_positions = np.arange(len(strata_order)) * group_spacing
    offsets_within = np.linspace(-spread / 2, spread / 2, n_bins)

    fig, ax = plt.subplots(figsize=(14, 9))

    bar_width = 0.25
    
    for bin_idx, lb in enumerate(all_bins):
        color = PALETTE[bin_idx % len(PALETTE)]
        sub = ndf[ndf["score_bin"] == lb]
    
        for s_idx, stratum in enumerate(strata_order):
            row = sub[sub["stratum"] == stratum]
            if row.empty:
                continue
    
            r = row.iloc[0]
            x = x_positions[s_idx] + offsets_within[bin_idx]
    
            y = r["pct_mean"]
            yerr = [[r["pct_mean"] - r["pct_lo"]],
                    [r["pct_hi"] - r["pct_mean"]]]
    
            ax.bar(
                x, y,
                width=bar_width,
                color=color,
                edgecolor="black",
                linewidth=0.8,
                zorder=3,
            )
    
            ax.errorbar(
                x, y,
                yerr=yerr,
                fmt="none",
                ecolor="black",
                capsize=4,
                capthick=1.6,
                linewidth=1.6,
                zorder=4,
            )

    ax.axhline(0, ls="--", color="black", lw=1.5, alpha=0.6, zorder=1)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        [TITLE_MAP.get(s, s) for s in strata_order],
        rotation=45, ha="right", fontsize=TICK_FONTSIZE - 4,
    )
    ax.set_ylabel("Normalized enrichment (%)", fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()
    forest_out = OUTDIR / "bar_scorestrat_normalized.png"
    fig.savefig(forest_out, dpi=600, bbox_inches="tight")
    fig.savefig(forest_out.with_suffix(".pdf"), bbox_inches="tight")
    plt.show()
    print(f"[OK] Forest plot: {forest_out}")

print("[DONE]")

## 13. Enrichment vs variant size

In [ ]:
# Summarise variant sizes for binning.
import pandas as pd
import numpy as np
path = "data/mutations/hprc_mutations"
df = pd.read_csv(path, sep="\t", usecols=["struct_class", "dominant_op", "ref_span", "alt_span"])

CLASSES = {"del", "ins", "complex", "substitution", "mnp"}
df = df[df["struct_class"].isin(CLASSES)].copy()

conditions = [
    df["struct_class"].isin(["substitution", "del"]) | (df["dominant_op"] == "del_like"),
    (df["struct_class"] == "ins") | (df["dominant_op"] == "ins_like"),
    df["dominant_op"] == "balanced",
]
choices = [df["ref_span"], df["alt_span"], df["ref_span"]]
df["event_size"] = np.select(conditions, choices, default=df[["ref_span", "alt_span"]].max(axis=1))

def reporting_label(row):
    if row["struct_class"] == "del":          return "DEL"
    elif row["struct_class"] == "ins":        return "INS"
    elif row["struct_class"] == "mnp": return "MNP"
    elif row["struct_class"] == "substitution": return "SUB"
    elif row["dominant_op"] == "del_like":    return "CDEL"
    elif row["dominant_op"] == "ins_like":    return "CINS"
    else:                                     return "CBAL"

df["category"] = df.apply(reporting_label, axis=1)

def size_stats(s):
    return pd.Series({
        "count": len(s), "min": s.min(), "q1": s.quantile(0.25),
        "median": s.median(), "mean": s.mean(),
        "q3": s.quantile(0.75), "max": s.max(), "std": s.std(),
    })

for cat in sorted(df["category"].unique()):
    stats = size_stats(df.loc[df["category"] == cat, "event_size"])
    print(f"\n--- {cat} (n = {int(stats['count']):,}) ---")
    for k, v in stats.items():
        print(f"  {k:8s}: {v:>12.1f}")

print(f"\n--- ALL (n = {len(df):,}) ---")
for k, v in size_stats(df["event_size"]).items():
    print(f"  {k:8s}: {v:>12.1f}")

In [ ]:
# Per-event Z-DNA enrichment vs mutation size (compute).
import matplotlib.pyplot as plt

CLASS_COLORS = {
    "DEL":  "#E63946",
    "INS":  "#457B9D",
    "CDEL": "#E9C46A",
    "CINS": "#2A9D8F",
    "SUB":  "#264653",
    "MNP":  "#3B7362",
}

CLASS_LABELS = {
    "DEL":  "Deletions",
    "INS":  "Insertions",
    "CDEL": "Complex (deletion-like)",
    "CINS": "Complex (insertion-like)",
    "SUB":  "Substitutions",
    "MNP":  "Multiple nucl. pol.",
}

WINDOW_STYLES = {
    10:  {"lw": 2.5, "markersize": 8, "alpha": 1.0, "ls": "-"},
    25: {"lw": 1.8, "markersize": 6, "alpha": 0.7, "ls": "--"},
    50: {"lw": 1.4, "markersize": 5, "alpha": 0.5, "ls": ":"},
}

LABEL_FONTSIZE = 22
TICK_FONTSIZE  = 18
TITLE_FONTSIZE = 24

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for cat in sorted(rdf["class"].unique()):
    cat_data = rdf[rdf["class"] == cat]

    bins_present = [b for b in BIN_ORDER
                    if b in cat_data["bin_label"].values]
    if len(bins_present) < 2:
        continue

    fig, ax = plt.subplots(figsize=(12, 6))
    color = CLASS_COLORS.get(cat, "#333333")
    x = np.arange(len(bins_present))

    for w in WINDOWS:
        sub = cat_data[cat_data["window"] == w]
        y_mean = []
        y_lo = []
        y_hi = []
        x_valid = []

        for i, b in enumerate(bins_present):
            row = sub[sub["bin_label"] == b]
            if row.empty:
                continue
            r = row.iloc[0]
            x_valid.append(i)
            y_mean.append(r["enrich_mean"])
            y_lo.append(r["enrich_lo"])
            y_hi.append(r["enrich_hi"])

        x_valid = np.array(x_valid)
        y_mean = np.array(y_mean)
        y_lo = np.array(y_lo)
        y_hi = np.array(y_hi)

        sty = WINDOW_STYLES[w]
        ax.errorbar(
            x_valid, y_mean,
            yerr=[y_mean - y_lo, y_hi - y_mean],
            fmt="o", color=color,
            lw=sty["lw"], markersize=sty["markersize"],
            alpha=sty["alpha"], linestyle=sty["ls"],
            capsize=4, capthick=1.5, zorder=3,
            label=f"±{w} bp",
        )
        ax.plot(x_valid, y_mean, color=color,
                lw=sty["lw"], alpha=sty["alpha"], ls=sty["ls"], zorder=2)

    ax.axhline(1, ls=":", color="black", lw=1.5, alpha=0.6)

    ax.set_xticks(x)
    ax.set_xticklabels(bins_present, rotation=45, ha="right",
                       fontsize=TICK_FONTSIZE - 4)

    ax.set_xlabel("Variant size range (bp)", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel("Z-DNA enrichment at anchor", fontsize=LABEL_FONTSIZE)
    ax.set_title(CLASS_LABELS.get(cat, cat),
                 fontsize=TITLE_FONTSIZE, fontweight="bold")
    ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)
    despine(ax)
    ax.legend(fontsize=TICK_FONTSIZE - 2, frameon=False)

    fig.tight_layout()
    fig.savefig(OUTDIR / f"size_vs_enrichment_{cat}.png",
                dpi=600, bbox_inches="tight")
    plt.show(fig)

fig, ax = plt.subplots(figsize=(14, 7))
group_gap = 2
x_cursor = 0
xtick_positions = []
xtick_labels_list = []
for cat in ["DEL", "INS", "CDEL", "CINS", "SUB"]:
    sub = rdf[(rdf["class"] == cat) & (rdf["window"] == 10)].copy()
    bins_present = [b for b in BIN_ORDER if b in sub["bin_label"].values]
    sub = sub.set_index("bin_label").loc[bins_present].reset_index()
    if len(sub) < 2:
        continue
    color = CLASS_COLORS.get(cat, "#333333")
    x = np.arange(len(sub)) + x_cursor
    ax.errorbar(
        x, sub["enrich_mean"],
        yerr=[sub["enrich_mean"] - sub["enrich_lo"],
              sub["enrich_hi"] - sub["enrich_mean"]],
        fmt="-o", color=color, lw=2.0, markersize=7,
        capsize=3, capthick=1.5, label=CLASS_LABELS.get(cat, cat), zorder=3,
    )
    center = x_cursor + len(sub) / 2 - 0.5
    ax.text(center, -0.08, CLASS_LABELS.get(cat, cat),
            transform=ax.get_xaxis_transform(),
            ha="center", va="top", fontsize=TICK_FONTSIZE - 4,
            fontweight="bold", color=color)
    for i, row in sub.iterrows():
        xtick_positions.append(x[i])
        xtick_labels_list.append(row["bin_label"])
    x_cursor += len(sub) + group_gap
ax.axhline(1, ls=":", color="black", lw=1.5, alpha=0.6)
ax.set_xticks(xtick_positions)
ax.set_xticklabels(xtick_labels_list, rotation=90, ha="center", fontsize=9)
ax.set_ylabel("Z-DNA enrichment at anchor (±10 bp)", fontsize=LABEL_FONTSIZE)
ax.set_title("Z-DNA enrichment vs. variant size",
             fontsize=TITLE_FONTSIZE, fontweight="bold")
ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)
despine(ax)
ax.legend(fontsize=TICK_FONTSIZE - 4, frameon=False, loc="upper right")

fig.tight_layout()
fig.savefig(OUTDIR / "size_vs_enrichment_ALL.png",
            dpi=600, bbox_inches="tight")
plt.show(fig)

print(f"\n[DONE] All outputs in {OUTDIR}")
print("[INFO] Plotting combined MNP / Substitutions...")

fig, ax = plt.subplots(figsize=(12, 6))

for w in WINDOWS:
    sub = rdf[(rdf["class"].isin(["MNP", "SUB"])) & (rdf["window"] == w)].copy()

    if sub.empty:
        continue

    agg = (
        sub.groupby("bin_label")
        .apply(lambda d: pd.Series({
            "enrich_mean": np.average(d["enrich_mean"], weights=d["n_muts"]),
            "enrich_lo":   np.average(d["enrich_lo"],   weights=d["n_muts"]),
            "enrich_hi":   np.average(d["enrich_hi"],   weights=d["n_muts"]),
            "n_muts":      d["n_muts"].sum(),
        }))
        .reset_index()
    )

    bins_present = [b for b in BIN_ORDER if b in agg["bin_label"].values]
    agg = agg.set_index("bin_label").loc[bins_present].reset_index()

    if len(agg) < 2:
        continue

    x = np.arange(len(agg))
    y_mean = agg["enrich_mean"].values
    y_lo   = agg["enrich_lo"].values
    y_hi   = agg["enrich_hi"].values

    sty = WINDOW_STYLES[w]

    ax.errorbar(
        x, y_mean,
        yerr=[y_mean - y_lo, y_hi - y_mean],
        fmt="o", color="#6A4C93",   # distinct color
        lw=sty["lw"], markersize=sty["markersize"],
        alpha=sty["alpha"], linestyle=sty["ls"],
        capsize=4, capthick=1.5,
        label=f"±{w} bp",
    )

    ax.plot(x, y_mean, color="#6A4C93",
            lw=sty["lw"], alpha=sty["alpha"], ls=sty["ls"])

ax.axhline(1, ls=":", color="black", lw=1.5, alpha=0.6)

ax.set_xticks(np.arange(len(bins_present)))
ax.set_xticklabels(bins_present, rotation=45, ha="right",
                   fontsize=TICK_FONTSIZE - 4)

ax.set_xlabel("Variant size range (bp)", fontsize=LABEL_FONTSIZE)
ax.set_ylabel("Z-DNA enrichment at anchor", fontsize=LABEL_FONTSIZE)
ax.set_title("MNP / Substitutions",
             fontsize=TITLE_FONTSIZE, fontweight="bold")

ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)
despine(ax)
ax.legend(fontsize=TICK_FONTSIZE - 2, frameon=False)

fig.tight_layout()
fig.savefig(OUTDIR / "size_vs_enrichment_MNP_SUB.png",
            dpi=600, bbox_inches="tight")
plt.show(fig)

In [ ]:
# Per-event Z-DNA enrichment vs size: size-binned block-bootstrap means.
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr

BASE = "data"

MASTER        = f"{BASE}/controls/CHM13_controls_MASTER.tsv"
MUT_BED       = f"{BASE}/mutations/hprc_mutations.bed.gz"
MUT_META      = f"{BASE}/mutations/hprc_mutations.meta.tsv.gz"
CHROM_LEN_TSV = f"{BASE}/chm13/chm13_seqreport_zdna.tsv"

TOL = "10%"
WINDOWS = [10, 25, 50]
N_BOOTS = 1000
CI_LOW, CI_HIGH = 2.5, 97.5
MIN_MUTS = 500

OUTDIR = Path(f"{BASE}/mutation_size_vs_zdna_eventlevel")
OUTDIR.mkdir(parents=True, exist_ok=True)

def build_bool_coverage(starts, ends, chrom_len):
    starts = np.clip(np.asarray(starts, dtype=np.int64), 0, chrom_len)
    ends   = np.clip(np.asarray(ends,   dtype=np.int64), 0, chrom_len)
    mask = ends > starts
    if not np.any(mask):
        return np.zeros(chrom_len, dtype=bool)
    starts, ends = starts[mask], ends[mask]
    diff = np.zeros(chrom_len + 1, dtype=np.int32)
    np.add.at(diff, starts,  1)
    np.add.at(diff, ends,   -1)
    return np.cumsum(diff[:-1]) > 0

DUAL_ANCHOR_STRATA = {"DEL_gt50bp", "CDEL_gt50bp", "SUB_50_200", "SUB_gt200"}
MIDPOINT_ANCHOR_STRATA = {"DEL_1bp", "DEL_2_5bp", "DEL_6_50bp",
                          "CDEL_1_5bp", "CDEL_6_50bp", "SUB_20_50"}
SINGLE_ANCHOR_STRATA = {
    "INS_1bp", "INS_2_5bp", "INS_6_50bp", "INS_51_200bp", "INS_gt200bp",
    "CINS_1_5bp", "CINS_6_50bp", "CINS_51_200bp", "CINS_gt200bp",
    "SNP", "MNP_2_20",
}

def assign_stratum(sc, dom, L):
    if sc == "snp":        return "SNP"
    if sc == "mnp":        return "MNP_2_20"
    if sc == "substitution":
        if L <= 50:   return "SUB_20_50"
        elif L <= 200: return "SUB_50_200"
        else:          return "SUB_gt200"
    if sc == "del":
        if L == 1:    return "DEL_1bp"
        elif L <= 5:  return "DEL_2_5bp"
        elif L <= 50: return "DEL_6_50bp"
        else:         return "DEL_gt50bp"
    if sc == "ins":
        if L == 1:    return "INS_1bp"
        elif L <= 5:  return "INS_2_5bp"
        elif L <= 50: return "INS_6_50bp"
        elif L <= 200: return "INS_51_200bp"
        else:         return "INS_gt200bp"
    if sc == "complex" and dom == "del_like":
        if L <= 5:   return "CDEL_1_5bp"
        elif L <= 50: return "CDEL_6_50bp"
        else:         return "CDEL_gt50bp"
    if sc == "complex" and dom == "ins_like":
        if L <= 5:   return "CINS_1_5bp"
        elif L <= 50: return "CINS_6_50bp"
        elif L <= 200: return "CINS_51_200bp"
        else:         return "CINS_gt200bp"
    return None

def assign_category(sc, dom):
    if sc == "del":            return "DEL"
    if sc == "ins":            return "INS"
    if sc == "substitution":   return "SUB"
    if sc == "mnp":            return "MNP"
    if sc == "complex" and dom == "del_like": return "CDEL"
    if sc == "complex" and dom == "ins_like": return "CINS"
    return None

def compute_anchors(starts, refs, strata):
    anchors = starts.copy()
    is_mid = np.isin(strata, list(MIDPOINT_ANCHOR_STRATA))
    if is_mid.any():
        anchors[is_mid] = starts[is_mid] + (refs[is_mid] - 1) // 2
    return anchors

SIZE_BOUNDARIES = [1, 2, 3, 6, 11, 21, 51, 101, 501]

BIN_ORDER = []
for i in range(len(SIZE_BOUNDARIES)):
    lo = SIZE_BOUNDARIES[i]
    if i + 1 < len(SIZE_BOUNDARIES):
        hi = SIZE_BOUNDARIES[i + 1] - 1
        BIN_ORDER.append(f"{lo}\u2013{hi}")
    else:
        BIN_ORDER.append(f"{lo}+")

def assign_size_bin(sizes):
    labels = np.empty(len(sizes), dtype=object)
    s = np.asarray(sizes)
    for i in range(len(SIZE_BOUNDARIES)):
        lo = SIZE_BOUNDARIES[i]
        if i + 1 < len(SIZE_BOUNDARIES):
            hi = SIZE_BOUNDARIES[i + 1] - 1
            mask = (s >= lo) & (s <= hi)
            labels[mask] = f"{lo}\u2013{hi}"
        else:
            mask = s >= lo
            labels[mask] = f"{lo}+"
    return labels

def chrom_block_bootstrap_mean(chroms, values, n_boot, ci_lo, ci_hi, rng):
    unique_chroms = np.unique(chroms)
    n_chroms = len(unique_chroms)
    if n_chroms < 3:
        return np.nan, np.nan, np.nan
    chrom_sums   = np.array([values[chroms == c].sum() for c in unique_chroms])
    chrom_counts = np.array([np.sum(chroms == c) for c in unique_chroms])
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n_chroms, n_chroms)
        boots[b] = chrom_sums[idx].sum() / chrom_counts[idx].sum()
    point = values.sum() / len(values)
    return point, np.percentile(boots, ci_lo), np.percentile(boots, ci_hi)

print("[INFO] Loading chromosome lengths...")
chrom_len_map = dict(
    pd.read_csv(CHROM_LEN_TSV, sep="\t",
                usecols=["ucsc_pansn_name", "gcf_seq_length"]).values
)

print("[INFO] Loading Z-DNA regions...")
m_all = pd.read_csv(MASTER, sep="\t")
m_all = m_all[(m_all["tol_level"] == TOL) & (m_all["has_control"])].drop_duplicates(
    ["Chromosome", "Start0", "End0"]
)

print("[INFO] Building Z-DNA coverage masks...")
z_cov = {}
for chrom, d in m_all.groupby("Chromosome"):
    L = int(chrom_len_map.get(chrom, 0))
    if L > 0:
        ivs = d[["Start0", "End0"]].to_numpy(dtype=np.int64)
        z_cov[chrom] = build_bool_coverage(ivs[:, 0], ivs[:, 1], L)

total_zdna_bases = sum(cov.sum() for cov in z_cov.values())
total_genome_bases = sum(int(chrom_len_map[c]) for c in z_cov)
ZDNA_GENOME_FRAC = total_zdna_bases / total_genome_bases
print(f"[INFO] Genome-wide Z-DNA fraction: {ZDNA_GENOME_FRAC:.6f} "
      f"({total_zdna_bases:,} / {total_genome_bases:,} bp)")

print("[INFO] Loading mutations...")
mut = pd.read_csv(MUT_BED, sep="\t", header=None,
                  names=["Chromosome", "Start", "End", "mut_id", "score", "strand"])
meta = pd.read_csv(MUT_META, sep="\t", compression="gzip")
need = ["mut_id", "struct_class", "dominant_op", "ref_span", "alt_span"]
mut = mut.merge(meta[need], on="mut_id", how="left").dropna()

mut["Start"]    = mut["Start"].astype(np.int64)
mut["ref_span"] = mut["ref_span"].astype(np.int64)
mut["alt_span"] = mut["alt_span"].astype(np.int64)

mut["event_size"] = np.where(
    (mut["struct_class"] == "ins") |
    ((mut["struct_class"] == "complex") & (mut["dominant_op"] == "ins_like")),
    mut["alt_span"], mut["ref_span"]
).astype(np.int64)

mut["stratum"] = [
    assign_stratum(sc, dom, L)
    for sc, dom, L in zip(mut["struct_class"], mut["dominant_op"], mut["event_size"])
]
mut = mut.dropna(subset=["stratum"]).copy()

mut["category"] = [
    assign_category(sc, dom)
    for sc, dom in zip(mut["struct_class"], mut["dominant_op"])
]
mut = mut.dropna(subset=["category"]).copy().reset_index(drop=True)

mut["anchor"] = compute_anchors(
    mut["Start"].values, mut["ref_span"].values, mut["stratum"].values,
)
print(f"  {len(mut):,} mutations after filtering")

print("[INFO] Computing per-mutation Z-DNA density...")

for w in WINDOWS:
    col = f"z_local_{w}"
    arr = np.full(len(mut), np.nan)

    for chrom, idx in mut.groupby("Chromosome", sort=False).groups.items():
        if chrom not in z_cov:
            continue
        cov = z_cov[chrom]
        L   = int(chrom_len_map[chrom])
        anchors = mut.loc[idx, "anchor"].values
        offsets = np.arange(-w, w + 1, dtype=np.int64)
        pos = anchors[:, None] + offsets[None, :]
        valid = (pos >= 0) & (pos < L)
        pos_safe = np.clip(pos, 0, L - 1)
        hits = cov[pos_safe] & valid
        n_valid = valid.sum(axis=1).astype(float)
        n_valid[n_valid == 0] = np.nan
        arr[idx.values] = hits.sum(axis=1) / n_valid

    mut[col] = arr
    mut[f"enrichment_{w}"] = mut[col] / ZDNA_GENOME_FRAC
    print(f"  \u00b1{w} bp done")

mut = mut.dropna(subset=[f"z_local_{w}" for w in WINDOWS]).copy()
print(f"  {len(mut):,} mutations with valid Z-DNA density")

save_cols = ["mut_id", "Chromosome", "anchor", "event_size", "category", "stratum"]
save_cols += [f"z_local_{w}" for w in WINDOWS]
save_cols += [f"enrichment_{w}" for w in WINDOWS]
mut[save_cols].to_csv(
    OUTDIR / "per_mutation_enrichment.tsv.gz", sep="\t", index=False,
    compression="gzip"
)

mut["size_bin"] = assign_size_bin(mut["event_size"].values)

print("[INFO] Computing chromosome-block bootstrap summaries...")
rng = np.random.default_rng(42)
results = []

for cat in sorted(mut["category"].unique()):
    cat_df = mut[mut["category"] == cat]
    present_bins = [b for b in BIN_ORDER if b in cat_df["size_bin"].values]
    print(f"\n  {cat}: {len(cat_df):,} mutations \u2192 {len(present_bins)} size bins")

    for sb in present_bins:
        bin_df = cat_df[cat_df["size_bin"] == sb]
        if len(bin_df) < MIN_MUTS:
            continue

        chroms_arr = bin_df["Chromosome"].values
        for w in WINDOWS:
            enrich_arr = bin_df[f"enrichment_{w}"].values
            point, lo, hi = chrom_block_bootstrap_mean(
                chroms_arr, enrich_arr, N_BOOTS, CI_LOW, CI_HIGH, rng
            )
            results.append({
                "class":       cat,
                "bin_label":   sb,
                "window":      w,
                "median_size": float(bin_df["event_size"].median()),
                "mean_size":   float(bin_df["event_size"].mean()),
                "size_lo":     int(bin_df["event_size"].min()),
                "size_hi":     int(bin_df["event_size"].max()),
                "n_muts":      len(bin_df),
                "enrich_mean": point,
                "enrich_lo":   lo,
                "enrich_hi":   hi,
            })
        print(f"    {sb}: n={len(bin_df):,}")

rdf = pd.DataFrame(results)
rdf.to_csv(OUTDIR / "size_vs_enrichment_summary.tsv", sep="\t", index=False)

print("\n" + "=" * 60)
print("PEAK ENRICHMENT PER CLASS (\u00b110 bp window)")
print("=" * 60)

peak_rows = []
for cat in sorted(rdf["class"].unique()):
    sub = rdf[(rdf["class"] == cat) & (rdf["window"] == 10)].copy()
    sub = sub.sort_values("median_size")
    if sub.empty:
        continue
    peak = sub.loc[sub["enrich_mean"].idxmax()]
    print(f"  {cat}: peak at {peak['bin_label']} bp  "
          f"(median {peak['median_size']:.0f} bp)  "
          f"enrich = {peak['enrich_mean']:.2f} "
          f"[{peak['enrich_lo']:.2f}\u2013{peak['enrich_hi']:.2f}]")
    peak_rows.append({
        "class":        cat,
        "peak_bin":     peak["bin_label"],
        "peak_median":  peak["median_size"],
        "peak_enrich":  peak["enrich_mean"],
        "peak_lo":      peak["enrich_lo"],
        "peak_hi":      peak["enrich_hi"],
    })

    if len(sub) >= 4:
        rho, pval = spearmanr(sub["median_size"], sub["enrich_mean"])
        if abs(rho) > 0.8:
            print(f"         monotonic trend: \u03c1 = {rho:+.3f}  p = {pval:.2e}")

pd.DataFrame(peak_rows).to_csv(
    OUTDIR / "peak_enrichment_summary.tsv", sep="\t", index=False
)

pd.Series(BIN_ORDER, name="bin_label").to_csv(
    OUTDIR / "bin_order.tsv", sep="\t", index=False
)

print(f"\n[DONE] All outputs in {OUTDIR}")

In [ ]:
# Plot Z-DNA enrichment vs variant size.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D
BASE = "data"
INDIR  = Path(f"{BASE}/mutation_size_vs_zdna_eventlevel")
OUTDIR = INDIR  # write plots alongside data (change if desired)

SUMMARY_TSV = INDIR / "size_vs_enrichment_summary.tsv"
BIN_ORDER_TSV = INDIR / "bin_order.tsv"

rdf = pd.read_csv(SUMMARY_TSV, sep="\t")
BIN_ORDER = pd.read_csv(BIN_ORDER_TSV, sep="\t")["bin_label"].tolist()

WINDOWS = sorted(rdf["window"].unique())
print(f"Loaded {len(rdf)} rows, {rdf['class'].nunique()} classes, "
      f"windows: {WINDOWS}")

CLASS_COLORS = {
    "DEL":  "#E63946",
    "INS":  "#457B9D",
    "CDEL": "#E9C46A",
    "CINS": "#2A9D8F",
    "SUB":  "#264653",
    "MNP":  "#3B7362",
}

CLASS_LABELS = {
    "DEL":  "Deletions",
    "INS":  "Insertions",
    "CDEL": "Complex deletions",
    "CINS": "Complex insertions",
    "SUB":  "Substitutions",
    "MNP":  "Multiple nucl. pol.",
}

WINDOW_STYLES = {
    10:  {"lw": 2.5, "markersize": 8, "alpha": 1.0, "ls": "-"},
    25:  {"lw": 1.8, "markersize": 6, "alpha": 0.7, "ls": "--"},
    50:  {"lw": 1.4, "markersize": 5, "alpha": 0.5, "ls": ":"},
}

LABEL_FONTSIZE = 22
TICK_FONTSIZE  = 18
TITLE_FONTSIZE = 22

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
def save_window_legend(outdir):
    handles = []

    for w in [10, 25, 50]:
        sty = WINDOW_STYLES[w]
        handles.append(
            Line2D(
                [0], [0],
                color="black",
                marker="o",
                linestyle=sty["ls"],
                linewidth=sty["lw"],
                markersize=sty["markersize"],
                alpha=sty["alpha"],
                label=f"±{w} bp",
            )
        )

    fig, ax = plt.subplots(figsize=(4.2, 1.2))
    ax.axis("off")

    ax.legend(
        handles=handles,
        loc="center",
        frameon=False,
        ncol=3,
        fontsize=14,
        handlelength=2.5,
        columnspacing=1.6,
        handletextpad=0.5,
    )

    fig.tight_layout()
    fig.savefig(outdir / "window_style_legend.png",
                dpi=600, bbox_inches="tight", transparent=True)

    plt.show()
    print("[SAVED] window_style_legend.png")

def clean_bin_label(label):
    s = str(label)

    if "+" in s:
        return s

    sep = "–" if "–" in s else "-"
    if sep not in s:
        return s

    lo, hi = s.split(sep, 1)
    return lo if lo == hi else s

save_window_legend(OUTDIR)
for cat in sorted(rdf["class"].unique()):
    cat_data = rdf[rdf["class"] == cat]
    bins_present = [b for b in BIN_ORDER if b in cat_data["bin_label"].values]
    if len(bins_present) < 2:
        continue

    fig, ax = plt.subplots(figsize=(8, 6))
    color = CLASS_COLORS.get(cat, "#333333")
    x = np.arange(len(bins_present))

    for w in WINDOWS:
        sub = cat_data[cat_data["window"] == w]
        x_valid, y_mean, y_lo, y_hi = [], [], [], []
        for i, b in enumerate(bins_present):
            row = sub[sub["bin_label"] == b]
            if row.empty:
                continue
            r = row.iloc[0]
            x_valid.append(i)
            y_mean.append(r["enrich_mean"])
            y_lo.append(r["enrich_lo"])
            y_hi.append(r["enrich_hi"])

        x_valid = np.array(x_valid)
        y_mean  = np.array(y_mean)
        y_lo    = np.array(y_lo)
        y_hi    = np.array(y_hi)

        sty = WINDOW_STYLES.get(w, WINDOW_STYLES[10])
        ax.errorbar(
            x_valid, y_mean,
            yerr=[y_mean - y_lo, y_hi - y_mean],
            fmt="o", color=color,
            lw=sty["lw"], markersize=sty["markersize"],
            alpha=sty["alpha"], linestyle=sty["ls"],
            capsize=4, capthick=1.5, zorder=3,
            label=f"\u00b1{w} bp",
        )
        ax.plot(x_valid, y_mean, color=color,
                lw=sty["lw"], alpha=sty["alpha"], ls=sty["ls"], zorder=2)

    ax.axhline(1, ls=":", color="black", lw=1.5, alpha=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels([clean_bin_label(b) for b in bins_present],
                       rotation=45, ha="right",
                       fontsize=TICK_FONTSIZE - 2)
    ax.set_xlabel("Variant size range (bp)", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel("Z-DNA enrichment at anchor", fontsize=LABEL_FONTSIZE)
    ax.set_title(CLASS_LABELS.get(cat, cat),
                 fontsize=TITLE_FONTSIZE)
    ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)
    despine(ax)
    fig.tight_layout()
    fig.savefig(OUTDIR / f"size_vs_enrichment_{cat}.png",
                dpi=600, bbox_inches="tight")
    plt.show()
    print(f"[SAVED] size_vs_enrichment_{cat}.png")

print("[INFO] Plotting combined MNP / Substitutions...")

fig, ax = plt.subplots(figsize=(8, 6))

for w in WINDOWS:
    sub = rdf[(rdf["class"].isin(["MNP", "SUB"])) & (rdf["window"] == w)].copy()
    if sub.empty:
        continue

    agg = (
        sub.groupby("bin_label")
        .apply(lambda d: pd.Series({
            "enrich_mean": np.average(d["enrich_mean"], weights=d["n_muts"]),
            "enrich_lo":   np.average(d["enrich_lo"],   weights=d["n_muts"]),
            "enrich_hi":   np.average(d["enrich_hi"],   weights=d["n_muts"]),
            "n_muts":      d["n_muts"].sum(),
        }))
        .reset_index()
    )

    bins_present = [b for b in BIN_ORDER if b in agg["bin_label"].values]
    agg = agg.set_index("bin_label").loc[bins_present].reset_index()
    if len(agg) < 2:
        continue

    x = np.arange(len(agg))
    y_mean = agg["enrich_mean"].values
    y_lo   = agg["enrich_lo"].values
    y_hi   = agg["enrich_hi"].values

    sty = WINDOW_STYLES.get(w, WINDOW_STYLES[10])
    ax.errorbar(
        x, y_mean, yerr=[y_mean - y_lo, y_hi - y_mean],
        fmt="o", color="#6A4C93",
        lw=sty["lw"], markersize=sty["markersize"],
        alpha=sty["alpha"], linestyle=sty["ls"],
        capsize=4, capthick=1.5, label=f"\u00b1{w} bp",
    )
    ax.plot(x, y_mean, color="#6A4C93",
            lw=sty["lw"], alpha=sty["alpha"], ls=sty["ls"])

ax.axhline(1, ls=":", color="black", lw=1.5, alpha=0.6)
ax.set_xticks(np.arange(len(bins_present)))
ax.set_xticklabels([clean_bin_label(b) for b in bins_present],
                   rotation=45, ha="right",
                   fontsize=TICK_FONTSIZE - 2)
ax.set_xlabel("Variant size range (bp)", fontsize=LABEL_FONTSIZE)
ax.set_ylabel("Z-DNA enrichment at anchor", fontsize=LABEL_FONTSIZE)
ax.set_title("MNP / Substitutions",
             fontsize=TITLE_FONTSIZE)
ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)
despine(ax)
fig.tight_layout()
fig.savefig(OUTDIR / "size_vs_enrichment_MNP_SUB.png",
            dpi=600, bbox_inches="tight")
plt.show()
print("[SAVED] size_vs_enrichment_MNP_SUB.png")
print(f"\n[DONE] All plots in {OUTDIR}")

## 14. Mutation enrichment within genomic features (CHM13)

In [ ]:
# Per-feature matching-validity check: confirm Z-DNA and matched controls are comparable within each feature.
import numpy as np
import pandas as pd
import pyranges as pr
from pathlib import Path

BASE = "data"

MASTER  = f"{BASE}/controls/CHM13_controls_MASTER.tsv"
FEATDIR = Path(f"{BASE}/chm13/annotation/"
                f"enrichment_investigation/features")

FEATURES_TO_CHECK = [
    "pc_CDS",
    "pc_exon",
    "pc_intron",
    "gene_body",
    "gene_protein_coding",
    "gene_lncRNA",
    "censat_hor",
    "censat_dhor",
    "censat_hsat1A",
    "censat_hsat1B",
    "censat_hsat2",
    "censat_hsat3",
    "censat_mon",
    "censat_gsat",
    "censat_bsat",
    "rmsk_LINE",
    "rmsk_SINE",
    "rmsk_LTR",
    "rmsk_DNA_transposon",
    "rmsk_Simple_repeat",
    "rmsk_Low_complexity",
    "rmsk_Retroposon",
    "telomere",
    "rDNA_arrays",
    "tRNA_arrays",
    "unannotated_intergenic",
]

print(f"Loading CHM13 Z-DNA / matched-control master table...")
m = pd.read_csv(MASTER, sep="\t")
m = m[(m["tol_level"] == "10%") & m["has_control"]].drop_duplicates(
    ["Chromosome", "Start0", "End0"]
)
print(f"  {len(m):,} matched Z/C pairs at 10% tolerance")

z_df0 = pd.DataFrame({
    "Chromosome": m["Chromosome"].astype(str),
    "Start": m["Start0"].astype(np.int64),
    "End": m["End0"].astype(np.int64),
})

c_df0 = pd.DataFrame({
    "Chromosome": m["Chromosome"].astype(str),
    "Start": m["Ctrl_Start"].astype(np.int64),
    "End": m["Ctrl_End"].astype(np.int64),
})

z = pr.PyRanges(z_df0)
c = pr.PyRanges(c_df0)

z_df = z.df.assign(length=lambda d: d["End"] - d["Start"])
c_df = c.df.assign(length=lambda d: d["End"] - d["Start"])
print(f"\nGenome-wide medians: Z={z_df['length'].median():.0f} bp, "
      f"C={c_df['length'].median():.0f} bp")
print(f"Genome-wide counts:  Z={len(z_df):,}, C={len(c_df):,}\n")

print(f"{'feature':<24} {'Z_n':>8} {'C_n':>8} {'Z_med':>8} {'C_med':>8} "
      f"{'ratio':>7}  status")
print("-" * 80)

rows = []
for feat_name in FEATURES_TO_CHECK:
    feat_path = FEATDIR / f"{feat_name}.merged.bed"
    if not feat_path.exists():
        print(f"  {feat_name:<24} (file not found, skipping)")
        continue

    fdf = pd.read_csv(
        feat_path,
        sep="\t",
        header=None,
        usecols=[0, 1, 2],
        names=["Chromosome", "Start", "End"],
    )
    
    f = pr.PyRanges(fdf)
    z_in = z.intersect(f).df
    c_in = c.intersect(f).df
    if len(z_in) == 0 or len(c_in) == 0:
        print(f"  {feat_name:<24} {len(z_in):>8} {len(c_in):>8}  "
              f"(insufficient overlap)")
        continue
    z_in = z_in.assign(length=lambda d: d["End"] - d["Start"])
    c_in = c_in.assign(length=lambda d: d["End"] - d["Start"])
    z_med, c_med = z_in["length"].median(), c_in["length"].median()
    ratio = z_med / c_med if c_med > 0 else np.nan
    if 0.7 <= ratio <= 1.4:
        status = "OK"
    elif 0.5 <= ratio <= 2.0:
        status = "WARN"
    else:
        status = "FAIL"
    print(f"  {feat_name:<24} {len(z_in):>8} {len(c_in):>8} "
          f"{z_med:>8.0f} {c_med:>8.0f} {ratio:>7.2f}  {status}")
    rows.append({
        "feature": feat_name, "z_n": len(z_in), "c_n": len(c_in),
        "z_median_bp": z_med, "c_median_bp": c_med,
        "length_ratio": ratio, "status": status,
    })

out_path = Path(f"{BASE}/coupling_matching_validity.tsv")
pd.DataFrame(rows).to_csv(out_path, sep="\t", index=False)
print(f"\nSaved: {out_path}")

In [ ]:
# Mutation x feature x Z-DNA coupling on independent (overlapping) features.
import numpy as np
import pandas as pd
import pyranges as pr
from pathlib import Path
from scipy.stats import binomtest
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

BASE     = "data"
MUT_BED  = f"{BASE}/mutations/hprc_mutations.bed.gz"
MUT_META = f"{BASE}/mutations/hprc_mutations.meta.tsv.gz"
MASTER   = f"{BASE}/controls/CHM13_controls_MASTER.tsv"
FEATDIR  = Path(f"{BASE}/chm13/annotation/"
                f"enrichment_investigation/features")
OUT_TSV  = f"{BASE}/coupling_zdna_mutation_features_chm13_8group.tsv"

FLANK = 10
MIN_MUTS_PER_CELL = 1000

FEATURES = [
    "pc_CDS", "pc_exon", "pc_intron", "gene_body",
    "gene_protein_coding", "gene_lncRNA", "gene_pseudogene",
    "censat_hor", "censat_hsat3",
    "censat_ct", "censat_censat", "telomere",
    "rmsk_LINE", "rmsk_SINE", "rmsk_LTR", "rmsk_DNA_transposon",
    "rmsk_Retroposon", "rmsk_Simple_repeat", "rmsk_Low_complexity",
    "rmsk_Satellite", "rDNA_arrays"
]

STRATUM_ORDER = [
    "SNV", "MNV", "small_del", "small_ins",
    "large_indel", "substitution", "complex_del", "complex_ins",
]

def assign_stratum(sc, dom, L):
    if sc == "snp":
        return "SNV"
    if sc == "mnp":
        return "MNV"
    if sc == "substitution":
        return "substitution"
    if sc == "del":
        return "small_del" if L <= 50 else "large_indel"
    if sc == "ins":
        return "small_ins" if L <= 50 else "large_indel"
    if sc == "complex" and dom == "del_like":
        return "complex_del"
    if sc == "complex" and dom == "ins_like":
        return "complex_ins"
    return None

def load_bed(feat_name):
    full = FEATDIR / f"{feat_name}.merged.bed"
    if not full.exists():
        print(f"  [missing] {full}")
        return None
    df = pd.read_csv(full, sep="\t", header=None, usecols=[0, 1, 2],
                     names=["Chromosome", "Start", "End"])
    if len(df) == 0:
        return None
    return pr.PyRanges(df).merge()

print("Loading CHM13 Z-DNA and matched-control intervals...")
m = pd.read_csv(MASTER, sep="\t")
m = m[(m["tol_level"] == "10%") & m["has_control"]].drop_duplicates(
    ["Chromosome", "Start0", "End0"]
)
print(f"  {len(m):,} matched Z/C pairs")

z_iv = pr.PyRanges(pd.DataFrame({
    "Chromosome": m["Chromosome"].astype(str),
    "Start": m["Start0"].astype(np.int64),
    "End": m["End0"].astype(np.int64),
}))
c_iv = pr.PyRanges(pd.DataFrame({
    "Chromosome": m["Chromosome"].astype(str),
    "Start": m["Ctrl_Start"].astype(np.int64),
    "End": m["Ctrl_End"].astype(np.int64),
}))

print(f"Expanding Z and C by +/-{FLANK} bp...")
z_e = z_iv.extend(FLANK).merge()
c_e = c_iv.extend(FLANK).merge()

print("Loading mutations...")
mut = pd.read_csv(MUT_BED, sep="\t", header=None,
                  names=["Chromosome", "Start", "End", "mut_id",
                         "score", "strand"])
meta = pd.read_csv(MUT_META, sep="\t",
                   usecols=["mut_id", "struct_class", "dominant_op",
                            "ref_span", "alt_span"])
mut = mut.merge(meta, on="mut_id", how="inner")
mut["event_size"] = np.where(
    (mut["struct_class"] == "ins") |
    ((mut["struct_class"] == "complex") & (mut["dominant_op"] == "ins_like")),
    mut["alt_span"], mut["ref_span"]
).astype(np.int64)

mut["stratum"] = [
    assign_stratum(s, d, l)
    for s, d, l in zip(mut["struct_class"], mut["dominant_op"], mut["event_size"])
]
mut = mut.dropna(subset=["stratum"]).reset_index(drop=True)
print(f"  {len(mut):,} mutations across {mut['stratum'].nunique()} collapsed strata")
print(mut["stratum"].value_counts().reindex(STRATUM_ORDER).to_string())

mut_pr = pr.PyRanges(pd.DataFrame({
    "Chromosome": mut["Chromosome"].astype(str),
    "Start": mut["Start"].astype(np.int64),
    "End": mut["End"].astype(np.int64),
    "mut_id": mut["mut_id"].astype(str),
}))

print(f"\nMarking Z/C overlap per mutation (anchor within +/-{FLANK} bp)...")
z_overlap_df = mut_pr.overlap(z_e).df
c_overlap_df = mut_pr.overlap(c_e).df
in_z = set(z_overlap_df["mut_id"]) if len(z_overlap_df) else set()
in_c = set(c_overlap_df["mut_id"]) if len(c_overlap_df) else set()
mut["hit_Z"] = mut["mut_id"].isin(in_z)
mut["hit_C"] = mut["mut_id"].isin(in_c)
print(f"  hit_Z: {mut['hit_Z'].sum():,}    hit_C: {mut['hit_C'].sum():,}")
print(f"  both:  {(mut['hit_Z'] & mut['hit_C']).sum():,}    "
      f"Z_only: {(mut['hit_Z'] & ~mut['hit_C']).sum():,}    "
      f"C_only: {(~mut['hit_Z'] & mut['hit_C']).sum():,}    "
      f"neither: {(~mut['hit_Z'] & ~mut['hit_C']).sum():,}")

print("\nPer-feature membership (independent, overlapping) + McNemar tests...")
results = []

for feat_name in FEATURES:
    iv = load_bed(feat_name)
    if iv is None:
        print(f"  {feat_name:<24}: skipped (no intervals)")
        continue

    feat_overlap_df = mut_pr.overlap(iv).df
    in_feat = set(feat_overlap_df["mut_id"]) if len(feat_overlap_df) else set()
    feat_mask = mut["mut_id"].isin(in_feat)
    n_feat = int(feat_mask.sum())
    print(f"  {feat_name:<24}: {n_feat:>10,} mutations overlap this feature")

    sub_feat = mut[feat_mask]

    for stratum_name in STRATUM_ORDER:
        sub = sub_feat[sub_feat["stratum"] == stratum_name]
        n_total = len(sub)

        if n_total < MIN_MUTS_PER_CELL:
            results.append({
                "feature": feat_name, "stratum": stratum_name,
                "n_total": n_total,
                "n_both": 0, "n_Z_only": 0, "n_C_only": 0, "n_neither": 0,
                "n_in_Z": 0, "n_in_C": 0,
                "z_frac": np.nan, "c_frac": np.nan,
                "OR": np.nan, "p_value": np.nan, "underpowered": True,
            })
            continue

        hz = sub["hit_Z"].values
        hc = sub["hit_C"].values
        a = int(( hz &  hc).sum())   # both
        b = int(( hz & ~hc).sum())   # Z only  (discordant)
        c = int((~hz &  hc).sum())   # C only  (discordant)
        d = int((~hz & ~hc).sum())   # neither

        or_val = (b / c) if c > 0 else (np.inf if b > 0 else np.nan)

        if b + c == 0:
            p = 1.0
        elif b + c < 25:
            p = binomtest(b, b + c, 0.5).pvalue
        else:
            mcnemar_table = np.array([[a, b], [c, d]])
            result = mcnemar(mcnemar_table, exact=False, correction=True)
            p = result.pvalue

        results.append({
            "feature": feat_name, "stratum": stratum_name,
            "n_total": n_total,
            "n_both": a, "n_Z_only": b, "n_C_only": c, "n_neither": d,
            "n_in_Z": a + b, "n_in_C": a + c,
            "z_frac": (a + b) / n_total,
            "c_frac": (a + c) / n_total,
            "OR": or_val,
            "p_value": p, "underpowered": False,
        })
res = pd.DataFrame(results)
mask_tested = ~res["underpowered"]
res["q_value"] = np.nan
if mask_tested.any():
    res.loc[mask_tested, "q_value"] = multipletests(
        res.loc[mask_tested, "p_value"].values,
        method="fdr_bh"
    )[1]
res["significant"] = res["q_value"] < 0.05
res["stratum"] = pd.Categorical(res["stratum"], categories=STRATUM_ORDER, ordered=True)
res["feature"] = pd.Categorical(res["feature"], categories=FEATURES, ordered=True)
res = res.sort_values(["feature", "stratum"]).reset_index(drop=True)
res.to_csv(OUT_TSV, sep="\t", index=False)
print(f"\nWrote: {OUT_TSV}")
n_tested = int(mask_tested.sum())
n_sig = int(res["significant"].sum())
print(f"\nTested {n_tested} / {len(res)} cells (McNemar); "
      f"{n_sig} significant at BH-FDR q < 0.05.\n")
pivot = res.pivot_table(index="feature", columns="stratum",
                        values="OR", observed=True).round(2)
print(pivot.to_string())
print("\n[DONE]")

In [ ]:
# Tabulate tested/underpowered features and plot the Z-DNA mutation-coupling heatmap.
shown = res[res["feature"].isin(FEATURE_ORDER)]

n_tested = (~shown["underpowered"]).sum()
n_sig = shown["significant"].sum()

print(n_tested, n_sig)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from pathlib import Path
BASE = Path("data")
COUPLING_TSV = BASE / "coupling_zdna_mutation_features_chm13_8group.tsv"
AUDIT_TSV    = BASE / "coupling_matching_validity.tsv"
OUT_PNG      = BASE / "coupling_zdna_mutation_features_chm13_8group_heatmap.png"
COUNT_RATIO_MIN = 0.5
COUNT_RATIO_MAX = 2.0
EXCLUDE_AUDIT_STATUSES = {"FAIL"}
MANUAL_EXCLUDE = {""}   # added per request
CMAP = "RdBu_r"   # diverging, centered at log2(OR)=0

STRATUM_ORDER = [
    "SNV", "MNV", "small_del", "small_ins",
    "large_indel", "substitution", "complex_del", "complex_ins",
]

STRATUM_LABELS = {
    "SNV": "SNV",
    "MNV": "MNV",
    "small_del": "Small del\n\u226450 bp",
    "small_ins": "Small ins\n\u226450 bp",
    "large_indel": "Large indel\n>50 bp",
    "substitution": "Substitution",
    "complex_del": "Complex\ndeletion",
    "complex_ins": "Complex\ninsertion",
}

FEATURE_ORDER = [
    "pc_CDS", "pc_exon", "pc_intron", "gene_body",
    "gene_protein_coding", "gene_lncRNA", "gene_pseudogene",
    "rmsk_LINE", "rmsk_SINE", "rmsk_LTR", "rmsk_DNA_transposon",
    "rmsk_Retroposon", "rmsk_Simple_repeat", "rmsk_Low_complexity",
    "rmsk_Satellite", "censat_hor", "censat_hsat3",
    "censat_ct", "censat_censat", "telomere",
    
]

FEATURE_LABELS = {
    "pc_CDS": "CDS", "pc_exon": "Exon", "pc_intron": "Intron",
    "gene_body": "Gene body", "gene_protein_coding": "Pr. cod. gene",
    "gene_lncRNA": "lncRNA", "gene_pseudogene": "Pseudogene",
    "rmsk_LINE": "LINE", "rmsk_SINE": "SINE", "rmsk_LTR": "LTR",
    "rmsk_DNA_transposon": "DNA transp.", "rmsk_Retroposon": "Retroposon",
    "rmsk_Simple_repeat": "Simple rep.", "rmsk_Low_complexity": "Low compl.",
    "rmsk_Satellite": "Sat (RMSK)", "censat_hor": "HOR", "censat_hsat3": "HSAT3",
    "censat_ct": "CT sat", "censat_censat": "CenSat",
    "telomere": "Telomere", "rDNA_arrays": "rDNA"
}

res = pd.read_csv(COUPLING_TSV, sep="\t")
audit = pd.read_csv(AUDIT_TSV, sep="\t")

audit = audit.copy()
audit["count_ratio_ZC"] = audit["z_n"] / audit["c_n"].replace(0, np.nan)

bad_audit_features = set(audit.loc[
    audit["status"].isin(EXCLUDE_AUDIT_STATUSES)
    | (audit["count_ratio_ZC"] < COUNT_RATIO_MIN)
    | (audit["count_ratio_ZC"] > COUNT_RATIO_MAX),
    "feature",
])
excluded = bad_audit_features | MANUAL_EXCLUDE

print("Features excluded (audit + manual):")
for f in sorted(excluded):
    src = []
    if f in bad_audit_features:
        src.append("audit")
    if f in MANUAL_EXCLUDE:
        src.append("manual")
    print(f"  {f:<24} ({', '.join(src)})")
print()

res = res[
    (~res["underpowered"])
    & (~res["feature"].isin(excluded))
].copy()

if len(res) == 0:
    raise RuntimeError("No testable cells remain after filtering.")

or_mat = res.pivot(index="stratum", columns="feature", values="OR")
p_mat  = res.pivot(index="stratum", columns="feature", values="q_value")

row_order = [s for s in STRATUM_ORDER if s in or_mat.index]
col_order = [f for f in FEATURE_ORDER if f in or_mat.columns]
or_mat = or_mat.loc[row_order, col_order]
p_mat  = p_mat.loc[row_order, col_order]

or_mat = or_mat.dropna(how="all", axis=0).dropna(how="all", axis=1)
p_mat  = p_mat.loc[or_mat.index, or_mat.columns]

log2_or = np.log2(or_mat)
print(f"Final matrix: {log2_or.shape[0]} strata x {log2_or.shape[1]} features")

finite = log2_or.replace([np.inf, -np.inf], np.nan).values
finite = finite[np.isfinite(finite)]
if len(finite) == 0:
    raise RuntimeError("No finite OR values.")
abs_lim = max(np.nanpercentile(np.abs(finite), 99), 0.1)
norm = TwoSlopeNorm(vmin=-abs_lim, vcenter=0, vmax=abs_lim)
cmap = plt.get_cmap(CMAP)

def text_color_for_value(value):
    if not np.isfinite(value):
        return "black"
    r, g, b, _ = cmap(norm(value))
    lum = 0.299 * r + 0.587 * g + 0.114 * b
    return "black" if lum > 0.55 else "white"

def sig_marker(p):
    if pd.isna(p): return ""
    if p < 1e-3:   return "***"
    if p < 1e-2:   return "**"
    if p < 0.05:   return "*"
    return ""

n_rows, n_cols = log2_or.shape
fig_width = max(14, 0.62 * n_cols + 5)
fig_height = max(5, 0.6 * n_rows + 1.5)

fig, ax = plt.subplots(figsize=(fig_width, fig_height))
im = ax.imshow(log2_or.values, cmap=cmap, norm=norm,
               aspect="auto", interpolation="nearest")
ax.set_xticks(np.arange(n_cols))
ax.set_xticklabels([FEATURE_LABELS.get(f, f) for f in log2_or.columns],
                   rotation=45, ha="right", fontsize=11)
ax.set_yticks(np.arange(n_rows))
ax.set_yticklabels([STRATUM_LABELS.get(s, s) for s in log2_or.index], fontsize=12)
for i in range(n_rows):
    for j in range(n_cols):
        raw = or_mat.iloc[i, j]
        val = log2_or.iloc[i, j]
        p = p_mat.iloc[i, j]

        if pd.isna(raw) or not np.isfinite(val):
            ax.text(
                j, i, "·",
                ha="center", va="center",
                fontsize=10,
                color="#999999"
            )
            continue

        cell_text = (
            f"{raw:.2f}{sig_marker(p)}"
            if np.isfinite(raw)
            else f"∞{sig_marker(p)}"
        )

        ax.text(
            j, i, cell_text,
            ha="center", va="center",
            fontsize=9,
            fontweight="bold",
            color=text_color_for_value(val)
        )

ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.5)
ax.tick_params(which="minor", bottom=False, left=False)

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("log₂ discordant OR (Z-only / C-only mutations)", fontsize=12)
cbar.ax.tick_params(labelsize=10)
ax.set_xlabel("Genomic feature", fontsize=13, labelpad=12)
ax.set_ylabel("Mutation class", fontsize=13, labelpad=10)
for spine in ax.spines.values():
    spine.set_visible(False)
fig.tight_layout()
fig.savefig(OUT_PNG, dpi=600, bbox_inches="tight")
fig.savefig(OUT_PNG.with_suffix(".pdf"), bbox_inches="tight")
plt.show()
print(f"\n[SAVED] {OUT_PNG}")

## 15. Allele-frequency (VAF) analysis

In [ ]:
# Prepare SNP allele-frequency inputs.
from pathlib import Path
import numpy as np
import pandas as pd

BASE = "data"

MUT_META = (
    f"{BASE}/mutations/"
    "hprc_mutations.meta.tsv.gz"
)

OUTDIR = Path(f"{BASE}/mutations")
OUTDIR.mkdir(parents=True, exist_ok=True)

OUT_SNP_AF = OUTDIR / "hprc_mutations"

MIN_AN_FOR_FLAG = 10

usecols = [
    "mut_id",
    "seqID",
    "start",
    "end",
    "struct_class",
    "reference",
    "variant",
    "AC",
    "AF",
    "AN",
    "NS",
    "TYPE",
    "subvariants",
    "is_multiallelic",
]

meta = pd.read_csv(
    MUT_META,
    sep="\t",
    compression="gzip",
    usecols=usecols,
)

snps = meta[
    (meta["struct_class"] == "snp")
    & (meta["reference"].astype(str).str.fullmatch("[ACGT]"))
    & (meta["variant"].astype(str).str.fullmatch("[ACGT]"))
].copy()

for col in ["start", "end", "AC", "AF", "AN", "NS"]:
    snps[col] = pd.to_numeric(snps[col], errors="coerce")

snps = snps.dropna(subset=["start", "AC", "AF", "AN"])

snps["start"] = snps["start"].astype(np.int64)
snps["end"]   = snps["end"].astype(np.int64)
snps["AC"]    = snps["AC"].astype(np.int64)
snps["AN"]    = snps["AN"].astype(np.int64)

snps["pos1"] = snps["start"] + 1

snps["snp_id"] = (
    snps["seqID"].astype(str)
    + ":"
    + snps["pos1"].astype(str)
    + ":"
    + snps["reference"].astype(str)
    + ">"
    + snps["variant"].astype(str)
)

snps["AF_from_AC_AN"] = snps["AC"] / snps["AN"]
snps["AF_delta"] = (snps["AF"] - snps["AF_from_AC_AN"]).abs()

snps["low_AN_flag"] = snps["AN"] < MIN_AN_FOR_FLAG

snps["frequency_class"] = np.select(
    [
        snps["AC"] == 1,
        snps["AF"] <= 0.01,
        (snps["AF"] > 0.01) & (snps["AF"] <= 0.05),
        snps["AF"] > 0.05,
    ],
    [
        "singleton",
        "rare_AF_le_0.01",
        "low_frequency_0.01_0.05",
        "common_AF_gt_0.05",
    ],
    default="unknown",
)

before = len(snps)
snps = snps.drop_duplicates(["seqID", "start", "reference", "variant"]).copy()
after = len(snps)

outcols = [
    "snp_id",
    "seqID",
    "start",
    "end",
    "pos1",
    "reference",
    "variant",
    "AC",
    "AF",
    "AF_from_AC_AN",
    "AF_delta",
    "AN",
    "NS",
    "low_AN_flag",
    "frequency_class",
    "mut_id",
    "TYPE",
    "subvariants",
    "is_multiallelic",
]

snps[outcols].to_csv(
    OUT_SNP_AF,
    sep="\t",
    index=False,
    compression="gzip",
)

print(f"[DONE] Wrote: {OUT_SNP_AF}")
print(f"SNP allele rows before duplicate removal: {before:,}")
print(f"SNP allele rows after duplicate removal:  {after:,}")
print(f"Removed duplicates:                     {before - after:,}")

print("\nAF sanity check:")
print(snps["AF_delta"].describe())

bad = snps[snps["AF_delta"] > 1e-6]
print(f"\nRows where INFO/AF differs from AC/AN by >1e-6: {len(bad):,}")

In [ ]:
# Assign SNPs to allele-frequency groups.
from pathlib import Path
import pandas as pd
import numpy as np

BASE = "data"

SNP_AF = (
    f"{BASE}/mutations/"
    "hprc_mutations"
)

OUTDIR = Path(f"{BASE}/mutations")
OUT_QUANT = OUTDIR / "hprc_mutations"
OUT_SUMMARY = OUTDIR / "hprc_mutations"

N_QUANTILES = 4
MIN_AN = 10

snps = pd.read_csv(SNP_AF, sep="\t", compression="gzip")

snps["AF"] = pd.to_numeric(snps["AF"], errors="coerce")
snps["AC"] = pd.to_numeric(snps["AC"], errors="coerce")
snps["AN"] = pd.to_numeric(snps["AN"], errors="coerce")

snps = snps.dropna(subset=["AF", "AC", "AN"]).copy()

print("============================================================")
print("All SNP allele-frequency rows")
print("============================================================")
print(f"n SNP alleles: {len(snps):,}")
print(f"n unique CHM13 positions: {snps[['seqID', 'start']].drop_duplicates().shape[0]:,}")
print(f"n multi-allelic SNP alleles: {snps['is_multiallelic'].sum():,}")
print(f"n low-AN rows, AN < {MIN_AN}: {(snps['AN'] < MIN_AN).sum():,}")

print("\nAF distribution:")
print(snps["AF"].describe(percentiles=[0.001, 0.005, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

print("\nAN distribution:")
print(snps["AN"].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

print("\nFrequency-class counts:")
print(snps["frequency_class"].value_counts(dropna=False))

q = snps[snps["AN"] >= MIN_AN].copy()

print("\n============================================================")
print(f"Quantile input after AN >= {MIN_AN}")
print("============================================================")
print(f"n SNP alleles: {len(q):,}")
print(f"removed low-AN SNP alleles: {len(snps) - len(q):,}")

q["_AF_rank"] = q["AF"].rank(method="first")

q["AF_quantile"] = pd.qcut(
    q["_AF_rank"],
    q=N_QUANTILES,
    labels=[f"Q{i}" for i in range(1, N_QUANTILES + 1)],
)

q = q.drop(columns=["_AF_rank"])

quant_summary = (
    q.groupby("AF_quantile", observed=True)
     .agg(
         n=("AF", "size"),
         AF_min=("AF", "min"),
         AF_q25=("AF", lambda x: x.quantile(0.25)),
         AF_median=("AF", "median"),
         AF_mean=("AF", "mean"),
         AF_q75=("AF", lambda x: x.quantile(0.75)),
         AF_max=("AF", "max"),
         AC_min=("AC", "min"),
         AC_median=("AC", "median"),
         AC_max=("AC", "max"),
         AN_min=("AN", "min"),
         AN_median=("AN", "median"),
         AN_max=("AN", "max"),
     )
     .reset_index()
)

print("\nEqual-count AF quantile summary:")
print(quant_summary)

q.to_csv(OUT_QUANT, sep="\t", index=False, compression="gzip")
quant_summary.to_csv(OUT_SUMMARY, sep="\t", index=False)

print(f"\n[DONE] Wrote SNP AF quantile table: {OUT_QUANT}")
print(f"[DONE] Wrote SNP AF summary table:   {OUT_SUMMARY}")

In [ ]:
# Z-DNA enrichment at SNP sites by allele-frequency group, CDS vs noncoding (block-bootstrap).
import numpy as np
import pandas as pd
from pathlib import Path

BASE = "data"

MASTER = f"{BASE}/controls/CHM13_controls_MASTER.tsv"
CHROM_LEN_TSV = f"{BASE}/chm13/chm13_seqreport_zdna.tsv"
GFF3 = (
    f"{BASE}/chm13/annotation/"
    "chm13_genes.gff3"
)

REGION_MODE = "noncoding"
SNP_AF_INPUT = (
    f"{BASE}/mutations/"
    "hprc_mutations"
)

TOL = "10%"
WINDOW_RADIUS = 500
N_BOOTS = 500
CI_LOW, CI_HIGH = 2.5, 97.5

MIN_SNPS_PER_GROUP = 500

MIN_AN = 10

BINNING_MODE = "custom"

N_EQUAL_GROUPS = 3

CUSTOM_AF_BINS = [
    ("very_low_AF_0_0.0025", 0.0,    0.0025),
    ("low_AF_0.0025_0.01",  0.0025, 0.01),
    ("mid_AF_0.01_0.05",    0.01,   0.05),
    ("high_AF_gt0.05",      0.05,   np.inf),
]

if BINNING_MODE == "equal_count":
    OUTDIR = Path(
        f"{BASE}/snp_af_equalcount{N_EQUAL_GROUPS}_{REGION_MODE}_profiles_tol10"
    )
elif BINNING_MODE == "custom":
    OUTDIR = Path(
        f"{BASE}/snp_af_custom_{REGION_MODE}_profiles_tol10"
    )
else:
    raise ValueError("BINNING_MODE must be 'equal_count' or 'custom'")

OUTDIR.mkdir(parents=True, exist_ok=True)

def chrom_block_bootstrap_windowmean_norm(counts_by_chr, n_boot):
    chroms = list(counts_by_chr)

    if len(chroms) == 0:
        raise ValueError("No chromosomes available for bootstrap")

    mat = np.stack([counts_by_chr[c].astype(float) for c in chroms], axis=0)
    rng = np.random.default_rng(42)

    boots = np.empty((n_boot, mat.shape[1]), dtype=float)

    for b in range(n_boot):
        idx = rng.integers(0, len(chroms), len(chroms))
        v = mat[idx].sum(axis=0)
        m = v.mean()
        boots[b] = v / m if m else np.full_like(v, np.nan, dtype=float)

    return (
        np.nanpercentile(boots, CI_LOW, axis=0),
        np.nanpercentile(boots, CI_HIGH, axis=0),
    )

def build_bool_coverage(starts, ends, chrom_len):
    starts = np.asarray(starts, dtype=np.int64)
    ends   = np.asarray(ends, dtype=np.int64)

    starts = np.clip(starts, 0, chrom_len)
    ends   = np.clip(ends, 0, chrom_len)

    mask = ends > starts
    if not np.any(mask):
        return np.zeros(chrom_len, dtype=bool)

    starts = starts[mask]
    ends   = ends[mask]

    diff = np.zeros(chrom_len + 1, dtype=np.int32)
    np.add.at(diff, starts, 1)
    np.add.at(diff, ends, -1)

    return np.cumsum(diff[:-1]) > 0

def load_merged_cds_intervals(gff3_path):

    cds_by_chr = {}

    opener = pd.io.common.get_handle(
        gff3_path,
        mode="r",
        compression="infer",
    ).handle

    with opener as f:
        for line in f:
            if not line or line.startswith("#"):
                continue

            fields = line.rstrip("\n").split("\t")
            if len(fields) < 9:
                continue

            chrom = fields[0]
            feature = fields[2]

            if feature != "CDS":
                continue

            start0 = int(fields[3]) - 1
            end0   = int(fields[4])

            if end0 <= start0:
                continue

            cds_by_chr.setdefault(chrom, []).append((start0, end0))

    merged = {}

    for chrom, intervals in cds_by_chr.items():
        intervals = sorted(intervals)

        out = []
        cur_start, cur_end = intervals[0]

        for start, end in intervals[1:]:
            if start <= cur_end:
                cur_end = max(cur_end, end)
            else:
                out.append((cur_start, cur_end))
                cur_start, cur_end = start, end

        out.append((cur_start, cur_end))

        merged[chrom] = np.asarray(out, dtype=np.int64)

    return merged

def filter_snps_to_intervals(snps, intervals_by_chr):

    keep_parts = []

    for chrom, d in snps.groupby("seqID", sort=False):
        if chrom not in intervals_by_chr:
            continue

        intervals = intervals_by_chr[chrom]
        if len(intervals) == 0:
            continue

        pos = d["start"].to_numpy(dtype=np.int64)

        starts = intervals[:, 0]
        ends   = intervals[:, 1]

        idx = np.searchsorted(starts, pos, side="right") - 1

        valid = idx >= 0
        inside = np.zeros(len(pos), dtype=bool)

        inside[valid] = pos[valid] < ends[idx[valid]]

        if inside.any():
            keep_parts.append(d.loc[inside])

    if not keep_parts:
        return snps.iloc[0:0].copy()

    return pd.concat(keep_parts, axis=0, ignore_index=True)

def filter_snps_outside_intervals(snps, intervals_by_chr):

    keep_parts = []

    for chrom, d in snps.groupby("seqID", sort=False):

        if chrom not in intervals_by_chr:
            keep_parts.append(d)
            continue

        intervals = intervals_by_chr[chrom]
        if len(intervals) == 0:
            keep_parts.append(d)
            continue

        pos = d["start"].to_numpy(dtype=np.int64)

        starts = intervals[:, 0]
        ends   = intervals[:, 1]

        idx = np.searchsorted(starts, pos, side="right") - 1

        valid = idx >= 0
        inside = np.zeros(len(pos), dtype=bool)

        inside[valid] = pos[valid] < ends[idx[valid]]

        outside = ~inside

        if outside.any():
            keep_parts.append(d.loc[outside])

    if not keep_parts:
        return snps.iloc[0:0].copy()

    return pd.concat(keep_parts, axis=0, ignore_index=True)
    
def assign_af_groups(snps):

    snps = snps.copy()

    if BINNING_MODE == "equal_count":

        if N_EQUAL_GROUPS < 2:
            raise ValueError("N_EQUAL_GROUPS must be >= 2")

        rank = snps["AF"].rank(method="first")

        labels = [f"Q{i}" for i in range(1, N_EQUAL_GROUPS + 1)]

        snps["AF_group"] = pd.qcut(
            rank,
            q=N_EQUAL_GROUPS,
            labels=labels,
        ).astype(str)

        snps["AF_group_mode"] = f"equal_count_{N_EQUAL_GROUPS}"

    elif BINNING_MODE == "custom":

        snps["AF_group"] = pd.NA

        for label, lo, hi in CUSTOM_AF_BINS:
            if hi == np.inf:
                mask = (snps["AF"] >= lo) & (snps["AF"] <= 1.0)
            else:
                mask = (snps["AF"] >= lo) & (snps["AF"] < hi)

            snps.loc[mask, "AF_group"] = label

        unassigned = snps["AF_group"].isna().sum()
        if unassigned:
            print(f"[WARN] {unassigned:,} SNP alleles were not assigned to any custom AF bin.")
            snps = snps.dropna(subset=["AF_group"]).copy()

        snps["AF_group"] = snps["AF_group"].astype(str)
        snps["AF_group_mode"] = "custom"

    else:
        raise ValueError("BINNING_MODE must be 'equal_count' or 'custom'")

    return snps

def summarize_groups(snps):
    summary = (
        snps.groupby("AF_group", sort=True)
            .agg(
                n=("AF", "size"),
                AF_min=("AF", "min"),
                AF_q25=("AF", lambda x: x.quantile(0.25)),
                AF_median=("AF", "median"),
                AF_mean=("AF", "mean"),
                AF_q75=("AF", lambda x: x.quantile(0.75)),
                AF_max=("AF", "max"),
                AC_min=("AC", "min"),
                AC_median=("AC", "median"),
                AC_max=("AC", "max"),
                AN_min=("AN", "min"),
                AN_median=("AN", "median"),
                AN_max=("AN", "max"),
            )
            .reset_index()
    )

    return summary

chrom_len_df = pd.read_csv(
    CHROM_LEN_TSV,
    sep="\t",
    usecols=["ucsc_pansn_name", "gcf_seq_length"],
)

chrom_len = dict(
    zip(
        chrom_len_df["ucsc_pansn_name"],
        chrom_len_df["gcf_seq_length"].astype(int),
    )
)

m = pd.read_csv(MASTER, sep="\t")

m = (
    m[(m["tol_level"] == TOL) & (m["has_control"])]
    .drop_duplicates(["Chromosome", "Start0", "End0"])
    .copy()
)

z_intervals = {
    chrom: d[["Start0", "End0"]].to_numpy(dtype=np.int64)
    for chrom, d in m.groupby("Chromosome", sort=False)
}

c_intervals = {
    chrom: d[["Ctrl_Start", "Ctrl_End"]].to_numpy(dtype=np.int64)
    for chrom, d in m.groupby("Chromosome", sort=False)
}

z_cov = {}
c_cov = {}

for chrom in sorted(set(z_intervals) & set(c_intervals) & set(chrom_len)):
    L = int(chrom_len[chrom])

    z_cov[chrom] = build_bool_coverage(
        z_intervals[chrom][:, 0],
        z_intervals[chrom][:, 1],
        L,
    )

    c_cov[chrom] = build_bool_coverage(
        c_intervals[chrom][:, 0],
        c_intervals[chrom][:, 1],
        L,
    )

print(f"Loaded coverage masks for {len(z_cov):,} chromosomes.")

snps = pd.read_csv(SNP_AF_INPUT, sep="\t", compression="gzip")

required_cols = {"seqID", "start", "AF", "AC", "AN"}
missing = required_cols - set(snps.columns)
if missing:
    raise ValueError(f"SNP_AF_INPUT is missing required columns: {sorted(missing)}")

snps["start"] = pd.to_numeric(snps["start"], errors="coerce")
snps["AF"]    = pd.to_numeric(snps["AF"], errors="coerce")
snps["AC"]    = pd.to_numeric(snps["AC"], errors="coerce")
snps["AN"]    = pd.to_numeric(snps["AN"], errors="coerce")

snps = snps.dropna(subset=["seqID", "start", "AF", "AC", "AN"]).copy()

snps["start"] = snps["start"].astype(np.int64)
snps["AC"]    = snps["AC"].astype(np.int64)
snps["AN"]    = snps["AN"].astype(np.int64)

snps = snps[(snps["AF"] >= 0) & (snps["AF"] <= 1)].copy()

n_before_zero_filter = len(snps)

n_af0 = (snps["AF"] == 0).sum()
n_ac0 = (snps["AC"] == 0).sum()
n_an0 = (snps["AN"] == 0).sum()
n_af0_or_ac0 = ((snps["AF"] == 0) | (snps["AC"] == 0)).sum()

print("\nZero-frequency / zero-count SNP alleles:")
print(f"SNP alleles before AF/AC zero filter: {n_before_zero_filter:,}")
print(f"AF == 0:                            {n_af0:,}")
print(f"AC == 0:                            {n_ac0:,}")
print(f"AN == 0:                            {n_an0:,}")
print(f"AF == 0 OR AC == 0:                 {n_af0_or_ac0:,}")

snps = snps[
    (snps["AF"] > 0)
    & (snps["AC"] > 0)
    & (snps["AN"] > 0)
].copy()

print(f"SNP alleles after AF/AC zero filter:  {len(snps):,}")
print(f"Removed by AF/AC zero filter:         {n_before_zero_filter - len(snps):,}")

if MIN_AN is not None and MIN_AN > 0:
    before = len(snps)
    snps = snps[snps["AN"] >= MIN_AN].copy()
    print(f"\nApplied AN filter: AN >= {MIN_AN}")
    print(f"Removed low-AN SNP alleles: {before - len(snps):,}")

if REGION_MODE not in {"genome", "cds", "noncoding"}:
    raise ValueError("REGION_MODE must be 'genome', 'cds', or 'noncoding'")

if REGION_MODE in {"cds", "noncoding"}:
    before = len(snps)

    cds_intervals = load_merged_cds_intervals(GFF3)

    n_cds_chroms = len(cds_intervals)
    n_cds_intervals = sum(len(v) for v in cds_intervals.values())
    n_cds_bp = sum((v[:, 1] - v[:, 0]).sum() for v in cds_intervals.values())

    print(f"Loaded merged CDS intervals from: {GFF3}")
    print(f"CDS chromosomes: {n_cds_chroms:,}")
    print(f"Merged CDS intervals: {n_cds_intervals:,}")
    print(f"Merged CDS bp: {n_cds_bp:,}")

    if REGION_MODE == "cds":
        snps = filter_snps_to_intervals(snps, cds_intervals)

        print("Applied region restriction: REGION_MODE = 'cds'")
        print(f"SNP alleles before CDS filter: {before:,}")
        print(f"SNP alleles after CDS filter:  {len(snps):,}")
        print(f"Removed non-CDS SNP alleles:   {before - len(snps):,}")

    elif REGION_MODE == "noncoding":
        snps = filter_snps_outside_intervals(snps, cds_intervals)

        print("Applied region restriction: REGION_MODE = 'noncoding'")
        print(f"SNP alleles before noncoding filter: {before:,}")
        print(f"SNP alleles after noncoding filter:  {len(snps):,}")
        print(f"Removed CDS SNP alleles:             {before - len(snps):,}")

    if len(snps) == 0:
        raise SystemExit(f"[ERROR] No SNP alleles remain after REGION_MODE = '{REGION_MODE}' filtering.")

else:
    print("Region restriction not applied: REGION_MODE = 'genome'")

snps = assign_af_groups(snps)

grouped_snps_path = OUTDIR / "snps_with_AF_groups_used.tsv.gz"
snps.to_csv(grouped_snps_path, sep="\t", index=False, compression="gzip")

group_summary = summarize_groups(snps)
group_summary_path = OUTDIR / "AF_group_summary.tsv"
group_summary.to_csv(group_summary_path, sep="\t", index=False)

print("\nAF group summary:")
print(group_summary)
print(f"\nWrote grouped SNP table: {grouped_snps_path}")
print(f"Wrote AF group summary: {group_summary_path}")

offsets = np.arange(-WINDOW_RADIUS, WINDOW_RADIUS + 1, dtype=np.int64)

summary_rows = []

for group_label, sub in snps.groupby("AF_group", sort=True):

    if len(sub) < MIN_SNPS_PER_GROUP:
        print(f"[SKIP] {group_label}: only {len(sub):,} SNP alleles")
        continue

    Z_by_chr = {}
    C_by_chr = {}

    Z_total = np.zeros(len(offsets), dtype=np.int64)
    C_total = np.zeros(len(offsets), dtype=np.int64)

    n_events_total = len(sub)
    n_events_used = 0

    for chrom, d in sub.groupby("seqID", sort=False):

        if chrom not in z_cov or chrom not in c_cov or chrom not in chrom_len:
            continue

        L = int(chrom_len[chrom])
        anchors = d["start"].to_numpy(dtype=np.int64)

        Zc = np.zeros(len(offsets), dtype=np.int64)
        Cc = np.zeros(len(offsets), dtype=np.int64)

        for i, k in enumerate(offsets):
            pos = anchors + k
            mask = (pos >= 0) & (pos < L)

            if np.any(mask):
                pos_valid = pos[mask]
                Zc[i] = z_cov[chrom][pos_valid].sum()
                Cc[i] = c_cov[chrom][pos_valid].sum()

        Z_total += Zc
        C_total += Cc

        Z_by_chr[chrom] = Zc
        C_by_chr[chrom] = Cc

        n_events_used += len(anchors)

    if len(Z_by_chr) == 0:
        print(f"[SKIP] {group_label}: no usable chromosomes")
        continue

    Z_mean = (
        Z_total / Z_total.mean()
        if Z_total.mean()
        else np.full(len(offsets), np.nan)
    )

    C_mean = (
        C_total / C_total.mean()
        if C_total.mean()
        else np.full(len(offsets), np.nan)
    )

    Z_lo, Z_hi = chrom_block_bootstrap_windowmean_norm(Z_by_chr, N_BOOTS)
    C_lo, C_hi = chrom_block_bootstrap_windowmean_norm(C_by_chr, N_BOOTS)

    af_min = sub["AF"].min()
    af_max = sub["AF"].max()
    af_med = sub["AF"].median()
    profile = pd.DataFrame({
        "offset": offsets,
        "Z_mean": Z_mean,
        "Z_lo": Z_lo,
        "Z_hi": Z_hi,
        "C_mean": C_mean,
        "C_lo": C_lo,
        "C_hi": C_hi,
        "AF_group": group_label,
        "AF_group_mode": sub["AF_group_mode"].iloc[0],
        "region_mode": REGION_MODE,
        "AF_min": af_min,
        "AF_median": af_med,
        "AF_max": af_max,
        "n_events_total": n_events_total,
        "n_events_used": n_events_used,
    })

    safe_group = str(group_label).replace("/", "_").replace(">", "gt").replace("<", "lt")
    out_profile = OUTDIR / f"profile_SNP_AF_{safe_group}.tsv"
    profile.to_csv(out_profile, sep="\t", index=False)

    chrom_rows = []

    for chrom in Z_by_chr:
        for i, off in enumerate(offsets):
            chrom_rows.append(
                (
                    chrom,
                    int(off),
                    int(Z_by_chr[chrom][i]),
                    int(C_by_chr[chrom][i]),
                    str(group_label),
                )
            )

    chrom_df = pd.DataFrame(
        chrom_rows,
        columns=["chrom", "offset", "Z_raw", "C_raw", "AF_group"],
    )

    out_bychrom = OUTDIR / f"profile_SNP_AF_{safe_group}_bychrom.tsv"
    chrom_df.to_csv(out_bychrom, sep="\t", index=False)

    window = profile[(profile["offset"] >= -5) & (profile["offset"] <= 5)]
    z_win = window["Z_mean"].mean()
    c_win = window["C_mean"].mean()
    zc_ratio = z_win / c_win if c_win > 0 else np.nan

    summary_rows.append({
        "AF_group": group_label,
        "AF_group_mode": sub["AF_group_mode"].iloc[0],
        "region_mode": REGION_MODE,
        "n_events_total": n_events_total,
        "n_events_used": n_events_used,
        "n_chromosomes": len(Z_by_chr),
        "AF_min": af_min,
        "AF_median": af_med,
        "AF_max": af_max,
        "mean_Z_raw": float(Z_total.mean()),
        "mean_C_raw": float(C_total.mean()),
        "Z_mean_pm5": float(z_win),
        "C_mean_pm5": float(c_win),
        "Z_over_C_pm5": float(zc_ratio),
    })

    print(
        f"[OK] {group_label}: "
        f"n={n_events_used:,}, "
        f"AF={af_min:.6g}–{af_max:.6g}, "
        f"median={af_med:.6g}, "
        f"±5bp Z/C={zc_ratio:.3f}"
    )
summary = pd.DataFrame(summary_rows).sort_values("AF_group")
summary.to_csv(OUTDIR / "profile_summary.tsv", sep="\t", index=False)
print("\n[DONE] SNP AF-group metaprofiles written.")
print(summary)

In [ ]:
# Barplot of CDS vs noncoding Z-DNA enrichment across AF groups with bootstrap CIs.
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE = "data"

BINNING_MODE = "custom"
N_EQUAL_GROUPS = 4

if BINNING_MODE == "custom":
    PROFILE_DIRS = {
        "cds": Path(f"{BASE}/snp_af_custom_cds_profiles_tol10"),
        "noncoding": Path(f"{BASE}/snp_af_custom_noncoding_profiles_tol10"),
    }
    OUTDIR = Path(f"{BASE}/snp_af_custom_cds_vs_noncoding_stats_plots_tol10")

elif BINNING_MODE == "equal_count":
    PROFILE_DIRS = {
        "cds": Path(f"{BASE}/snp_af_equalcount{N_EQUAL_GROUPS}_cds_profiles_tol10"),
        "noncoding": Path(f"{BASE}/snp_af_equalcount{N_EQUAL_GROUPS}_noncoding_profiles_tol10"),
    }
    OUTDIR = Path(f"{BASE}/snp_af_equalcount{N_EQUAL_GROUPS}_cds_vs_noncoding_stats_plots_tol10")

else:
    raise ValueError("BINNING_MODE must be 'custom' or 'equal_count'")

OUTDIR.mkdir(parents=True, exist_ok=True)

CENTER_RADIUS = 5
N_BOOTS = 10000
SEED = 42
CI_LOW, CI_HIGH = 2.5, 97.5

LABEL_FS = 18
TICK_FS = 14
TITLE_FS = 18

REGION_COLORS = {
    "cds": "#1f77b4",
    "noncoding": "#ff7f0e",
}

PREFERRED_GROUP_ORDER = [
    "very_low_AF_0_0.0025",
    "low_AF_0.0025_0.01",
    "mid_AF_0.01_0.05",
    "high_AF_gt0.05",
]

def safe_group_name(label):
    return str(label).replace("/", "_").replace(">", "gt").replace("<", "lt")

def format_p(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return f"{p:.2e}"
    return f"{p:.4f}"

def p_to_stars(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

def compute_zc_pm_from_bychrom(chrom_df, chroms=None):

    if chroms is not None:
        chrom_df = chrom_df[chrom_df["chrom"].isin(chroms)].copy()

    if chrom_df.empty:
        return np.nan

    wide = (
        chrom_df.groupby("offset", sort=True)[["Z_raw", "C_raw"]]
        .sum()
        .reset_index()
    )

    z = wide["Z_raw"].to_numpy(dtype=float)
    c = wide["C_raw"].to_numpy(dtype=float)
    offsets = wide["offset"].to_numpy(dtype=int)

    if z.mean() == 0 or c.mean() == 0:
        return np.nan

    z_norm = z / z.mean()
    c_norm = c / c.mean()

    center = (offsets >= -CENTER_RADIUS) & (offsets <= CENTER_RADIUS)

    z_pm = z_norm[center].mean()
    c_pm = c_norm[center].mean()

    return z_pm / c_pm if c_pm > 0 else np.nan

def bootstrap_zc(chrom_df, n_boot=N_BOOTS, seed=SEED):

    rng = np.random.default_rng(seed)
    chroms = np.array(sorted(chrom_df["chrom"].unique()))

    observed = compute_zc_pm_from_bychrom(chrom_df)

    boots = np.empty(n_boot, dtype=float)

    for b in range(n_boot):
        sampled = rng.choice(chroms, size=len(chroms), replace=True)

        sampled_parts = []
        for i, chrom in enumerate(sampled):
            tmp = chrom_df[chrom_df["chrom"] == chrom].copy()
            tmp["chrom"] = f"{chrom}__boot{i}"
            sampled_parts.append(tmp)

        boot_df = pd.concat(sampled_parts, axis=0, ignore_index=True)
        boots[b] = compute_zc_pm_from_bychrom(boot_df)

    lo, hi = np.nanpercentile(boots, [CI_LOW, CI_HIGH])

    return observed, lo, hi, boots

def bootstrap_difference(chrom_df_a, chrom_df_b, paired=True, n_boot=N_BOOTS, seed=SEED):

    rng = np.random.default_rng(seed)

    chroms_a = set(chrom_df_a["chrom"].unique())
    chroms_b = set(chrom_df_b["chrom"].unique())

    if paired:
        chroms = np.array(sorted(chroms_a & chroms_b))
    else:
        chroms_a = np.array(sorted(chroms_a))
        chroms_b = np.array(sorted(chroms_b))

    obs_a = compute_zc_pm_from_bychrom(chrom_df_a)
    obs_b = compute_zc_pm_from_bychrom(chrom_df_b)
    obs_delta = obs_a - obs_b

    deltas = np.empty(n_boot, dtype=float)

    for b in range(n_boot):
        if paired:
            sampled = rng.choice(chroms, size=len(chroms), replace=True)

            parts_a = []
            parts_b = []

            for i, chrom in enumerate(sampled):
                ta = chrom_df_a[chrom_df_a["chrom"] == chrom].copy()
                tb = chrom_df_b[chrom_df_b["chrom"] == chrom].copy()

                ta["chrom"] = f"{chrom}__boot{i}"
                tb["chrom"] = f"{chrom}__boot{i}"

                parts_a.append(ta)
                parts_b.append(tb)

            ba = pd.concat(parts_a, axis=0, ignore_index=True)
            bb = pd.concat(parts_b, axis=0, ignore_index=True)

        else:
            sampled_a = rng.choice(chroms_a, size=len(chroms_a), replace=True)
            sampled_b = rng.choice(chroms_b, size=len(chroms_b), replace=True)

            parts_a = []
            parts_b = []

            for i, chrom in enumerate(sampled_a):
                ta = chrom_df_a[chrom_df_a["chrom"] == chrom].copy()
                ta["chrom"] = f"{chrom}__boot{i}"
                parts_a.append(ta)

            for i, chrom in enumerate(sampled_b):
                tb = chrom_df_b[chrom_df_b["chrom"] == chrom].copy()
                tb["chrom"] = f"{chrom}__boot{i}"
                parts_b.append(tb)

            ba = pd.concat(parts_a, axis=0, ignore_index=True)
            bb = pd.concat(parts_b, axis=0, ignore_index=True)

        deltas[b] = compute_zc_pm_from_bychrom(ba) - compute_zc_pm_from_bychrom(bb)

    lo, hi = np.nanpercentile(deltas, [CI_LOW, CI_HIGH])

    if obs_delta >= 0:
        p = 2 * np.nanmean(deltas <= 0)
    else:
        p = 2 * np.nanmean(deltas >= 0)

    p = min(float(p), 1.0)

    return obs_delta, lo, hi, p

summaries = {}
bychrom = {}

for region, dpath in PROFILE_DIRS.items():

    summary_path = dpath / "profile_summary.tsv"
    if not summary_path.exists():
        raise SystemExit(f"[ERROR] Missing summary for {region}: {summary_path}")

    s = pd.read_csv(summary_path, sep="\t")
    summaries[region] = s

    bychrom[region] = {}

    for group in s["AF_group"].astype(str):
        path = dpath / f"profile_SNP_AF_{safe_group_name(group)}_bychrom.tsv"

        if not path.exists():
            print(f"[WARN] Missing bychrom file for {region}, {group}: {path}")
            continue

        df = pd.read_csv(path, sep="\t")
        bychrom[region][group] = df

print("Loaded groups:")
for region in PROFILE_DIRS:
    print(region, sorted(bychrom[region]))

groups = sorted(set(bychrom["cds"]) & set(bychrom["noncoding"]))

if PREFERRED_GROUP_ORDER:
    groups = [g for g in PREFERRED_GROUP_ORDER if g in groups] + [
        g for g in groups if g not in PREFERRED_GROUP_ORDER
    ]

if not groups:
    raise SystemExit("[ERROR] No common AF groups between CDS and noncoding.")

print("\nCommon groups:")
for g in groups:
    print(f"  {g}")

bar_rows = []
boot_store = {}

for region in ["cds", "noncoding"]:
    boot_store[region] = {}

    for group in groups:
        df = bychrom[region][group]

        obs, lo, hi, boots = bootstrap_zc(
            df,
            n_boot=N_BOOTS,
            seed=SEED,
        )

        boot_store[region][group] = boots

        srow = summaries[region][summaries[region]["AF_group"].astype(str) == group].iloc[0]

        bar_rows.append({
            "region_mode": region,
            "AF_group": group,
            "Z_over_C_pm5": obs,
            "ci_lo": lo,
            "ci_hi": hi,
            "err_lo": obs - lo,
            "err_hi": hi - obs,
            "AF_min": srow["AF_min"],
            "AF_median": srow["AF_median"],
            "AF_max": srow["AF_max"],
            "n_events_used": srow["n_events_used"],
        })

bar_df = pd.DataFrame(bar_rows)
bar_df.to_csv(OUTDIR / "cds_noncoding_bar_values_with_bootstrap_ci.tsv", sep="\t", index=False)

print("\nBar values:")
print(bar_df)

between_rows = []

for group in groups:
    delta, lo, hi, p = bootstrap_difference(
        bychrom["cds"][group],
        bychrom["noncoding"][group],
        paired=True,
        n_boot=N_BOOTS,
        seed=SEED,
    )

    between_rows.append({
        "test_type": "between_region_same_AF_group",
        "AF_group": group,
        "contrast": "cds_minus_noncoding",
        "delta_Z_over_C_pm5": delta,
        "ci_lo": lo,
        "ci_hi": hi,
        "p_boot_two_sided": p,
        "signif": p_to_stars(p),
    })

between_df = pd.DataFrame(between_rows)
between_df.to_csv(OUTDIR / "tests_cds_vs_noncoding_by_AF_group.tsv", sep="\t", index=False)

print("\nCDS vs noncoding tests:")
print(between_df)

within_rows = []

for region in ["cds", "noncoding"]:
    for g1, g2 in itertools.combinations(groups, 2):
        delta, lo, hi, p = bootstrap_difference(
            bychrom[region][g1],
            bychrom[region][g2],
            paired=True,
            n_boot=N_BOOTS,
            seed=SEED,
        )

        within_rows.append({
            "test_type": "within_region_between_AF_groups",
            "region_mode": region,
            "contrast": f"{g1}_minus_{g2}",
            "group_1": g1,
            "group_2": g2,
            "delta_Z_over_C_pm5": delta,
            "ci_lo": lo,
            "ci_hi": hi,
            "p_boot_two_sided": p,
            "signif": p_to_stars(p),
        })

within_df = pd.DataFrame(within_rows)
within_df.to_csv(OUTDIR / "tests_within_region_between_AF_groups.tsv", sep="\t", index=False)

print("\nWithin-region AF-group tests:")
print(within_df)

fig, ax = plt.subplots(figsize=(max(9, 2.2 * len(groups)), 6.2))

x = np.arange(len(groups))
width = 0.36

for j, region in enumerate(["cds", "noncoding"]):
    d = (
        bar_df[bar_df["region_mode"] == region]
        .set_index("AF_group")
        .loc[groups]
        .reset_index()
    )

    xpos = x + (-width / 2 if region == "cds" else width / 2)

    y = d["Z_over_C_pm5"].to_numpy(float)
    yerr = np.vstack([
        d["err_lo"].to_numpy(float),
        d["err_hi"].to_numpy(float),
    ])

    bars = ax.bar(
        xpos,
        y,
        yerr=yerr,
        capsize=4,
        width=width,
        color=REGION_COLORS[region],
        edgecolor="black",
        linewidth=0.8,
        label=region,
    )

    for bar, val in zip(bars, y):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            val + 0.015,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=10,
            rotation=90,
        )

ax.axhline(1, ls=":", color="black", lw=1.5)

ax.set_xticks(x)
ax.set_xticklabels(groups, rotation=25, ha="right", fontsize=TICK_FS)

ax.set_ylabel("Z / Control enrichment, ±5 bp", fontsize=LABEL_FS)
ax.set_title("SNP Z-DNA enrichment by AF group: CDS vs noncoding", fontsize=TITLE_FS)

ax.tick_params(axis="y", labelsize=TICK_FS)
ax.legend(frameon=False, fontsize=13)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

out_plot = OUTDIR / "barplot_cds_vs_noncoding_AF_groups_bootstrap_CI.png"
fig.savefig(out_plot, dpi=600, bbox_inches="tight")
plt.close(fig)

print(f"\nWrote: {out_plot}")

print("\n[DONE]")

### 15.1 Region-stratified odds ratios and metaprofiles

In [ ]:
# Z-DNA enrichment at SNP sites by AF bin: within/between-region odds ratios with BH-corrected tests.
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

BASE = "data"
MASTER = f"{BASE}/controls/CHM13_controls_MASTER.tsv"
CHROM_LEN_TSV = f"{BASE}/chm13/chm13_seqreport_zdna.tsv"
GFF3 = (f"{BASE}/chm13/annotation/"
        "chm13_genes.gff3")
SNP_AF_INPUT = (f"{BASE}/mutations/"
                "hprc_mutations")

TOL = "10%"
WINDOW_RADIUS = 500
PM = 5
N_BOOTS = 2000
CI_LOW, CI_HIGH = 2.5, 97.5
MIN_SNPS_PER_GROUP = 500
BOOT_SEED = 42
REGIONS = ["cds", "noncoding"]

MIN_AN_GRID = [10, 100, 250, 400]
SINGLETON_GRID = [False, True]          # True => drop AC == 1

PRIMARY_MIN_AN = 250
PRIMARY_EXCLUDE_SINGLETONS = False

OR_MODE = "conditioned"                 # "snp_universe" or "conditioned"

CUSTOM_AF_BINS = [
    ("very_low_AF_0_0.0025", 0.0,    0.0025),   # NB: == singletons at high AN
    ("low_AF_0.0025_0.01",   0.0025, 0.01),
    ("mid_AF_0.01_0.05",     0.01,   0.05),
    ("high_AF_gt0.05",       0.05,   np.inf),
]
BIN_ORDER = [l for l, _, _ in CUSTOM_AF_BINS]

OUTDIR = Path(f"{BASE}/snp_af_cds_vs_noncoding_stats_tol10_v2")

LABEL_FS, TICK_FS, TITLE_FS = 18, 14, 16
GROUP_COLORS = ["#d62728", "#ff7f0e", "#bcbd22", "#2ca02c",
                "#17becf", "#1f77b4", "#9467bd", "#8c564b"]
REGION_COLOR = {"cds": "#1f77b4", "noncoding": "#ff7f0e"}

def build_bool_coverage(starts, ends, chrom_len):
    starts = np.clip(np.asarray(starts, dtype=np.int64), 0, chrom_len)
    ends = np.clip(np.asarray(ends, dtype=np.int64), 0, chrom_len)
    mask = ends > starts
    if not np.any(mask):
        return np.zeros(chrom_len, dtype=bool)
    starts, ends = starts[mask], ends[mask]
    diff = np.zeros(chrom_len + 1, dtype=np.int32)
    np.add.at(diff, starts, 1)
    np.add.at(diff, ends, -1)
    return np.cumsum(diff[:-1]) > 0

def load_merged_cds_intervals(gff3_path):
    cds = {}
    handle = pd.io.common.get_handle(gff3_path, mode="r", compression="infer").handle
    with handle as f:
        for line in f:
            if not line or line.startswith("#"):
                continue
            fl = line.rstrip("\n").split("\t")
            if len(fl) < 9 or fl[2] != "CDS":
                continue
            s0, e0 = int(fl[3]) - 1, int(fl[4])
            if e0 > s0:
                cds.setdefault(fl[0], []).append((s0, e0))
    merged = {}
    for c, iv in cds.items():
        iv = sorted(iv); out = []; cs, ce = iv[0]
        for s, e in iv[1:]:
            if s <= ce:
                ce = max(ce, e)
            else:
                out.append((cs, ce)); cs, ce = s, e
        out.append((cs, ce))
        merged[c] = np.asarray(out, dtype=np.int64)
    return merged

def filter_snps_by_region(snps, cds_intervals, region_mode):
    keep = []
    for chrom, d in snps.groupby("seqID", sort=False):
        iv = cds_intervals.get(chrom)
        if iv is None or len(iv) == 0:
            if region_mode == "noncoding":
                keep.append(d)
            continue
        pos = d["start"].to_numpy(np.int64)
        idx = np.searchsorted(iv[:, 0], pos, side="right") - 1
        valid = idx >= 0
        inside = np.zeros(len(pos), dtype=bool)
        inside[valid] = pos[valid] < iv[idx[valid], 1]
        sel = inside if region_mode == "cds" else ~inside
        if sel.any():
            keep.append(d.loc[sel])
    return pd.concat(keep, ignore_index=True) if keep else snps.iloc[0:0].copy()

def assign_af_groups(snps):
    snps = snps.copy(); snps["AF_group"] = pd.NA
    for label, lo, hi in CUSTOM_AF_BINS:
        m = (snps["AF"] >= lo) & ((snps["AF"] <= 1.0) if hi == np.inf else (snps["AF"] < hi))
        snps.loc[m, "AF_group"] = label
    return snps.dropna(subset=["AF_group"]).copy()

def boot_p_twosided(diff):
    v = diff[np.isfinite(diff)]
    if v.size == 0:
        return np.nan
    n = v.size
    n_le = int(np.sum(v <= 0)); n_ge = int(np.sum(v >= 0))
    return float(min(1.0, 2.0 * (min(n_le, n_ge) + 1) / (n + 1)))

def benjamini_hochberg(pvals):
    p = np.asarray(pvals, float); out = np.full_like(p, np.nan)
    idx = np.where(np.isfinite(p))[0]
    if idx.size == 0:
        return out
    pp = p[idx]; order = np.argsort(pp); ranked = pp[order]; m = len(ranked)
    adj = np.clip(np.minimum.accumulate((ranked * m / np.arange(1, m + 1))[::-1])[::-1], 0, 1)
    res = np.empty(m); res[order] = adj; out[idx] = res
    return out

def stars(p):
    if not np.isfinite(p):
        return "n/a"
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 5e-2 else "ns"

def run_difference_tests(boot_log, observed, region_labels, shared_bins, family):
    rows_trend, rows_pair, rows_between = [], [], []
    raw_p, slots = [], []   # slots: (list_ref_idx, row)

    for region in REGIONS:
        labs = region_labels[region]
        if len(labs) >= 3:
            xr = np.arange(len(labs))
            L = np.vstack([boot_log[(region, l)] for l in labs])
            slopes = np.array([np.polyfit(xr, L[:, b], 1)[0]
                               if np.all(np.isfinite(L[:, b])) else np.nan
                               for b in range(L.shape[1])])
            p = boot_p_twosided(slopes); sl = slopes[np.isfinite(slopes)]
            r = {"family": family, "region": region,
                 "slope_log_per_binrank": float(np.median(sl)) if sl.size else np.nan,
                 "slope_ci_lo": float(np.percentile(sl, CI_LOW)) if sl.size else np.nan,
                 "slope_ci_hi": float(np.percentile(sl, CI_HIGH)) if sl.size else np.nan,
                 "p_raw": p}
            rows_trend.append(r); raw_p.append(p); slots.append(r)

    for region in REGIONS:
        labs = region_labels[region]
        for i in range(len(labs)):
            for j in range(i + 1, len(labs)):
                d = boot_log[(region, labs[i])] - boot_log[(region, labs[j])]
                p = boot_p_twosided(d); dv = d[np.isfinite(d)]
                r = {"family": family, "region": region,
                     "bin_a": labs[i], "bin_b": labs[j],
                     "log2_a_over_b": float(np.median(dv) / np.log(2)) if dv.size else np.nan,
                     "p_raw": p}
                rows_pair.append(r); raw_p.append(p); slots.append(r)

    for l in shared_bins:
        d = boot_log[("cds", l)] - boot_log[("noncoding", l)]
        p = boot_p_twosided(d); dv = d[np.isfinite(d)]
        r = {"family": family, "AF_bin": l,
             "log2_cds_over_noncoding": float(np.median(dv) / np.log(2)) if dv.size else np.nan,
             "p_raw": p}
        rows_between.append(r); raw_p.append(p); slots.append(r)

    padj = benjamini_hochberg(raw_p)
    for k, r in enumerate(slots):
        r["p_adj"] = padj[k]
    return rows_trend, rows_pair, rows_between

def or_cells_for_config(snps, cds_intervals, z_cov, c_cov, cds_cov, chroms,
                        min_an, exclude_singletons, mode):
    s = snps[snps["AN"] >= min_an] if min_an else snps
    if exclude_singletons:
        s = s[s["AC"] > 1]
    ci = {c: i for i, c in enumerate(chroms)}
    cells, region_labels = {}, {}
    for region in REGIONS:
        sr = assign_af_groups(filter_snps_by_region(s, cds_intervals, region))
        labs_here = []
        for label in BIN_ORDER:
            sub = sr[sr["AF_group"] == label]
            if len(sub) < MIN_SNPS_PER_GROUP:
                continue
            nZ = np.zeros(len(chroms), np.int64)
            nC = np.zeros(len(chroms), np.int64)
            for chrom, d in sub.groupby("seqID", sort=False):
                if chrom not in ci:
                    continue
                a = d["start"].to_numpy(np.int64)
                if mode == "conditioned":
                    in_region = (cds_cov[chrom][a] if region == "cds"
                                 else ~cds_cov[chrom][a])
                    nZ[ci[chrom]] = (z_cov[chrom][a] & in_region).sum()
                    nC[ci[chrom]] = (c_cov[chrom][a] & in_region).sum()
                else:  # snp_universe
                    nZ[ci[chrom]] = z_cov[chrom][a].sum()
                    nC[ci[chrom]] = c_cov[chrom][a].sum()
            cells[(region, label)] = {"Z": nZ, "C": nC, "n": int(len(sub))}
            labs_here.append(label)
        region_labels[region] = labs_here
    return cells, region_labels

def or_paired_bootstrap(cells, n_boot, n_chroms, seed):
    rng = np.random.default_rng(seed)
    keys = list(cells.keys())
    observed = {}
    for k in keys:
        z, c = cells[k]["Z"].sum(), cells[k]["C"].sum()
        observed[k] = (z / c) if c > 0 else np.nan
    boot_log = {k: np.full(n_boot, np.nan) for k in keys}
    for b in range(n_boot):
        idx = rng.integers(0, n_chroms, n_chroms)
        for k in keys:
            z = cells[k]["Z"][idx].sum(); c = cells[k]["C"][idx].sum()
            if c > 0 and z > 0:
                boot_log[k][b] = np.log(z / c)
    return observed, boot_log

def build_region_matrices(snps_region, chroms, z_cov, c_cov, chrom_len, offsets):
    ci = {c: i for i, c in enumerate(chroms)}; n_off = len(offsets); out = {}
    for label in BIN_ORDER:
        sub = snps_region[snps_region["AF_group"] == label]
        if len(sub) < MIN_SNPS_PER_GROUP:
            continue
        Z = np.zeros((len(chroms), n_off), np.int64)
        C = np.zeros((len(chroms), n_off), np.int64)
        for chrom, d in sub.groupby("seqID", sort=False):
            if chrom not in ci:
                continue
            r = ci[chrom]; L = int(chrom_len[chrom]); a = d["start"].to_numpy(np.int64)
            zc, cc = z_cov[chrom], c_cov[chrom]
            for i, k in enumerate(offsets):
                pos = a + k; mm = (pos >= 0) & (pos < L)
                if mm.any():
                    pv = pos[mm]; Z[r, i] = zc[pv].sum(); C[r, i] = cc[pv].sum()
        out[label] = {"Z": Z, "C": C, "n": int(len(sub)),
                      "af_min": float(sub["AF"].min()),
                      "af_med": float(sub["AF"].median()),
                      "af_max": float(sub["AF"].max())}
    return out

def meta_ratio(zsum, csum, center, half):
    zm, cm = zsum.mean(), csum.mean()
    if zm <= 0 or cm <= 0:
        return np.nan
    zp = (zsum / zm)[center - half:center + half + 1].mean()
    cp = (csum / cm)[center - half:center + half + 1].mean()
    return (zp / cp) if cp > 0 else np.nan

def meta_paired_bootstrap(cells, center, half, n_boot, n_chroms, seed):
    rng = np.random.default_rng(seed); keys = list(cells.keys())
    observed = {k: meta_ratio(cells[k]["Z"].sum(0), cells[k]["C"].sum(0), center, half)
                for k in keys}
    boot_log = {k: np.full(n_boot, np.nan) for k in keys}
    for b in range(n_boot):
        idx = rng.integers(0, n_chroms, n_chroms)
        for k in keys:
            r = meta_ratio(cells[k]["Z"][idx].sum(0), cells[k]["C"][idx].sum(0), center, half)
            boot_log[k][b] = np.log(r) if (r is not None and r > 0) else np.nan
    return observed, boot_log

def _ci(boot_arr):
    v = boot_arr[np.isfinite(boot_arr)]
    if v.size == 0:
        return np.nan, np.nan
    return float(np.exp(np.percentile(v, CI_LOW))), float(np.exp(np.percentile(v, CI_HIGH)))

def despine(ax):
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

def plot_between(labels, observed, cis, between_padj, ylabel, title, outpath):
    fig, ax = plt.subplots(figsize=(max(7.5, 2.2 * len(labels)), 5.8))
    x = np.arange(len(labels)); w = 0.38
    for j, region in enumerate(REGIONS):
        vals = np.array([observed[(region, l)] for l in labels])
        lo = np.array([cis[(region, l)][0] for l in labels])
        hi = np.array([cis[(region, l)][1] for l in labels])
        ax.bar(x + (j - 0.5) * w, vals, width=w,
               yerr=np.vstack([vals - lo, hi - vals]), capsize=4,
               color=REGION_COLOR[region], edgecolor="black", linewidth=0.8, label=region)
    ax.axhline(1, ls=":", color="black", lw=1.5)
    ymax = max(cis[(r, l)][1] for r in REGIONS for l in labels)
    for i, l in enumerate(labels):
        ax.text(i, ymax * 1.04, stars(between_padj.get(l, np.nan)),
                ha="center", va="bottom", fontsize=13)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=LABEL_FS); ax.set_title(title, fontsize=TITLE_FS)
    ax.tick_params(axis="y", labelsize=TICK_FS); ax.legend(frameon=False, fontsize=12)
    despine(ax); fig.tight_layout(); fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig); print(f"Wrote: {outpath}")

def plot_within(region, labels, observed, cis, ns, colors, trend_p, ylabel, outpath):
    fig, ax = plt.subplots(figsize=(max(6.5, 1.7 * len(labels)), 5.6))
    x = np.arange(len(labels))
    vals = np.array([observed[(region, l)] for l in labels])
    lo = np.array([cis[(region, l)][0] for l in labels])
    hi = np.array([cis[(region, l)][1] for l in labels])
    ax.bar(x, vals, yerr=np.vstack([vals - lo, hi - vals]), capsize=5, width=0.65,
           color=[colors[l] for l in labels], edgecolor="black", linewidth=0.8)
    ax.axhline(1, ls=":", color="black", lw=1.5)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=LABEL_FS)
    ax.set_title(f"{region} — trend p={trend_p:.3g} ({stars(trend_p)})", fontsize=TITLE_FS)
    ax.tick_params(axis="y", labelsize=TICK_FS)
    for xi, v, hh, n in zip(x, vals, hi, [ns[(region, l)] for l in labels]):
        ax.text(xi, hh, f"{v:.3f}\n(n={n:,})", ha="center", va="bottom", fontsize=10)
    despine(ax); fig.tight_layout(); fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig); print(f"Wrote: {outpath}")

def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)
    offsets = np.arange(-WINDOW_RADIUS, WINDOW_RADIUS + 1, dtype=np.int64)
    center = int(np.where(offsets == 0)[0][0])

    cl = pd.read_csv(CHROM_LEN_TSV, sep="\t",
                     usecols=["ucsc_pansn_name", "gcf_seq_length"])
    chrom_len = dict(zip(cl["ucsc_pansn_name"], cl["gcf_seq_length"].astype(int)))

    m = pd.read_csv(MASTER, sep="\t")
    m = (m[(m["tol_level"] == TOL) & (m["has_control"])]
         .drop_duplicates(["Chromosome", "Start0", "End0"]).copy())
    z_int = {c: d[["Start0", "End0"]].to_numpy(np.int64) for c, d in m.groupby("Chromosome", sort=False)}
    c_int = {c: d[["Ctrl_Start", "Ctrl_End"]].to_numpy(np.int64) for c, d in m.groupby("Chromosome", sort=False)}

    cds_intervals = load_merged_cds_intervals(GFF3)

    z_cov, c_cov, cds_cov = {}, {}, {}
    for chrom in sorted(set(z_int) & set(c_int) & set(chrom_len)):
        L = int(chrom_len[chrom])
        z_cov[chrom] = build_bool_coverage(z_int[chrom][:, 0], z_int[chrom][:, 1], L)
        c_cov[chrom] = build_bool_coverage(c_int[chrom][:, 0], c_int[chrom][:, 1], L)
        iv = cds_intervals.get(chrom)
        cds_cov[chrom] = (build_bool_coverage(iv[:, 0], iv[:, 1], L)
                          if iv is not None and len(iv) else np.zeros(L, bool))
    CHROMS = sorted(z_cov.keys()); nC = len(CHROMS)
    print(f"Coverage masks for {nC} chromosomes.")

    snps = pd.read_csv(SNP_AF_INPUT, sep="\t", compression="gzip")
    for col in ["start", "AF", "AC", "AN"]:
        snps[col] = pd.to_numeric(snps[col], errors="coerce")
    snps = snps.dropna(subset=["seqID", "start", "AF", "AC", "AN"]).copy()
    snps["start"] = snps["start"].astype(np.int64)
    snps = snps[(snps["AF"] > 0) & (snps["AF"] <= 1) &
                (snps["AC"] > 0) & (snps["AN"] > 0)].copy()
    print(f"AN distribution: median={snps['AN'].median():.0f}, "
          f"mode={snps['AN'].mode().iloc[0]:.0f}, max={snps['AN'].max():.0f}")

    stability = []
    for min_an in MIN_AN_GRID:
        for excl in SINGLETON_GRID:
            cells, rlab = or_cells_for_config(snps, cds_intervals, z_cov, c_cov,
                                              cds_cov, CHROMS, min_an, excl, OR_MODE)
            if not cells:
                continue
            obs, blog = or_paired_bootstrap(cells, N_BOOTS, nC, BOOT_SEED)
            shared = [l for l in BIN_ORDER if all((r, l) in cells for r in REGIONS)]
            tr, _, bw = run_difference_tests(blog, obs, rlab, shared, family="overlapOR")
            for r in tr:
                stability.append({"min_an": min_an, "exclude_singletons": excl,
                                  "region": r["region"], "trend_slope": r["slope_log_per_binrank"],
                                  "trend_p_adj": r["p_adj"]})
            for r in bw:
                stability.append({"min_an": min_an, "exclude_singletons": excl,
                                  "region": f"between:{r['AF_bin']}",
                                  "trend_slope": r["log2_cds_over_noncoding"],
                                  "trend_p_adj": r["p_adj"]})
    pd.DataFrame(stability).to_csv(OUTDIR / "stability_sweep_overlapOR.tsv",
                                   sep="\t", index=False)
    print(f"Wrote stability sweep ({OR_MODE}) -> stability_sweep_overlapOR.tsv")

    or_cells, or_rlab = or_cells_for_config(snps, cds_intervals, z_cov, c_cov, cds_cov,
                                            CHROMS, PRIMARY_MIN_AN,
                                            PRIMARY_EXCLUDE_SINGLETONS, OR_MODE)
    or_obs, or_blog = or_paired_bootstrap(or_cells, N_BOOTS, nC, BOOT_SEED)
    or_cis = {k: _ci(or_blog[k]) for k in or_cells}
    or_ns = {k: or_cells[k]["n"] for k in or_cells}
    or_shared = [l for l in BIN_ORDER if all((r, l) in or_cells for r in REGIONS)]
    or_tr, or_pr, or_bw = run_difference_tests(or_blog, or_obs, or_rlab, or_shared, "overlapOR")
    pd.DataFrame(or_tr).to_csv(OUTDIR / "OR_trend.tsv", sep="\t", index=False)
    pd.DataFrame(or_pr).to_csv(OUTDIR / "OR_pairwise.tsv", sep="\t", index=False)
    pd.DataFrame(or_bw).to_csv(OUTDIR / "OR_between.tsv", sep="\t", index=False)
    pal = {l: GROUP_COLORS[i % len(GROUP_COLORS)] for i, l in enumerate(BIN_ORDER)}
    tp = {r["region"]: r["p_adj"] for r in or_tr}
    for region in REGIONS:
        if or_rlab[region]:
            plot_within(region, or_rlab[region], or_obs, or_cis, or_ns, pal,
                        tp.get(region, np.nan), f"Z/Control overlap OR ({OR_MODE})",
                        OUTDIR / f"OR_within_{region}.png")
    if or_shared:
        plot_between(or_shared, or_obs, or_cis,
                     {r["AF_bin"]: r["p_adj"] for r in or_bw},
                     f"Z/Control overlap OR ({OR_MODE})",
                     "CDS vs noncoding overlap OR per AF bin", OUTDIR / "OR_between.png")
    pd.DataFrame([{"region": r, "AF_bin": b, "ratio": or_obs[(r, b)],
                   "ci_lo": or_cis[(r, b)][0], "ci_hi": or_cis[(r, b)][1],
                   "n": or_ns[(r, b)]}
                  for (r, b) in or_cells]).to_csv(OUTDIR / "OR_ratio_with_CI.tsv",
                                                  sep="\t", index=False)
    
    pd.DataFrame([{"region": k[0], "AF_bin": k[1], "n": meta_meta[k]["n"],
                   "af_min": meta_meta[k]["af_min"], "af_med": meta_meta[k]["af_med"],
                   "af_max": meta_meta[k]["af_max"], "ratio": meta_obs[k],
                   "ci_lo": meta_cis[k][0], "ci_hi": meta_cis[k][1]}
                  for k in meta_cells]).to_csv(OUTDIR / "meta_ratio_with_CI.tsv",
                                               sep="\t", index=False)
    meta_cells, meta_rlab, meta_meta = {}, {}, {}
    s = snps[snps["AN"] >= PRIMARY_MIN_AN] if PRIMARY_MIN_AN else snps
    if PRIMARY_EXCLUDE_SINGLETONS:
        s = s[s["AC"] > 1]
    for region in REGIONS:
        sr = assign_af_groups(filter_snps_by_region(s, cds_intervals, region))
        mats = build_region_matrices(sr, CHROMS, z_cov, c_cov, chrom_len, offsets)
        meta_rlab[region] = [l for l in BIN_ORDER if l in mats]
        for l, d in mats.items():
            meta_cells[(region, l)] = {"Z": d["Z"], "C": d["C"]}
            meta_meta[(region, l)] = d
    meta_obs, meta_blog = meta_paired_bootstrap(meta_cells, center, PM, N_BOOTS, nC, BOOT_SEED)
    meta_cis = {k: _ci(meta_blog[k]) for k in meta_cells}
    meta_ns = {k: meta_meta[k]["n"] for k in meta_cells}
    meta_shared = [l for l in BIN_ORDER if all((r, l) in meta_cells for r in REGIONS)]
    mtr, mpr, mbw = run_difference_tests(meta_blog, meta_obs, meta_rlab, meta_shared, "metaprofile")
    pd.DataFrame([{**meta_meta[k], "region": k[0], "AF_bin": k[1],
                   "ratio": meta_obs[k], "ci_lo": meta_cis[k][0], "ci_hi": meta_cis[k][1]}
                  for k in meta_cells]).to_csv(OUTDIR / "meta_ratio_with_CI.tsv", sep="\t", index=False)
    pd.DataFrame(mtr).to_csv(OUTDIR / "meta_trend.tsv", sep="\t", index=False)
    pd.DataFrame(mpr).to_csv(OUTDIR / "meta_pairwise.tsv", sep="\t", index=False)
    pd.DataFrame(mbw).to_csv(OUTDIR / "meta_between.tsv", sep="\t", index=False)
    mtp = {r["region"]: r["p_adj"] for r in mtr}
    for region in REGIONS:
        if meta_rlab[region]:
            plot_within(region, meta_rlab[region], meta_obs, meta_cis, meta_ns, pal,
                        mtp.get(region, np.nan), f"Z/Control (±{PM} bp peak)",
                        OUTDIR / f"meta_within_{region}.png")
    if meta_shared:
        plot_between(meta_shared, meta_obs, meta_cis,
                     {r["AF_bin"]: r["p_adj"] for r in mbw},
                     f"Z/Control (±{PM} bp peak)",
                     "CDS vs noncoding peak ratio per AF bin", OUTDIR / "meta_between.png")

    print(f"\n[DONE] {OUTDIR}")
    print(f"Bootstrap p floor (corrected) = 2/(N+1) = {2/(N_BOOTS+1):.2g}; no p=0 reported.")
    print("Primary metaprofile config: "
          f"MIN_AN={PRIMARY_MIN_AN}, exclude_singletons={PRIMARY_EXCLUDE_SINGLETONS}. "
          "Read stability_sweep_overlapOR.tsv before trusting the trend.")
if __name__ == "__main__":
    main()

In [ ]:
# Publication figures for the Z-DNA / SNP allele-frequency analysis.
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path

INDIR = Path("data/snp_af_cds_vs_noncoding_stats_tol10_v2")
OUTDIR = INDIR / "figures"
STYLE = "pointrange"          # "pointrange" or "bar"
TREND_ANNOT = True            # show trend p_adj in within-region panels
FORMATS = ["pdf", "png"]
DPI = 600

REGIONS = ["cds", "noncoding"]
BIN_ORDER = ["very_low_AF_0_0.0025", "low_AF_0.0025_0.01",
             "mid_AF_0.01_0.05", "high_AF_gt0.05"]

NICE_LABEL = {
    "very_low_AF_0_0.0025": "AF < 0.25%",     # singletons (AF = 1/AN)
    "low_AF_0.0025_0.01":   "0.25–1%",
    "mid_AF_0.01_0.05":     "1–5%",
    "high_AF_gt0.05":       "AF > 5%",
}
REGION_LABEL = {"cds": "CDS", "noncoding": "Noncoding"}
Y_LABEL = {"meta": "Z-DNA / control\n(±5 bp peak ratio)",
           "OR":   "Z-DNA / control\noverlap odds ratio"}

C_CDS, C_NONCODING = "#0072B2", "#E69F00"
C_SINGLE = "#444444"          # single-series point/bar colour

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 13,
    "axes.linewidth": 1.1, "axes.edgecolor": "#222222",
    "xtick.major.size": 4, "ytick.major.size": 4,
    "xtick.major.width": 1.0, "ytick.major.width": 1.0,
    "savefig.bbox": "tight", "pdf.fonttype": 42, "ps.fonttype": 42,
})

def despine(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

def stars(p):
    if not np.isfinite(p):
        return "n.s."
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 5e-2 else "n.s."

def save(fig, stem):
    OUTDIR.mkdir(parents=True, exist_ok=True)
    for ext in FORMATS:
        out = OUTDIR / f"{stem}.{ext}"
        fig.savefig(out, dpi=DPI)
        print(f"Wrote: {out}")
    plt.close(fig)

def load_family(prefix):
    rpath = INDIR / f"{prefix}_ratio_with_CI.tsv"
    if not rpath.exists():
        print(f"[WARN] {rpath.name} missing — skipping {prefix} family. "
              f"(For OR, add the OR_ratio_with_CI.tsv dump to the analysis script.)")
        return None
    rdf = pd.read_csv(rpath, sep="\t", usecols=["region", "AF_bin", "ratio", "ci_lo", "ci_hi", "n"])
    rdf["AF_bin"] = rdf["AF_bin"].astype(str)

    trend = {}
    tpath = INDIR / f"{prefix}_trend.tsv"
    if tpath.exists():
        td = pd.read_csv(tpath, sep="\t")
        trend = dict(zip(td["region"].astype(str), td["p_adj"]))

    between = {}
    bpath = INDIR / f"{prefix}_between.tsv"
    if bpath.exists():
        bd = pd.read_csv(bpath, sep="\t")
        between = dict(zip(bd["AF_bin"].astype(str), bd["p_adj"]))

    return rdf, trend, between

def bins_present(rdf, region):
    have = set(rdf[rdf["region"] == region]["AF_bin"])
    return [b for b in BIN_ORDER if b in have]

def row(rdf, region, b):
    r = rdf[(rdf["region"] == region) & (rdf["AF_bin"] == b)].iloc[0]
    return float(r["ratio"]), float(r["ci_lo"]), float(r["ci_hi"]), int(r["n"])

def plot_within(rdf, trend, region, prefix):
    labs = bins_present(rdf, region)
    if not labs:
        return
    x = np.arange(len(labs))
    vals = np.array([row(rdf, region, b)[0] for b in labs])
    lo = np.array([row(rdf, region, b)[1] for b in labs])
    hi = np.array([row(rdf, region, b)[2] for b in labs])
    yerr = np.vstack([vals - lo, hi - vals])

    fig, ax = plt.subplots(figsize=(4.6, 4.0))
    ax.axhline(1.0, ls=(0, (4, 4)), color="#999999", lw=1.2, zorder=0)
    ax.yaxis.grid(True, color="#EEEEEE", lw=1.0, zorder=0)
    ax.set_axisbelow(True)

    if STYLE == "bar":
        ax.bar(x, vals, yerr=yerr, capsize=4, width=0.62, color=C_SINGLE,
               edgecolor="black", linewidth=0.8, error_kw=dict(lw=1.4), zorder=3)
        ax.set_ylim(0, max(hi) * 1.12)
    else:  # pointrange
        ax.plot(x, vals, "-", color=C_SINGLE, lw=1.1, alpha=0.5, zorder=2)
        ax.errorbar(x, vals, yerr=yerr, fmt="o", ms=8, color=C_SINGLE,
                    ecolor=C_SINGLE, elinewidth=1.6, capsize=4, capthick=1.6, zorder=3)
        pad = (max(hi) - min(lo)) * 0.18 + 1e-9
        ax.set_ylim(min(min(lo), 1.0) - pad, max(hi) + pad)

    ax.set_xticks(x)
    ax.set_xticklabels([NICE_LABEL.get(b, b) for b in labs], rotation=20, ha="right")
    ax.set_xlabel("Allele frequency")
    ax.set_ylabel(Y_LABEL[prefix])

    if TREND_ANNOT and region in trend:
        p = trend[region]
        ax.text(0.04, 0.96, f"trend {stars(p)}\n$p_{{adj}}$={p:.3g}",
                transform=ax.transAxes, va="top", ha="left", fontsize=11,
                color="#222222")
    despine(ax)
    save(fig, f"{prefix}_within_{region}")

def plot_between(rdf, between, prefix):
    labs = [b for b in BIN_ORDER
            if all(((rdf["region"] == r) & (rdf["AF_bin"] == b)).any() for r in REGIONS)]
    if not labs:
        return
    x = np.arange(len(labs))

    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    ax.axhline(1.0, ls=(0, (4, 4)), color="#999999", lw=1.2, zorder=0)
    ax.yaxis.grid(True, color="#EEEEEE", lw=1.0, zorder=0)
    ax.set_axisbelow(True)

    rcol = {"cds": C_CDS, "noncoding": C_NONCODING}
    top = 0.0
    if STYLE == "bar":
        w = 0.38
        for j, region in enumerate(REGIONS):
            vals = np.array([row(rdf, region, b)[0] for b in labs])
            lo = np.array([row(rdf, region, b)[1] for b in labs])
            hi = np.array([row(rdf, region, b)[2] for b in labs])
            ax.bar(x + (j - 0.5) * w, vals, width=w,
                   yerr=np.vstack([vals - lo, hi - vals]), capsize=3,
                   color=rcol[region], edgecolor="black", linewidth=0.7,
                   error_kw=dict(lw=1.2), label=REGION_LABEL[region], zorder=3)
            top = max(top, hi.max())
        ax.set_ylim(0, top * 1.16)
        star_y = lambda: top * 1.05
    else:  # pointrange, dodged
        off = 0.13
        allhi, alllo = [], []
        for j, region in enumerate(REGIONS):
            dx = (j - 0.5) * 2 * off
            vals = np.array([row(rdf, region, b)[0] for b in labs])
            lo = np.array([row(rdf, region, b)[1] for b in labs])
            hi = np.array([row(rdf, region, b)[2] for b in labs])
            ax.errorbar(x + dx, vals, yerr=np.vstack([vals - lo, hi - vals]),
                        fmt="o", ms=7, color=rcol[region], ecolor=rcol[region],
                        elinewidth=1.6, capsize=4, capthick=1.6,
                        label=REGION_LABEL[region], zorder=3)
            allhi += list(hi); alllo += list(lo)
        top = max(allhi)
        pad = (top - min(alllo)) * 0.18
        ax.set_ylim(min(min(alllo), 1.0) - pad, top + pad)
        star_y = lambda: top + pad * 0.4

    for i, b in enumerate(labs):
        ax.text(i, star_y(), stars(between.get(b, np.nan)),
                ha="center", va="bottom", fontsize=12, color="#222222")
    ax.set_xticks(x)
    ax.set_xticklabels([NICE_LABEL.get(b, b) for b in labs], rotation=20, ha="right")
    ax.set_xlabel("Allele frequency")
    ax.set_ylabel(Y_LABEL[prefix])
    ax.legend(frameon=False, ncol=2, fontsize=12, handletextpad=0.4,
              loc="lower center", bbox_to_anchor=(0.5, 1.02))
    despine(ax)
    save(fig, f"{prefix}_between")
def main():
    print(f"Reading results from: {INDIR}\nStyle: {STYLE}\n")
    for prefix in ["meta", "OR"]:
        loaded = load_family(prefix)
        if loaded is None:
            continue
        rdf, trend, between = loaded
        for region in REGIONS:
            plot_within(rdf, trend, region, prefix)
        plot_between(rdf, between, prefix)
    print(f"\n[DONE] Figures in {OUTDIR}")
if __name__ == "__main__":
    main()